# Bank Customer Churn Benchmarking: End-to-End Reproducible Pipeline

Run this notebook **from top to bottom**.  
Only edit `DATASET_PATH` in the next cell if the dataset is stored in a different Kaggle input folder.

The pipeline is CPU-only and contains all development, locked evaluation, statistical, explainability, robustness, subgroup, computational, and supplementary analyses used by the study.


In [ ]:
from pathlib import Path
import shutil
import pandas as pd

# ============================================================
# USER CONFIGURATION
# Edit only this path when reproducing the study.
# ============================================================
DATASET_PATH = Path(
    "/kaggle/input/datasets/mathchi/churn-for-bank-customers/churn.csv"
)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH}\n"
        "Update DATASET_PATH in this cell, then rerun the notebook."
    )


# Start from a clean reproducibility workspace so that old Optuna
# studies or saved artifacts cannot be reused accidentally.
PIPELINE_DIRECTORIES = [
    "classical_hpo",
    "classical_repeated_cv",
    "deep_hpo",
    "deep_repeated_cv",
    "unified_benchmark",
    "imbalance_ablation",
    "mlp_architecture_ablation",
    "calibration_threshold",
    "calibration_threshold_locked",
    "final_locked_holdout",
    "statistical_inference",
    "explainability_audit",
    "explainability_shap",
    "robustness_audit",
    "subgroup_fairness_audit",
    "computational_efficiency_audit",
    "publication_outputs"
]

for directory_name in PIPELINE_DIRECTORIES:
    shutil.rmtree(
        Path("/kaggle/working") / directory_name,
        ignore_errors=True
    )

for filename in [
    "split_manifest.csv",
    "experiment_config.json",
    "sanity_baseline_results.csv",
    "sanity_oof_predictions.csv",
    "environment_record.json"
]:
    filepath = Path("/kaggle/working") / filename
    if filepath.exists():
        filepath.unlink()


df = pd.read_csv(DATASET_PATH)

print("Dataset:", DATASET_PATH)
print("Shape:", df.shape)
df.info()


In [ ]:
# ============================================================
# REPRODUCIBILITY DEPENDENCIES
# Install the model-specific packages used by the study.
# ============================================================

import sys
import subprocess
import importlib.metadata

REQUIRED_PACKAGES = [
    "xgboost==3.2.0",
    "lightgbm==4.6.0",
    "catboost==1.2.10",
    "optuna==4.9.0",
    "shap==0.51.0",
    "imbalanced-learn==0.14.1",
    "pytorch-tabnet==4.1.0",
    "pytabkit[models]==1.7.3",
    "faiss-cpu==1.14.3",
    "psutil"
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--upgrade-strategy",
    "only-if-needed",
    *REQUIRED_PACKAGES
])

print("Required modeling packages are available.")
for distribution_name in [
    "xgboost",
    "lightgbm",
    "catboost",
    "optuna",
    "shap",
    "imbalanced-learn",
    "pytorch-tabnet",
    "pytabkit",
    "faiss-cpu"
]:
    print(
        f"{distribution_name}: "
        f"{importlib.metadata.version(distribution_name)}"
    )


In [ ]:
# ============================================================
# RAW DATA INTEGRITY AUDIT
# ============================================================

import pandas as pd
import numpy as np
import hashlib
import sklearn

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Giữ một bản raw bất biến để đối chiếu về sau
df_raw = df.copy(deep=True)

TARGET = "Exited"
ID_COLS = ["RowNumber", "CustomerId", "Surname"]

EXPECTED_COLUMNS = [
    "RowNumber", "CustomerId", "Surname", "CreditScore",
    "Geography", "Gender", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary", "Exited"
]

print("=" * 80)
print("1. DATASET IDENTITY")
print("=" * 80)

print("Shape:", df_raw.shape)
print("Exact expected schema:", list(df_raw.columns) == EXPECTED_COLUMNS)

print("\nMissing expected columns:")
print(sorted(set(EXPECTED_COLUMNS) - set(df_raw.columns)))

print("\nUnexpected columns:")
print(sorted(set(df_raw.columns) - set(EXPECTED_COLUMNS)))

# Hash giúp xác định chính xác phiên bản file đã dùng
file_path = DATASET_PATH

with open(file_path, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

print("\nSHA-256:", sha256)
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)


print("\n" + "=" * 80)
print("2. MISSING, INFINITE AND CONSTANT VALUES")
print("=" * 80)

quality_table = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_n": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(4),
    "unique_n": df_raw.nunique(dropna=False)
})

print(quality_table)

numeric_cols = df_raw.select_dtypes(include=np.number).columns
infinite_counts = np.isinf(df_raw[numeric_cols]).sum()

print("\nInfinite values:")
print(infinite_counts[infinite_counts > 0] if infinite_counts.sum() > 0 else "None")

constant_cols = [
    col for col in df_raw.columns
    if df_raw[col].nunique(dropna=False) <= 1
]

print("\nConstant columns:", constant_cols)


print("\n" + "=" * 80)
print("3. DUPLICATES AND IDENTIFIER CHECKS")
print("=" * 80)

print("Fully duplicated rows:", df_raw.duplicated().sum())
print("Duplicated RowNumber:", df_raw["RowNumber"].duplicated().sum())
print("Duplicated CustomerId:", df_raw["CustomerId"].duplicated().sum())

# Kiểm tra các quan sát giống nhau về predictor + target,
# không xét các trường định danh
analysis_cols = [
    col for col in df_raw.columns
    if col not in ID_COLS
]

print(
    "Duplicates excluding identifiers:",
    df_raw.duplicated(subset=analysis_cols).sum()
)

print("RowNumber unique:", df_raw["RowNumber"].is_unique)
print("CustomerId unique:", df_raw["CustomerId"].is_unique)


print("\n" + "=" * 80)
print("4. TARGET DISTRIBUTION")
print("=" * 80)

target_counts = df_raw[TARGET].value_counts(dropna=False).sort_index()
target_pct = (
    df_raw[TARGET]
    .value_counts(normalize=True, dropna=False)
    .sort_index()
    .mul(100)
    .round(4)
)

target_table = pd.DataFrame({
    "count": target_counts,
    "percentage": target_pct
})

print(target_table)
print("\nTarget unique values:", sorted(df_raw[TARGET].dropna().unique().tolist()))


print("\n" + "=" * 80)
print("5. CATEGORICAL VALUES")
print("=" * 80)

for col in ["Geography", "Gender"]:
    print(f"\n{col}:")
    print(df_raw[col].value_counts(dropna=False))


print("\n" + "=" * 80)
print("6. BINARY AND DISCRETE FEATURE VALUES")
print("=" * 80)

for col in ["HasCrCard", "IsActiveMember", "NumOfProducts", "Tenure"]:
    print(f"\n{col}:")
    print(df_raw[col].value_counts(dropna=False).sort_index())


print("\n" + "=" * 80)
print("7. NUMERICAL SUMMARY")
print("=" * 80)

summary_cols = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "EstimatedSalary"
]

print(
    df_raw[summary_cols]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .T
)


print("\n" + "=" * 80)
print("8. TECHNICAL SANITY CHECKS")
print("=" * 80)

sanity_checks = {
    "Exited contains only 0/1":
        set(df_raw["Exited"].unique()).issubset({0, 1}),

    "HasCrCard contains only 0/1":
        set(df_raw["HasCrCard"].unique()).issubset({0, 1}),

    "IsActiveMember contains only 0/1":
        set(df_raw["IsActiveMember"].unique()).issubset({0, 1}),

    "CreditScore is non-negative":
        (df_raw["CreditScore"] >= 0).all(),

    "Age is positive":
        (df_raw["Age"] > 0).all(),

    "Tenure is non-negative":
        (df_raw["Tenure"] >= 0).all(),

    "Balance is non-negative":
        (df_raw["Balance"] >= 0).all(),

    "NumOfProducts is positive":
        (df_raw["NumOfProducts"] > 0).all(),

    "EstimatedSalary is non-negative":
        (df_raw["EstimatedSalary"] >= 0).all()
}

for check, result in sanity_checks.items():
    print(f"{check}: {result}")

In [ ]:
# ============================================================
# LEAKAGE AND UNIVARIATE SIGNAL AUDIT
# ============================================================

import pandas as pd
import numpy as np

from scipy.stats import pointbiserialr, spearmanr

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

TARGET = "Exited"
ID_COLS = ["RowNumber", "CustomerId", "Surname"]

MODEL_FEATURES = [
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

CATEGORICAL_FEATURES = ["Geography", "Gender"]
DISCRETE_FEATURES = [
    "Tenure",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember"
]
CONTINUOUS_FEATURES = [
    "CreditScore",
    "Age",
    "Balance",
    "EstimatedSalary"
]

y = df_raw[TARGET].astype(int)

print("=" * 90)
print("1. FEATURE POLICY")
print("=" * 90)

print("Excluded identifiers:", ID_COLS)
print("Official predictors:", MODEL_FEATURES)
print("Number of predictors:", len(MODEL_FEATURES))


# ------------------------------------------------------------
# Direct equality / proxy checks
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("2. DIRECT TARGET-PROXY CHECKS")
print("=" * 90)

proxy_rows = []

for col in df_raw.columns:
    if col == TARGET:
        continue

    if pd.api.types.is_numeric_dtype(df_raw[col]):
        values = df_raw[col]

        exact_match = (
            (values == y).mean()
            if set(values.unique()).issubset({0, 1})
            else np.nan
        )

        inverse_match = (
            (values == (1 - y)).mean()
            if set(values.unique()).issubset({0, 1})
            else np.nan
        )

        unique_ratio = values.nunique() / len(values)

        proxy_rows.append({
            "feature": col,
            "unique_n": values.nunique(),
            "unique_ratio": unique_ratio,
            "exact_target_match": exact_match,
            "inverse_target_match": inverse_match
        })

proxy_table = pd.DataFrame(proxy_rows)

print(
    proxy_table
    .sort_values(
        by="exact_target_match",
        ascending=False,
        na_position="last"
    )
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Target rates by categorical and discrete features
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("3. CHURN RATE BY CATEGORICAL/DISCRETE FEATURES")
print("=" * 90)

for col in CATEGORICAL_FEATURES + DISCRETE_FEATURES:
    table = (
        df_raw
        .groupby(col, dropna=False)[TARGET]
        .agg(["count", "sum", "mean"])
        .rename(columns={
            "sum": "churn_n",
            "mean": "churn_rate"
        })
    )

    table["churn_rate"] = table["churn_rate"].round(4)

    print(f"\n--- {col} ---")
    print(table)


# ------------------------------------------------------------
# Important structural conditions
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("4. STRUCTURAL CONDITIONS")
print("=" * 90)

balance_zero_table = (
    df_raw
    .assign(BalanceZero=(df_raw["Balance"] == 0).astype(int))
    .groupby("BalanceZero")[TARGET]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "sum": "churn_n",
        "mean": "churn_rate"
    })
)

balance_zero_table["churn_rate"] = (
    balance_zero_table["churn_rate"].round(4)
)

print("\nBalance equal to zero:")
print(balance_zero_table)

rare_product_table = (
    df_raw
    .groupby("NumOfProducts")[TARGET]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "sum": "churn_n",
        "mean": "churn_rate"
    })
)

rare_product_table["churn_rate"] = (
    rare_product_table["churn_rate"].round(4)
)

print("\nNumOfProducts:")
print(rare_product_table)


# ------------------------------------------------------------
# Numerical associations with target
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("5. NUMERICAL ASSOCIATION WITH TARGET")
print("=" * 90)

association_rows = []

for col in CONTINUOUS_FEATURES + ["Tenure", "NumOfProducts"]:
    pb_corr, pb_p = pointbiserialr(df_raw[col], y)
    sp_corr, sp_p = spearmanr(df_raw[col], y)

    association_rows.append({
        "feature": col,
        "point_biserial_r": pb_corr,
        "point_biserial_p": pb_p,
        "spearman_rho": sp_corr,
        "spearman_p": sp_p
    })

association_table = pd.DataFrame(association_rows)

print(
    association_table
    .sort_values(
        "point_biserial_r",
        key=lambda s: s.abs(),
        ascending=False
    )
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Leakage-aware one-feature out-of-fold performance
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("6. SINGLE-FEATURE OUT-OF-FOLD PREDICTIVE POWER")
print("=" * 90)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

audit_features = [
    "RowNumber",
    "CustomerId",
    "Surname"
] + MODEL_FEATURES

univariate_results = []

for feature in audit_features:

    is_categorical = (
        df_raw[feature].dtype == "object"
        or feature in DISCRETE_FEATURES
    )

    if is_categorical:
        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "feature",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=True
                    ),
                    [feature]
                )
            ],
            remainder="drop"
        )
    else:
        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "feature",
                    StandardScaler(),
                    [feature]
                )
            ],
            remainder="drop"
        )

    logistic_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        (
            "model",
            LogisticRegression(
                solver="liblinear",
                max_iter=2000,
                random_state=42
            )
        )
    ])

    tree_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=3,
                min_samples_leaf=30,
                random_state=42
            )
        )
    ])

    logistic_oof = cross_val_predict(
        logistic_pipeline,
        df_raw[[feature]],
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    tree_oof = cross_val_predict(
        tree_pipeline,
        df_raw[[feature]],
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    logistic_auc = roc_auc_score(y, logistic_oof)
    logistic_pr_auc = average_precision_score(y, logistic_oof)

    tree_auc = roc_auc_score(y, tree_oof)
    tree_pr_auc = average_precision_score(y, tree_oof)

    best_auc = max(logistic_auc, tree_auc)

    if best_auc >= 0.95:
        risk_flag = "CRITICAL"
    elif best_auc >= 0.85:
        risk_flag = "HIGH"
    elif best_auc >= 0.75:
        risk_flag = "MODERATE"
    else:
        risk_flag = "LOW"

    univariate_results.append({
        "feature": feature,
        "role": (
            "EXCLUDED_IDENTIFIER"
            if feature in ID_COLS
            else "MODEL_FEATURE"
        ),
        "logistic_roc_auc": logistic_auc,
        "logistic_pr_auc": logistic_pr_auc,
        "tree_roc_auc": tree_auc,
        "tree_pr_auc": tree_pr_auc,
        "best_single_feature_auc": best_auc,
        "leakage_risk_flag": risk_flag
    })

univariate_table = (
    pd.DataFrame(univariate_results)
    .sort_values(
        "best_single_feature_auc",
        ascending=False
    )
)

print(
    univariate_table
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# Compact automatic summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("7. AUTOMATIC AUDIT SUMMARY")
print("=" * 90)

critical_features = univariate_table.loc[
    univariate_table["leakage_risk_flag"] == "CRITICAL",
    "feature"
].tolist()

high_risk_features = univariate_table.loc[
    univariate_table["leakage_risk_flag"] == "HIGH",
    "feature"
].tolist()

print("Critical single-feature predictors:", critical_features)
print("High-risk single-feature predictors:", high_risk_features)

print(
    "\nReference prevalence / no-skill PR-AUC:",
    round(y.mean(), 4)
)

In [ ]:
# ============================================================
# LOCKED DEVELOPMENT / TEST SPLIT
# ============================================================

import json
import random
import hashlib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Global experiment configuration
# ------------------------------------------------------------

GLOBAL_SEED = 42
TEST_SIZE = 0.20
TARGET = "Exited"

MODEL_FEATURES = [
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

ID_COLS = ["RowNumber", "CustomerId", "Surname"]

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# ------------------------------------------------------------
# 2. Create immutable modeling table
# ------------------------------------------------------------

X = df_raw[MODEL_FEATURES].copy()
y = df_raw[TARGET].astype(int).copy()

all_indices = np.arange(len(df_raw))

dev_idx, test_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=GLOBAL_SEED,
    shuffle=True
)

X_dev = X.iloc[dev_idx].copy()
X_test = X.iloc[test_idx].copy()

y_dev = y.iloc[dev_idx].copy()
y_test = y.iloc[test_idx].copy()

# IDs retained only for audit/tracking
id_dev = df_raw.iloc[dev_idx][ID_COLS].copy()
id_test = df_raw.iloc[test_idx][ID_COLS].copy()

# ------------------------------------------------------------
# 3. Split integrity checks
# ------------------------------------------------------------

print("=" * 85)
print("1. SPLIT SHAPES")
print("=" * 85)

print("Full X:", X.shape)
print("Development X:", X_dev.shape)
print("Test X:", X_test.shape)

print("\nDevelopment y:", y_dev.shape)
print("Test y:", y_test.shape)


print("\n" + "=" * 85)
print("2. CLASS DISTRIBUTION")
print("=" * 85)

def class_summary(y_part, name):
    counts = y_part.value_counts().sort_index()
    percentages = (
        y_part.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )

    return pd.DataFrame({
        "split": name,
        "class": counts.index,
        "count": counts.values,
        "percentage": percentages.values
    })

distribution_table = pd.concat([
    class_summary(y, "Full"),
    class_summary(y_dev, "Development"),
    class_summary(y_test, "Test")
], ignore_index=True)

print(
    distribution_table
    .round({"percentage": 4})
    .to_string(index=False)
)


print("\n" + "=" * 85)
print("3. OVERLAP AND COVERAGE CHECKS")
print("=" * 85)

print(
    "Development/test index overlap:",
    len(set(dev_idx).intersection(set(test_idx)))
)

print(
    "Total unique indices covered:",
    len(set(dev_idx).union(set(test_idx)))
)

print(
    "All observations covered exactly once:",
    len(set(dev_idx).union(set(test_idx))) == len(df_raw)
)

print(
    "CustomerId overlap:",
    len(
        set(id_dev["CustomerId"])
        .intersection(set(id_test["CustomerId"]))
    )
)

print(
    "Development CustomerId unique:",
    id_dev["CustomerId"].is_unique
)

print(
    "Test CustomerId unique:",
    id_test["CustomerId"].is_unique
)


print("\n" + "=" * 85)
print("4. FEATURE DOMAIN COVERAGE")
print("=" * 85)

for col in ["Geography", "Gender", "Tenure", "NumOfProducts",
            "HasCrCard", "IsActiveMember"]:

    dev_values = set(X_dev[col].unique())
    test_values = set(X_test[col].unique())

    unseen_in_test = test_values - dev_values

    print(
        f"{col}: "
        f"dev={sorted(dev_values)}, "
        f"test={sorted(test_values)}, "
        f"unseen_test_values={sorted(unseen_in_test)}"
    )


# ------------------------------------------------------------
# 5. Stable split fingerprints
# ------------------------------------------------------------

def index_fingerprint(indices):
    arr = np.sort(np.asarray(indices, dtype=np.int64))
    return hashlib.sha256(arr.tobytes()).hexdigest()

dev_fingerprint = index_fingerprint(dev_idx)
test_fingerprint = index_fingerprint(test_idx)

print("\n" + "=" * 85)
print("5. SPLIT FINGERPRINTS")
print("=" * 85)

print("Development index SHA-256:", dev_fingerprint)
print("Test index SHA-256:", test_fingerprint)


# ------------------------------------------------------------
# 6. Save reproducibility artifacts
# ------------------------------------------------------------

split_manifest = pd.DataFrame({
    "original_index": all_indices,
    "CustomerId": df_raw["CustomerId"].values,
    "split": np.where(
        np.isin(all_indices, test_idx),
        "test",
        "development"
    )
})

split_manifest = split_manifest.sort_values(
    "original_index"
).reset_index(drop=True)

split_manifest.to_csv(
    "/kaggle/working/split_manifest.csv",
    index=False
)

experiment_config = {
    "global_seed": GLOBAL_SEED,
    "test_size": TEST_SIZE,
    "target": TARGET,
    "model_features": MODEL_FEATURES,
    "excluded_identifiers": ID_COLS,
    "development_size": int(len(dev_idx)),
    "test_size_n": int(len(test_idx)),
    "development_index_sha256": dev_fingerprint,
    "test_index_sha256": test_fingerprint,
    "dataset_sha256": (
        "3996cd1fa372e0db0cd9c0ebac35bbd4e8e3c65"
        "fb942bb010c826e7b1eeef0a0"
    )
}

with open(
    "/kaggle/working/experiment_config.json",
    "w"
) as f:
    json.dump(experiment_config, f, indent=4)

print("\nSaved:")
print("/kaggle/working/split_manifest.csv")
print("/kaggle/working/experiment_config.json")

In [ ]:
# ============================================================
# LEAKAGE-SAFE SANITY BASELINES
# Development set only
# ============================================================

import time
import json
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    log_loss,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Feature groups
# ------------------------------------------------------------

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]


# ------------------------------------------------------------
# 2. Leakage-safe preprocessing
# ------------------------------------------------------------

linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None,
                sparse_output=True
            ),
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop"
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None,
                sparse_output=True
            ),
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop"
)


# ------------------------------------------------------------
# 3. Baseline models
# No SMOTE and no class weighting at this stage
# ------------------------------------------------------------

models = {
    "Dummy": Pipeline([
        ("preprocessor", clone(linear_preprocessor)),
        (
            "model",
            DummyClassifier(
                strategy="prior",
                random_state=GLOBAL_SEED
            )
        )
    ]),

    "LogisticRegression": Pipeline([
        ("preprocessor", clone(linear_preprocessor)),
        (
            "model",
            LogisticRegression(
                solver="liblinear",
                penalty="l2",
                C=1.0,
                max_iter=3000,
                random_state=GLOBAL_SEED
            )
        )
    ]),

    "RandomForest": Pipeline([
        ("preprocessor", clone(tree_preprocessor)),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_features="sqrt",
                min_samples_leaf=5,
                class_weight=None,
                n_jobs=-1,
                random_state=GLOBAL_SEED
            )
        )
    ])
}


# ------------------------------------------------------------
# 4. Fixed development CV
# ------------------------------------------------------------

sanity_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=GLOBAL_SEED
)


# ------------------------------------------------------------
# 5. Metric function
# ------------------------------------------------------------

def evaluate_probabilities(y_true, y_prob, threshold=0.50):
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Brier_Score": brier_score_loss(
            y_true,
            y_prob
        ),
        "Log_Loss": log_loss(
            y_true,
            y_prob,
            labels=[0, 1]
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


# ------------------------------------------------------------
# 6. Generate out-of-fold predictions
# ------------------------------------------------------------

results = []
oof_predictions = pd.DataFrame({
    "original_index": X_dev.index,
    "y_true": y_dev.values
})

for model_name, pipeline in models.items():

    print("=" * 80)
    print(f"Running: {model_name}")
    print("=" * 80)

    start_time = time.perf_counter()

    oof_prob = cross_val_predict(
        estimator=pipeline,
        X=X_dev,
        y=y_dev,
        cv=sanity_cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    elapsed = time.perf_counter() - start_time

    metrics = evaluate_probabilities(
        y_true=y_dev.values,
        y_prob=oof_prob,
        threshold=0.50
    )

    metrics["Model"] = model_name
    metrics["CV_Time_Seconds"] = elapsed

    results.append(metrics)
    oof_predictions[f"{model_name}_prob"] = oof_prob

    print(
        f"ROC-AUC: {metrics['ROC_AUC']:.4f} | "
        f"PR-AUC: {metrics['PR_AUC']:.4f} | "
        f"F1: {metrics['F1']:.4f} | "
        f"MCC: {metrics['MCC']:.4f}"
    )

    print(
        f"Confusion matrix: "
        f"TN={metrics['TN']}, "
        f"FP={metrics['FP']}, "
        f"FN={metrics['FN']}, "
        f"TP={metrics['TP']}"
    )

    print(f"CV time: {elapsed:.2f} seconds\n")


# ------------------------------------------------------------
# 7. Consolidated results
# ------------------------------------------------------------

sanity_results = pd.DataFrame(results)

ordered_columns = [
    "Model",
    "ROC_AUC",
    "PR_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "Brier_Score",
    "Log_Loss",
    "TN",
    "FP",
    "FN",
    "TP",
    "CV_Time_Seconds"
]

sanity_results = sanity_results[ordered_columns]

print("\n" + "=" * 100)
print("SANITY BASELINE RESULTS — 5-FOLD OOF ON DEVELOPMENT SET")
print("=" * 100)

print(
    sanity_results
    .sort_values("ROC_AUC", ascending=False)
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 8. Automatic leakage and validity checks
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("AUTOMATIC SANITY CHECKS")
print("=" * 100)

dummy_row = sanity_results.loc[
    sanity_results["Model"] == "Dummy"
].iloc[0]

print(
    "Expected no-skill PR-AUC:",
    round(y_dev.mean(), 4)
)

print(
    "Observed Dummy PR-AUC:",
    round(dummy_row["PR_AUC"], 4)
)

for _, row in sanity_results.iterrows():

    if row["Model"] == "Dummy":
        continue

    if row["ROC_AUC"] >= 0.95:
        status = "WARNING: suspiciously high"
    elif row["ROC_AUC"] >= 0.75:
        status = "Plausible predictive signal"
    else:
        status = "Weak predictive signal"

    print(
        f"{row['Model']}: "
        f"ROC-AUC={row['ROC_AUC']:.4f} → {status}"
    )


# ------------------------------------------------------------
# 9. Save development OOF artifacts
# ------------------------------------------------------------

sanity_results.to_csv(
    "/kaggle/working/sanity_baseline_results.csv",
    index=False
)

oof_predictions.to_csv(
    "/kaggle/working/sanity_oof_predictions.csv",
    index=False
)

print("\nSaved:")
print("/kaggle/working/sanity_baseline_results.csv")
print("/kaggle/working/sanity_oof_predictions.csv")

In [ ]:
# ============================================================
# KAGGLE ENVIRONMENT AND PACKAGE AUDIT
# ============================================================

import os
import sys
import json
import platform
import subprocess
import importlib.util
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import psutil
except ImportError:
    psutil = None


# ------------------------------------------------------------
# 1. System information
# ------------------------------------------------------------

print("=" * 90)
print("1. SYSTEM INFORMATION")
print("=" * 90)

print("Python version:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Processor:", platform.processor() or "Not reported")
print("Logical CPU cores:", os.cpu_count())

if psutil is not None:
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    print(f"Total RAM: {total_ram_gb:.2f} GB")
else:
    total_ram_gb = None
    print("Total RAM: psutil unavailable")


# ------------------------------------------------------------
# 2. GPU / CUDA availability
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("2. GPU AND CUDA")
print("=" * 90)

gpu_info = {
    "torch_installed": False,
    "cuda_available": False,
    "cuda_version": None,
    "gpu_name": None,
    "gpu_count": 0
}

try:
    import torch

    gpu_info["torch_installed"] = True
    gpu_info["cuda_available"] = torch.cuda.is_available()
    gpu_info["cuda_version"] = torch.version.cuda
    gpu_info["gpu_count"] = torch.cuda.device_count()

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())

    if torch.cuda.is_available():
        gpu_info["gpu_name"] = torch.cuda.get_device_name(0)

        print("GPU name:", torch.cuda.get_device_name(0))
        print(
            "GPU memory allocated:",
            round(torch.cuda.memory_allocated(0) / 1024**3, 4),
            "GB"
        )
        print(
            "GPU memory reserved:",
            round(torch.cuda.memory_reserved(0) / 1024**3, 4),
            "GB"
        )

except Exception as exc:
    print("PyTorch/GPU check failed:", repr(exc))


print("\nNVIDIA-SMI:")

try:
    nvidia_output = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True,
        timeout=15
    )

    if nvidia_output.returncode == 0:
        print(nvidia_output.stdout.strip())
    else:
        print("No active NVIDIA GPU detected.")

except Exception:
    print("nvidia-smi unavailable.")


# ------------------------------------------------------------
# 3. Package availability and versions
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("3. PACKAGE AVAILABILITY")
print("=" * 90)

package_map = {
    "numpy": ("numpy", "numpy"),
    "pandas": ("pandas", "pandas"),
    "scipy": ("scipy", "scipy"),
    "scikit-learn": ("sklearn", "scikit-learn"),
    "xgboost": ("xgboost", "xgboost"),
    "lightgbm": ("lightgbm", "lightgbm"),
    "catboost": ("catboost", "catboost"),
    "optuna": ("optuna", "optuna"),
    "torch": ("torch", "torch"),
    "shap": ("shap", "shap"),
    "imbalanced-learn": ("imblearn", "imbalanced-learn"),
    "pytorch-tabnet": ("pytorch_tabnet", "pytorch-tabnet"),
    "rtdl-revisiting-models": (
        "rtdl_revisiting_models",
        "rtdl-revisiting-models"
    )
}

package_rows = []

for display_name, (import_name, distribution_name) in package_map.items():

    installed = importlib.util.find_spec(import_name) is not None

    if installed:
        try:
            version = importlib.metadata.version(distribution_name)
        except importlib.metadata.PackageNotFoundError:
            version = "Installed, version unresolved"
    else:
        version = None

    package_rows.append({
        "package": display_name,
        "import_name": import_name,
        "installed": installed,
        "version": version
    })

package_table = pd.DataFrame(package_rows)

print(package_table.to_string(index=False))


# ------------------------------------------------------------
# 4. Core package import test
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("4. CORE IMPORT TEST")
print("=" * 90)

core_imports = {}

for package_name, import_name in {
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm",
    "CatBoost": "catboost",
    "Optuna": "optuna",
    "SHAP": "shap",
    "Imbalanced-learn": "imblearn"
}.items():

    try:
        module = __import__(import_name)
        core_imports[package_name] = "OK"
        print(f"{package_name}: OK")
    except Exception as exc:
        core_imports[package_name] = repr(exc)
        print(f"{package_name}: FAILED -> {repr(exc)}")


# ------------------------------------------------------------
# 5. Save environment record
# ------------------------------------------------------------

environment_record = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "logical_cpu_cores": os.cpu_count(),
    "total_ram_gb": total_ram_gb,
    "gpu": gpu_info,
    "packages": package_rows,
    "core_import_test": core_imports
}

environment_path = Path(
    "/kaggle/working/environment_record.json"
)

with open(environment_path, "w") as f:
    json.dump(
        environment_record,
        f,
        indent=4,
        default=str
    )

print("\n" + "=" * 90)
print("5. SAVED ARTIFACT")
print("=" * 90)

print(environment_path)

In [ ]:
# ============================================================
# CLASSICAL MODEL HYPERPARAMETER OPTIMIZATION
# Development set only, CPU only
# ============================================================

import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

GLOBAL_SEED = 42
N_TRIALS = 30
N_THREADS = 4

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

ARTIFACT_DIR = Path("/kaggle/working/classical_hpo")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Fixed folds shared by every model
# ------------------------------------------------------------

hpo_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=GLOBAL_SEED
)

CV_SPLITS = list(
    hpo_cv.split(X_dev, y_dev)
)

print("Number of fixed CV folds:", len(CV_SPLITS))

for fold_id, (train_idx, valid_idx) in enumerate(CV_SPLITS, start=1):
    print(
        f"Fold {fold_id}: "
        f"train={len(train_idx)}, "
        f"validation={len(valid_idx)}, "
        f"validation prevalence={y_dev.iloc[valid_idx].mean():.4f}"
    )


# ------------------------------------------------------------
# 3. Preprocessing factories
# ------------------------------------------------------------

def make_linear_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )


def make_tree_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                "passthrough",
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )


# ------------------------------------------------------------
# 4. Generic CV evaluators
# ------------------------------------------------------------

def evaluate_preprocessed_model(
    model,
    preprocessor_factory
):
    fold_scores = []

    for train_idx, valid_idx in CV_SPLITS:

        X_train_fold = X_dev.iloc[train_idx]
        X_valid_fold = X_dev.iloc[valid_idx]

        y_train_fold = y_dev.iloc[train_idx]
        y_valid_fold = y_dev.iloc[valid_idx]

        preprocessor = preprocessor_factory()

        X_train_processed = preprocessor.fit_transform(
            X_train_fold
        )

        X_valid_processed = preprocessor.transform(
            X_valid_fold
        )

        model.fit(
            X_train_processed,
            y_train_fold
        )

        valid_prob = model.predict_proba(
            X_valid_processed
        )[:, 1]

        fold_auc = roc_auc_score(
            y_valid_fold,
            valid_prob
        )

        fold_scores.append(fold_auc)

        del (
            X_train_processed,
            X_valid_processed,
            preprocessor
        )

        gc.collect()

    return np.asarray(fold_scores)


def evaluate_catboost_model(model):
    fold_scores = []

    for train_idx, valid_idx in CV_SPLITS:

        X_train_fold = X_dev.iloc[train_idx].copy()
        X_valid_fold = X_dev.iloc[valid_idx].copy()

        y_train_fold = y_dev.iloc[train_idx]
        y_valid_fold = y_dev.iloc[valid_idx]

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=CATEGORICAL_FEATURES,
            verbose=False
        )

        valid_prob = model.predict_proba(
            X_valid_fold
        )[:, 1]

        fold_auc = roc_auc_score(
            y_valid_fold,
            valid_prob
        )

        fold_scores.append(fold_auc)

        gc.collect()

    return np.asarray(fold_scores)


# ------------------------------------------------------------
# 5. Objective functions
# ------------------------------------------------------------

def objective_logistic(trial):

    params = {
        "C": trial.suggest_float(
            "C",
            1e-4,
            1e3,
            log=True
        ),
        "penalty": trial.suggest_categorical(
            "penalty",
            ["l1", "l2"]
        )
    }

    model = LogisticRegression(
        solver="liblinear",
        max_iter=5000,
        class_weight=None,
        random_state=GLOBAL_SEED,
        **params
    )

    scores = evaluate_preprocessed_model(
        model=model,
        preprocessor_factory=make_linear_preprocessor
    )

    trial.set_user_attr(
        "fold_scores",
        scores.tolist()
    )

    trial.set_user_attr(
        "cv_std",
        float(scores.std(ddof=1))
    )

    return float(scores.mean())


def objective_random_forest(trial):

    max_depth_choice = trial.suggest_categorical(
        "max_depth",
        [None, 5, 8, 12, 16, 20]
    )

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            200,
            600,
            step=50
        ),
        "max_depth": max_depth_choice,
        "min_samples_split": trial.suggest_int(
            "min_samples_split",
            2,
            20
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            12
        ),
        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", 0.5, 0.8]
        )
    }

    model = RandomForestClassifier(
        class_weight=None,
        bootstrap=True,
        n_jobs=N_THREADS,
        random_state=GLOBAL_SEED,
        **params
    )

    scores = evaluate_preprocessed_model(
        model=model,
        preprocessor_factory=make_tree_preprocessor
    )

    trial.set_user_attr(
        "fold_scores",
        scores.tolist()
    )

    trial.set_user_attr(
        "cv_std",
        float(scores.std(ddof=1))
    )

    return float(scores.mean())


def objective_xgboost(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            150,
            700,
            step=50
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            8
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.25,
            log=True
        ),
        "min_child_weight": trial.suggest_float(
            "min_child_weight",
            1.0,
            15.0,
            log=True
        ),
        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.60,
            1.00
        ),
        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            5.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            5.0,
            log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            20.0,
            log=True
        )
    }

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=1.0,
        n_jobs=N_THREADS,
        random_state=GLOBAL_SEED,
        verbosity=0,
        **params
    )

    scores = evaluate_preprocessed_model(
        model=model,
        preprocessor_factory=make_tree_preprocessor
    )

    trial.set_user_attr(
        "fold_scores",
        scores.tolist()
    )

    trial.set_user_attr(
        "cv_std",
        float(scores.std(ddof=1))
    )

    return float(scores.mean())


def objective_lightgbm(trial):

    max_depth = trial.suggest_int(
        "max_depth",
        3,
        10
    )

    maximum_leaves = min(
        (2 ** max_depth) - 1,
        127
    )

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            150,
            700,
            step=50
        ),
        "max_depth": max_depth,
        "num_leaves": trial.suggest_int(
            "num_leaves",
            7,
            maximum_leaves
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.25,
            log=True
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            10,
            100
        ),
        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.60,
            1.00
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            5.0,
            log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            20.0,
            log=True
        )
    }

    model = LGBMClassifier(
        objective="binary",
        class_weight=None,
        subsample_freq=1,
        random_state=GLOBAL_SEED,
        n_jobs=N_THREADS,
        verbosity=-1,
        **params
    )

    scores = evaluate_preprocessed_model(
        model=model,
        preprocessor_factory=make_tree_preprocessor
    )

    trial.set_user_attr(
        "fold_scores",
        scores.tolist()
    )

    trial.set_user_attr(
        "cv_std",
        float(scores.std(ddof=1))
    )

    return float(scores.mean())


def objective_catboost(trial):

    params = {
        "iterations": trial.suggest_int(
            "iterations",
            200,
            700,
            step=50
        ),
        "depth": trial.suggest_int(
            "depth",
            4,
            10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.25,
            log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1.0,
            20.0,
            log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength",
            1e-3,
            5.0,
            log=True
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0.0,
            5.0
        ),
        "border_count": trial.suggest_categorical(
            "border_count",
            [32, 64, 128]
        )
    }

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        bootstrap_type="Bayesian",
        auto_class_weights=None,
        random_seed=GLOBAL_SEED,
        thread_count=N_THREADS,
        allow_writing_files=False,
        verbose=False,
        **params
    )

    scores = evaluate_catboost_model(model)

    trial.set_user_attr(
        "fold_scores",
        scores.tolist()
    )

    trial.set_user_attr(
        "cv_std",
        float(scores.std(ddof=1))
    )

    return float(scores.mean())


# ------------------------------------------------------------
# 6. Run studies sequentially
# ------------------------------------------------------------

OBJECTIVES = {
    "LogisticRegression": objective_logistic,
    "RandomForest": objective_random_forest,
    "XGBoost": objective_xgboost,
    "LightGBM": objective_lightgbm,
    "CatBoost": objective_catboost
}

studies = {}
best_model_records = []

total_start_time = time.perf_counter()

for model_name, objective_function in OBJECTIVES.items():

    print("\n" + "=" * 100)
    print(f"HYPERPARAMETER OPTIMIZATION: {model_name}")
    print("=" * 100)

    model_start_time = time.perf_counter()

    sampler = optuna.samplers.TPESampler(
        seed=GLOBAL_SEED
    )

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name=model_name
    )

    study.optimize(
        objective_function,
        n_trials=N_TRIALS,
        n_jobs=1,
        show_progress_bar=True,
        gc_after_trial=True
    )

    elapsed_seconds = (
        time.perf_counter() - model_start_time
    )

    studies[model_name] = study

    best_trial = study.best_trial

    best_model_records.append({
        "Model": model_name,
        "Best_CV_ROC_AUC": best_trial.value,
        "Best_CV_SD": best_trial.user_attrs.get(
            "cv_std",
            np.nan
        ),
        "Best_Trial_Number": best_trial.number,
        "Trials": len(study.trials),
        "HPO_Time_Seconds": elapsed_seconds,
        "Best_Params": best_trial.params
    })

    print(
        f"\nBest mean ROC-AUC: "
        f"{best_trial.value:.6f}"
    )

    print(
        f"Fold SD: "
        f"{best_trial.user_attrs.get('cv_std', np.nan):.6f}"
    )

    print("Best parameters:")
    print(
        json.dumps(
            best_trial.params,
            indent=4,
            default=str
        )
    )

    trials_df = study.trials_dataframe()

    trials_df.to_csv(
        ARTIFACT_DIR / f"{model_name}_optuna_trials.csv",
        index=False
    )

    with open(
        ARTIFACT_DIR / f"{model_name}_best_params.json",
        "w"
    ) as f:
        json.dump(
            {
                "best_value": best_trial.value,
                "best_params": best_trial.params,
                "best_trial_number": best_trial.number,
                "fold_scores": best_trial.user_attrs.get(
                    "fold_scores"
                ),
                "cv_std": best_trial.user_attrs.get(
                    "cv_std"
                ),
                "hpo_time_seconds": elapsed_seconds
            },
            f,
            indent=4,
            default=str
        )


# ------------------------------------------------------------
# 7. HPO summary
# ------------------------------------------------------------

total_elapsed = (
    time.perf_counter() - total_start_time
)

hpo_summary = pd.DataFrame(
    best_model_records
)

hpo_summary = hpo_summary.sort_values(
    "Best_CV_ROC_AUC",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 110)
print("CLASSICAL HPO SUMMARY")
print("=" * 110)

print(
    hpo_summary[
        [
            "Model",
            "Best_CV_ROC_AUC",
            "Best_CV_SD",
            "Best_Trial_Number",
            "Trials",
            "HPO_Time_Seconds"
        ]
    ]
    .round(6)
    .to_string(index=False)
)

print(
    f"\nTotal HPO time: "
    f"{total_elapsed / 60:.2f} minutes"
)

hpo_summary.to_pickle(
    ARTIFACT_DIR / "classical_hpo_summary.pkl"
)

hpo_summary.drop(
    columns=["Best_Params"]
).to_csv(
    ARTIFACT_DIR / "classical_hpo_summary.csv",
    index=False
)

with open(
    ARTIFACT_DIR / "all_best_params.json",
    "w"
) as f:
    json.dump(
        {
            row["Model"]: row["Best_Params"]
            for row in best_model_records
        },
        f,
        indent=4,
        default=str
    )

print("\nSaved directory:")
print(ARTIFACT_DIR)

In [ ]:
# ============================================================
# REPEATED STRATIFIED CV FOR CLASSICAL MODELS
# 5 folds × 3 repeats, development set only
# CPU only
# ============================================================

import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    log_loss,
    confusion_matrix
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

GLOBAL_SEED = 42
N_SPLITS = 5
N_REPEATS = 3
N_THREADS = 4

OUTPUT_DIR = Path(
    "/kaggle/working/classical_repeated_cv"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]


# ------------------------------------------------------------
# 2. Recover locked best parameters
# ------------------------------------------------------------

MODEL_NAMES = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost"
]

if "studies" in globals():
    best_params_map = {
        model_name: studies[model_name].best_params
        for model_name in MODEL_NAMES
    }
else:
    with open(
        "/kaggle/working/classical_hpo/all_best_params.json",
        "r"
    ) as f:
        best_params_map = json.load(f)

print("=" * 95)
print("LOCKED BEST PARAMETERS")
print("=" * 95)

for model_name, params in best_params_map.items():
    print(f"\n{model_name}")
    print(json.dumps(params, indent=4, default=str))


# ------------------------------------------------------------
# 3. Preprocessing factories
# ------------------------------------------------------------

def make_linear_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )


def make_tree_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                "passthrough",
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )


# ------------------------------------------------------------
# 4. Model factory
# New seed for each split captures algorithmic variability
# ------------------------------------------------------------

def build_model(model_name, seed):

    params = best_params_map.get(model_name, {})

    if model_name == "Dummy":
        return DummyClassifier(
            strategy="prior",
            random_state=seed
        )

    if model_name == "LogisticRegression":
        return LogisticRegression(
            solver="liblinear",
            max_iter=5000,
            class_weight=None,
            random_state=seed,
            **params
        )

    if model_name == "RandomForest":
        return RandomForestClassifier(
            bootstrap=True,
            class_weight=None,
            n_jobs=N_THREADS,
            random_state=seed,
            **params
        )

    if model_name == "XGBoost":
        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=1.0,
            n_jobs=N_THREADS,
            random_state=seed,
            verbosity=0,
            **params
        )

    if model_name == "LightGBM":
        return LGBMClassifier(
            objective="binary",
            class_weight=None,
            subsample_freq=1,
            n_jobs=N_THREADS,
            random_state=seed,
            verbosity=-1,
            **params
        )

    if model_name == "CatBoost":
        return CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            bootstrap_type="Bayesian",
            auto_class_weights=None,
            random_seed=seed,
            thread_count=N_THREADS,
            allow_writing_files=False,
            verbose=False,
            **params
        )

    raise ValueError(f"Unknown model: {model_name}")


# ------------------------------------------------------------
# 5. Metric utilities
# ------------------------------------------------------------

def expected_calibration_error(
    y_true,
    y_prob,
    n_bins=10
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(
        y_prob,
        bin_edges[1:-1],
        right=True
    )

    ece = 0.0

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id

        if mask.sum() == 0:
            continue

        mean_confidence = y_prob[mask].mean()
        observed_rate = y_true[mask].mean()

        ece += (
            mask.mean()
            * abs(mean_confidence - observed_rate)
        )

    return float(ece)


def probability_metrics(y_true, y_prob):
    return {
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(
            y_true,
            y_prob
        ),
        "Brier_Score": brier_score_loss(
            y_true,
            y_prob
        ),
        "Log_Loss": log_loss(
            y_true,
            y_prob,
            labels=[0, 1]
        ),
        "ECE_10": expected_calibration_error(
            y_true,
            y_prob,
            n_bins=10
        )
    }


def threshold_metrics(
    y_true,
    y_prob,
    threshold=0.50
):
    y_pred = (
        np.asarray(y_prob) >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp)

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),
        "MCC": matthews_corrcoef(
            y_true,
            y_pred
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


# ------------------------------------------------------------
# 6. Fixed repeated CV splits
# ------------------------------------------------------------

repeated_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=GLOBAL_SEED
)

CV_SPLITS = list(
    repeated_cv.split(X_dev, y_dev)
)

print("\n" + "=" * 95)
print("REPEATED CV CONFIGURATION")
print("=" * 95)

print("Number of folds:", N_SPLITS)
print("Number of repeats:", N_REPEATS)
print("Total validation evaluations:", len(CV_SPLITS))


# ------------------------------------------------------------
# 7. Evaluate all models
# ------------------------------------------------------------

ALL_MODELS = [
    "Dummy",
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost"
]

fold_records = []
oof_records = []

total_start = time.perf_counter()

for model_name in ALL_MODELS:

    print("\n" + "=" * 95)
    print(f"RUNNING REPEATED CV: {model_name}")
    print("=" * 95)

    model_start = time.perf_counter()

    for split_id, (train_pos, valid_pos) in enumerate(
        CV_SPLITS
    ):

        repeat_id = split_id // N_SPLITS + 1
        fold_id = split_id % N_SPLITS + 1
        split_seed = GLOBAL_SEED + split_id

        X_train_fold = X_dev.iloc[train_pos]
        X_valid_fold = X_dev.iloc[valid_pos]

        y_train_fold = y_dev.iloc[train_pos]
        y_valid_fold = y_dev.iloc[valid_pos]

        model = build_model(
            model_name=model_name,
            seed=split_seed
        )

        fit_start = time.perf_counter()

        if model_name == "CatBoost":

            model.fit(
                X_train_fold,
                y_train_fold,
                cat_features=CATEGORICAL_FEATURES,
                verbose=False
            )

            fit_time = (
                time.perf_counter() - fit_start
            )

            predict_start = time.perf_counter()

            valid_prob = model.predict_proba(
                X_valid_fold
            )[:, 1]

        else:

            if model_name in [
                "Dummy",
                "LogisticRegression"
            ]:
                preprocessor = (
                    make_linear_preprocessor()
                )
            else:
                preprocessor = (
                    make_tree_preprocessor()
                )

            X_train_processed = (
                preprocessor.fit_transform(
                    X_train_fold
                )
            )

            X_valid_processed = (
                preprocessor.transform(
                    X_valid_fold
                )
            )

            model.fit(
                X_train_processed,
                y_train_fold
            )

            fit_time = (
                time.perf_counter() - fit_start
            )

            predict_start = time.perf_counter()

            valid_prob = model.predict_proba(
                X_valid_processed
            )[:, 1]

        predict_time = (
            time.perf_counter() - predict_start
        )

        metrics = probability_metrics(
            y_valid_fold.values,
            valid_prob
        )

        fold_records.append({
            "Model": model_name,
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Split_Seed": split_seed,
            "Validation_N": len(valid_pos),
            "Validation_Prevalence":
                y_valid_fold.mean(),
            **metrics,
            "Fit_Time_Seconds": fit_time,
            "Predict_Time_Seconds":
                predict_time
        })

        for local_row, probability in zip(
            valid_pos,
            valid_prob
        ):
            oof_records.append({
                "Model": model_name,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Development_Position":
                    int(local_row),
                "Original_Index":
                    int(X_dev.index[local_row]),
                "y_true":
                    int(y_dev.iloc[local_row]),
                "y_prob": float(probability)
            })

        print(
            f"Repeat {repeat_id}, Fold {fold_id}: "
            f"ROC-AUC={metrics['ROC_AUC']:.4f}, "
            f"PR-AUC={metrics['PR_AUC']:.4f}"
        )

        del model

        if model_name != "CatBoost":
            del (
                preprocessor,
                X_train_processed,
                X_valid_processed
            )

        gc.collect()

    model_elapsed = (
        time.perf_counter() - model_start
    )

    print(
        f"{model_name} completed in "
        f"{model_elapsed:.2f} seconds"
    )


# ------------------------------------------------------------
# 8. Fold-level summary
# ------------------------------------------------------------

fold_results = pd.DataFrame(fold_records)
oof_long = pd.DataFrame(oof_records)

summary_metrics = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "ECE_10",
    "Fit_Time_Seconds",
    "Predict_Time_Seconds"
]

fold_summary = (
    fold_results
    .groupby("Model")[summary_metrics]
    .agg(["mean", "std"])
)

fold_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in fold_summary.columns
]

fold_summary = (
    fold_summary
    .reset_index()
    .sort_values(
        "ROC_AUC_mean",
        ascending=False
    )
)


print("\n" + "=" * 120)
print("REPEATED CV FOLD-LEVEL SUMMARY")
print("=" * 120)

display_columns = [
    "Model",
    "ROC_AUC_mean",
    "ROC_AUC_std",
    "PR_AUC_mean",
    "PR_AUC_std",
    "Brier_Score_mean",
    "Brier_Score_std",
    "ECE_10_mean",
    "ECE_10_std",
    "Fit_Time_Seconds_mean",
    "Predict_Time_Seconds_mean"
]

print(
    fold_summary[display_columns]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. Average each observation's three OOF predictions
# ------------------------------------------------------------

mean_oof = (
    oof_long
    .groupby(
        [
            "Model",
            "Development_Position",
            "Original_Index",
            "y_true"
        ],
        as_index=False
    )
    .agg(
        Mean_OOF_Probability=("y_prob", "mean"),
        SD_OOF_Probability=("y_prob", "std"),
        Prediction_Count=("y_prob", "count")
    )
)

assert (
    mean_oof["Prediction_Count"] == N_REPEATS
).all()


# ------------------------------------------------------------
# 10. Aggregate OOF metrics
# ------------------------------------------------------------

aggregate_records = []

for model_name in ALL_MODELS:

    model_oof = (
        mean_oof[
            mean_oof["Model"] == model_name
        ]
        .sort_values("Development_Position")
    )

    y_true_model = model_oof[
        "y_true"
    ].values

    y_prob_model = model_oof[
        "Mean_OOF_Probability"
    ].values

    prob_metrics = probability_metrics(
        y_true_model,
        y_prob_model
    )

    class_metrics = threshold_metrics(
        y_true_model,
        y_prob_model,
        threshold=0.50
    )

    aggregate_records.append({
        "Model": model_name,
        **prob_metrics,
        **class_metrics,
        "Mean_Probability_SD_Across_Repeats":
            model_oof[
                "SD_OOF_Probability"
            ].mean()
    })

aggregate_results = (
    pd.DataFrame(aggregate_records)
    .sort_values(
        "ROC_AUC",
        ascending=False
    )
)


print("\n" + "=" * 135)
print("AGGREGATED MEAN OOF RESULTS")
print("Each customer receives one OOF prediction per repeat; the three probabilities are averaged.")
print("=" * 135)

aggregate_columns = [
    "Model",
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "ECE_10",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "TN",
    "FP",
    "FN",
    "TP",
    "Mean_Probability_SD_Across_Repeats"
]

print(
    aggregate_results[aggregate_columns]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 11. Save artifacts
# ------------------------------------------------------------

fold_results.to_csv(
    OUTPUT_DIR / "fold_level_metrics.csv",
    index=False
)

fold_summary.to_csv(
    OUTPUT_DIR / "fold_level_summary.csv",
    index=False
)

oof_long.to_csv(
    OUTPUT_DIR / "repeated_oof_predictions_long.csv",
    index=False
)

mean_oof.to_csv(
    OUTPUT_DIR / "mean_oof_predictions.csv",
    index=False
)

aggregate_results.to_csv(
    OUTPUT_DIR / "aggregate_mean_oof_metrics.csv",
    index=False
)

total_elapsed = (
    time.perf_counter() - total_start
)

print("\nSaved directory:")
print(OUTPUT_DIR)

print(
    f"\nTotal repeated-CV time: "
    f"{total_elapsed / 60:.2f} minutes"
)

In [ ]:
# ============================================================
# DEEP MODEL HYPERPARAMETER OPTIMIZATION
# 15 trials/model × fixed 3-fold CV
# Development set only, CPU only
# ============================================================

import copy
import gc
import json
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from pytorch_tabnet.tab_model import TabNetClassifier
from pytabkit import (
    FTT_D_Classifier,
    RealTabR_D_Classifier
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ------------------------------------------------------------
# 1. Global configuration
# ------------------------------------------------------------

GLOBAL_SEED = 42
N_TRIALS = 15
N_SPLITS = 3
N_THREADS = 4
PYTABKIT_FOLD_TIME_LIMIT = 240  # seconds

OUTPUT_DIR = Path("/kaggle/working/deep_hpo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMP_DIR = OUTPUT_DIR / "temporary_training_files"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = OUTPUT_DIR / "deep_hpo.db"
STORAGE_URL = f"sqlite:///{DB_PATH}"

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(N_THREADS)

try:
    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )
except Exception:
    pass


# ------------------------------------------------------------
# 2. Fixed HPO folds shared across all models
# ------------------------------------------------------------

deep_hpo_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=GLOBAL_SEED
)

DEEP_HPO_SPLITS = list(
    deep_hpo_cv.split(X_dev, y_dev)
)

print("=" * 100)
print("DEEP HPO CONFIGURATION")
print("=" * 100)

print("Models: MLP, TabNet, FTTransformer, RealTabR")
print("Trials per model:", N_TRIALS)
print("CV folds:", N_SPLITS)
print("Device: CPU")
print("Persistent database:", DB_PATH)

for fold_id, (train_pos, valid_pos) in enumerate(
    DEEP_HPO_SPLITS,
    start=1
):
    print(
        f"Fold {fold_id}: "
        f"train={len(train_pos)}, "
        f"validation={len(valid_pos)}, "
        f"validation prevalence="
        f"{y_dev.iloc[valid_pos].mean():.4f}"
    )


# ------------------------------------------------------------
# 3. Shared utilities
# ------------------------------------------------------------

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def summarize_fold_predictions(
    y_true,
    y_prob
):
    return {
        "roc_auc": roc_auc_score(
            y_true,
            y_prob
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_prob
        ),
        "brier": brier_score_loss(
            y_true,
            y_prob
        )
    }


def finalize_trial_metrics(
    trial,
    fold_metrics
):
    roc_scores = np.array([
        item["roc_auc"]
        for item in fold_metrics
    ])

    pr_scores = np.array([
        item["pr_auc"]
        for item in fold_metrics
    ])

    brier_scores = np.array([
        item["brier"]
        for item in fold_metrics
    ])

    trial.set_user_attr(
        "fold_roc_auc",
        roc_scores.tolist()
    )

    trial.set_user_attr(
        "fold_pr_auc",
        pr_scores.tolist()
    )

    trial.set_user_attr(
        "fold_brier",
        brier_scores.tolist()
    )

    trial.set_user_attr(
        "roc_auc_std",
        float(roc_scores.std(ddof=1))
    )

    trial.set_user_attr(
        "mean_pr_auc",
        float(pr_scores.mean())
    )

    trial.set_user_attr(
        "mean_brier",
        float(brier_scores.mean())
    )

    return float(roc_scores.mean())


# ============================================================
# 4. CUSTOM MLP
# ============================================================

class TunableMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout,
        use_batch_norm,
        activation_name
    ):
        super().__init__()

        activation_classes = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "silu": nn.SiLU
        }

        activation_class = activation_classes[
            activation_name
        ]

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:

            layers.append(
                nn.Linear(
                    previous_dim,
                    hidden_dim
                )
            )

            if use_batch_norm:
                layers.append(
                    nn.BatchNorm1d(hidden_dim)
                )

            layers.append(
                activation_class()
            )

            if dropout > 0:
                layers.append(
                    nn.Dropout(dropout)
                )

            previous_dim = hidden_dim

        layers.append(
            nn.Linear(previous_dim, 1)
        )

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)


def objective_mlp(trial):

    hidden_name = trial.suggest_categorical(
        "hidden_structure",
        [
            "64-32",
            "128-64",
            "128-64-32",
            "256-128-64"
        ]
    )

    hidden_dims = tuple(
        int(value)
        for value in hidden_name.split("-")
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.50
    )

    use_batch_norm = trial.suggest_categorical(
        "use_batch_norm",
        [True, False]
    )

    activation_name = trial.suggest_categorical(
        "activation",
        ["relu", "gelu", "silu"]
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        3e-3,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-2,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [128, 256, 512]
    )

    max_epochs = 80
    patience = 10

    fold_metrics = []
    fold_epochs = []
    fold_parameter_counts = []

    for fold_id, (train_pos, valid_pos) in enumerate(
        DEEP_HPO_SPLITS,
        start=1
    ):
        fold_seed = (
            GLOBAL_SEED
            + trial.number * 100
            + fold_id
        )

        seed_everything(fold_seed)

        X_train_fold = X_dev.iloc[train_pos]
        X_valid_fold = X_dev.iloc[valid_pos]

        y_train_fold = (
            y_dev.iloc[train_pos]
            .astype(np.float32)
            .to_numpy()
        )

        y_valid_fold = (
            y_dev.iloc[valid_pos]
            .astype(int)
            .to_numpy()
        )

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    StandardScaler(),
                    NUMERIC_FEATURES
                ),
                (
                    "categorical",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    ),
                    CATEGORICAL_FEATURES
                )
            ],
            remainder="drop"
        )

        X_train_processed = (
            preprocessor
            .fit_transform(X_train_fold)
            .astype(np.float32)
        )

        X_valid_processed = (
            preprocessor
            .transform(X_valid_fold)
            .astype(np.float32)
        )

        train_dataset = TensorDataset(
            torch.from_numpy(
                X_train_processed
            ),
            torch.from_numpy(
                y_train_fold
            )
        )

        generator = torch.Generator()
        generator.manual_seed(fold_seed)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            generator=generator,
            drop_last=False
        )

        model = TunableMLP(
            input_dim=X_train_processed.shape[1],
            hidden_dims=hidden_dims,
            dropout=dropout,
            use_batch_norm=use_batch_norm,
            activation_name=activation_name
        )

        parameter_count = sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )

        criterion = nn.BCEWithLogitsLoss()

        X_valid_tensor = torch.from_numpy(
            X_valid_processed
        )

        best_auc = -np.inf
        best_state = None
        epochs_without_improvement = 0
        completed_epochs = 0

        for epoch in range(1, max_epochs + 1):

            model.train()

            for batch_X, batch_y in train_loader:

                optimizer.zero_grad()

                logits = model(batch_X)

                loss = criterion(
                    logits,
                    batch_y
                )

                loss.backward()
                optimizer.step()

            model.eval()

            with torch.no_grad():
                valid_prob_epoch = torch.sigmoid(
                    model(X_valid_tensor)
                ).numpy()

            valid_auc = roc_auc_score(
                y_valid_fold,
                valid_prob_epoch
            )

            completed_epochs = epoch

            if valid_auc > best_auc + 1e-5:
                best_auc = valid_auc
                best_state = copy.deepcopy(
                    model.state_dict()
                )
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if (
                epochs_without_improvement
                >= patience
            ):
                break

        model.load_state_dict(best_state)
        model.eval()

        with torch.no_grad():
            valid_prob = torch.sigmoid(
                model(X_valid_tensor)
            ).numpy()

        fold_metrics.append(
            summarize_fold_predictions(
                y_valid_fold,
                valid_prob
            )
        )

        fold_epochs.append(completed_epochs)
        fold_parameter_counts.append(
            parameter_count
        )

        del (
            model,
            optimizer,
            preprocessor,
            X_train_processed,
            X_valid_processed,
            train_dataset,
            train_loader
        )

        gc.collect()

    trial.set_user_attr(
        "completed_epochs",
        fold_epochs
    )

    trial.set_user_attr(
        "parameter_count",
        int(np.median(fold_parameter_counts))
    )

    return finalize_trial_metrics(
        trial,
        fold_metrics
    )


# ============================================================
# 5. TABNET
# ============================================================

def objective_tabnet(trial):

    n_d = trial.suggest_categorical(
        "n_d",
        [8, 16, 24, 32]
    )

    n_steps = trial.suggest_int(
        "n_steps",
        3,
        6
    )

    gamma = trial.suggest_float(
        "gamma",
        1.0,
        2.0
    )

    lambda_sparse = trial.suggest_float(
        "lambda_sparse",
        1e-6,
        1e-2,
        log=True
    )

    cat_emb_dim = trial.suggest_categorical(
        "cat_emb_dim",
        [1, 2, 3, 4]
    )

    n_independent = trial.suggest_int(
        "n_independent",
        1,
        3
    )

    n_shared = trial.suggest_int(
        "n_shared",
        1,
        3
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-3,
        3e-2,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [128, 256, 512]
    )

    virtual_batch_size = (
        batch_size // 4
    )

    fold_metrics = []
    fold_epochs = []
    fold_parameter_counts = []

    for fold_id, (train_pos, valid_pos) in enumerate(
        DEEP_HPO_SPLITS,
        start=1
    ):
        fold_seed = (
            GLOBAL_SEED
            + trial.number * 100
            + fold_id
        )

        seed_everything(fold_seed)

        X_train_fold = X_dev.iloc[train_pos]
        X_valid_fold = X_dev.iloc[valid_pos]

        y_train_fold = (
            y_dev.iloc[train_pos]
            .astype(int)
            .to_numpy()
        )

        y_valid_fold = (
            y_dev.iloc[valid_pos]
            .astype(int)
            .to_numpy()
        )

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    StandardScaler(),
                    NUMERIC_FEATURES
                ),
                (
                    "categorical",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1
                    ),
                    CATEGORICAL_FEATURES
                )
            ],
            remainder="drop"
        )

        X_train_processed = (
            preprocessor
            .fit_transform(X_train_fold)
            .astype(np.float32)
        )

        X_valid_processed = (
            preprocessor
            .transform(X_valid_fold)
            .astype(np.float32)
        )

        numeric_count = len(
            NUMERIC_FEATURES
        )

        cat_idxs = list(
            range(
                numeric_count,
                numeric_count
                + len(CATEGORICAL_FEATURES)
            )
        )

        encoder = (
            preprocessor
            .named_transformers_["categorical"]
        )

        X_train_processed[
            :, cat_idxs
        ] += 1

        X_valid_processed[
            :, cat_idxs
        ] += 1

        cat_dims = [
            len(categories) + 1
            for categories
            in encoder.categories_
        ]

        model = TabNetClassifier(
            n_d=n_d,
            n_a=n_d,
            n_steps=n_steps,
            gamma=gamma,
            cat_idxs=cat_idxs,
            cat_dims=cat_dims,
            cat_emb_dim=cat_emb_dim,
            n_independent=n_independent,
            n_shared=n_shared,
            lambda_sparse=lambda_sparse,
            seed=fold_seed,
            optimizer_fn=torch.optim.Adam,
            optimizer_params={
                "lr": learning_rate
            },
            mask_type="sparsemax",
            device_name="cpu",
            verbose=0
        )

        model.fit(
            X_train=X_train_processed,
            y_train=y_train_fold,
            eval_set=[
                (
                    X_valid_processed,
                    y_valid_fold
                )
            ],
            eval_name=["validation"],
            eval_metric=["auc"],
            max_epochs=100,
            patience=12,
            batch_size=batch_size,
            virtual_batch_size=(
                virtual_batch_size
            ),
            num_workers=0,
            drop_last=False,
            pin_memory=False,
            compute_importance=False
        )

        valid_prob = model.predict_proba(
            X_valid_processed
        )[:, 1]

        fold_metrics.append(
            summarize_fold_predictions(
                y_valid_fold,
                valid_prob
            )
        )

        completed_epochs = len(
            model.history["loss"]
        )

        parameter_count = sum(
            parameter.numel()
            for parameter
            in model.network.parameters()
            if parameter.requires_grad
        )

        fold_epochs.append(completed_epochs)
        fold_parameter_counts.append(
            parameter_count
        )

        del (
            model,
            preprocessor,
            X_train_processed,
            X_valid_processed
        )

        gc.collect()

    trial.set_user_attr(
        "completed_epochs",
        fold_epochs
    )

    trial.set_user_attr(
        "parameter_count",
        int(np.median(fold_parameter_counts))
    )

    return finalize_trial_metrics(
        trial,
        fold_metrics
    )


# ============================================================
# 6. FT-TRANSFORMER
# ============================================================

def objective_fttransformer(trial):

    d_token = trial.suggest_categorical(
        "module_d_token",
        [32, 64, 128]
    )

    n_heads = trial.suggest_categorical(
        "module_n_heads",
        [4, 8]
    )

    if d_token % n_heads != 0:
        raise optuna.TrialPruned(
            "d_token must be divisible by n_heads"
        )

    params = {
        "module_d_token": d_token,
        "module_d_ffn_factor":
            trial.suggest_float(
                "module_d_ffn_factor",
                1.0,
                2.5
            ),
        "module_n_layers":
            trial.suggest_int(
                "module_n_layers",
                1,
                4
            ),
        "module_n_heads": n_heads,
        "module_attention_dropout":
            trial.suggest_float(
                "module_attention_dropout",
                0.0,
                0.30
            ),
        "module_ffn_dropout":
            trial.suggest_float(
                "module_ffn_dropout",
                0.0,
                0.30
            ),
        "module_residual_dropout":
            trial.suggest_float(
                "module_residual_dropout",
                0.0,
                0.20
            ),
        "lr": trial.suggest_float(
            "lr",
            1e-4,
            3e-3,
            log=True
        ),
        "optimizer_weight_decay":
            trial.suggest_float(
                "optimizer_weight_decay",
                1e-6,
                1e-3,
                log=True
            ),
        "batch_size":
            trial.suggest_categorical(
                "batch_size",
                [128, 256, 512]
            )
    }

    fold_metrics = []

    for fold_id, (train_pos, valid_pos) in enumerate(
        DEEP_HPO_SPLITS,
        start=1
    ):
        fold_seed = (
            GLOBAL_SEED
            + trial.number * 100
            + fold_id
        )

        fold_temp = (
            TEMP_DIR
            / f"FTT_trial_{trial.number}"
            / f"fold_{fold_id}"
        )

        fold_temp.mkdir(
            parents=True,
            exist_ok=True
        )

        try:
            model = FTT_D_Classifier(
                **params,
                max_epochs=60,
                es_patience=8,
                lr_scheduler=True,
                lr_patience=4,
                use_checkpoints=False,
                device="cpu",
                random_state=fold_seed,
                n_cv=1,
                n_refit=0,
                n_repeats=1,
                val_fraction=0.20,
                n_threads=N_THREADS,
                tmp_folder=fold_temp,
                verbose=0,
                verbosity=0
            )

            X_train_fold = (
                X_dev.iloc[train_pos]
                .copy()
                .reset_index(drop=True)
            )

            X_valid_fold = (
                X_dev.iloc[valid_pos]
                .copy()
                .reset_index(drop=True)
            )

            y_train_fold = (
                y_dev.iloc[train_pos]
                .astype(int)
                .to_numpy()
            )

            y_valid_fold = (
                y_dev.iloc[valid_pos]
                .astype(int)
                .to_numpy()
            )

            model.fit(
                X_train_fold,
                y_train_fold,
                X_val=X_valid_fold,
                y_val=y_valid_fold,
                cat_col_names=CATEGORICAL_FEATURES,
                time_to_fit_in_seconds=(
                    PYTABKIT_FOLD_TIME_LIMIT
                )
            )

            valid_prob = (
                model.predict_proba(
                    X_valid_fold
                )[:, 1]
            )

            fold_metrics.append(
                summarize_fold_predictions(
                    y_valid_fold,
                    valid_prob
                )
            )

            del model
            gc.collect()

        except Exception as exc:
            trial.set_user_attr(
                f"fold_{fold_id}_error",
                repr(exc)
            )
            raise optuna.TrialPruned(
                repr(exc)
            )

        finally:
            shutil.rmtree(
                fold_temp,
                ignore_errors=True
            )

    return finalize_trial_metrics(
        trial,
        fold_metrics
    )


# ============================================================
# 7. REALTABR
# ============================================================

def objective_realtabr(trial):

    params = {
        "d_main": trial.suggest_categorical(
            "d_main",
            [64, 128, 256]
        ),
        "d_multiplier":
            trial.suggest_categorical(
                "d_multiplier",
                [2, 4]
            ),
        "encoder_n_blocks":
            trial.suggest_int(
                "encoder_n_blocks",
                1,
                3
            ),
        "predictor_n_blocks":
            trial.suggest_int(
                "predictor_n_blocks",
                1,
                3
            ),
        "context_size":
            trial.suggest_categorical(
                "context_size",
                [16, 32, 64]
            ),
        "context_dropout":
            trial.suggest_float(
                "context_dropout",
                0.0,
                0.30
            ),
        "dropout0":
            trial.suggest_float(
                "dropout0",
                0.0,
                0.30
            ),
        "dropout1":
            trial.suggest_float(
                "dropout1",
                0.0,
                0.30
            ),
        "batch_size":
            trial.suggest_categorical(
                "batch_size",
                [128, 256, 512]
            )
    }

    fold_metrics = []

    for fold_id, (train_pos, valid_pos) in enumerate(
        DEEP_HPO_SPLITS,
        start=1
    ):
        fold_seed = (
            GLOBAL_SEED
            + trial.number * 100
            + fold_id
        )

        fold_temp = (
            TEMP_DIR
            / f"RealTabR_trial_{trial.number}"
            / f"fold_{fold_id}"
        )

        fold_temp.mkdir(
            parents=True,
            exist_ok=True
        )

        try:
            model = RealTabR_D_Classifier(
                **params,
                n_epochs=40,
                eval_batch_size=512,
                candidate_encoding_batch_size=512,
                patience=6,
                memory_efficient=True,
                device="cpu",
                random_state=fold_seed,
                n_cv=1,
                n_refit=0,
                n_repeats=1,
                val_fraction=0.20,
                n_threads=N_THREADS,
                tmp_folder=fold_temp,
                verbosity=0
            )

            X_train_fold = (
                X_dev.iloc[train_pos]
                .copy()
                .reset_index(drop=True)
            )

            X_valid_fold = (
                X_dev.iloc[valid_pos]
                .copy()
                .reset_index(drop=True)
            )

            y_train_fold = (
                y_dev.iloc[train_pos]
                .astype(int)
                .to_numpy()
            )

            y_valid_fold = (
                y_dev.iloc[valid_pos]
                .astype(int)
                .to_numpy()
            )

            model.fit(
                X_train_fold,
                y_train_fold,
                X_val=X_valid_fold,
                y_val=y_valid_fold,
                cat_col_names=CATEGORICAL_FEATURES,
                time_to_fit_in_seconds=(
                    PYTABKIT_FOLD_TIME_LIMIT
                )
            )

            valid_prob = (
                model.predict_proba(
                    X_valid_fold
                )[:, 1]
            )

            fold_metrics.append(
                summarize_fold_predictions(
                    y_valid_fold,
                    valid_prob
                )
            )

            del model
            gc.collect()

        except Exception as exc:
            trial.set_user_attr(
                f"fold_{fold_id}_error",
                repr(exc)
            )
            raise optuna.TrialPruned(
                repr(exc)
            )

        finally:
            shutil.rmtree(
                fold_temp,
                ignore_errors=True
            )

    return finalize_trial_metrics(
        trial,
        fold_metrics
    )


# ------------------------------------------------------------
# 8. Run persistent Optuna studies
# ------------------------------------------------------------

OBJECTIVES = {
    "MLP": objective_mlp,
    "TabNet": objective_tabnet,
    "FTTransformer": objective_fttransformer,
    "RealTabR": objective_realtabr
}

# Change this list to resume or run selected models.
MODELS_TO_RUN = [
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

study_records = []
deep_studies = {}

overall_start = time.perf_counter()

for model_name in MODELS_TO_RUN:

    print("\n" + "=" * 110)
    print(f"DEEP HPO: {model_name}")
    print("=" * 110)

    sampler = optuna.samplers.TPESampler(
        seed=GLOBAL_SEED
    )

    study = optuna.create_study(
        study_name=f"deep_{model_name}",
        direction="maximize",
        sampler=sampler,
        storage=STORAGE_URL,
        load_if_exists=True
    )

    deep_studies[model_name] = study

    trials_already_present = len(
        study.trials
    )

    remaining_trials = max(
        0,
        N_TRIALS - trials_already_present
    )

    print(
        "Trials already stored:",
        trials_already_present
    )

    print(
        "Trials remaining:",
        remaining_trials
    )

    model_start = time.perf_counter()

    if remaining_trials > 0:
        study.optimize(
            OBJECTIVES[model_name],
            n_trials=remaining_trials,
            n_jobs=1,
            show_progress_bar=True,
            gc_after_trial=True
        )

    model_elapsed = (
        time.perf_counter() - model_start
    )

    complete_trials = [
        trial
        for trial in study.trials
        if trial.state
        == optuna.trial.TrialState.COMPLETE
    ]

    pruned_trials = [
        trial
        for trial in study.trials
        if trial.state
        == optuna.trial.TrialState.PRUNED
    ]

    failed_trials = [
        trial
        for trial in study.trials
        if trial.state
        == optuna.trial.TrialState.FAIL
    ]

    if not complete_trials:
        print(
            f"No completed trials for {model_name}."
        )
        continue

    best_trial = study.best_trial

    record = {
        "Model": model_name,
        "Best_CV_ROC_AUC":
            best_trial.value,
        "Best_CV_SD":
            best_trial.user_attrs.get(
                "roc_auc_std",
                np.nan
            ),
        "Best_CV_PR_AUC":
            best_trial.user_attrs.get(
                "mean_pr_auc",
                np.nan
            ),
        "Best_CV_Brier":
            best_trial.user_attrs.get(
                "mean_brier",
                np.nan
            ),
        "Best_Trial_Number":
            best_trial.number,
        "Completed_Trials":
            len(complete_trials),
        "Pruned_Trials":
            len(pruned_trials),
        "Failed_Trials":
            len(failed_trials),
        "Session_Time_Seconds":
            model_elapsed,
        "Best_Params":
            best_trial.params
    }

    study_records.append(record)

    print(
        f"\nBest ROC-AUC: "
        f"{record['Best_CV_ROC_AUC']:.6f}"
    )

    print(
        f"Fold SD: "
        f"{record['Best_CV_SD']:.6f}"
    )

    print(
        f"Mean PR-AUC: "
        f"{record['Best_CV_PR_AUC']:.6f}"
    )

    print(
        f"Mean Brier: "
        f"{record['Best_CV_Brier']:.6f}"
    )

    print("Best parameters:")
    print(
        json.dumps(
            best_trial.params,
            indent=4,
            default=str
        )
    )

    study.trials_dataframe().to_csv(
        OUTPUT_DIR
        / f"{model_name}_optuna_trials.csv",
        index=False
    )

    with open(
        OUTPUT_DIR
        / f"{model_name}_best_params.json",
        "w"
    ) as f:
        json.dump(
            {
                "best_value":
                    best_trial.value,
                "best_params":
                    best_trial.params,
                "best_trial_number":
                    best_trial.number,
                "user_attrs":
                    best_trial.user_attrs,
                "completed_trials":
                    len(complete_trials),
                "pruned_trials":
                    len(pruned_trials),
                "failed_trials":
                    len(failed_trials)
            },
            f,
            indent=4,
            default=str
        )


# ------------------------------------------------------------
# 9. Consolidated summary
# ------------------------------------------------------------

deep_hpo_summary = pd.DataFrame(
    study_records
)

if not deep_hpo_summary.empty:

    deep_hpo_summary = (
        deep_hpo_summary
        .sort_values(
            "Best_CV_ROC_AUC",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print("\n" + "=" * 125)
    print("DEEP MODEL HPO SUMMARY")
    print("=" * 125)

    display_columns = [
        "Model",
        "Best_CV_ROC_AUC",
        "Best_CV_SD",
        "Best_CV_PR_AUC",
        "Best_CV_Brier",
        "Best_Trial_Number",
        "Completed_Trials",
        "Pruned_Trials",
        "Failed_Trials",
        "Session_Time_Seconds"
    ]

    print(
        deep_hpo_summary[
            display_columns
        ]
        .round(6)
        .to_string(index=False)
    )

    deep_hpo_summary.to_pickle(
        OUTPUT_DIR
        / "deep_hpo_summary.pkl"
    )

    deep_hpo_summary.drop(
        columns=["Best_Params"]
    ).to_csv(
        OUTPUT_DIR
        / "deep_hpo_summary.csv",
        index=False
    )

    with open(
        OUTPUT_DIR
        / "all_deep_best_params.json",
        "w"
    ) as f:
        json.dump(
            {
                row["Model"]:
                    row["Best_Params"]
                for row
                in study_records
            },
            f,
            indent=4,
            default=str
        )

total_elapsed = (
    time.perf_counter() - overall_start
)

print(
    f"\nTotal current-session HPO time: "
    f"{total_elapsed / 60:.2f} minutes"
)

print("\nSaved directory:")
print(OUTPUT_DIR)

print("\nPersistent Optuna database:")
print(DB_PATH)

In [ ]:
# ============================================================
# NESTED EARLY-STOPPING REPEATED CV
# Deep models: 5 folds × 3 repeats
# Development set only, CPU only
# Outer validation folds remain untouched
# ============================================================

import copy
import gc
import json
import random
import shutil
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    train_test_split
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    log_loss,
    confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import (
    DataLoader,
    TensorDataset
)

from pytorch_tabnet.tab_model import TabNetClassifier
from pytabkit import (
    FTT_D_Classifier,
    RealTabR_D_Classifier
)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

GLOBAL_SEED = 42
N_SPLITS = 5
N_REPEATS = 3
INNER_VALIDATION_SIZE = 0.15
N_THREADS = 4

OUTPUT_DIR = Path(
    "/kaggle/working/deep_repeated_cv"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEMP_DIR = OUTPUT_DIR / "temporary_files"
TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(N_THREADS)

try:
    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )
except Exception:
    pass


# ------------------------------------------------------------
# 2. Load locked best parameters
# ------------------------------------------------------------

with open(
    "/kaggle/working/deep_hpo/"
    "all_deep_best_params.json",
    "r"
) as f:
    best_params = json.load(f)

print("=" * 100)
print("LOCKED DEEP-MODEL PARAMETERS")
print("=" * 100)

for model_name, params in best_params.items():
    print(f"\n{model_name}")
    print(
        json.dumps(
            params,
            indent=4
        )
    )


# ------------------------------------------------------------
# 3. Outer repeated CV
# Same outer splits as classical repeated CV
# ------------------------------------------------------------

outer_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=GLOBAL_SEED
)

OUTER_SPLITS = list(
    outer_cv.split(X_dev, y_dev)
)

print("\n" + "=" * 100)
print("REPEATED CV CONFIGURATION")
print("=" * 100)

print("Outer folds:", N_SPLITS)
print("Repeats:", N_REPEATS)
print("Total outer evaluations:", len(OUTER_SPLITS))
print("Inner early-stopping fraction:", INNER_VALIDATION_SIZE)
print("Device: CPU")


# ------------------------------------------------------------
# 4. Metric utilities
# ------------------------------------------------------------

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def expected_calibration_error(
    y_true,
    y_prob,
    n_bins=10
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    bin_ids = np.digitize(
        y_prob,
        edges[1:-1],
        right=True
    )

    ece = 0.0

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id

        if mask.sum() == 0:
            continue

        confidence = y_prob[mask].mean()
        observed = y_true[mask].mean()

        ece += (
            mask.mean()
            * abs(confidence - observed)
        )

    return float(ece)


def probability_metrics(
    y_true,
    y_prob
):
    y_prob = np.clip(
        np.asarray(y_prob),
        1e-7,
        1 - 1e-7
    )

    return {
        "ROC_AUC": roc_auc_score(
            y_true,
            y_prob
        ),
        "PR_AUC": average_precision_score(
            y_true,
            y_prob
        ),
        "Brier_Score": brier_score_loss(
            y_true,
            y_prob
        ),
        "Log_Loss": log_loss(
            y_true,
            y_prob,
            labels=[0, 1]
        ),
        "ECE_10": expected_calibration_error(
            y_true,
            y_prob,
            n_bins=10
        )
    }


def threshold_metrics(
    y_true,
    y_prob,
    threshold=0.50
):
    y_pred = (
        np.asarray(y_prob) >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),
        "MCC": matthews_corrcoef(
            y_true,
            y_pred
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


# ------------------------------------------------------------
# 5. Custom MLP architecture
# ------------------------------------------------------------

class TunableMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout,
        use_batch_norm,
        activation_name
    ):
        super().__init__()

        activation_classes = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "silu": nn.SiLU
        }

        activation_class = (
            activation_classes[
                activation_name
            ]
        )

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:

            layers.append(
                nn.Linear(
                    previous_dim,
                    hidden_dim
                )
            )

            if use_batch_norm:
                layers.append(
                    nn.BatchNorm1d(
                        hidden_dim
                    )
                )

            layers.append(
                activation_class()
            )

            if dropout > 0:
                layers.append(
                    nn.Dropout(dropout)
                )

            previous_dim = hidden_dim

        layers.append(
            nn.Linear(previous_dim, 1)
        )

        self.network = nn.Sequential(
            *layers
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


# ------------------------------------------------------------
# 6. Prepare results containers
# ------------------------------------------------------------

MODEL_NAMES = [
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

fold_records = []
oof_records = []
error_records = []

overall_start = time.perf_counter()


# ------------------------------------------------------------
# 7. Repeated nested evaluation
# ------------------------------------------------------------

for model_name in MODEL_NAMES:

    print("\n" + "=" * 105)
    print(f"RUNNING: {model_name}")
    print("=" * 105)

    model_start = time.perf_counter()

    for split_id, (
        outer_train_pos,
        outer_valid_pos
    ) in enumerate(OUTER_SPLITS):

        repeat_id = (
            split_id // N_SPLITS + 1
        )

        fold_id = (
            split_id % N_SPLITS + 1
        )

        split_seed = (
            GLOBAL_SEED + split_id
        )

        seed_everything(split_seed)

        # Outer validation remains untouched.
        X_outer_train = (
            X_dev.iloc[outer_train_pos]
            .copy()
            .reset_index(drop=True)
        )

        y_outer_train = (
            y_dev.iloc[outer_train_pos]
            .astype(int)
            .reset_index(drop=True)
        )

        X_outer_valid = (
            X_dev.iloc[outer_valid_pos]
            .copy()
            .reset_index(drop=True)
        )

        y_outer_valid = (
            y_dev.iloc[outer_valid_pos]
            .astype(int)
            .to_numpy()
        )

        outer_original_indices = (
            X_dev.iloc[outer_valid_pos]
            .index
            .to_numpy()
        )

        # Inner split only for early stopping.
        inner_positions = np.arange(
            len(X_outer_train)
        )

        (
            inner_train_pos,
            inner_stop_pos
        ) = train_test_split(
            inner_positions,
            test_size=INNER_VALIDATION_SIZE,
            stratify=y_outer_train,
            random_state=split_seed
        )

        X_inner_train = (
            X_outer_train.iloc[
                inner_train_pos
            ]
            .copy()
            .reset_index(drop=True)
        )

        y_inner_train = (
            y_outer_train.iloc[
                inner_train_pos
            ]
            .astype(int)
            .to_numpy()
        )

        X_inner_stop = (
            X_outer_train.iloc[
                inner_stop_pos
            ]
            .copy()
            .reset_index(drop=True)
        )

        y_inner_stop = (
            y_outer_train.iloc[
                inner_stop_pos
            ]
            .astype(int)
            .to_numpy()
        )

        fit_start = time.perf_counter()
        completed_epochs = np.nan
        parameter_count = np.nan

        fold_temp = (
            TEMP_DIR
            / model_name
            / f"repeat_{repeat_id}"
            / f"fold_{fold_id}"
        )

        fold_temp.mkdir(
            parents=True,
            exist_ok=True
        )

        try:

            # =================================================
            # MLP
            # =================================================

            if model_name == "MLP":

                params = best_params["MLP"]

                hidden_dims = tuple(
                    int(value)
                    for value in params[
                        "hidden_structure"
                    ].split("-")
                )

                preprocessor = ColumnTransformer(
                    transformers=[
                        (
                            "numeric",
                            StandardScaler(),
                            NUMERIC_FEATURES
                        ),
                        (
                            "categorical",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False
                            ),
                            CATEGORICAL_FEATURES
                        )
                    ],
                    remainder="drop"
                )

                X_train_processed = (
                    preprocessor
                    .fit_transform(
                        X_inner_train
                    )
                    .astype(np.float32)
                )

                X_stop_processed = (
                    preprocessor
                    .transform(
                        X_inner_stop
                    )
                    .astype(np.float32)
                )

                X_valid_processed = (
                    preprocessor
                    .transform(
                        X_outer_valid
                    )
                    .astype(np.float32)
                )

                train_dataset = TensorDataset(
                    torch.from_numpy(
                        X_train_processed
                    ),
                    torch.from_numpy(
                        y_inner_train
                        .astype(np.float32)
                    )
                )

                generator = torch.Generator()
                generator.manual_seed(
                    split_seed
                )

                train_loader = DataLoader(
                    train_dataset,
                    batch_size=params[
                        "batch_size"
                    ],
                    shuffle=True,
                    num_workers=0,
                    generator=generator,
                    drop_last=False
                )

                model = TunableMLP(
                    input_dim=
                        X_train_processed.shape[1],
                    hidden_dims=hidden_dims,
                    dropout=params["dropout"],
                    use_batch_norm=params[
                        "use_batch_norm"
                    ],
                    activation_name=params[
                        "activation"
                    ]
                )

                parameter_count = sum(
                    parameter.numel()
                    for parameter
                    in model.parameters()
                    if parameter.requires_grad
                )

                optimizer = torch.optim.AdamW(
                    model.parameters(),
                    lr=params[
                        "learning_rate"
                    ],
                    weight_decay=params[
                        "weight_decay"
                    ]
                )

                criterion = (
                    nn.BCEWithLogitsLoss()
                )

                X_stop_tensor = (
                    torch.from_numpy(
                        X_stop_processed
                    )
                )

                max_epochs = 80
                patience = 10

                best_inner_auc = -np.inf
                best_state = None
                no_improvement = 0

                for epoch in range(
                    1,
                    max_epochs + 1
                ):

                    model.train()

                    for (
                        batch_X,
                        batch_y
                    ) in train_loader:

                        optimizer.zero_grad()

                        logits = model(batch_X)

                        loss = criterion(
                            logits,
                            batch_y
                        )

                        loss.backward()
                        optimizer.step()

                    model.eval()

                    with torch.no_grad():

                        stop_prob = (
                            torch.sigmoid(
                                model(
                                    X_stop_tensor
                                )
                            )
                            .cpu()
                            .numpy()
                        )

                    inner_auc = roc_auc_score(
                        y_inner_stop,
                        stop_prob
                    )

                    completed_epochs = epoch

                    if (
                        inner_auc
                        > best_inner_auc + 1e-5
                    ):
                        best_inner_auc = (
                            inner_auc
                        )

                        best_state = copy.deepcopy(
                            model.state_dict()
                        )

                        no_improvement = 0

                    else:
                        no_improvement += 1

                    if (
                        no_improvement
                        >= patience
                    ):
                        break

                model.load_state_dict(
                    best_state
                )

                model.eval()

                X_outer_tensor = (
                    torch.from_numpy(
                        X_valid_processed
                    )
                )

                fit_time = (
                    time.perf_counter()
                    - fit_start
                )

                predict_start = (
                    time.perf_counter()
                )

                with torch.no_grad():

                    outer_prob = (
                        torch.sigmoid(
                            model(
                                X_outer_tensor
                            )
                        )
                        .cpu()
                        .numpy()
                    )

                predict_time = (
                    time.perf_counter()
                    - predict_start
                )


            # =================================================
            # TabNet
            # =================================================

            elif model_name == "TabNet":

                params = best_params[
                    "TabNet"
                ]

                preprocessor = ColumnTransformer(
                    transformers=[
                        (
                            "numeric",
                            StandardScaler(),
                            NUMERIC_FEATURES
                        ),
                        (
                            "categorical",
                            OrdinalEncoder(
                                handle_unknown=
                                    "use_encoded_value",
                                unknown_value=-1
                            ),
                            CATEGORICAL_FEATURES
                        )
                    ],
                    remainder="drop"
                )

                X_train_processed = (
                    preprocessor
                    .fit_transform(
                        X_inner_train
                    )
                    .astype(np.float32)
                )

                X_stop_processed = (
                    preprocessor
                    .transform(
                        X_inner_stop
                    )
                    .astype(np.float32)
                )

                X_valid_processed = (
                    preprocessor
                    .transform(
                        X_outer_valid
                    )
                    .astype(np.float32)
                )

                numeric_count = len(
                    NUMERIC_FEATURES
                )

                cat_idxs = list(
                    range(
                        numeric_count,
                        numeric_count
                        + len(
                            CATEGORICAL_FEATURES
                        )
                    )
                )

                encoder = (
                    preprocessor
                    .named_transformers_[
                        "categorical"
                    ]
                )

                X_train_processed[
                    :, cat_idxs
                ] += 1

                X_stop_processed[
                    :, cat_idxs
                ] += 1

                X_valid_processed[
                    :, cat_idxs
                ] += 1

                cat_dims = [
                    len(categories) + 1
                    for categories
                    in encoder.categories_
                ]

                model = TabNetClassifier(
                    n_d=params["n_d"],
                    n_a=params["n_d"],
                    n_steps=params[
                        "n_steps"
                    ],
                    gamma=params["gamma"],
                    cat_idxs=cat_idxs,
                    cat_dims=cat_dims,
                    cat_emb_dim=params[
                        "cat_emb_dim"
                    ],
                    n_independent=params[
                        "n_independent"
                    ],
                    n_shared=params[
                        "n_shared"
                    ],
                    lambda_sparse=params[
                        "lambda_sparse"
                    ],
                    seed=split_seed,
                    optimizer_fn=
                        torch.optim.Adam,
                    optimizer_params={
                        "lr": params[
                            "learning_rate"
                        ]
                    },
                    mask_type="sparsemax",
                    device_name="cpu",
                    verbose=0
                )

                model.fit(
                    X_train=
                        X_train_processed,
                    y_train=y_inner_train,
                    eval_set=[
                        (
                            X_stop_processed,
                            y_inner_stop
                        )
                    ],
                    eval_name=[
                        "early_stopping"
                    ],
                    eval_metric=["auc"],
                    max_epochs=100,
                    patience=12,
                    batch_size=params[
                        "batch_size"
                    ],
                    virtual_batch_size=max(
                        16,
                        params[
                            "batch_size"
                        ] // 4
                    ),
                    num_workers=0,
                    drop_last=False,
                    pin_memory=False,
                    compute_importance=False
                )

                completed_epochs = len(
                    model.history["loss"]
                )

                parameter_count = sum(
                    parameter.numel()
                    for parameter
                    in model.network.parameters()
                    if parameter.requires_grad
                )

                fit_time = (
                    time.perf_counter()
                    - fit_start
                )

                predict_start = (
                    time.perf_counter()
                )

                outer_prob = (
                    model.predict_proba(
                        X_valid_processed
                    )[:, 1]
                )

                predict_time = (
                    time.perf_counter()
                    - predict_start
                )


            # =================================================
            # FT-Transformer
            # =================================================

            elif model_name == "FTTransformer":

                params = best_params[
                    "FTTransformer"
                ]

                model = FTT_D_Classifier(
                    **params,
                    max_epochs=60,
                    es_patience=8,
                    lr_scheduler=True,
                    lr_patience=4,
                    use_checkpoints=False,
                    device="cpu",
                    random_state=split_seed,
                    n_cv=1,
                    n_refit=0,
                    n_repeats=1,
                    val_fraction=0.20,
                    n_threads=N_THREADS,
                    tmp_folder=fold_temp,
                    verbose=0,
                    verbosity=0
                )

                model.fit(
                    X_inner_train,
                    y_inner_train,
                    X_val=X_inner_stop,
                    y_val=y_inner_stop,
                    cat_col_names=
                        CATEGORICAL_FEATURES,
                    time_to_fit_in_seconds=600
                )

                fit_time = (
                    time.perf_counter()
                    - fit_start
                )

                predict_start = (
                    time.perf_counter()
                )

                outer_prob = (
                    model.predict_proba(
                        X_outer_valid
                    )[:, 1]
                )

                predict_time = (
                    time.perf_counter()
                    - predict_start
                )


            # =================================================
            # RealTabR
            # =================================================

            elif model_name == "RealTabR":

                params = best_params[
                    "RealTabR"
                ]

                model = RealTabR_D_Classifier(
                    **params,
                    n_epochs=40,
                    eval_batch_size=512,
                    candidate_encoding_batch_size=512,
                    patience=6,
                    memory_efficient=True,
                    device="cpu",
                    random_state=split_seed,
                    n_cv=1,
                    n_refit=0,
                    n_repeats=1,
                    val_fraction=0.20,
                    n_threads=N_THREADS,
                    tmp_folder=fold_temp,
                    verbosity=0
                )

                model.fit(
                    X_inner_train,
                    y_inner_train,
                    X_val=X_inner_stop,
                    y_val=y_inner_stop,
                    cat_col_names=
                        CATEGORICAL_FEATURES,
                    time_to_fit_in_seconds=600
                )

                fit_time = (
                    time.perf_counter()
                    - fit_start
                )

                predict_start = (
                    time.perf_counter()
                )

                outer_prob = (
                    model.predict_proba(
                        X_outer_valid
                    )[:, 1]
                )

                predict_time = (
                    time.perf_counter()
                    - predict_start
                )

            else:
                raise ValueError(
                    f"Unknown model: "
                    f"{model_name}"
                )

            outer_prob = np.asarray(
                outer_prob,
                dtype=float
            ).reshape(-1)

            if (
                len(outer_prob)
                != len(y_outer_valid)
            ):
                raise ValueError(
                    "Prediction length mismatch."
                )

            if not np.isfinite(
                outer_prob
            ).all():
                raise ValueError(
                    "Non-finite probabilities."
                )

            outer_prob = np.clip(
                outer_prob,
                1e-7,
                1 - 1e-7
            )

            metrics = probability_metrics(
                y_outer_valid,
                outer_prob
            )

            fold_records.append({
                "Model": model_name,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Split_Seed": split_seed,
                "Outer_Train_N":
                    len(outer_train_pos),
                "Inner_Train_N":
                    len(inner_train_pos),
                "Inner_Stop_N":
                    len(inner_stop_pos),
                "Outer_Validation_N":
                    len(outer_valid_pos),
                "Outer_Prevalence":
                    y_outer_valid.mean(),
                **metrics,
                "Fit_Time_Seconds":
                    fit_time,
                "Predict_Time_Seconds":
                    predict_time,
                "Completed_Epochs":
                    completed_epochs,
                "Parameter_Count":
                    parameter_count
            })

            for (
                development_position,
                original_index,
                y_value,
                probability
            ) in zip(
                outer_valid_pos,
                outer_original_indices,
                y_outer_valid,
                outer_prob
            ):
                oof_records.append({
                    "Model": model_name,
                    "Repeat": repeat_id,
                    "Fold": fold_id,
                    "Development_Position":
                        int(
                            development_position
                        ),
                    "Original_Index":
                        int(original_index),
                    "y_true":
                        int(y_value),
                    "y_prob":
                        float(probability)
                })

            print(
                f"Repeat {repeat_id}, "
                f"Fold {fold_id}: "
                f"ROC-AUC="
                f"{metrics['ROC_AUC']:.4f}, "
                f"PR-AUC="
                f"{metrics['PR_AUC']:.4f}, "
                f"fit={fit_time:.1f}s"
            )

        except Exception as exc:

            error_records.append({
                "Model": model_name,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Error": repr(exc)
            })

            print(
                f"FAILED: Repeat {repeat_id}, "
                f"Fold {fold_id}: {repr(exc)}"
            )

            raise

        finally:

            shutil.rmtree(
                fold_temp,
                ignore_errors=True
            )

            local_names = [
                "model",
                "optimizer",
                "preprocessor",
                "train_dataset",
                "train_loader",
                "X_train_processed",
                "X_stop_processed",
                "X_valid_processed"
            ]

            for local_name in local_names:
                if local_name in locals():
                    del locals()[local_name]

            gc.collect()

    model_elapsed = (
        time.perf_counter()
        - model_start
    )

    print(
        f"\n{model_name} completed in "
        f"{model_elapsed / 60:.2f} minutes"
    )


# ------------------------------------------------------------
# 8. Fold-level summary
# ------------------------------------------------------------

fold_results = pd.DataFrame(
    fold_records
)

oof_long = pd.DataFrame(
    oof_records
)

summary_metrics = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "ECE_10",
    "Fit_Time_Seconds",
    "Predict_Time_Seconds"
]

fold_summary = (
    fold_results
    .groupby("Model")[
        summary_metrics
    ]
    .agg(["mean", "std"])
)

fold_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat
    in fold_summary.columns
]

fold_summary = (
    fold_summary
    .reset_index()
    .sort_values(
        "ROC_AUC_mean",
        ascending=False
    )
)

print("\n" + "=" * 125)
print("DEEP REPEATED-CV FOLD SUMMARY")
print("=" * 125)

display_columns = [
    "Model",
    "ROC_AUC_mean",
    "ROC_AUC_std",
    "PR_AUC_mean",
    "PR_AUC_std",
    "Brier_Score_mean",
    "Brier_Score_std",
    "ECE_10_mean",
    "ECE_10_std",
    "Fit_Time_Seconds_mean",
    "Predict_Time_Seconds_mean"
]

print(
    fold_summary[
        display_columns
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. Average three OOF predictions per observation
# ------------------------------------------------------------

mean_oof = (
    oof_long
    .groupby(
        [
            "Model",
            "Development_Position",
            "Original_Index",
            "y_true"
        ],
        as_index=False
    )
    .agg(
        Mean_OOF_Probability=(
            "y_prob",
            "mean"
        ),
        SD_OOF_Probability=(
            "y_prob",
            "std"
        ),
        Prediction_Count=(
            "y_prob",
            "count"
        )
    )
)

assert (
    mean_oof["Prediction_Count"]
    == N_REPEATS
).all()


# ------------------------------------------------------------
# 10. Aggregate OOF metrics
# ------------------------------------------------------------

aggregate_records = []

for model_name in MODEL_NAMES:

    model_oof = (
        mean_oof[
            mean_oof["Model"]
            == model_name
        ]
        .sort_values(
            "Development_Position"
        )
    )

    y_true_model = (
        model_oof["y_true"]
        .to_numpy()
    )

    y_prob_model = (
        model_oof[
            "Mean_OOF_Probability"
        ]
        .to_numpy()
    )

    aggregate_records.append({
        "Model": model_name,
        **probability_metrics(
            y_true_model,
            y_prob_model
        ),
        **threshold_metrics(
            y_true_model,
            y_prob_model,
            threshold=0.50
        ),
        "Mean_Probability_SD_Across_Repeats":
            model_oof[
                "SD_OOF_Probability"
            ].mean()
    })

aggregate_results = (
    pd.DataFrame(
        aggregate_records
    )
    .sort_values(
        "ROC_AUC",
        ascending=False
    )
)

print("\n" + "=" * 140)
print("DEEP AGGREGATED MEAN OOF RESULTS")
print("=" * 140)

aggregate_columns = [
    "Model",
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "ECE_10",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "TN",
    "FP",
    "FN",
    "TP",
    "Mean_Probability_SD_Across_Repeats"
]

print(
    aggregate_results[
        aggregate_columns
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 11. Epoch and parameter summary
# ------------------------------------------------------------

training_summary = (
    fold_results
    .groupby("Model")
    .agg(
        Mean_Fit_Time=(
            "Fit_Time_Seconds",
            "mean"
        ),
        SD_Fit_Time=(
            "Fit_Time_Seconds",
            "std"
        ),
        Mean_Predict_Time=(
            "Predict_Time_Seconds",
            "mean"
        ),
        Mean_Completed_Epochs=(
            "Completed_Epochs",
            "mean"
        ),
        Median_Parameter_Count=(
            "Parameter_Count",
            "median"
        )
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("DEEP TRAINING SUMMARY")
print("=" * 110)

print(
    training_summary
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 12. Save artifacts
# ------------------------------------------------------------

fold_results.to_csv(
    OUTPUT_DIR
    / "deep_fold_level_metrics.csv",
    index=False
)

fold_summary.to_csv(
    OUTPUT_DIR
    / "deep_fold_summary.csv",
    index=False
)

oof_long.to_csv(
    OUTPUT_DIR
    / "deep_repeated_oof_long.csv",
    index=False
)

mean_oof.to_csv(
    OUTPUT_DIR
    / "deep_mean_oof_predictions.csv",
    index=False
)

aggregate_results.to_csv(
    OUTPUT_DIR
    / "deep_aggregate_mean_oof_metrics.csv",
    index=False
)

training_summary.to_csv(
    OUTPUT_DIR
    / "deep_training_summary.csv",
    index=False
)

pd.DataFrame(
    error_records
).to_csv(
    OUTPUT_DIR
    / "deep_repeated_cv_errors.csv",
    index=False
)

total_elapsed = (
    time.perf_counter()
    - overall_start
)

print("\nSaved directory:")
print(OUTPUT_DIR)

print(
    f"\nTotal elapsed time: "
    f"{total_elapsed / 60:.2f} minutes"
)

print("\nErrors:")
print(
    error_records
    if error_records
    else "None"
)

In [ ]:
# ============================================================
# UNIFIED CROSS-VALIDATION BENCHMARK
# Primary result: mean ± SD across repeated CV folds
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT_DIR = Path(
    "/kaggle/working/unified_benchmark"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Load primary fold-level summaries
# ------------------------------------------------------------

classical_fold = pd.read_csv(
    "/kaggle/working/classical_repeated_cv/"
    "fold_level_summary.csv"
)

deep_fold = pd.read_csv(
    "/kaggle/working/deep_repeated_cv/"
    "deep_fold_summary.csv"
)

classical_fold["Model_Family"] = "Classical"
deep_fold["Model_Family"] = "Deep"

primary_cv = pd.concat(
    [
        classical_fold,
        deep_fold
    ],
    ignore_index=True,
    sort=False
)

# ------------------------------------------------------------
# 2. Add ranking and efficiency fields
# ------------------------------------------------------------

primary_cv["ROC_AUC_Rank"] = (
    primary_cv["ROC_AUC_mean"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

primary_cv["PR_AUC_Rank"] = (
    primary_cv["PR_AUC_mean"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

best_auc = primary_cv[
    "ROC_AUC_mean"
].max()

primary_cv["ROC_AUC_Delta_From_Best"] = (
    best_auc
    - primary_cv["ROC_AUC_mean"]
)

xgb_fit_time = primary_cv.loc[
    primary_cv["Model"] == "XGBoost",
    "Fit_Time_Seconds_mean"
].iloc[0]

primary_cv[
    "Fit_Time_Ratio_vs_XGBoost"
] = (
    primary_cv["Fit_Time_Seconds_mean"]
    / xgb_fit_time
)

primary_cv = primary_cv.sort_values(
    [
        "ROC_AUC_mean",
        "PR_AUC_mean"
    ],
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 3. Retention policy
# ------------------------------------------------------------

retention_map = {
    "Dummy": "No-skill reference",
    "LogisticRegression": "Interpretable linear baseline",
    "RandomForest": "Bagging baseline",
    "XGBoost": "Primary classical ablation model",
    "LightGBM": "Strong boosting comparator",
    "CatBoost": "Native-categorical boosting comparator",
    "MLP": "Primary deep ablation model",
    "TabNet": "Original manuscript model",
    "FTTransformer": "Transformer comparator",
    "RealTabR": "Original manuscript model"
}

primary_cv["Experimental_Role"] = (
    primary_cv["Model"]
    .map(retention_map)
)

# ------------------------------------------------------------
# 4. Print principal benchmark
# ------------------------------------------------------------

display_columns = [
    "ROC_AUC_Rank",
    "Model",
    "Model_Family",
    "ROC_AUC_mean",
    "ROC_AUC_std",
    "PR_AUC_mean",
    "PR_AUC_std",
    "Brier_Score_mean",
    "ECE_10_mean",
    "Fit_Time_Seconds_mean",
    "Predict_Time_Seconds_mean",
    "Fit_Time_Ratio_vs_XGBoost",
    "ROC_AUC_Delta_From_Best",
    "Experimental_Role"
]

print("=" * 165)
print("UNIFIED PRIMARY REPEATED-CV BENCHMARK")
print("Primary estimates are means across 15 validation folds.")
print("=" * 165)

print(
    primary_cv[
        display_columns
    ]
    .round(6)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 5. Load averaged OOF diagnostics separately
# These are not the primary single-model CV estimates.
# ------------------------------------------------------------

classical_oof = pd.read_csv(
    "/kaggle/working/classical_repeated_cv/"
    "aggregate_mean_oof_metrics.csv"
)

deep_oof = pd.read_csv(
    "/kaggle/working/deep_repeated_cv/"
    "deep_aggregate_mean_oof_metrics.csv"
)

cross_fitted_ensemble_oof = pd.concat(
    [
        classical_oof,
        deep_oof
    ],
    ignore_index=True,
    sort=False
)

cross_fitted_ensemble_oof = (
    cross_fitted_ensemble_oof
    .sort_values(
        "ROC_AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 120)
print("CROSS-FITTED THREE-REPEAT OOF DIAGNOSTICS")
print("Diagnostic only: each observation's three OOF probabilities were averaged.")
print("=" * 120)

print(
    cross_fitted_ensemble_oof[
        [
            "Model",
            "ROC_AUC",
            "PR_AUC",
            "Brier_Score",
            "ECE_10",
            "MCC",
            "Mean_Probability_SD_Across_Repeats"
        ]
    ]
    .round(6)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Save artifacts
# ------------------------------------------------------------

primary_cv.to_csv(
    OUTPUT_DIR
    / "unified_primary_repeated_cv.csv",
    index=False
)

cross_fitted_ensemble_oof.to_csv(
    OUTPUT_DIR
    / "cross_fitted_ensemble_oof_diagnostics.csv",
    index=False
)

ablation_plan = pd.DataFrame({
    "Ablation": [
        "Imbalance handling",
        "Imbalance handling",
        "MLP architecture"
    ],
    "Model": [
        "XGBoost",
        "MLP",
        "MLP"
    ],
    "Compared_Conditions": [
        "None vs class weighting vs SMOTENC",
        "None vs class weighting vs SMOTENC",
        "Dropout and Batch Normalization combinations"
    ]
})

ablation_plan.to_csv(
    OUTPUT_DIR / "ablation_plan.csv",
    index=False
)

print("\nSaved directory:")
print(OUTPUT_DIR)

In [ ]:
# ============================================================
# IMBALANCE-HANDLING ABLATION
# Models: XGBoost and MLP
# Strategies: None, ClassWeight, SMOTENC
# 5 folds × 3 repeats, development data only
# ============================================================

import copy
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    train_test_split
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from imblearn.over_sampling import SMOTENC
from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

GLOBAL_SEED = 42
N_SPLITS = 5
N_REPEATS = 3
N_THREADS = 4
INNER_VALIDATION_SIZE = 0.15

OUTPUT_DIR = Path(
    "/kaggle/working/imbalance_ablation"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

STRATEGIES = [
    "None",
    "ClassWeight",
    "SMOTENC"
]

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(N_THREADS)


# ------------------------------------------------------------
# 2. Load locked parameters
# ------------------------------------------------------------

with open(
    "/kaggle/working/classical_hpo/"
    "all_best_params.json",
    "r"
) as f:
    classical_params = json.load(f)

with open(
    "/kaggle/working/deep_hpo/"
    "all_deep_best_params.json",
    "r"
) as f:
    deep_params = json.load(f)

XGB_PARAMS = classical_params["XGBoost"]
MLP_PARAMS = deep_params["MLP"]

print("=" * 100)
print("LOCKED PARAMETERS")
print("=" * 100)

print("\nXGBoost:")
print(json.dumps(XGB_PARAMS, indent=4))

print("\nMLP:")
print(json.dumps(MLP_PARAMS, indent=4))


# ------------------------------------------------------------
# 3. Shared outer folds
# ------------------------------------------------------------

outer_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=GLOBAL_SEED
)

OUTER_SPLITS = list(
    outer_cv.split(X_dev, y_dev)
)

print("\nOuter evaluations:", len(OUTER_SPLITS))


# ------------------------------------------------------------
# 4. Utilities
# ------------------------------------------------------------

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def expected_calibration_error(
    y_true,
    y_prob,
    n_bins=10
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    edges = np.linspace(0, 1, n_bins + 1)

    bin_ids = np.digitize(
        y_prob,
        edges[1:-1],
        right=True
    )

    ece = 0.0

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id

        if mask.sum() == 0:
            continue

        ece += (
            mask.mean()
            * abs(
                y_prob[mask].mean()
                - y_true[mask].mean()
            )
        )

    return float(ece)


def evaluate_probabilities(
    y_true,
    y_prob,
    threshold=0.50
):
    y_prob = np.clip(
        np.asarray(y_prob),
        1e-7,
        1 - 1e-7
    )

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "ROC_AUC": roc_auc_score(
            y_true,
            y_prob
        ),
        "PR_AUC": average_precision_score(
            y_true,
            y_prob
        ),
        "Brier_Score": brier_score_loss(
            y_true,
            y_prob
        ),
        "Log_Loss": log_loss(
            y_true,
            y_prob,
            labels=[0, 1]
        ),
        "ECE_10": expected_calibration_error(
            y_true,
            y_prob,
            n_bins=10
        ),
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),
        "MCC": matthews_corrcoef(
            y_true,
            y_pred
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


# ------------------------------------------------------------
# 5. Standard preprocessing
# ------------------------------------------------------------

def fit_standard_preprocessor(
    X_train,
    X_evaluation,
    model_type
):
    if model_type == "MLP":
        numeric_transformer = StandardScaler()
        sparse_output = False
    else:
        numeric_transformer = "passthrough"
        sparse_output = True

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_transformer,
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=sparse_output
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )

    X_train_processed = (
        preprocessor.fit_transform(X_train)
    )

    X_evaluation_processed = (
        preprocessor.transform(X_evaluation)
    )

    if model_type == "MLP":
        X_train_processed = (
            X_train_processed.astype(np.float32)
        )

        X_evaluation_processed = (
            X_evaluation_processed.astype(
                np.float32
            )
        )

    return (
        X_train_processed,
        X_evaluation_processed,
        preprocessor
    )


# ------------------------------------------------------------
# 6. SMOTENC preprocessing
# ------------------------------------------------------------

def fit_smotenc_preprocessing(
    X_train,
    y_train,
    evaluation_sets,
    model_type,
    random_state
):
    ordinal = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    train_numeric = (
        X_train[NUMERIC_FEATURES]
        .to_numpy(dtype=float)
    )

    train_categorical = ordinal.fit_transform(
        X_train[CATEGORICAL_FEATURES]
    )

    # Code 0 is reserved for unseen categories.
    train_categorical += 1

    train_mixed = np.hstack([
        train_numeric,
        train_categorical
    ])

    categorical_indices = list(
        range(
            len(NUMERIC_FEATURES),
            len(NUMERIC_FEATURES)
            + len(CATEGORICAL_FEATURES)
        )
    )

    smote = SMOTENC(
        categorical_features=
            categorical_indices,
        sampling_strategy=1.0,
        random_state=random_state,
        k_neighbors=5
    )

    X_resampled, y_resampled = (
        smote.fit_resample(
            train_mixed,
            np.asarray(y_train)
        )
    )

    numeric_resampled = X_resampled[
        :, :len(NUMERIC_FEATURES)
    ]

    categorical_resampled = X_resampled[
        :, len(NUMERIC_FEATURES):
    ]

    one_hot = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=(
            model_type == "XGBoost"
        )
    )

    one_hot.fit(
        categorical_resampled
    )

    if model_type == "MLP":
        scaler = StandardScaler()
        scaler.fit(numeric_resampled)

        train_final = np.hstack([
            scaler.transform(
                numeric_resampled
            ),
            one_hot.transform(
                categorical_resampled
            )
        ]).astype(np.float32)

    else:
        scaler = None

        train_final = sp.hstack(
            [
                sp.csr_matrix(
                    numeric_resampled
                ),
                one_hot.transform(
                    categorical_resampled
                )
            ],
            format="csr"
        )

    transformed_evaluations = []

    for evaluation_df in evaluation_sets:
        evaluation_numeric = (
            evaluation_df[
                NUMERIC_FEATURES
            ].to_numpy(dtype=float)
        )

        evaluation_categorical = (
            ordinal.transform(
                evaluation_df[
                    CATEGORICAL_FEATURES
                ]
            )
        )

        evaluation_categorical += 1

        if model_type == "MLP":
            evaluation_final = np.hstack([
                scaler.transform(
                    evaluation_numeric
                ),
                one_hot.transform(
                    evaluation_categorical
                )
            ]).astype(np.float32)

        else:
            evaluation_final = sp.hstack(
                [
                    sp.csr_matrix(
                        evaluation_numeric
                    ),
                    one_hot.transform(
                        evaluation_categorical
                    )
                ],
                format="csr"
            )

        transformed_evaluations.append(
            evaluation_final
        )

    return (
        train_final,
        np.asarray(y_resampled),
        transformed_evaluations,
        {
            "ordinal_encoder": ordinal,
            "one_hot_encoder": one_hot,
            "scaler": scaler,
            "smote": smote
        }
    )


# ------------------------------------------------------------
# 7. MLP definition
# ------------------------------------------------------------

class BankingMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout,
        use_batch_norm,
        activation_name
    ):
        super().__init__()

        activations = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "silu": nn.SiLU
        }

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(
                nn.Linear(
                    previous_dim,
                    hidden_dim
                )
            )

            if use_batch_norm:
                layers.append(
                    nn.BatchNorm1d(
                        hidden_dim
                    )
                )

            layers.append(
                activations[
                    activation_name
                ]()
            )

            if dropout > 0:
                layers.append(
                    nn.Dropout(dropout)
                )

            previous_dim = hidden_dim

        layers.append(
            nn.Linear(previous_dim, 1)
        )

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)


# ------------------------------------------------------------
# 8. Result containers
# ------------------------------------------------------------

fold_records = []
oof_records = []

overall_start = time.perf_counter()


# ============================================================
# 9. XGBOOST ABLATION
# ============================================================

print("\n" + "=" * 105)
print("XGBOOST IMBALANCE ABLATION")
print("=" * 105)

for split_id, (
    train_pos,
    valid_pos
) in enumerate(OUTER_SPLITS):

    repeat_id = split_id // N_SPLITS + 1
    fold_id = split_id % N_SPLITS + 1
    split_seed = GLOBAL_SEED + split_id

    X_train_fold = (
        X_dev.iloc[train_pos]
        .copy()
        .reset_index(drop=True)
    )

    X_valid_fold = (
        X_dev.iloc[valid_pos]
        .copy()
        .reset_index(drop=True)
    )

    y_train_fold = (
        y_dev.iloc[train_pos]
        .astype(int)
        .to_numpy()
    )

    y_valid_fold = (
        y_dev.iloc[valid_pos]
        .astype(int)
        .to_numpy()
    )

    original_indices = (
        X_dev.iloc[valid_pos]
        .index.to_numpy()
    )

    for strategy in STRATEGIES:
        seed_everything(split_seed)

        fit_start = time.perf_counter()

        if strategy == "SMOTENC":
            (
                X_train_processed,
                y_train_used,
                transformed_sets,
                preprocessing_objects
            ) = fit_smotenc_preprocessing(
                X_train=X_train_fold,
                y_train=y_train_fold,
                evaluation_sets=[
                    X_valid_fold
                ],
                model_type="XGBoost",
                random_state=split_seed
            )

            X_valid_processed = (
                transformed_sets[0]
            )

            scale_pos_weight = 1.0

        else:
            (
                X_train_processed,
                X_valid_processed,
                preprocessing_objects
            ) = fit_standard_preprocessor(
                X_train_fold,
                X_valid_fold,
                model_type="XGBoost"
            )

            y_train_used = y_train_fold

            if strategy == "ClassWeight":
                negative_n = (
                    y_train_fold == 0
                ).sum()

                positive_n = (
                    y_train_fold == 1
                ).sum()

                scale_pos_weight = (
                    negative_n / positive_n
                )

            else:
                scale_pos_weight = 1.0

        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=
                scale_pos_weight,
            n_jobs=N_THREADS,
            random_state=split_seed,
            verbosity=0,
            **XGB_PARAMS
        )

        model.fit(
            X_train_processed,
            y_train_used
        )

        fit_time = (
            time.perf_counter()
            - fit_start
        )

        predict_start = time.perf_counter()

        valid_prob = model.predict_proba(
            X_valid_processed
        )[:, 1]

        predict_time = (
            time.perf_counter()
            - predict_start
        )

        metrics = evaluate_probabilities(
            y_valid_fold,
            valid_prob
        )

        fold_records.append({
            "Model": "XGBoost",
            "Strategy": strategy,
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Split_Seed": split_seed,
            "Training_N_Before":
                len(y_train_fold),
            "Training_N_After":
                len(y_train_used),
            "Training_Positive_Rate_After":
                np.mean(y_train_used),
            **metrics,
            "Fit_Time_Seconds": fit_time,
            "Predict_Time_Seconds":
                predict_time
        })

        for (
            development_position,
            original_index,
            y_value,
            probability
        ) in zip(
            valid_pos,
            original_indices,
            y_valid_fold,
            valid_prob
        ):
            oof_records.append({
                "Model": "XGBoost",
                "Strategy": strategy,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Development_Position":
                    int(development_position),
                "Original_Index":
                    int(original_index),
                "y_true": int(y_value),
                "y_prob": float(probability)
            })

        print(
            f"Repeat {repeat_id}, "
            f"Fold {fold_id}, "
            f"{strategy}: "
            f"ROC={metrics['ROC_AUC']:.4f}, "
            f"PR={metrics['PR_AUC']:.4f}, "
            f"Brier={metrics['Brier_Score']:.4f}"
        )

        del (
            model,
            X_train_processed,
            X_valid_processed
        )

        gc.collect()


# ============================================================
# 10. MLP ABLATION
# ============================================================

print("\n" + "=" * 105)
print("MLP IMBALANCE ABLATION")
print("=" * 105)

hidden_dims = tuple(
    int(value)
    for value in MLP_PARAMS[
        "hidden_structure"
    ].split("-")
)

for split_id, (
    outer_train_pos,
    outer_valid_pos
) in enumerate(OUTER_SPLITS):

    repeat_id = split_id // N_SPLITS + 1
    fold_id = split_id % N_SPLITS + 1
    split_seed = GLOBAL_SEED + split_id

    X_outer_train = (
        X_dev.iloc[outer_train_pos]
        .copy()
        .reset_index(drop=True)
    )

    y_outer_train = (
        y_dev.iloc[outer_train_pos]
        .astype(int)
        .reset_index(drop=True)
    )

    X_outer_valid = (
        X_dev.iloc[outer_valid_pos]
        .copy()
        .reset_index(drop=True)
    )

    y_outer_valid = (
        y_dev.iloc[outer_valid_pos]
        .astype(int)
        .to_numpy()
    )

    original_indices = (
        X_dev.iloc[outer_valid_pos]
        .index.to_numpy()
    )

    inner_positions = np.arange(
        len(X_outer_train)
    )

    (
        inner_train_pos,
        inner_stop_pos
    ) = train_test_split(
        inner_positions,
        test_size=INNER_VALIDATION_SIZE,
        stratify=y_outer_train,
        random_state=split_seed
    )

    X_inner_train = (
        X_outer_train.iloc[
            inner_train_pos
        ]
        .copy()
        .reset_index(drop=True)
    )

    y_inner_train = (
        y_outer_train.iloc[
            inner_train_pos
        ]
        .astype(int)
        .to_numpy()
    )

    X_inner_stop = (
        X_outer_train.iloc[
            inner_stop_pos
        ]
        .copy()
        .reset_index(drop=True)
    )

    y_inner_stop = (
        y_outer_train.iloc[
            inner_stop_pos
        ]
        .astype(int)
        .to_numpy()
    )

    for strategy in STRATEGIES:
        seed_everything(split_seed)

        fit_start = time.perf_counter()

        if strategy == "SMOTENC":
            (
                X_train_processed,
                y_train_used,
                transformed_sets,
                preprocessing_objects
            ) = fit_smotenc_preprocessing(
                X_train=X_inner_train,
                y_train=y_inner_train,
                evaluation_sets=[
                    X_inner_stop,
                    X_outer_valid
                ],
                model_type="MLP",
                random_state=split_seed
            )

            X_stop_processed = (
                transformed_sets[0]
            )

            X_valid_processed = (
                transformed_sets[1]
            )

        else:
            standard_preprocessor = (
                ColumnTransformer(
                    transformers=[
                        (
                            "numeric",
                            StandardScaler(),
                            NUMERIC_FEATURES
                        ),
                        (
                            "categorical",
                            OneHotEncoder(
                                handle_unknown=
                                    "ignore",
                                sparse_output=False
                            ),
                            CATEGORICAL_FEATURES
                        )
                    ],
                    remainder="drop"
                )
            )

            X_train_processed = (
                standard_preprocessor
                .fit_transform(
                    X_inner_train
                )
                .astype(np.float32)
            )

            X_stop_processed = (
                standard_preprocessor
                .transform(
                    X_inner_stop
                )
                .astype(np.float32)
            )

            X_valid_processed = (
                standard_preprocessor
                .transform(
                    X_outer_valid
                )
                .astype(np.float32)
            )

            y_train_used = y_inner_train

        train_dataset = TensorDataset(
            torch.from_numpy(
                X_train_processed
            ),
            torch.from_numpy(
                y_train_used.astype(
                    np.float32
                )
            )
        )

        generator = torch.Generator()
        generator.manual_seed(split_seed)

        train_loader = DataLoader(
            train_dataset,
            batch_size=MLP_PARAMS[
                "batch_size"
            ],
            shuffle=True,
            num_workers=0,
            generator=generator,
            drop_last=False
        )

        model = BankingMLP(
            input_dim=
                X_train_processed.shape[1],
            hidden_dims=hidden_dims,
            dropout=MLP_PARAMS[
                "dropout"
            ],
            use_batch_norm=MLP_PARAMS[
                "use_batch_norm"
            ],
            activation_name=MLP_PARAMS[
                "activation"
            ]
        )

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=MLP_PARAMS[
                "learning_rate"
            ],
            weight_decay=MLP_PARAMS[
                "weight_decay"
            ]
        )

        if strategy == "ClassWeight":
            negative_n = (
                y_train_used == 0
            ).sum()

            positive_n = (
                y_train_used == 1
            ).sum()

            positive_weight = (
                negative_n / positive_n
            )

            criterion = (
                nn.BCEWithLogitsLoss(
                    pos_weight=torch.tensor(
                        positive_weight,
                        dtype=torch.float32
                    )
                )
            )

        else:
            criterion = (
                nn.BCEWithLogitsLoss()
            )

        X_stop_tensor = torch.from_numpy(
            X_stop_processed
        )

        X_valid_tensor = torch.from_numpy(
            X_valid_processed
        )

        best_inner_auc = -np.inf
        best_state = None
        no_improvement = 0
        completed_epochs = 0

        for epoch in range(1, 81):
            model.train()

            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()

                logits = model(batch_X)

                loss = criterion(
                    logits,
                    batch_y
                )

                loss.backward()
                optimizer.step()

            model.eval()

            with torch.no_grad():
                stop_prob = torch.sigmoid(
                    model(X_stop_tensor)
                ).numpy()

            stop_auc = roc_auc_score(
                y_inner_stop,
                stop_prob
            )

            completed_epochs = epoch

            if (
                stop_auc
                > best_inner_auc + 1e-5
            ):
                best_inner_auc = stop_auc

                best_state = copy.deepcopy(
                    model.state_dict()
                )

                no_improvement = 0
            else:
                no_improvement += 1

            if no_improvement >= 10:
                break

        model.load_state_dict(best_state)
        model.eval()

        fit_time = (
            time.perf_counter()
            - fit_start
        )

        predict_start = time.perf_counter()

        with torch.no_grad():
            valid_prob = torch.sigmoid(
                model(X_valid_tensor)
            ).numpy()

        predict_time = (
            time.perf_counter()
            - predict_start
        )

        metrics = evaluate_probabilities(
            y_outer_valid,
            valid_prob
        )

        fold_records.append({
            "Model": "MLP",
            "Strategy": strategy,
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Split_Seed": split_seed,
            "Training_N_Before":
                len(y_inner_train),
            "Training_N_After":
                len(y_train_used),
            "Training_Positive_Rate_After":
                np.mean(y_train_used),
            "Completed_Epochs":
                completed_epochs,
            **metrics,
            "Fit_Time_Seconds": fit_time,
            "Predict_Time_Seconds":
                predict_time
        })

        for (
            development_position,
            original_index,
            y_value,
            probability
        ) in zip(
            outer_valid_pos,
            original_indices,
            y_outer_valid,
            valid_prob
        ):
            oof_records.append({
                "Model": "MLP",
                "Strategy": strategy,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Development_Position":
                    int(development_position),
                "Original_Index":
                    int(original_index),
                "y_true": int(y_value),
                "y_prob": float(probability)
            })

        print(
            f"Repeat {repeat_id}, "
            f"Fold {fold_id}, "
            f"{strategy}: "
            f"ROC={metrics['ROC_AUC']:.4f}, "
            f"PR={metrics['PR_AUC']:.4f}, "
            f"Brier={metrics['Brier_Score']:.4f}, "
            f"epochs={completed_epochs}"
        )

        del (
            model,
            optimizer,
            train_dataset,
            train_loader,
            X_train_processed,
            X_stop_processed,
            X_valid_processed
        )

        gc.collect()


# ------------------------------------------------------------
# 11. Fold-level summary
# ------------------------------------------------------------

fold_results = pd.DataFrame(
    fold_records
)

oof_long = pd.DataFrame(
    oof_records
)

summary_metrics = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "ECE_10",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "Fit_Time_Seconds"
]

fold_summary = (
    fold_results
    .groupby(
        ["Model", "Strategy"]
    )[summary_metrics]
    .agg(["mean", "std"])
)

fold_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat
    in fold_summary.columns
]

fold_summary = (
    fold_summary
    .reset_index()
    .sort_values(
        ["Model", "ROC_AUC_mean"],
        ascending=[True, False]
    )
)

print("\n" + "=" * 150)
print("IMBALANCE ABLATION FOLD SUMMARY")
print("=" * 150)

display_columns = [
    "Model",
    "Strategy",
    "ROC_AUC_mean",
    "ROC_AUC_std",
    "PR_AUC_mean",
    "PR_AUC_std",
    "Brier_Score_mean",
    "ECE_10_mean",
    "Precision_mean",
    "Recall_mean",
    "F1_mean",
    "Balanced_Accuracy_mean",
    "MCC_mean",
    "Fit_Time_Seconds_mean"
]

print(
    fold_summary[
        display_columns
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 12. Delta against no-treatment baseline
# ------------------------------------------------------------

delta_records = []

for model_name in ["XGBoost", "MLP"]:
    model_rows = fold_summary[
        fold_summary["Model"]
        == model_name
    ].copy()

    baseline = model_rows[
        model_rows["Strategy"]
        == "None"
    ].iloc[0]

    for _, row in model_rows.iterrows():
        delta_records.append({
            "Model": model_name,
            "Strategy": row["Strategy"],
            "Delta_ROC_AUC":
                row["ROC_AUC_mean"]
                - baseline[
                    "ROC_AUC_mean"
                ],
            "Delta_PR_AUC":
                row["PR_AUC_mean"]
                - baseline[
                    "PR_AUC_mean"
                ],
            "Delta_Brier":
                row["Brier_Score_mean"]
                - baseline[
                    "Brier_Score_mean"
                ],
            "Delta_ECE":
                row["ECE_10_mean"]
                - baseline[
                    "ECE_10_mean"
                ],
            "Delta_Recall":
                row["Recall_mean"]
                - baseline[
                    "Recall_mean"
                ],
            "Delta_MCC":
                row["MCC_mean"]
                - baseline[
                    "MCC_mean"
                ]
        })

delta_table = pd.DataFrame(
    delta_records
)

print("\n" + "=" * 115)
print("DELTA VERSUS NO IMBALANCE TREATMENT")
print("For Brier and ECE, negative values indicate improvement.")
print("=" * 115)

print(
    delta_table
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 13. Mean OOF predictions across repeats
# ------------------------------------------------------------

mean_oof = (
    oof_long
    .groupby(
        [
            "Model",
            "Strategy",
            "Development_Position",
            "Original_Index",
            "y_true"
        ],
        as_index=False
    )
    .agg(
        Mean_OOF_Probability=(
            "y_prob",
            "mean"
        ),
        SD_OOF_Probability=(
            "y_prob",
            "std"
        ),
        Prediction_Count=(
            "y_prob",
            "count"
        )
    )
)

assert (
    mean_oof["Prediction_Count"]
    == N_REPEATS
).all()


# ------------------------------------------------------------
# 14. Save artifacts
# ------------------------------------------------------------

fold_results.to_csv(
    OUTPUT_DIR
    / "imbalance_fold_metrics.csv",
    index=False
)

fold_summary.to_csv(
    OUTPUT_DIR
    / "imbalance_fold_summary.csv",
    index=False
)

delta_table.to_csv(
    OUTPUT_DIR
    / "imbalance_delta_vs_none.csv",
    index=False
)

oof_long.to_csv(
    OUTPUT_DIR
    / "imbalance_oof_long.csv",
    index=False
)

mean_oof.to_csv(
    OUTPUT_DIR
    / "imbalance_mean_oof.csv",
    index=False
)

total_elapsed = (
    time.perf_counter()
    - overall_start
)

print("\nSaved directory:")
print(OUTPUT_DIR)

print(
    f"\nTotal elapsed time: "
    f"{total_elapsed / 60:.2f} minutes"
)

In [ ]:
# ============================================================
# MLP ARCHITECTURE ABLATION
# Controlled 2 × 2 factorial:
# Dropout OFF/ON × Batch Normalization OFF/ON
# 5 folds × 3 repeats, development set only
# ============================================================

import copy
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    train_test_split
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

GLOBAL_SEED = 42
N_SPLITS = 5
N_REPEATS = 3
INNER_VALIDATION_SIZE = 0.15
N_THREADS = 4

OUTPUT_DIR = Path(
    "/kaggle/working/mlp_architecture_ablation"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(N_THREADS)

try:
    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )
except Exception:
    pass


# ------------------------------------------------------------
# 2. Load locked MLP parameters
# ------------------------------------------------------------

with open(
    "/kaggle/working/deep_hpo/"
    "all_deep_best_params.json",
    "r"
) as f:
    deep_params = json.load(f)

MLP_PARAMS = deep_params["MLP"]

TUNED_DROPOUT = float(
    MLP_PARAMS["dropout"]
)

HIDDEN_DIMS = tuple(
    int(value)
    for value in MLP_PARAMS[
        "hidden_structure"
    ].split("-")
)

print("=" * 100)
print("LOCKED MLP PARAMETERS")
print("=" * 100)

print(json.dumps(MLP_PARAMS, indent=4))

print("\nTuned dropout:", TUNED_DROPOUT)
print("Hidden dimensions:", HIDDEN_DIMS)


# ------------------------------------------------------------
# 3. Architecture conditions
# ------------------------------------------------------------

ARCHITECTURES = {
    "D0_BN0": {
        "Dropout_Enabled": False,
        "BatchNorm_Enabled": False,
        "Dropout_Value": 0.0
    },
    "D1_BN0_TunedBaseline": {
        "Dropout_Enabled": True,
        "BatchNorm_Enabled": False,
        "Dropout_Value": TUNED_DROPOUT
    },
    "D0_BN1": {
        "Dropout_Enabled": False,
        "BatchNorm_Enabled": True,
        "Dropout_Value": 0.0
    },
    "D1_BN1": {
        "Dropout_Enabled": True,
        "BatchNorm_Enabled": True,
        "Dropout_Value": TUNED_DROPOUT
    }
}

print("\n" + "=" * 100)
print("ARCHITECTURE CONDITIONS")
print("=" * 100)

for name, config in ARCHITECTURES.items():
    print(name, config)


# ------------------------------------------------------------
# 4. Shared repeated CV splits
# ------------------------------------------------------------

outer_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=GLOBAL_SEED
)

OUTER_SPLITS = list(
    outer_cv.split(X_dev, y_dev)
)

print("\nTotal outer evaluations:", len(OUTER_SPLITS))


# ------------------------------------------------------------
# 5. Utilities
# ------------------------------------------------------------

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def expected_calibration_error(
    y_true,
    y_prob,
    n_bins=10
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    edges = np.linspace(0, 1, n_bins + 1)

    bin_ids = np.digitize(
        y_prob,
        edges[1:-1],
        right=True
    )

    ece = 0.0

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id

        if mask.sum() == 0:
            continue

        ece += (
            mask.mean()
            * abs(
                y_prob[mask].mean()
                - y_true[mask].mean()
            )
        )

    return float(ece)


def evaluate_probabilities(
    y_true,
    y_prob,
    threshold=0.50
):
    y_prob = np.clip(
        np.asarray(y_prob),
        1e-7,
        1 - 1e-7
    )

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "ROC_AUC": roc_auc_score(
            y_true,
            y_prob
        ),
        "PR_AUC": average_precision_score(
            y_true,
            y_prob
        ),
        "Brier_Score": brier_score_loss(
            y_true,
            y_prob
        ),
        "Log_Loss": log_loss(
            y_true,
            y_prob,
            labels=[0, 1]
        ),
        "ECE_10": expected_calibration_error(
            y_true,
            y_prob,
            n_bins=10
        ),
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),
        "MCC": matthews_corrcoef(
            y_true,
            y_pred
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


# ------------------------------------------------------------
# 6. MLP definition
# ------------------------------------------------------------

class ArchitectureMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout,
        use_batch_norm,
        activation_name
    ):
        super().__init__()

        activation_classes = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "silu": nn.SiLU
        }

        activation_class = activation_classes[
            activation_name
        ]

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:

            layers.append(
                nn.Linear(
                    previous_dim,
                    hidden_dim
                )
            )

            if use_batch_norm:
                layers.append(
                    nn.BatchNorm1d(
                        hidden_dim
                    )
                )

            layers.append(
                activation_class()
            )

            if dropout > 0:
                layers.append(
                    nn.Dropout(dropout)
                )

            previous_dim = hidden_dim

        layers.append(
            nn.Linear(previous_dim, 1)
        )

        self.network = nn.Sequential(
            *layers
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


# ------------------------------------------------------------
# 7. Evaluation loop
# ------------------------------------------------------------

fold_records = []
oof_records = []

overall_start = time.perf_counter()

for split_id, (
    outer_train_pos,
    outer_valid_pos
) in enumerate(OUTER_SPLITS):

    repeat_id = split_id // N_SPLITS + 1
    fold_id = split_id % N_SPLITS + 1
    split_seed = GLOBAL_SEED + split_id

    X_outer_train = (
        X_dev.iloc[outer_train_pos]
        .copy()
        .reset_index(drop=True)
    )

    y_outer_train = (
        y_dev.iloc[outer_train_pos]
        .astype(int)
        .reset_index(drop=True)
    )

    X_outer_valid = (
        X_dev.iloc[outer_valid_pos]
        .copy()
        .reset_index(drop=True)
    )

    y_outer_valid = (
        y_dev.iloc[outer_valid_pos]
        .astype(int)
        .to_numpy()
    )

    original_indices = (
        X_dev.iloc[outer_valid_pos]
        .index
        .to_numpy()
    )

    inner_positions = np.arange(
        len(X_outer_train)
    )

    (
        inner_train_pos,
        inner_stop_pos
    ) = train_test_split(
        inner_positions,
        test_size=INNER_VALIDATION_SIZE,
        stratify=y_outer_train,
        random_state=split_seed
    )

    X_inner_train = (
        X_outer_train.iloc[inner_train_pos]
        .copy()
        .reset_index(drop=True)
    )

    y_inner_train = (
        y_outer_train.iloc[inner_train_pos]
        .astype(int)
        .to_numpy()
    )

    X_inner_stop = (
        X_outer_train.iloc[inner_stop_pos]
        .copy()
        .reset_index(drop=True)
    )

    y_inner_stop = (
        y_outer_train.iloc[inner_stop_pos]
        .astype(int)
        .to_numpy()
    )

    # Fit preprocessing once per split.
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )

    X_train_processed = (
        preprocessor
        .fit_transform(X_inner_train)
        .astype(np.float32)
    )

    X_stop_processed = (
        preprocessor
        .transform(X_inner_stop)
        .astype(np.float32)
    )

    X_valid_processed = (
        preprocessor
        .transform(X_outer_valid)
        .astype(np.float32)
    )

    X_stop_tensor = torch.from_numpy(
        X_stop_processed
    )

    X_valid_tensor = torch.from_numpy(
        X_valid_processed
    )

    for architecture_name, architecture in (
        ARCHITECTURES.items()
    ):
        seed_everything(split_seed)

        train_dataset = TensorDataset(
            torch.from_numpy(
                X_train_processed
            ),
            torch.from_numpy(
                y_inner_train.astype(
                    np.float32
                )
            )
        )

        generator = torch.Generator()
        generator.manual_seed(split_seed)

        train_loader = DataLoader(
            train_dataset,
            batch_size=MLP_PARAMS[
                "batch_size"
            ],
            shuffle=True,
            num_workers=0,
            generator=generator,
            drop_last=False
        )

        model = ArchitectureMLP(
            input_dim=
                X_train_processed.shape[1],
            hidden_dims=HIDDEN_DIMS,
            dropout=architecture[
                "Dropout_Value"
            ],
            use_batch_norm=architecture[
                "BatchNorm_Enabled"
            ],
            activation_name=MLP_PARAMS[
                "activation"
            ]
        )

        parameter_count = sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=MLP_PARAMS[
                "learning_rate"
            ],
            weight_decay=MLP_PARAMS[
                "weight_decay"
            ]
        )

        criterion = nn.BCEWithLogitsLoss()

        best_inner_auc = -np.inf
        best_state = None
        no_improvement = 0
        completed_epochs = 0

        fit_start = time.perf_counter()

        for epoch in range(1, 81):

            model.train()

            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()

                logits = model(batch_X)

                loss = criterion(
                    logits,
                    batch_y
                )

                loss.backward()
                optimizer.step()

            model.eval()

            with torch.no_grad():
                stop_prob = torch.sigmoid(
                    model(X_stop_tensor)
                ).cpu().numpy()

            inner_auc = roc_auc_score(
                y_inner_stop,
                stop_prob
            )

            completed_epochs = epoch

            if (
                inner_auc
                > best_inner_auc + 1e-5
            ):
                best_inner_auc = inner_auc

                best_state = copy.deepcopy(
                    model.state_dict()
                )

                no_improvement = 0
            else:
                no_improvement += 1

            if no_improvement >= 10:
                break

        model.load_state_dict(best_state)
        model.eval()

        fit_time = (
            time.perf_counter()
            - fit_start
        )

        predict_start = time.perf_counter()

        with torch.no_grad():
            valid_prob = torch.sigmoid(
                model(X_valid_tensor)
            ).cpu().numpy()

        predict_time = (
            time.perf_counter()
            - predict_start
        )

        metrics = evaluate_probabilities(
            y_outer_valid,
            valid_prob
        )

        fold_records.append({
            "Architecture":
                architecture_name,
            "Dropout_Enabled":
                architecture[
                    "Dropout_Enabled"
                ],
            "BatchNorm_Enabled":
                architecture[
                    "BatchNorm_Enabled"
                ],
            "Dropout_Value":
                architecture[
                    "Dropout_Value"
                ],
            "Repeat": repeat_id,
            "Fold": fold_id,
            "Split_Seed": split_seed,
            "Completed_Epochs":
                completed_epochs,
            "Best_Inner_ROC_AUC":
                best_inner_auc,
            "Parameter_Count":
                parameter_count,
            **metrics,
            "Fit_Time_Seconds":
                fit_time,
            "Predict_Time_Seconds":
                predict_time
        })

        for (
            development_position,
            original_index,
            y_value,
            probability
        ) in zip(
            outer_valid_pos,
            original_indices,
            y_outer_valid,
            valid_prob
        ):
            oof_records.append({
                "Architecture":
                    architecture_name,
                "Repeat": repeat_id,
                "Fold": fold_id,
                "Development_Position":
                    int(
                        development_position
                    ),
                "Original_Index":
                    int(original_index),
                "y_true": int(y_value),
                "y_prob": float(probability)
            })

        print(
            f"Repeat {repeat_id}, "
            f"Fold {fold_id}, "
            f"{architecture_name}: "
            f"ROC={metrics['ROC_AUC']:.4f}, "
            f"PR={metrics['PR_AUC']:.4f}, "
            f"Brier={metrics['Brier_Score']:.4f}, "
            f"epochs={completed_epochs}"
        )

        del (
            model,
            optimizer,
            train_dataset,
            train_loader
        )

        gc.collect()

    del (
        preprocessor,
        X_train_processed,
        X_stop_processed,
        X_valid_processed
    )

    gc.collect()


# ------------------------------------------------------------
# 8. Fold-level summary
# ------------------------------------------------------------

fold_results = pd.DataFrame(
    fold_records
)

oof_long = pd.DataFrame(
    oof_records
)

summary_metrics = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "ECE_10",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "Fit_Time_Seconds",
    "Predict_Time_Seconds",
    "Completed_Epochs",
    "Parameter_Count"
]

architecture_summary = (
    fold_results
    .groupby(
        [
            "Architecture",
            "Dropout_Enabled",
            "BatchNorm_Enabled",
            "Dropout_Value"
        ]
    )[summary_metrics]
    .agg(["mean", "std"])
)

architecture_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat
    in architecture_summary.columns
]

architecture_summary = (
    architecture_summary
    .reset_index()
    .sort_values(
        "ROC_AUC_mean",
        ascending=False
    )
)


print("\n" + "=" * 155)
print("MLP ARCHITECTURE ABLATION SUMMARY")
print("=" * 155)

display_columns = [
    "Architecture",
    "Dropout_Enabled",
    "BatchNorm_Enabled",
    "Dropout_Value",
    "ROC_AUC_mean",
    "ROC_AUC_std",
    "PR_AUC_mean",
    "PR_AUC_std",
    "Brier_Score_mean",
    "ECE_10_mean",
    "F1_mean",
    "MCC_mean",
    "Completed_Epochs_mean",
    "Parameter_Count_mean",
    "Fit_Time_Seconds_mean"
]

print(
    architecture_summary[
        display_columns
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. Paired descriptive deltas versus tuned baseline
# ------------------------------------------------------------

BASELINE_NAME = "D1_BN0_TunedBaseline"

baseline_folds = (
    fold_results[
        fold_results["Architecture"]
        == BASELINE_NAME
    ][
        [
            "Repeat",
            "Fold",
            "ROC_AUC",
            "PR_AUC",
            "Brier_Score",
            "ECE_10",
            "F1",
            "MCC"
        ]
    ]
    .rename(
        columns={
            column: f"Baseline_{column}"
            for column in [
                "ROC_AUC",
                "PR_AUC",
                "Brier_Score",
                "ECE_10",
                "F1",
                "MCC"
            ]
        }
    )
)

paired = fold_results.merge(
    baseline_folds,
    on=["Repeat", "Fold"],
    how="left"
)

for metric in [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "ECE_10",
    "F1",
    "MCC"
]:
    paired[f"Delta_{metric}"] = (
        paired[metric]
        - paired[f"Baseline_{metric}"]
    )

delta_summary = (
    paired
    .groupby("Architecture")[
        [
            "Delta_ROC_AUC",
            "Delta_PR_AUC",
            "Delta_Brier_Score",
            "Delta_ECE_10",
            "Delta_F1",
            "Delta_MCC"
        ]
    ]
    .agg(["mean", "std"])
)

delta_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat
    in delta_summary.columns
]

delta_summary = (
    delta_summary
    .reset_index()
    .sort_values(
        "Delta_ROC_AUC_mean",
        ascending=False
    )
)

print("\n" + "=" * 125)
print("PAIRED DELTAS VERSUS TUNED MLP BASELINE")
print("For Brier and ECE, negative deltas indicate improvement.")
print("=" * 125)

print(
    delta_summary
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. Factorial descriptive effects
# ------------------------------------------------------------

factor_table = (
    fold_results
    .groupby(
        [
            "Dropout_Enabled",
            "BatchNorm_Enabled"
        ]
    )[
        [
            "ROC_AUC",
            "PR_AUC",
            "Brier_Score",
            "ECE_10"
        ]
    ]
    .mean()
    .reset_index()
)

dropout_on = factor_table[
    factor_table["Dropout_Enabled"]
]["ROC_AUC"].mean()

dropout_off = factor_table[
    ~factor_table["Dropout_Enabled"]
]["ROC_AUC"].mean()

bn_on = factor_table[
    factor_table["BatchNorm_Enabled"]
]["ROC_AUC"].mean()

bn_off = factor_table[
    ~factor_table["BatchNorm_Enabled"]
]["ROC_AUC"].mean()

factorial_effects = pd.DataFrame({
    "Factor": [
        "Dropout main effect",
        "BatchNorm main effect"
    ],
    "Mean_ROC_AUC_Effect": [
        dropout_on - dropout_off,
        bn_on - bn_off
    ]
})

print("\n" + "=" * 90)
print("DESCRIPTIVE FACTORIAL EFFECTS")
print("=" * 90)

print(
    factorial_effects
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 11. Save artifacts
# ------------------------------------------------------------

fold_results.to_csv(
    OUTPUT_DIR
    / "mlp_architecture_fold_metrics.csv",
    index=False
)

architecture_summary.to_csv(
    OUTPUT_DIR
    / "mlp_architecture_summary.csv",
    index=False
)

paired.to_csv(
    OUTPUT_DIR
    / "mlp_architecture_paired_fold_data.csv",
    index=False
)

delta_summary.to_csv(
    OUTPUT_DIR
    / "mlp_architecture_delta_vs_baseline.csv",
    index=False
)

factorial_effects.to_csv(
    OUTPUT_DIR
    / "mlp_architecture_factorial_effects.csv",
    index=False
)

oof_long.to_csv(
    OUTPUT_DIR
    / "mlp_architecture_oof_long.csv",
    index=False
)

total_elapsed = (
    time.perf_counter()
    - overall_start
)

print("\nSaved directory:")
print(OUTPUT_DIR)

print(
    f"\nTotal elapsed time: "
    f"{total_elapsed / 60:.2f} minutes"
)

In [ ]:
# ============================================================
# CROSS-FITTED CALIBRATION AND THRESHOLD LOCKING
# Uses Repeat 1 OOF predictions only
# Development data only
# Locked test set remains untouched
# ============================================================

import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

CALIBRATION_SEED = 2026
N_CALIBRATION_FOLDS = 5
PROBABILITY_EPSILON = 1e-6

OUTPUT_DIR = Path(
    "/kaggle/working/calibration_threshold"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_ORDER = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

CALIBRATION_METHODS = [
    "None",
    "Sigmoid",
    "Isotonic"
]


# ------------------------------------------------------------
# 2. Load single-repeat OOF predictions
# ------------------------------------------------------------

classical_oof_long = pd.read_csv(
    "/kaggle/working/classical_repeated_cv/"
    "repeated_oof_predictions_long.csv"
)

deep_oof_long = pd.read_csv(
    "/kaggle/working/deep_repeated_cv/"
    "deep_repeated_oof_long.csv"
)

classical_repeat_1 = classical_oof_long[
    classical_oof_long["Repeat"] == 1
].copy()

deep_repeat_1 = deep_oof_long[
    deep_oof_long["Repeat"] == 1
].copy()

single_repeat_oof = pd.concat(
    [
        classical_repeat_1,
        deep_repeat_1
    ],
    ignore_index=True
)

single_repeat_oof = single_repeat_oof[
    single_repeat_oof["Model"].isin(
        MODEL_ORDER
    )
].copy()


# ------------------------------------------------------------
# 3. Integrity checks
# ------------------------------------------------------------

print("=" * 100)
print("SINGLE-REPEAT OOF INTEGRITY CHECK")
print("=" * 100)

integrity_records = []

for model_name in MODEL_ORDER:

    model_data = (
        single_repeat_oof[
            single_repeat_oof["Model"]
            == model_name
        ]
        .sort_values(
            "Development_Position"
        )
        .reset_index(drop=True)
    )

    integrity_records.append({
        "Model": model_name,
        "Rows": len(model_data),
        "Unique_Development_Positions":
            model_data[
                "Development_Position"
            ].nunique(),
        "Unique_Original_Indices":
            model_data[
                "Original_Index"
            ].nunique(),
        "Positive_Rate":
            model_data["y_true"].mean(),
        "Probability_Min":
            model_data["y_prob"].min(),
        "Probability_Max":
            model_data["y_prob"].max(),
        "Missing_Probabilities":
            model_data["y_prob"].isna().sum()
    })

integrity_table = pd.DataFrame(
    integrity_records
)

print(
    integrity_table
    .round(6)
    .to_string(index=False)
)

assert (
    integrity_table["Rows"] == len(X_dev)
).all()

assert (
    integrity_table[
        "Unique_Development_Positions"
    ] == len(X_dev)
).all()

assert (
    integrity_table[
        "Missing_Probabilities"
    ] == 0
).all()


# ------------------------------------------------------------
# 4. Utility functions
# ------------------------------------------------------------

def clip_probabilities(probabilities):
    return np.clip(
        np.asarray(
            probabilities,
            dtype=float
        ),
        PROBABILITY_EPSILON,
        1 - PROBABILITY_EPSILON
    )


def probability_to_logit(probabilities):
    probabilities = clip_probabilities(
        probabilities
    )

    return np.log(
        probabilities
        / (1 - probabilities)
    )


def expected_calibration_error(
    y_true,
    y_prob,
    n_bins=10
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    bin_ids = np.digitize(
        y_prob,
        edges[1:-1],
        right=True
    )

    ece = 0.0

    for bin_id in range(n_bins):

        mask = bin_ids == bin_id

        if mask.sum() == 0:
            continue

        observed_rate = y_true[mask].mean()
        predicted_rate = y_prob[mask].mean()

        ece += (
            mask.mean()
            * abs(
                observed_rate
                - predicted_rate
            )
        )

    return float(ece)


def probability_metrics(
    y_true,
    y_prob
):
    y_prob = clip_probabilities(
        y_prob
    )

    return {
        "ROC_AUC": roc_auc_score(
            y_true,
            y_prob
        ),
        "PR_AUC": average_precision_score(
            y_true,
            y_prob
        ),
        "Brier_Score": brier_score_loss(
            y_true,
            y_prob
        ),
        "Log_Loss": log_loss(
            y_true,
            y_prob,
            labels=[0, 1]
        ),
        "ECE_10": expected_calibration_error(
            y_true,
            y_prob,
            n_bins=10
        ),
        "Mean_Probability":
            float(y_prob.mean())
    }


def classification_metrics(
    y_true,
    y_prob,
    threshold
):
    y_pred = (
        np.asarray(y_prob) >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),
        "MCC": matthews_corrcoef(
            y_true,
            y_pred
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


def fit_calibrator(
    method,
    raw_probability,
    y_true
):
    raw_probability = clip_probabilities(
        raw_probability
    )

    if method == "None":
        return None

    if method == "Sigmoid":

        calibrator = LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=2000,
            random_state=CALIBRATION_SEED
        )

        calibrator.fit(
            probability_to_logit(
                raw_probability
            ).reshape(-1, 1),
            y_true
        )

        return calibrator

    if method == "Isotonic":

        calibrator = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            increasing=True,
            out_of_bounds="clip"
        )

        calibrator.fit(
            raw_probability,
            y_true
        )

        return calibrator

    raise ValueError(
        f"Unknown calibration method: {method}"
    )


def apply_calibrator(
    method,
    calibrator,
    raw_probability
):
    raw_probability = clip_probabilities(
        raw_probability
    )

    if method == "None":
        return raw_probability

    if method == "Sigmoid":

        return calibrator.predict_proba(
            probability_to_logit(
                raw_probability
            ).reshape(-1, 1)
        )[:, 1]

    if method == "Isotonic":

        return calibrator.predict(
            raw_probability
        )

    raise ValueError(
        f"Unknown calibration method: {method}"
    )


def optimize_mcc_threshold(
    y_true,
    y_prob
):
    threshold_grid = np.linspace(
        0.001,
        0.999,
        999
    )

    threshold_records = []

    for threshold in threshold_grid:

        metrics = classification_metrics(
            y_true,
            y_prob,
            threshold
        )

        threshold_records.append(
            metrics
        )

    threshold_table = pd.DataFrame(
        threshold_records
    )

    threshold_table[
        "Distance_From_0_5"
    ] = abs(
        threshold_table["Threshold"]
        - 0.50
    )

    best_row = (
        threshold_table
        .sort_values(
            [
                "MCC",
                "Balanced_Accuracy",
                "F1",
                "Distance_From_0_5"
            ],
            ascending=[
                False,
                False,
                False,
                True
            ]
        )
        .iloc[0]
    )

    return (
        best_row.to_dict(),
        threshold_table
    )


# ------------------------------------------------------------
# 5. Fixed meta-level calibration folds
# ------------------------------------------------------------

calibration_cv = StratifiedKFold(
    n_splits=N_CALIBRATION_FOLDS,
    shuffle=True,
    random_state=CALIBRATION_SEED
)


# ------------------------------------------------------------
# 6. Cross-fitted calibration comparison
# ------------------------------------------------------------

calibration_summary_records = []
calibrated_prediction_records = []

for model_name in MODEL_ORDER:

    print("\n" + "=" * 100)
    print(f"CROSS-FITTED CALIBRATION: {model_name}")
    print("=" * 100)

    model_data = (
        single_repeat_oof[
            single_repeat_oof["Model"]
            == model_name
        ]
        .sort_values(
            "Development_Position"
        )
        .reset_index(drop=True)
    )

    y_model = (
        model_data["y_true"]
        .astype(int)
        .to_numpy()
    )

    raw_probability = (
        model_data["y_prob"]
        .astype(float)
        .to_numpy()
    )

    positions = model_data[
        "Development_Position"
    ].to_numpy()

    original_indices = model_data[
        "Original_Index"
    ].to_numpy()

    calibration_splits = list(
        calibration_cv.split(
            raw_probability.reshape(-1, 1),
            y_model
        )
    )

    for method in CALIBRATION_METHODS:

        cross_fitted_probability = np.zeros(
            len(model_data),
            dtype=float
        )

        for calibration_fold, (
            train_position,
            validation_position
        ) in enumerate(
            calibration_splits,
            start=1
        ):

            calibrator = fit_calibrator(
                method=method,
                raw_probability=raw_probability[
                    train_position
                ],
                y_true=y_model[
                    train_position
                ]
            )

            calibrated_validation = (
                apply_calibrator(
                    method=method,
                    calibrator=calibrator,
                    raw_probability=
                        raw_probability[
                            validation_position
                        ]
                )
            )

            cross_fitted_probability[
                validation_position
            ] = calibrated_validation

        metrics = probability_metrics(
            y_model,
            cross_fitted_probability
        )

        calibration_summary_records.append({
            "Model": model_name,
            "Calibration_Method": method,
            **metrics
        })

        for (
            position,
            original_index,
            y_value,
            raw_prob,
            calibrated_prob
        ) in zip(
            positions,
            original_indices,
            y_model,
            raw_probability,
            cross_fitted_probability
        ):
            calibrated_prediction_records.append({
                "Model": model_name,
                "Calibration_Method": method,
                "Development_Position":
                    int(position),
                "Original_Index":
                    int(original_index),
                "y_true": int(y_value),
                "Raw_OOF_Probability":
                    float(raw_prob),
                "Calibrated_OOF_Probability":
                    float(calibrated_prob)
            })

        print(
            f"{method}: "
            f"Brier={metrics['Brier_Score']:.6f}, "
            f"LogLoss={metrics['Log_Loss']:.6f}, "
            f"ECE={metrics['ECE_10']:.6f}"
        )


calibration_summary = pd.DataFrame(
    calibration_summary_records
)

calibrated_predictions = pd.DataFrame(
    calibrated_prediction_records
)


# ------------------------------------------------------------
# 7. Select calibration method
# Primary criterion: lowest cross-fitted Brier score
# Tie-breakers: LogLoss, ECE, simpler method
# ------------------------------------------------------------

method_complexity = {
    "None": 0,
    "Sigmoid": 1,
    "Isotonic": 2
}

calibration_summary[
    "Method_Complexity"
] = calibration_summary[
    "Calibration_Method"
].map(method_complexity)

selected_method_records = []

for model_name in MODEL_ORDER:

    model_methods = calibration_summary[
        calibration_summary["Model"]
        == model_name
    ].copy()

    selected_row = (
        model_methods
        .sort_values(
            [
                "Brier_Score",
                "Log_Loss",
                "ECE_10",
                "Method_Complexity"
            ],
            ascending=True
        )
        .iloc[0]
    )

    raw_row = model_methods[
        model_methods[
            "Calibration_Method"
        ] == "None"
    ].iloc[0]

    selected_method_records.append({
        "Model": model_name,
        "Selected_Calibration":
            selected_row[
                "Calibration_Method"
            ],
        "Raw_Brier":
            raw_row["Brier_Score"],
        "Selected_Brier":
            selected_row["Brier_Score"],
        "Delta_Brier":
            selected_row["Brier_Score"]
            - raw_row["Brier_Score"],
        "Raw_Log_Loss":
            raw_row["Log_Loss"],
        "Selected_Log_Loss":
            selected_row["Log_Loss"],
        "Delta_Log_Loss":
            selected_row["Log_Loss"]
            - raw_row["Log_Loss"],
        "Raw_ECE":
            raw_row["ECE_10"],
        "Selected_ECE":
            selected_row["ECE_10"],
        "Delta_ECE":
            selected_row["ECE_10"]
            - raw_row["ECE_10"]
    })

selected_methods = pd.DataFrame(
    selected_method_records
)


# ------------------------------------------------------------
# 8. Threshold selection using selected
# cross-fitted calibrated probabilities
# ------------------------------------------------------------

threshold_records = []
selected_oof_records = []
threshold_grid_records = []
calibrator_manifest = {}

for model_name in MODEL_ORDER:

    selected_method = (
        selected_methods.loc[
            selected_methods["Model"]
            == model_name,
            "Selected_Calibration"
        ]
        .iloc[0]
    )

    selected_predictions = (
        calibrated_predictions[
            (
                calibrated_predictions["Model"]
                == model_name
            )
            & (
                calibrated_predictions[
                    "Calibration_Method"
                ] == selected_method
            )
        ]
        .sort_values(
            "Development_Position"
        )
        .reset_index(drop=True)
    )

    y_model = (
        selected_predictions["y_true"]
        .astype(int)
        .to_numpy()
    )

    selected_probability = (
        selected_predictions[
            "Calibrated_OOF_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    raw_probability = (
        selected_predictions[
            "Raw_OOF_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    best_threshold, threshold_table = (
        optimize_mcc_threshold(
            y_model,
            selected_probability
        )
    )

    metrics_at_05 = classification_metrics(
        y_model,
        selected_probability,
        threshold=0.50
    )

    threshold_records.append({
        "Model": model_name,
        "Selected_Calibration":
            selected_method,
        "Locked_MCC_Threshold":
            best_threshold["Threshold"],
        "MCC_At_Locked_Threshold":
            best_threshold["MCC"],
        "Balanced_Accuracy_At_Locked_Threshold":
            best_threshold[
                "Balanced_Accuracy"
            ],
        "F1_At_Locked_Threshold":
            best_threshold["F1"],
        "Precision_At_Locked_Threshold":
            best_threshold["Precision"],
        "Recall_At_Locked_Threshold":
            best_threshold["Recall"],
        "Specificity_At_Locked_Threshold":
            best_threshold["Specificity"],
        "MCC_At_0_5":
            metrics_at_05["MCC"],
        "F1_At_0_5":
            metrics_at_05["F1"],
        "Recall_At_0_5":
            metrics_at_05["Recall"],
        "Specificity_At_0_5":
            metrics_at_05["Specificity"]
    })

    threshold_table["Model"] = model_name

    threshold_grid_records.append(
        threshold_table
    )

    selected_predictions[
        "Selected_Calibration"
    ] = selected_method

    selected_predictions[
        "Locked_MCC_Threshold"
    ] = best_threshold["Threshold"]

    selected_oof_records.append(
        selected_predictions
    )

    # Fit final calibrator on all development
    # single-repeat OOF predictions.
    final_calibrator = fit_calibrator(
        method=selected_method,
        raw_probability=raw_probability,
        y_true=y_model
    )

    calibrator_path = (
        OUTPUT_DIR
        / f"{model_name}_calibrator.joblib"
    )

    joblib.dump(
        {
            "model_name": model_name,
            "method": selected_method,
            "probability_epsilon":
                PROBABILITY_EPSILON,
            "calibrator": final_calibrator
        },
        calibrator_path
    )

    calibrator_manifest[model_name] = {
        "method": selected_method,
        "calibrator_path":
            str(calibrator_path),
        "locked_mcc_threshold":
            float(
                best_threshold["Threshold"]
            )
    }


threshold_plan = pd.DataFrame(
    threshold_records
)

selected_calibrated_oof = pd.concat(
    selected_oof_records,
    ignore_index=True
)

threshold_grid_all = pd.concat(
    threshold_grid_records,
    ignore_index=True
)


# ------------------------------------------------------------
# 9. Print results
# ------------------------------------------------------------

print("\n" + "=" * 125)
print("CROSS-FITTED CALIBRATION COMPARISON")
print("Lower Brier, LogLoss and ECE values are better.")
print("=" * 125)

print(
    calibration_summary[
        [
            "Model",
            "Calibration_Method",
            "ROC_AUC",
            "PR_AUC",
            "Brier_Score",
            "Log_Loss",
            "ECE_10",
            "Mean_Probability"
        ]
    ]
    .sort_values(
        [
            "Model",
            "Brier_Score"
        ]
    )
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 125)
print("SELECTED CALIBRATION METHODS")
print("=" * 125)

print(
    selected_methods
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 150)
print("LOCKED CALIBRATION AND THRESHOLD PLAN")
print("Thresholds maximize MCC on development cross-fitted calibrated probabilities.")
print("=" * 150)

print(
    threshold_plan
    .sort_values("Model")
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. Save artifacts
# ------------------------------------------------------------

integrity_table.to_csv(
    OUTPUT_DIR
    / "single_repeat_oof_integrity.csv",
    index=False
)

single_repeat_oof.to_csv(
    OUTPUT_DIR
    / "single_repeat_raw_oof_predictions.csv",
    index=False
)

calibration_summary.to_csv(
    OUTPUT_DIR
    / "cross_fitted_calibration_comparison.csv",
    index=False
)

calibrated_predictions.to_csv(
    OUTPUT_DIR
    / "all_cross_fitted_calibrated_predictions.csv",
    index=False
)

selected_methods.to_csv(
    OUTPUT_DIR
    / "selected_calibration_methods.csv",
    index=False
)

threshold_plan.to_csv(
    OUTPUT_DIR
    / "locked_calibration_threshold_plan.csv",
    index=False
)

selected_calibrated_oof.to_csv(
    OUTPUT_DIR
    / "selected_calibrated_oof_predictions.csv",
    index=False
)

threshold_grid_all.to_csv(
    OUTPUT_DIR
    / "all_threshold_search_grids.csv",
    index=False
)

with open(
    OUTPUT_DIR
    / "calibrator_threshold_manifest.json",
    "w"
) as f:
    json.dump(
        calibrator_manifest,
        f,
        indent=4
    )

print("\nSaved directory:")
print(OUTPUT_DIR)

In [ ]:
# ============================================================
# FIXED CONSERVATIVE CALIBRATION LOCK
#
# Fix:
# pandas may interpret the literal string "None" as NaN
# when loading CSV files.
#
# Development OOF only.
# Locked test set remains untouched.
# ============================================================

import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

INPUT_DIR = Path(
    "/kaggle/working/calibration_threshold"
)

OUTPUT_DIR = Path(
    "/kaggle/working/calibration_threshold_locked"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CALIBRATION_SEED = 2026
PROBABILITY_EPSILON = 1e-6

MODEL_ORDER = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

METHOD_COMPLEXITY = {
    "None": 0,
    "Sigmoid": 1,
    "Isotonic": 2
}


# ------------------------------------------------------------
# 2. Load Stage artifacts
#
# keep_default_na=False prevents the literal string
# "None" from being converted to NaN.
# ------------------------------------------------------------

calibration_summary = pd.read_csv(
    INPUT_DIR
    / "cross_fitted_calibration_comparison.csv",
    keep_default_na=False
)

calibrated_predictions = pd.read_csv(
    INPUT_DIR
    / "all_cross_fitted_calibrated_predictions.csv",
    keep_default_na=False
)


# Extra defensive cleaning in case the DataFrames
# were loaded previously with default NA conversion.
for dataframe in [
    calibration_summary,
    calibrated_predictions
]:
    dataframe["Calibration_Method"] = (
        dataframe["Calibration_Method"]
        .replace("", "None")
        .fillna("None")
        .astype(str)
    )


# ------------------------------------------------------------
# 3. Input integrity checks
# ------------------------------------------------------------

print("=" * 100)
print("CALIBRATION FILE INTEGRITY CHECK")
print("=" * 100)

print("\nCalibration methods found:")
print(
    calibration_summary[
        "Calibration_Method"
    ].value_counts(dropna=False)
)

integrity_records = []

for model_name in MODEL_ORDER:

    model_summary = calibration_summary[
        calibration_summary["Model"]
        == model_name
    ]

    model_predictions = calibrated_predictions[
        calibrated_predictions["Model"]
        == model_name
    ]

    none_summary_rows = (
        model_summary[
            "Calibration_Method"
        ] == "None"
    ).sum()

    none_prediction_rows = (
        model_predictions[
            "Calibration_Method"
        ] == "None"
    ).sum()

    integrity_records.append({
        "Model": model_name,
        "Calibration_Summary_Rows":
            len(model_summary),
        "Raw_None_Summary_Rows":
            int(none_summary_rows),
        "Prediction_Rows":
            len(model_predictions),
        "Raw_None_Prediction_Rows":
            int(none_prediction_rows)
    })

integrity_table = pd.DataFrame(
    integrity_records
)

print("\n" + integrity_table.to_string(index=False))

assert (
    integrity_table[
        "Calibration_Summary_Rows"
    ] == 3
).all(), (
    "Each model should have None, Sigmoid "
    "and Isotonic summary rows."
)

assert (
    integrity_table[
        "Raw_None_Summary_Rows"
    ] == 1
).all(), (
    'A model is missing its "None" summary row.'
)

assert (
    integrity_table[
        "Raw_None_Prediction_Rows"
    ] == len(X_dev)
).all(), (
    'A model is missing its "None" OOF predictions.'
)


# ------------------------------------------------------------
# 4. Conservative calibration selection
#
# A calibrated method is accepted only when it improves
# both Brier score and Log Loss versus raw probabilities.
# ------------------------------------------------------------

selection_records = []

for model_name in MODEL_ORDER:

    model_results = (
        calibration_summary[
            calibration_summary["Model"]
            == model_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    raw_candidates = model_results[
        model_results["Calibration_Method"]
        == "None"
    ]

    if len(raw_candidates) != 1:
        raise ValueError(
            f"{model_name}: expected exactly one "
            f'Calibration_Method="None" row, '
            f"found {len(raw_candidates)}."
        )

    raw_row = raw_candidates.iloc[0]

    raw_brier = float(
        raw_row["Brier_Score"]
    )

    raw_log_loss = float(
        raw_row["Log_Loss"]
    )

    eligible = model_results[
        (
            model_results[
                "Calibration_Method"
            ] != "None"
        )
        &
        (
            model_results[
                "Brier_Score"
            ] < raw_brier
        )
        &
        (
            model_results[
                "Log_Loss"
            ] < raw_log_loss
        )
    ].copy()

    if eligible.empty:

        selected_row = raw_row

        selection_reason = (
            "No calibrated method improved both "
            "Brier score and Log Loss."
        )

    else:

        eligible[
            "Method_Complexity"
        ] = (
            eligible[
                "Calibration_Method"
            ].map(METHOD_COMPLEXITY)
        )

        selected_row = (
            eligible
            .sort_values(
                [
                    "Brier_Score",
                    "Log_Loss",
                    "Method_Complexity"
                ],
                ascending=True
            )
            .iloc[0]
        )

        selection_reason = (
            "Improved both Brier score "
            "and Log Loss."
        )

    selection_records.append({
        "Model": model_name,
        "Locked_Calibration":
            selected_row[
                "Calibration_Method"
            ],
        "Raw_Brier":
            raw_brier,
        "Locked_Brier":
            float(
                selected_row[
                    "Brier_Score"
                ]
            ),
        "Delta_Brier":
            float(
                selected_row[
                    "Brier_Score"
                ] - raw_brier
            ),
        "Raw_Log_Loss":
            raw_log_loss,
        "Locked_Log_Loss":
            float(
                selected_row[
                    "Log_Loss"
                ]
            ),
        "Delta_Log_Loss":
            float(
                selected_row[
                    "Log_Loss"
                ] - raw_log_loss
            ),
        "Raw_ECE":
            float(
                raw_row["ECE_10"]
            ),
        "Locked_ECE":
            float(
                selected_row["ECE_10"]
            ),
        "Delta_ECE":
            float(
                selected_row["ECE_10"]
                - raw_row["ECE_10"]
            ),
        "Selection_Reason":
            selection_reason
    })


locked_methods = pd.DataFrame(
    selection_records
)


# ------------------------------------------------------------
# 5. Calibration utilities
# ------------------------------------------------------------

def clip_probabilities(probabilities):

    return np.clip(
        np.asarray(
            probabilities,
            dtype=float
        ),
        PROBABILITY_EPSILON,
        1 - PROBABILITY_EPSILON
    )


def probability_to_logit(probabilities):

    probabilities = clip_probabilities(
        probabilities
    )

    return np.log(
        probabilities
        / (1 - probabilities)
    )


def fit_final_calibrator(
    method,
    raw_probability,
    y_true
):

    raw_probability = clip_probabilities(
        raw_probability
    )

    if method == "None":
        return None

    if method == "Sigmoid":

        calibrator = LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=2000,
            random_state=CALIBRATION_SEED
        )

        calibrator.fit(
            probability_to_logit(
                raw_probability
            ).reshape(-1, 1),
            y_true
        )

        return calibrator

    if method == "Isotonic":

        calibrator = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            increasing=True,
            out_of_bounds="clip"
        )

        calibrator.fit(
            raw_probability,
            y_true
        )

        return calibrator

    raise ValueError(
        f"Unknown calibration method: {method}"
    )


# ------------------------------------------------------------
# 6. Threshold utilities
# ------------------------------------------------------------

def classification_metrics(
    y_true,
    y_probability,
    threshold
):

    y_prediction = (
        np.asarray(y_probability)
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "Threshold": float(threshold),

        "Accuracy": accuracy_score(
            y_true,
            y_prediction
        ),

        "Precision": precision_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "Specificity": specificity,

        "F1": f1_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_prediction
            ),

        "MCC": matthews_corrcoef(
            y_true,
            y_prediction
        ),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


def optimize_mcc_threshold(
    y_true,
    y_probability
):

    threshold_records = []

    for threshold in np.linspace(
        0.001,
        0.999,
        999
    ):

        threshold_records.append(
            classification_metrics(
                y_true=y_true,
                y_probability=y_probability,
                threshold=threshold
            )
        )

    threshold_table = pd.DataFrame(
        threshold_records
    )

    threshold_table[
        "Distance_From_0_5"
    ] = abs(
        threshold_table["Threshold"]
        - 0.50
    )

    best_row = (
        threshold_table
        .sort_values(
            [
                "MCC",
                "Balanced_Accuracy",
                "F1",
                "Distance_From_0_5"
            ],
            ascending=[
                False,
                False,
                False,
                True
            ]
        )
        .iloc[0]
    )

    return (
        best_row.to_dict(),
        threshold_table
    )


# ------------------------------------------------------------
# 7. Recompute thresholds using locked methods
# ------------------------------------------------------------

threshold_records = []
threshold_grid_records = []
locked_prediction_records = []

manifest = {}

for model_name in MODEL_ORDER:

    locked_method = (
        locked_methods.loc[
            locked_methods["Model"]
            == model_name,
            "Locked_Calibration"
        ]
        .iloc[0]
    )

    model_predictions = (
        calibrated_predictions[
            (
                calibrated_predictions[
                    "Model"
                ] == model_name
            )
            &
            (
                calibrated_predictions[
                    "Calibration_Method"
                ] == locked_method
            )
        ]
        .sort_values(
            "Development_Position"
        )
        .reset_index(drop=True)
    )

    if len(model_predictions) != len(X_dev):
        raise ValueError(
            f"{model_name}, {locked_method}: "
            f"expected {len(X_dev)} predictions, "
            f"found {len(model_predictions)}."
        )

    y_model = (
        model_predictions["y_true"]
        .astype(int)
        .to_numpy()
    )

    raw_probability = (
        model_predictions[
            "Raw_OOF_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    locked_probability = (
        model_predictions[
            "Calibrated_OOF_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    (
        best_threshold,
        threshold_table
    ) = optimize_mcc_threshold(
        y_true=y_model,
        y_probability=locked_probability
    )

    metrics_at_05 = (
        classification_metrics(
            y_true=y_model,
            y_probability=
                locked_probability,
            threshold=0.50
        )
    )

    threshold_records.append({
        "Model": model_name,

        "Locked_Calibration":
            locked_method,

        "Locked_MCC_Threshold":
            best_threshold[
                "Threshold"
            ],

        "MCC_At_Locked_Threshold":
            best_threshold["MCC"],

        "Balanced_Accuracy_At_Locked_Threshold":
            best_threshold[
                "Balanced_Accuracy"
            ],

        "F1_At_Locked_Threshold":
            best_threshold["F1"],

        "Precision_At_Locked_Threshold":
            best_threshold[
                "Precision"
            ],

        "Recall_At_Locked_Threshold":
            best_threshold["Recall"],

        "Specificity_At_Locked_Threshold":
            best_threshold[
                "Specificity"
            ],

        "MCC_At_0_5":
            metrics_at_05["MCC"],

        "F1_At_0_5":
            metrics_at_05["F1"],

        "Recall_At_0_5":
            metrics_at_05["Recall"],

        "Specificity_At_0_5":
            metrics_at_05[
                "Specificity"
            ]
    })

    threshold_table[
        "Model"
    ] = model_name

    threshold_grid_records.append(
        threshold_table
    )

    model_predictions[
        "Locked_Calibration"
    ] = locked_method

    model_predictions[
        "Locked_MCC_Threshold"
    ] = best_threshold[
        "Threshold"
    ]

    locked_prediction_records.append(
        model_predictions
    )

    final_calibrator = (
        fit_final_calibrator(
            method=locked_method,
            raw_probability=
                raw_probability,
            y_true=y_model
        )
    )

    calibrator_path = (
        OUTPUT_DIR
        / f"{model_name}_calibrator.joblib"
    )

    joblib.dump(
        {
            "model_name":
                model_name,

            "method":
                locked_method,

            "probability_epsilon":
                PROBABILITY_EPSILON,

            "calibrator":
                final_calibrator
        },
        calibrator_path
    )

    manifest[model_name] = {
        "calibration_method":
            locked_method,

        "locked_mcc_threshold":
            float(
                best_threshold[
                    "Threshold"
                ]
            ),

        "calibrator_path":
            str(calibrator_path)
    }


locked_threshold_plan = (
    pd.DataFrame(
        threshold_records
    )
    .sort_values("Model")
    .reset_index(drop=True)
)

locked_predictions = pd.concat(
    locked_prediction_records,
    ignore_index=True
)

threshold_grids = pd.concat(
    threshold_grid_records,
    ignore_index=True
)


# ------------------------------------------------------------
# 8. Expected-method verification
# ------------------------------------------------------------

expected_methods = {
    "LogisticRegression": "None",
    "RandomForest": "None",
    "XGBoost": "None",
    "LightGBM": "Sigmoid",
    "CatBoost": "Sigmoid",
    "MLP": "None",
    "TabNet": "None",
    "FTTransformer": "Sigmoid",
    "RealTabR": "Isotonic"
}

observed_methods = dict(
    zip(
        locked_methods["Model"],
        locked_methods[
            "Locked_Calibration"
        ]
    )
)


# ------------------------------------------------------------
# 9. Print results
# ------------------------------------------------------------

print("\n" + "=" * 125)
print("REVISED LOCKED CALIBRATION METHODS")
print("=" * 125)

print(
    locked_methods
    .round(6)
    .to_string(index=False)
)


print("\nExpected method check:")

all_matches = True

for model_name in MODEL_ORDER:

    expected = expected_methods[
        model_name
    ]

    observed = observed_methods[
        model_name
    ]

    match = observed == expected
    all_matches &= match

    print(
        f"{model_name}: "
        f"observed={observed}, "
        f"expected={expected}, "
        f"match={match}"
    )

assert all_matches, (
    "At least one calibration method "
    "does not match the expected result."
)


print("\n" + "=" * 150)
print(
    "REVISED LOCKED CALIBRATION "
    "AND THRESHOLD PLAN"
)
print("=" * 150)

print(
    locked_threshold_plan
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. Save corrected locked artifacts
# ------------------------------------------------------------

integrity_table.to_csv(
    OUTPUT_DIR
    / "calibration_file_integrity.csv",
    index=False
)

locked_methods.to_csv(
    OUTPUT_DIR
    / "locked_calibration_methods.csv",
    index=False
)

locked_threshold_plan.to_csv(
    OUTPUT_DIR
    / "locked_calibration_threshold_plan.csv",
    index=False
)

locked_predictions.to_csv(
    OUTPUT_DIR
    / "locked_calibrated_oof_predictions.csv",
    index=False
)

threshold_grids.to_csv(
    OUTPUT_DIR
    / "locked_threshold_search_grids.csv",
    index=False
)

with open(
    OUTPUT_DIR
    / "locked_calibration_threshold_manifest.json",
    "w"
) as file:

    json.dump(
        manifest,
        file,
        indent=4
    )


print("\nSaved directory:")
print(OUTPUT_DIR)


# ============================================================
# FINAL CALIBRATION-LOCK VERIFICATION
# ============================================================
# ------------------------------------------------------------
# 1. Correct expected calibration methods
# ------------------------------------------------------------

expected_methods = {
    "LogisticRegression": "None",
    "RandomForest": "Sigmoid",
    "XGBoost": "None",
    "LightGBM": "Sigmoid",
    "CatBoost": "Sigmoid",
    "MLP": "None",
    "TabNet": "None",
    "FTTransformer": "Sigmoid",
    "RealTabR": "Isotonic"
}

observed_methods = dict(
    zip(
        locked_methods["Model"],
        locked_methods["Locked_Calibration"]
    )
)


# ------------------------------------------------------------
# 2. Verify corrected expectation
# ------------------------------------------------------------

print("=" * 125)
print("CORRECTED EXPECTED METHOD CHECK")
print("=" * 125)

all_matches = True

for model_name in MODEL_ORDER:

    expected = expected_methods[model_name]
    observed = observed_methods[model_name]
    match = observed == expected

    all_matches &= match

    print(
        f"{model_name}: "
        f"observed={observed}, "
        f"expected={expected}, "
        f"match={match}"
    )

assert all_matches, (
    "At least one calibration method remains inconsistent."
)


# ------------------------------------------------------------
# 3. Print final locked calibration methods
# ------------------------------------------------------------

print("\n" + "=" * 125)
print("FINAL LOCKED CALIBRATION METHODS")
print("=" * 125)

print(
    locked_methods
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 4. Print final threshold plan
# ------------------------------------------------------------

print("\n" + "=" * 150)
print("FINAL LOCKED CALIBRATION AND THRESHOLD PLAN")
print("=" * 150)

print(
    locked_threshold_plan
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Save all artifacts that were computed before the error
# ------------------------------------------------------------

integrity_table.to_csv(
    OUTPUT_DIR
    / "calibration_file_integrity.csv",
    index=False
)

locked_methods.to_csv(
    OUTPUT_DIR
    / "locked_calibration_methods.csv",
    index=False
)

locked_threshold_plan.to_csv(
    OUTPUT_DIR
    / "locked_calibration_threshold_plan.csv",
    index=False
)

locked_predictions.to_csv(
    OUTPUT_DIR
    / "locked_calibrated_oof_predictions.csv",
    index=False
)

threshold_grids.to_csv(
    OUTPUT_DIR
    / "locked_threshold_search_grids.csv",
    index=False
)

with open(
    OUTPUT_DIR
    / "locked_calibration_threshold_manifest.json",
    "w"
) as file:

    json.dump(
        manifest,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 6. Confirm files exist
# ------------------------------------------------------------

required_files = [
    "calibration_file_integrity.csv",
    "locked_calibration_methods.csv",
    "locked_calibration_threshold_plan.csv",
    "locked_calibrated_oof_predictions.csv",
    "locked_threshold_search_grids.csv",
    "locked_calibration_threshold_manifest.json"
]

print("\n" + "=" * 100)
print("SAVED ARTIFACT CHECK")
print("=" * 100)

for filename in required_files:

    filepath = OUTPUT_DIR / filename

    print(
        f"{filename}: "
        f"exists={filepath.exists()}, "
        f"size={filepath.stat().st_size if filepath.exists() else 0} bytes"
    )

    assert filepath.exists()
    assert filepath.stat().st_size > 0


print("\nStage completed successfully.")
print("Saved directory:")
print(OUTPUT_DIR)


In [ ]:
# ============================================================
# FINAL MODEL TRAINING AND TEST PREDICTION
#
# IMPORTANT:
# - Hyperparameters are already locked.
# - Calibration methods are already locked.
# - Thresholds are already locked.
# - This cell does NOT access y_test.
# - It only trains final models and produces raw probabilities.
# ============================================================

import copy
import gc
import json
import joblib
import os
import random
import shutil
import threading
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from pytorch_tabnet.tab_model import TabNetClassifier
from pytabkit import (
    FTT_D_Classifier,
    RealTabR_D_Classifier
)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

GLOBAL_SEED = 42
DEEP_SPLIT_SEED = 2027
DEEP_VALIDATION_SIZE = 0.15
N_THREADS = 4

OUTPUT_DIR = Path(
    "/kaggle/working/final_locked_holdout"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR = OUTPUT_DIR / "model_artifacts"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEMP_DIR = OUTPUT_DIR / "temporary_files"
TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_METRICS_PATH = (
    OUTPUT_DIR
    / "final_locked_test_metrics.csv"
)

if FINAL_METRICS_PATH.exists():
    raise RuntimeError(
        "The locked test has already been evaluated. "
        "Do not overwrite the final results."
    )

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

EXPECTED_MODELS = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(N_THREADS)

try:
    torch.use_deterministic_algorithms(
        True,
        warn_only=True
    )
except Exception:
    pass


# ------------------------------------------------------------
# 2. Locked dataset integrity checks
# ------------------------------------------------------------

print("=" * 100)
print("LOCKED DATASET INTEGRITY")
print("=" * 100)

assert len(X_dev) == 8000
assert len(X_test) == 2000
assert len(y_dev) == 8000

assert set(X_dev.index).isdisjoint(
    set(X_test.index)
)

assert set(
    id_dev["CustomerId"]
).isdisjoint(
    set(id_test["CustomerId"])
)

assert list(X_dev.columns) == list(X_test.columns)

print("Development observations:", len(X_dev))
print("Test observations:", len(X_test))
print("Feature count:", X_dev.shape[1])
print("Index overlap: 0")
print("CustomerId overlap: 0")
print("Test labels accessed: NO")


# ------------------------------------------------------------
# 3. Load all locked artifacts
# ------------------------------------------------------------

with open(
    "/kaggle/working/classical_hpo/"
    "all_best_params.json",
    "r"
) as file:
    classical_params = json.load(file)

with open(
    "/kaggle/working/deep_hpo/"
    "all_deep_best_params.json",
    "r"
) as file:
    deep_params = json.load(file)

with open(
    "/kaggle/working/"
    "calibration_threshold_locked/"
    "locked_calibration_threshold_manifest.json",
    "r"
) as file:
    calibration_manifest = json.load(file)

assert set(calibration_manifest) == set(
    EXPECTED_MODELS
)

print("\nLocked parameter and calibration files loaded.")


# ------------------------------------------------------------
# 4. Fixed deep-learning validation split
# ------------------------------------------------------------

development_positions = np.arange(
    len(X_dev)
)

(
    deep_train_positions,
    deep_stop_positions
) = train_test_split(
    development_positions,
    test_size=DEEP_VALIDATION_SIZE,
    stratify=y_dev,
    random_state=DEEP_SPLIT_SEED
)

X_deep_train = (
    X_dev.iloc[deep_train_positions]
    .copy()
    .reset_index(drop=True)
)

y_deep_train = (
    y_dev.iloc[deep_train_positions]
    .astype(int)
    .to_numpy()
)

X_deep_stop = (
    X_dev.iloc[deep_stop_positions]
    .copy()
    .reset_index(drop=True)
)

y_deep_stop = (
    y_dev.iloc[deep_stop_positions]
    .astype(int)
    .to_numpy()
)

deep_split_manifest = pd.DataFrame({
    "Development_Position":
        np.concatenate([
            deep_train_positions,
            deep_stop_positions
        ]),
    "Original_Index":
        np.concatenate([
            X_dev.iloc[
                deep_train_positions
            ].index.to_numpy(),
            X_dev.iloc[
                deep_stop_positions
            ].index.to_numpy()
        ]),
    "Role":
        (
            ["Final_Deep_Train"]
            * len(deep_train_positions)
            +
            ["Final_Deep_Early_Stop"]
            * len(deep_stop_positions)
        )
})

deep_split_manifest.to_csv(
    OUTPUT_DIR
    / "final_deep_split_manifest.csv",
    index=False
)

print("\n" + "=" * 100)
print("FINAL DEEP MODEL SPLIT")
print("=" * 100)

print("Deep training rows:", len(X_deep_train))
print("Early-stopping rows:", len(X_deep_stop))

print(
    "Training prevalence:",
    round(y_deep_train.mean(), 6)
)

print(
    "Early-stopping prevalence:",
    round(y_deep_stop.mean(), 6)
)


# ------------------------------------------------------------
# 5. Preprocessing factories
# ------------------------------------------------------------

def make_linear_preprocessor():

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )


def make_tree_preprocessor():

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                "passthrough",
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )


# ------------------------------------------------------------
# 6. Direct peak-RSS monitoring for the original final runs
# ------------------------------------------------------------

class RSSPeakMonitor:
    """Sample process-tree resident memory during one operation."""

    def __init__(self, sample_interval_seconds=0.002):
        self.sample_interval_seconds = sample_interval_seconds
        self._stop_event = threading.Event()
        self._thread = None
        self.baseline_bytes = 0
        self.peak_bytes = 0

    @staticmethod
    def _process_tree_rss_bytes():
        process = psutil.Process(os.getpid())
        total = 0

        try:
            total += process.memory_info().rss
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

        try:
            children = process.children(recursive=True)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            children = []

        for child in children:
            try:
                total += child.memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass

        return total

    def start(self):
        gc.collect()
        self._stop_event.clear()
        self.baseline_bytes = self._process_tree_rss_bytes()
        self.peak_bytes = self.baseline_bytes

        def _sample():
            while not self._stop_event.is_set():
                rss = self._process_tree_rss_bytes()
                if rss > self.peak_bytes:
                    self.peak_bytes = rss
                time.sleep(self.sample_interval_seconds)

        self._thread = threading.Thread(
            target=_sample,
            daemon=True
        )
        self._thread.start()
        return self

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join()

        self.peak_bytes = max(
            self.peak_bytes,
            self._process_tree_rss_bytes()
        )

        scale = 1024 ** 2

        return {
            "Baseline_RSS_MB":
                self.baseline_bytes / scale,
            "Peak_RSS_MB":
                self.peak_bytes / scale,
            "Incremental_Peak_RSS_MB":
                max(
                    0,
                    self.peak_bytes
                    - self.baseline_bytes
                ) / scale,
            "Sampling_Interval_ms":
                self.sample_interval_seconds * 1000.0
        }


# ------------------------------------------------------------
# 7. Result containers
# ------------------------------------------------------------

raw_test_probabilities = {}
training_records = []
artifact_records = []


def validate_probability_vector(
    model_name,
    probabilities
):

    probabilities = np.asarray(
        probabilities,
        dtype=float
    ).reshape(-1)

    if len(probabilities) != len(X_test):
        raise ValueError(
            f"{model_name}: expected "
            f"{len(X_test)} probabilities, "
            f"received {len(probabilities)}."
        )

    if not np.isfinite(
        probabilities
    ).all():
        raise ValueError(
            f"{model_name}: non-finite probability."
        )

    if (
        probabilities.min() < -1e-7
        or probabilities.max() > 1 + 1e-7
    ):
        raise ValueError(
            f"{model_name}: probabilities "
            f"outside [0, 1]."
        )

    return np.clip(
        probabilities,
        1e-7,
        1 - 1e-7
    )


def save_artifact_safely(
    model_name,
    object_to_save,
    filepath
):

    try:
        joblib.dump(
            object_to_save,
            filepath
        )

        artifact_records.append({
            "Model": model_name,
            "Artifact_Path": str(filepath),
            "Save_Status": "SUCCESS",
            "Error": None
        })

    except Exception as exc:

        artifact_records.append({
            "Model": model_name,
            "Artifact_Path": str(filepath),
            "Save_Status": "FAILED",
            "Error": repr(exc)
        })


def register_result(
    model_name,
    probabilities,
    fit_time,
    prediction_time,
    training_rows,
    early_stopping_rows=0,
    completed_epochs=np.nan,
    parameter_count=np.nan
):

    probabilities = (
        validate_probability_vector(
            model_name,
            probabilities
        )
    )

    raw_test_probabilities[
        model_name
    ] = probabilities

    training_records.append({
        "Model": model_name,
        "Training_Rows": training_rows,
        "Early_Stopping_Rows":
            early_stopping_rows,
        "Completed_Epochs":
            completed_epochs,
        "Parameter_Count":
            parameter_count,
        "Fit_Time_Seconds":
            fit_time,
        "Prediction_Time_Seconds":
            prediction_time
    })

    print(
        f"{model_name}: SUCCESS | "
        f"fit={fit_time:.2f}s | "
        f"predict={prediction_time:.4f}s | "
        f"mean probability="
        f"{probabilities.mean():.6f}"
    )


# ============================================================
# 7. LOGISTIC REGRESSION
# ============================================================

print("\n" + "=" * 100)
print("FINAL TRAINING: LogisticRegression")
print("=" * 100)

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

logistic_preprocessor = (
    make_linear_preprocessor()
)

X_dev_processed = (
    logistic_preprocessor
    .fit_transform(X_dev)
)

logistic_model = LogisticRegression(
    solver="liblinear",
    max_iter=5000,
    class_weight=None,
    random_state=GLOBAL_SEED,
    **classical_params[
        "LogisticRegression"
    ]
)

logistic_model.fit(
    X_dev_processed,
    y_dev
)

logistic_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

X_test_processed = (
    logistic_preprocessor
    .transform(X_test)
)

logistic_probability = (
    logistic_model
    .predict_proba(
        X_test_processed
    )[:, 1]
)

logistic_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="LogisticRegression",
    probabilities=logistic_probability,
    fit_time=logistic_fit_time,
    prediction_time=
        logistic_predict_time,
    training_rows=len(X_dev)
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


save_artifact_safely(
    "LogisticRegression",
    {
        "preprocessor":
            logistic_preprocessor,
        "model":
            logistic_model
    },
    MODEL_DIR
    / "LogisticRegression.joblib"
)

del (
    X_dev_processed,
    X_test_processed,
    logistic_model,
    logistic_preprocessor
)

gc.collect()


# ============================================================
# 8. RANDOM FOREST
# ============================================================

print("\n" + "=" * 100)
print("FINAL TRAINING: RandomForest")
print("=" * 100)

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

rf_preprocessor = make_tree_preprocessor()

X_dev_processed = (
    rf_preprocessor
    .fit_transform(X_dev)
)

rf_model = RandomForestClassifier(
    bootstrap=True,
    class_weight=None,
    n_jobs=N_THREADS,
    random_state=GLOBAL_SEED,
    **classical_params[
        "RandomForest"
    ]
)

rf_model.fit(
    X_dev_processed,
    y_dev
)

rf_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

X_test_processed = (
    rf_preprocessor
    .transform(X_test)
)

rf_probability = (
    rf_model
    .predict_proba(
        X_test_processed
    )[:, 1]
)

rf_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="RandomForest",
    probabilities=rf_probability,
    fit_time=rf_fit_time,
    prediction_time=rf_predict_time,
    training_rows=len(X_dev)
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


save_artifact_safely(
    "RandomForest",
    {
        "preprocessor":
            rf_preprocessor,
        "model":
            rf_model
    },
    MODEL_DIR
    / "RandomForest.joblib"
)

del (
    X_dev_processed,
    X_test_processed,
    rf_model,
    rf_preprocessor
)

gc.collect()


# ============================================================
# 9. XGBOOST
# ============================================================

print("\n" + "=" * 100)
print("FINAL TRAINING: XGBoost")
print("=" * 100)

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

xgb_preprocessor = make_tree_preprocessor()

X_dev_processed = (
    xgb_preprocessor
    .fit_transform(X_dev)
)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=1.0,
    n_jobs=N_THREADS,
    random_state=GLOBAL_SEED,
    verbosity=0,
    **classical_params["XGBoost"]
)

xgb_model.fit(
    X_dev_processed,
    y_dev
)

xgb_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

X_test_processed = (
    xgb_preprocessor
    .transform(X_test)
)

xgb_probability = (
    xgb_model
    .predict_proba(
        X_test_processed
    )[:, 1]
)

xgb_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="XGBoost",
    probabilities=xgb_probability,
    fit_time=xgb_fit_time,
    prediction_time=xgb_predict_time,
    training_rows=len(X_dev)
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


save_artifact_safely(
    "XGBoost",
    {
        "preprocessor":
            xgb_preprocessor,
        "model":
            xgb_model
    },
    MODEL_DIR / "XGBoost.joblib"
)

del (
    X_dev_processed,
    X_test_processed,
    xgb_model,
    xgb_preprocessor
)

gc.collect()


# ============================================================
# 10. LIGHTGBM
# ============================================================

print("\n" + "=" * 100)
print("FINAL TRAINING: LightGBM")
print("=" * 100)

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

lgbm_preprocessor = (
    make_tree_preprocessor()
)

X_dev_processed = (
    lgbm_preprocessor
    .fit_transform(X_dev)
)

lgbm_model = LGBMClassifier(
    objective="binary",
    class_weight=None,
    subsample_freq=1,
    n_jobs=N_THREADS,
    random_state=GLOBAL_SEED,
    verbosity=-1,
    **classical_params["LightGBM"]
)

lgbm_model.fit(
    X_dev_processed,
    y_dev
)

lgbm_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

X_test_processed = (
    lgbm_preprocessor
    .transform(X_test)
)

lgbm_probability = (
    lgbm_model
    .predict_proba(
        X_test_processed
    )[:, 1]
)

lgbm_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="LightGBM",
    probabilities=lgbm_probability,
    fit_time=lgbm_fit_time,
    prediction_time=lgbm_predict_time,
    training_rows=len(X_dev)
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


save_artifact_safely(
    "LightGBM",
    {
        "preprocessor":
            lgbm_preprocessor,
        "model":
            lgbm_model
    },
    MODEL_DIR / "LightGBM.joblib"
)

del (
    X_dev_processed,
    X_test_processed,
    lgbm_model,
    lgbm_preprocessor
)

gc.collect()


# ============================================================
# 11. CATBOOST
# ============================================================

print("\n" + "=" * 100)
print("FINAL TRAINING: CatBoost")
print("=" * 100)

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

catboost_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    bootstrap_type="Bayesian",
    auto_class_weights=None,
    random_seed=GLOBAL_SEED,
    thread_count=N_THREADS,
    allow_writing_files=False,
    verbose=False,
    **classical_params["CatBoost"]
)

catboost_model.fit(
    X_dev,
    y_dev,
    cat_features=CATEGORICAL_FEATURES,
    verbose=False
)

catboost_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

catboost_probability = (
    catboost_model
    .predict_proba(X_test)[:, 1]
)

catboost_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="CatBoost",
    probabilities=catboost_probability,
    fit_time=catboost_fit_time,
    prediction_time=
        catboost_predict_time,
    training_rows=len(X_dev)
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


save_artifact_safely(
    "CatBoost",
    catboost_model,
    MODEL_DIR / "CatBoost.joblib"
)

del catboost_model
gc.collect()


# ============================================================
# 12. FINAL MLP
# ============================================================

class FinalMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout,
        use_batch_norm,
        activation_name
    ):
        super().__init__()

        activation_classes = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "silu": nn.SiLU
        }

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:

            layers.append(
                nn.Linear(
                    previous_dim,
                    hidden_dim
                )
            )

            if use_batch_norm:
                layers.append(
                    nn.BatchNorm1d(
                        hidden_dim
                    )
                )

            layers.append(
                activation_classes[
                    activation_name
                ]()
            )

            if dropout > 0:
                layers.append(
                    nn.Dropout(dropout)
                )

            previous_dim = hidden_dim

        layers.append(
            nn.Linear(previous_dim, 1)
        )

        self.network = nn.Sequential(
            *layers
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


print("\n" + "=" * 100)
print("FINAL TRAINING: MLP")
print("=" * 100)

mlp_params = deep_params["MLP"]

mlp_hidden_dims = tuple(
    int(value)
    for value in mlp_params[
        "hidden_structure"
    ].split("-")
)

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

mlp_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop"
)

X_train_mlp = (
    mlp_preprocessor
    .fit_transform(X_deep_train)
    .astype(np.float32)
)

X_stop_mlp = (
    mlp_preprocessor
    .transform(X_deep_stop)
    .astype(np.float32)
)

X_test_mlp = (
    mlp_preprocessor
    .transform(X_test)
    .astype(np.float32)
)

mlp_dataset = TensorDataset(
    torch.from_numpy(X_train_mlp),
    torch.from_numpy(
        y_deep_train.astype(np.float32)
    )
)

mlp_generator = torch.Generator()
mlp_generator.manual_seed(
    DEEP_SPLIT_SEED
)

mlp_loader = DataLoader(
    mlp_dataset,
    batch_size=mlp_params[
        "batch_size"
    ],
    shuffle=True,
    num_workers=0,
    generator=mlp_generator,
    drop_last=False
)

mlp_model = FinalMLP(
    input_dim=X_train_mlp.shape[1],
    hidden_dims=mlp_hidden_dims,
    dropout=mlp_params["dropout"],
    use_batch_norm=mlp_params[
        "use_batch_norm"
    ],
    activation_name=mlp_params[
        "activation"
    ]
)

mlp_parameter_count = sum(
    parameter.numel()
    for parameter
    in mlp_model.parameters()
    if parameter.requires_grad
)

mlp_optimizer = torch.optim.AdamW(
    mlp_model.parameters(),
    lr=mlp_params[
        "learning_rate"
    ],
    weight_decay=mlp_params[
        "weight_decay"
    ]
)

mlp_criterion = nn.BCEWithLogitsLoss()

X_stop_mlp_tensor = torch.from_numpy(
    X_stop_mlp
)

best_mlp_auc = -np.inf
best_mlp_state = None
mlp_no_improvement = 0
mlp_completed_epochs = 0

for epoch in range(1, 81):

    mlp_model.train()

    for batch_X, batch_y in mlp_loader:

        mlp_optimizer.zero_grad()

        logits = mlp_model(batch_X)

        loss = mlp_criterion(
            logits,
            batch_y
        )

        loss.backward()
        mlp_optimizer.step()

    mlp_model.eval()

    with torch.no_grad():

        stop_probability = torch.sigmoid(
            mlp_model(
                X_stop_mlp_tensor
            )
        ).cpu().numpy()

    from sklearn.metrics import roc_auc_score

    stop_auc = roc_auc_score(
        y_deep_stop,
        stop_probability
    )

    mlp_completed_epochs = epoch

    if stop_auc > best_mlp_auc + 1e-5:

        best_mlp_auc = stop_auc

        best_mlp_state = copy.deepcopy(
            mlp_model.state_dict()
        )

        mlp_no_improvement = 0

    else:
        mlp_no_improvement += 1

    if mlp_no_improvement >= 10:
        break

mlp_model.load_state_dict(
    best_mlp_state
)

mlp_model.eval()

mlp_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

with torch.no_grad():

    mlp_probability = torch.sigmoid(
        mlp_model(
            torch.from_numpy(
                X_test_mlp
            )
        )
    ).cpu().numpy()

mlp_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="MLP",
    probabilities=mlp_probability,
    fit_time=mlp_fit_time,
    prediction_time=mlp_predict_time,
    training_rows=len(X_deep_train),
    early_stopping_rows=
        len(X_deep_stop),
    completed_epochs=
        mlp_completed_epochs,
    parameter_count=
        mlp_parameter_count
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


torch.save(
    {
        "state_dict":
            mlp_model.state_dict(),
        "input_dim":
            X_train_mlp.shape[1],
        "hidden_dims":
            mlp_hidden_dims,
        "dropout":
            mlp_params["dropout"],
        "use_batch_norm":
            mlp_params[
                "use_batch_norm"
            ],
        "activation":
            mlp_params["activation"],
        "completed_epochs":
            mlp_completed_epochs,
        "best_early_stopping_auc":
            best_mlp_auc
    },
    MODEL_DIR / "MLP_state.pt"
)

save_artifact_safely(
    "MLP_Preprocessor",
    mlp_preprocessor,
    MODEL_DIR
    / "MLP_preprocessor.joblib"
)

del (
    mlp_model,
    mlp_optimizer,
    mlp_dataset,
    mlp_loader,
    X_train_mlp,
    X_stop_mlp,
    X_test_mlp
)

gc.collect()


# ============================================================
# 13. FINAL TABNET
# ============================================================

print("\n" + "=" * 100)
print("FINAL TRAINING: TabNet")
print("=" * 100)

tabnet_params = deep_params[
    "TabNet"
]

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

tabnet_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES
            ),
            (
                "categorical",
                OrdinalEncoder(
                    handle_unknown=
                        "use_encoded_value",
                    unknown_value=-1
                ),
                CATEGORICAL_FEATURES
            )
        ],
        remainder="drop"
    )
)

X_train_tabnet = (
    tabnet_preprocessor
    .fit_transform(X_deep_train)
    .astype(np.float32)
)

X_stop_tabnet = (
    tabnet_preprocessor
    .transform(X_deep_stop)
    .astype(np.float32)
)

X_test_tabnet = (
    tabnet_preprocessor
    .transform(X_test)
    .astype(np.float32)
)

numeric_count = len(
    NUMERIC_FEATURES
)

tabnet_cat_idxs = list(
    range(
        numeric_count,
        numeric_count
        + len(CATEGORICAL_FEATURES)
    )
)

tabnet_encoder = (
    tabnet_preprocessor
    .named_transformers_[
        "categorical"
    ]
)

X_train_tabnet[
    :, tabnet_cat_idxs
] += 1

X_stop_tabnet[
    :, tabnet_cat_idxs
] += 1

X_test_tabnet[
    :, tabnet_cat_idxs
] += 1

tabnet_cat_dims = [
    len(categories) + 1
    for categories
    in tabnet_encoder.categories_
]

tabnet_model = TabNetClassifier(
    n_d=tabnet_params["n_d"],
    n_a=tabnet_params["n_d"],
    n_steps=tabnet_params[
        "n_steps"
    ],
    gamma=tabnet_params["gamma"],
    cat_idxs=tabnet_cat_idxs,
    cat_dims=tabnet_cat_dims,
    cat_emb_dim=tabnet_params[
        "cat_emb_dim"
    ],
    n_independent=tabnet_params[
        "n_independent"
    ],
    n_shared=tabnet_params[
        "n_shared"
    ],
    lambda_sparse=tabnet_params[
        "lambda_sparse"
    ],
    seed=DEEP_SPLIT_SEED,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={
        "lr": tabnet_params[
            "learning_rate"
        ]
    },
    mask_type="sparsemax",
    device_name="cpu",
    verbose=0
)

tabnet_model.fit(
    X_train=X_train_tabnet,
    y_train=y_deep_train,
    eval_set=[
        (
            X_stop_tabnet,
            y_deep_stop
        )
    ],
    eval_name=["early_stopping"],
    eval_metric=["auc"],
    max_epochs=100,
    patience=12,
    batch_size=tabnet_params[
        "batch_size"
    ],
    virtual_batch_size=max(
        16,
        tabnet_params[
            "batch_size"
        ] // 4
    ),
    num_workers=0,
    drop_last=False,
    pin_memory=False,
    compute_importance=True
)

tabnet_completed_epochs = len(
    tabnet_model.history["loss"]
)

tabnet_parameter_count = sum(
    parameter.numel()
    for parameter
    in tabnet_model.network.parameters()
    if parameter.requires_grad
)

tabnet_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

tabnet_probability = (
    tabnet_model
    .predict_proba(
        X_test_tabnet
    )[:, 1]
)

tabnet_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="TabNet",
    probabilities=tabnet_probability,
    fit_time=tabnet_fit_time,
    prediction_time=
        tabnet_predict_time,
    training_rows=len(X_deep_train),
    early_stopping_rows=
        len(X_deep_stop),
    completed_epochs=
        tabnet_completed_epochs,
    parameter_count=
        tabnet_parameter_count
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


save_artifact_safely(
    "TabNet_Preprocessor",
    tabnet_preprocessor,
    MODEL_DIR
    / "TabNet_preprocessor.joblib"
)

try:
    tabnet_model.save_model(
        str(
            MODEL_DIR
            / "TabNet_model"
        )
    )

    artifact_records.append({
        "Model": "TabNet",
        "Artifact_Path":
            str(
                MODEL_DIR
                / "TabNet_model.zip"
            ),
        "Save_Status": "SUCCESS",
        "Error": None
    })

except Exception as exc:

    artifact_records.append({
        "Model": "TabNet",
        "Artifact_Path":
            str(
                MODEL_DIR
                / "TabNet_model.zip"
            ),
        "Save_Status": "FAILED",
        "Error": repr(exc)
    })

del (
    tabnet_model,
    tabnet_preprocessor,
    X_train_tabnet,
    X_stop_tabnet,
    X_test_tabnet
)

gc.collect()


# ============================================================
# 14. FINAL FT-TRANSFORMER
# ============================================================

print("\n" + "=" * 100)
print("FINAL TRAINING: FTTransformer")
print("=" * 100)

ftt_params = deep_params[
    "FTTransformer"
]

ftt_temp = (
    TEMP_DIR
    / "FTTransformer"
)

ftt_temp.mkdir(
    parents=True,
    exist_ok=True
)

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

ftt_model = FTT_D_Classifier(
    **ftt_params,
    max_epochs=60,
    es_patience=8,
    lr_scheduler=True,
    lr_patience=4,
    use_checkpoints=False,
    device="cpu",
    random_state=DEEP_SPLIT_SEED,
    n_cv=1,
    n_refit=0,
    n_repeats=1,
    val_fraction=0.20,
    n_threads=N_THREADS,
    tmp_folder=ftt_temp,
    verbose=0,
    verbosity=0
)

ftt_model.fit(
    X_deep_train,
    y_deep_train,
    X_val=X_deep_stop,
    y_val=y_deep_stop,
    cat_col_names=
        CATEGORICAL_FEATURES,
    time_to_fit_in_seconds=900
)

ftt_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

ftt_probability = (
    ftt_model
    .predict_proba(X_test)[:, 1]
)

ftt_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="FTTransformer",
    probabilities=ftt_probability,
    fit_time=ftt_fit_time,
    prediction_time=ftt_predict_time,
    training_rows=len(X_deep_train),
    early_stopping_rows=
        len(X_deep_stop)
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


save_artifact_safely(
    "FTTransformer",
    ftt_model,
    MODEL_DIR
    / "FTTransformer.joblib"
)

del ftt_model

shutil.rmtree(
    ftt_temp,
    ignore_errors=True
)

gc.collect()


# ============================================================
# 15. FINAL REALTABR
# ============================================================

print("\n" + "=" * 100)
print("FINAL TRAINING: RealTabR")
print("=" * 100)

realtabr_params = deep_params[
    "RealTabR"
]

realtabr_temp = (
    TEMP_DIR / "RealTabR"
)

realtabr_temp.mkdir(
    parents=True,
    exist_ok=True
)

fit_memory_monitor = RSSPeakMonitor().start()
fit_start = time.perf_counter()

realtabr_model = RealTabR_D_Classifier(
    **realtabr_params,
    n_epochs=40,
    eval_batch_size=512,
    candidate_encoding_batch_size=512,
    patience=6,
    memory_efficient=True,
    device="cpu",
    random_state=DEEP_SPLIT_SEED,
    n_cv=1,
    n_refit=0,
    n_repeats=1,
    val_fraction=0.20,
    n_threads=N_THREADS,
    tmp_folder=realtabr_temp,
    verbosity=0
)

realtabr_model.fit(
    X_deep_train,
    y_deep_train,
    X_val=X_deep_stop,
    y_val=y_deep_stop,
    cat_col_names=
        CATEGORICAL_FEATURES,
    time_to_fit_in_seconds=900
)

realtabr_fit_time = (
    time.perf_counter()
    - fit_start
)

fit_memory = fit_memory_monitor.stop()

prediction_memory_monitor = RSSPeakMonitor().start()
predict_start = time.perf_counter()

realtabr_probability = (
    realtabr_model
    .predict_proba(X_test)[:, 1]
)

realtabr_predict_time = (
    time.perf_counter()
    - predict_start
)

prediction_memory = prediction_memory_monitor.stop()

register_result(
    model_name="RealTabR",
    probabilities=
        realtabr_probability,
    fit_time=realtabr_fit_time,
    prediction_time=
        realtabr_predict_time,
    training_rows=len(X_deep_train),
    early_stopping_rows=
        len(X_deep_stop)
)

training_records[-1].update({
    "Training_Baseline_RSS_MB":
        fit_memory["Baseline_RSS_MB"],
    "Training_Peak_RSS_MB":
        fit_memory["Peak_RSS_MB"],
    "Training_Incremental_Peak_RSS_MB":
        fit_memory["Incremental_Peak_RSS_MB"],
    "Inference_Baseline_RSS_MB":
        prediction_memory["Baseline_RSS_MB"],
    "Inference_Peak_RSS_MB":
        prediction_memory["Peak_RSS_MB"],
    "Inference_Incremental_Peak_RSS_MB":
        prediction_memory["Incremental_Peak_RSS_MB"],
    "Memory_Sampling_Interval_ms":
        fit_memory["Sampling_Interval_ms"]
})


save_artifact_safely(
    "RealTabR",
    realtabr_model,
    MODEL_DIR / "RealTabR.joblib"
)

# Keep the exact final RealTabR estimator in memory until the
# supplementary SHAP stage. Its fitted object is not reliably
# serializable, so explaining this in-memory instance avoids
# post-hoc reconstruction drift.
gc.collect()


# ------------------------------------------------------------
# 16. Final prediction integrity
# ------------------------------------------------------------

assert set(
    raw_test_probabilities
) == set(EXPECTED_MODELS)

training_summary = (
    pd.DataFrame(training_records)
    .sort_values(
        "Fit_Time_Seconds"
    )
    .reset_index(drop=True)
)

artifact_summary = pd.DataFrame(
    artifact_records
)

prediction_integrity_records = []

for model_name in EXPECTED_MODELS:

    probabilities = (
        raw_test_probabilities[
            model_name
        ]
    )

    prediction_integrity_records.append({
        "Model": model_name,
        "Rows": len(probabilities),
        "Finite_Count":
            int(
                np.isfinite(
                    probabilities
                ).sum()
            ),
        "Probability_Min":
            probabilities.min(),
        "Probability_Max":
            probabilities.max(),
        "Probability_Mean":
            probabilities.mean(),
        "Probability_SD":
            probabilities.std(),
        "Unique_Probabilities":
            np.unique(
                probabilities
            ).size
    })

prediction_integrity = pd.DataFrame(
    prediction_integrity_records
)


# ------------------------------------------------------------
# 17. Save raw test probabilities
# No y_test column is included.
# ------------------------------------------------------------

raw_prediction_table = pd.DataFrame({
    "Original_Index":
        X_test.index.to_numpy(),
    "CustomerId":
        id_test["CustomerId"]
        .to_numpy()
})

for model_name in EXPECTED_MODELS:

    raw_prediction_table[
        f"{model_name}_Raw_Probability"
    ] = raw_test_probabilities[
        model_name
    ]

raw_prediction_table.to_csv(
    OUTPUT_DIR
    / "final_test_raw_probabilities.csv",
    index=False
)

training_summary.to_csv(
    OUTPUT_DIR
    / "final_model_training_summary.csv",
    index=False
)

prediction_integrity.to_csv(
    OUTPUT_DIR
    / "final_test_prediction_integrity.csv",
    index=False
)

artifact_summary.to_csv(
    OUTPUT_DIR
    / "final_model_artifact_status.csv",
    index=False
)


# ------------------------------------------------------------
# 18. Print final pre-evaluation checks
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("FINAL MODEL TRAINING SUMMARY")
print("These are training diagnostics, not final performance results.")
print("=" * 120)

print(
    training_summary
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 120)
print("TEST PREDICTION INTEGRITY")
print("No test labels have been used.")
print("=" * 120)

print(
    prediction_integrity
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 120)
print("MODEL ARTIFACT SAVE STATUS")
print("=" * 120)

print(
    artifact_summary
    .to_string(index=False)
)


print("\nSaved directory:")
print(OUTPUT_DIR)

print("\nLocked test labels accessed: NO")
print("Stage completed successfully.")

In [ ]:
# ============================================================
# ONE-TIME LOCKED TEST EVALUATION
#
# This is the first and only evaluation using y_test.
# No model, calibrator, hyperparameter, or threshold
# may be changed after examining these results.
# ============================================================

import hashlib
import json
import joblib

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Paths and lock
# ------------------------------------------------------------

INPUT_DIR = Path(
    "/kaggle/working/final_locked_holdout"
)

CALIBRATION_DIR = Path(
    "/kaggle/working/calibration_threshold_locked"
)

OUTPUT_DIR = INPUT_DIR

RAW_PREDICTION_PATH = (
    INPUT_DIR
    / "final_test_raw_probabilities.csv"
)

FINAL_METRICS_PATH = (
    OUTPUT_DIR
    / "final_locked_test_metrics.csv"
)

FINAL_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "final_locked_test_predictions.csv"
)

CALIBRATION_EFFECT_PATH = (
    OUTPUT_DIR
    / "final_test_calibration_effects.csv"
)

EVALUATION_LOCK_PATH = (
    OUTPUT_DIR
    / "test_evaluation_lock.json"
)

if FINAL_METRICS_PATH.exists():
    raise RuntimeError(
        "Final locked-test metrics already exist. "
        "Do not evaluate the test set again."
    )

if EVALUATION_LOCK_PATH.exists():
    raise RuntimeError(
        "A locked-test evaluation record already exists."
    )


MODEL_ORDER = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

PROBABILITY_EPSILON = 1e-6
TOP_FRACTION = 0.20


# ------------------------------------------------------------
# 2. Load and verify raw test predictions
# ------------------------------------------------------------

raw_predictions = pd.read_csv(
    RAW_PREDICTION_PATH
)

expected_probability_columns = [
    f"{model_name}_Raw_Probability"
    for model_name in MODEL_ORDER
]

required_columns = [
    "Original_Index",
    "CustomerId",
    *expected_probability_columns
]

missing_columns = [
    column
    for column in required_columns
    if column not in raw_predictions.columns
]

if missing_columns:
    raise ValueError(
        f"Missing prediction columns: {missing_columns}"
    )

assert len(raw_predictions) == len(X_test) == 2000

assert np.array_equal(
    raw_predictions[
        "Original_Index"
    ].to_numpy(),
    X_test.index.to_numpy()
), "Raw prediction order does not match X_test."

assert np.array_equal(
    raw_predictions[
        "CustomerId"
    ].to_numpy(),
    id_test[
        "CustomerId"
    ].to_numpy()
), "CustomerId order does not match id_test."

for column in expected_probability_columns:

    probability = raw_predictions[
        column
    ].to_numpy(dtype=float)

    assert len(probability) == 2000
    assert np.isfinite(probability).all()
    assert probability.min() >= 0.0
    assert probability.max() <= 1.0


print("=" * 105)
print("FINAL TEST PREDICTION ORDER CHECK")
print("=" * 105)

print("Rows:", len(raw_predictions))
print("Original index order matches:", True)
print("CustomerId order matches:", True)
print("All probability vectors valid:", True)


# ------------------------------------------------------------
# 3. Load locked calibration and threshold manifest
# ------------------------------------------------------------

with open(
    CALIBRATION_DIR
    / "locked_calibration_threshold_manifest.json",
    "r"
) as file:
    locked_manifest = json.load(file)

assert set(locked_manifest) == set(
    MODEL_ORDER
)

print("\n" + "=" * 105)
print("LOCKED DECISION MANIFEST")
print("=" * 105)

for model_name in MODEL_ORDER:

    model_manifest = locked_manifest[
        model_name
    ]

    print(
        f"{model_name}: "
        f"calibration="
        f"{model_manifest['calibration_method']}, "
        f"threshold="
        f"{model_manifest['locked_mcc_threshold']:.3f}"
    )


# ------------------------------------------------------------
# 4. Calibration utilities
# ------------------------------------------------------------

def clip_probabilities(probabilities):

    return np.clip(
        np.asarray(
            probabilities,
            dtype=float
        ),
        PROBABILITY_EPSILON,
        1 - PROBABILITY_EPSILON
    )


def probability_to_logit(probabilities):

    probabilities = clip_probabilities(
        probabilities
    )

    return np.log(
        probabilities
        / (1.0 - probabilities)
    )


def apply_locked_calibrator(
    method,
    calibrator,
    raw_probability
):

    raw_probability = clip_probabilities(
        raw_probability
    )

    if method == "None":
        return raw_probability

    if method == "Sigmoid":

        calibrated = calibrator.predict_proba(
            probability_to_logit(
                raw_probability
            ).reshape(-1, 1)
        )[:, 1]

        return clip_probabilities(
            calibrated
        )

    if method == "Isotonic":

        calibrated = calibrator.predict(
            raw_probability
        )

        return clip_probabilities(
            calibrated
        )

    raise ValueError(
        f"Unknown calibration method: {method}"
    )


# ------------------------------------------------------------
# 5. Evaluation utilities
# ------------------------------------------------------------

def expected_calibration_error(
    y_true,
    y_probability,
    n_bins=10
):

    y_true = np.asarray(y_true)
    y_probability = np.asarray(
        y_probability
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    bin_ids = np.digitize(
        y_probability,
        bin_edges[1:-1],
        right=True
    )

    ece = 0.0

    for bin_id in range(n_bins):

        mask = bin_ids == bin_id

        if mask.sum() == 0:
            continue

        predicted_rate = (
            y_probability[mask].mean()
        )

        observed_rate = (
            y_true[mask].mean()
        )

        ece += (
            mask.mean()
            * abs(
                predicted_rate
                - observed_rate
            )
        )

    return float(ece)


def probability_metrics(
    y_true,
    y_probability
):

    y_probability = clip_probabilities(
        y_probability
    )

    return {
        "ROC_AUC": roc_auc_score(
            y_true,
            y_probability
        ),

        "PR_AUC": average_precision_score(
            y_true,
            y_probability
        ),

        "Brier_Score": brier_score_loss(
            y_true,
            y_probability
        ),

        "Log_Loss": log_loss(
            y_true,
            y_probability,
            labels=[0, 1]
        ),

        "ECE_10": expected_calibration_error(
            y_true,
            y_probability,
            n_bins=10
        ),

        "Mean_Probability":
            float(
                y_probability.mean()
            )
    }


def classification_metrics(
    y_true,
    y_probability,
    threshold
):

    y_prediction = (
        np.asarray(y_probability)
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "Locked_Threshold":
            float(threshold),

        "Accuracy": accuracy_score(
            y_true,
            y_prediction
        ),

        "Precision": precision_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "Specificity": specificity,

        "F1": f1_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_prediction
            ),

        "MCC": matthews_corrcoef(
            y_true,
            y_prediction
        ),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


def top_fraction_metrics(
    y_true,
    y_probability,
    fraction=0.20
):

    y_true = np.asarray(y_true)

    y_probability = np.asarray(
        y_probability
    )

    top_n = int(
        np.ceil(
            len(y_true) * fraction
        )
    )

    ranked_positions = np.argsort(
        -y_probability
    )

    selected_positions = (
        ranked_positions[:top_n]
    )

    selected_y = y_true[
        selected_positions
    ]

    total_positives = y_true.sum()

    captured_positives = (
        selected_y.sum()
    )

    recall_at_fraction = (
        captured_positives
        / total_positives
        if total_positives > 0
        else np.nan
    )

    population_prevalence = (
        y_true.mean()
    )

    selected_prevalence = (
        selected_y.mean()
    )

    lift_at_fraction = (
        selected_prevalence
        / population_prevalence
        if population_prevalence > 0
        else np.nan
    )

    return {
        "Top_Fraction": fraction,
        "Top_N": top_n,
        "Captured_Churners":
            int(captured_positives),
        "Recall_At_Top_20pct":
            recall_at_fraction,
        "Lift_At_Top_20pct":
            lift_at_fraction
    }


# ------------------------------------------------------------
# 6. OPEN LOCKED TEST LABELS
# ------------------------------------------------------------

print("\n" + "=" * 105)
print("OPENING LOCKED TEST LABELS")
print("=" * 105)

# This is the first test-label access in the final protocol.
y_test_array = (
    y_test
    .astype(int)
    .to_numpy()
)

assert len(y_test_array) == 2000
assert set(np.unique(y_test_array)) == {0, 1}

print("Test observations:", len(y_test_array))
print("Test churners:", int(y_test_array.sum()))
print("Test non-churners:", int((y_test_array == 0).sum()))
print("Test prevalence:", round(y_test_array.mean(), 6))
print("Locked test is now OPENED and must not be reused for tuning.")


# ------------------------------------------------------------
# 7. Apply locked calibration and thresholds
# ------------------------------------------------------------

final_metric_records = []
calibration_effect_records = []

final_predictions = pd.DataFrame({
    "Original_Index":
        raw_predictions[
            "Original_Index"
        ].to_numpy(),

    "CustomerId":
        raw_predictions[
            "CustomerId"
        ].to_numpy(),

    "y_true":
        y_test_array
})


for model_name in MODEL_ORDER:

    raw_probability = (
        raw_predictions[
            f"{model_name}_Raw_Probability"
        ]
        .to_numpy(dtype=float)
    )

    model_manifest = (
        locked_manifest[
            model_name
        ]
    )

    calibration_method = (
        model_manifest[
            "calibration_method"
        ]
    )

    locked_threshold = float(
        model_manifest[
            "locked_mcc_threshold"
        ]
    )

    calibrator_path = Path(
        model_manifest[
            "calibrator_path"
        ]
    )

    if not calibrator_path.exists():
        raise FileNotFoundError(
            f"Missing calibrator artifact: "
            f"{calibrator_path}"
        )

    calibrator_bundle = joblib.load(
        calibrator_path
    )

    assert (
        calibrator_bundle["method"]
        == calibration_method
    )

    calibrator = calibrator_bundle[
        "calibrator"
    ]

    calibrated_probability = (
        apply_locked_calibrator(
            method=calibration_method,
            calibrator=calibrator,
            raw_probability=raw_probability
        )
    )

    raw_probability_results = (
        probability_metrics(
            y_test_array,
            raw_probability
        )
    )

    calibrated_probability_results = (
        probability_metrics(
            y_test_array,
            calibrated_probability
        )
    )

    locked_classification_results = (
        classification_metrics(
            y_true=y_test_array,
            y_probability=
                calibrated_probability,
            threshold=locked_threshold
        )
    )

    default_classification_results = (
        classification_metrics(
            y_true=y_test_array,
            y_probability=
                calibrated_probability,
            threshold=0.50
        )
    )

    business_results = (
        top_fraction_metrics(
            y_true=y_test_array,
            y_probability=
                calibrated_probability,
            fraction=TOP_FRACTION
        )
    )

    final_metric_records.append({
        "Model": model_name,
        "Calibration_Method":
            calibration_method,

        **calibrated_probability_results,

        **locked_classification_results,

        "MCC_At_0_5":
            default_classification_results[
                "MCC"
            ],

        "F1_At_0_5":
            default_classification_results[
                "F1"
            ],

        "Recall_At_0_5":
            default_classification_results[
                "Recall"
            ],

        "Specificity_At_0_5":
            default_classification_results[
                "Specificity"
            ],

        **business_results
    })

    calibration_effect_records.append({
        "Model": model_name,
        "Calibration_Method":
            calibration_method,

        "Raw_ROC_AUC":
            raw_probability_results[
                "ROC_AUC"
            ],

        "Calibrated_ROC_AUC":
            calibrated_probability_results[
                "ROC_AUC"
            ],

        "Delta_ROC_AUC":
            calibrated_probability_results[
                "ROC_AUC"
            ]
            - raw_probability_results[
                "ROC_AUC"
            ],

        "Raw_PR_AUC":
            raw_probability_results[
                "PR_AUC"
            ],

        "Calibrated_PR_AUC":
            calibrated_probability_results[
                "PR_AUC"
            ],

        "Delta_PR_AUC":
            calibrated_probability_results[
                "PR_AUC"
            ]
            - raw_probability_results[
                "PR_AUC"
            ],

        "Raw_Brier":
            raw_probability_results[
                "Brier_Score"
            ],

        "Calibrated_Brier":
            calibrated_probability_results[
                "Brier_Score"
            ],

        "Delta_Brier":
            calibrated_probability_results[
                "Brier_Score"
            ]
            - raw_probability_results[
                "Brier_Score"
            ],

        "Raw_Log_Loss":
            raw_probability_results[
                "Log_Loss"
            ],

        "Calibrated_Log_Loss":
            calibrated_probability_results[
                "Log_Loss"
            ],

        "Delta_Log_Loss":
            calibrated_probability_results[
                "Log_Loss"
            ]
            - raw_probability_results[
                "Log_Loss"
            ],

        "Raw_ECE":
            raw_probability_results[
                "ECE_10"
            ],

        "Calibrated_ECE":
            calibrated_probability_results[
                "ECE_10"
            ],

        "Delta_ECE":
            calibrated_probability_results[
                "ECE_10"
            ]
            - raw_probability_results[
                "ECE_10"
            ]
    })

    final_prediction = (
        calibrated_probability
        >= locked_threshold
    ).astype(int)

    final_predictions[
        f"{model_name}_Raw_Probability"
    ] = raw_probability

    final_predictions[
        f"{model_name}_Calibrated_Probability"
    ] = calibrated_probability

    final_predictions[
        f"{model_name}_Locked_Prediction"
    ] = final_prediction


# ------------------------------------------------------------
# 8. Construct final result tables
# ------------------------------------------------------------

final_metrics = (
    pd.DataFrame(
        final_metric_records
    )
    .sort_values(
        [
            "ROC_AUC",
            "PR_AUC"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

final_metrics[
    "ROC_AUC_Rank"
] = (
    final_metrics["ROC_AUC"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

final_metrics[
    "PR_AUC_Rank"
] = (
    final_metrics["PR_AUC"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

calibration_effects = (
    pd.DataFrame(
        calibration_effect_records
    )
    .sort_values("Model")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Print one-time final results
# ------------------------------------------------------------

print("\n" + "=" * 175)
print("FINAL LOCKED TEST PERFORMANCE")
print("All probabilities, calibrators and thresholds were locked before y_test was opened.")
print("=" * 175)

final_display_columns = [
    "ROC_AUC_Rank",
    "PR_AUC_Rank",
    "Model",
    "Calibration_Method",
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "ECE_10",
    "Locked_Threshold",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "TN",
    "FP",
    "FN",
    "TP",
    "Recall_At_Top_20pct",
    "Lift_At_Top_20pct"
]

print(
    final_metrics[
        final_display_columns
    ]
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 145)
print("FINAL TEST CALIBRATION EFFECTS")
print("For Brier, Log Loss and ECE, negative deltas indicate improvement.")
print("=" * 145)

print(
    calibration_effects
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. Save immutable final artifacts
# ------------------------------------------------------------

final_metrics.to_csv(
    FINAL_METRICS_PATH,
    index=False
)

calibration_effects.to_csv(
    CALIBRATION_EFFECT_PATH,
    index=False
)

final_predictions.to_csv(
    FINAL_PREDICTIONS_PATH,
    index=False
)


def sha256_file(filepath):

    digest = hashlib.sha256()

    with open(filepath, "rb") as file:

        for block in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


evaluation_record = {
    "evaluated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "test_rows":
        int(len(y_test_array)),

    "test_positive_count":
        int(y_test_array.sum()),

    "test_prevalence":
        float(y_test_array.mean()),

    "models":
        MODEL_ORDER,

    "raw_predictions_sha256":
        sha256_file(
            RAW_PREDICTION_PATH
        ),

    "final_predictions_sha256":
        sha256_file(
            FINAL_PREDICTIONS_PATH
        ),

    "final_metrics_sha256":
        sha256_file(
            FINAL_METRICS_PATH
        ),

    "post_test_tuning_allowed":
        False
}

with open(
    EVALUATION_LOCK_PATH,
    "w"
) as file:

    json.dump(
        evaluation_record,
        file,
        indent=4
    )


print("\n" + "=" * 105)
print("FINAL TEST EVALUATION LOCK")
print("=" * 105)

print(
    json.dumps(
        evaluation_record,
        indent=4
    )
)

print("\nSaved:")
print(FINAL_METRICS_PATH)
print(CALIBRATION_EFFECT_PATH)
print(FINAL_PREDICTIONS_PATH)
print(EVALUATION_LOCK_PATH)

print(
    "\nThe locked test has been evaluated. "
    "No post-test model or threshold changes are permitted."
)

In [ ]:
# ============================================================
# POST-HOC STATISTICAL ANALYSIS
# LOCKED ARTIFACT VERIFICATION
# + MODEL-LEVEL BOOTSTRAP 95% CONFIDENCE INTERVALS
#
# Uses saved locked-test predictions only.
# No model training or post-test tuning.
# ============================================================

from pathlib import Path
import hashlib
import json
import math
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

BOOTSTRAP_SEED = 2028
N_BOOTSTRAPS = 5000
CONFIDENCE_LEVEL = 0.95
TOP_FRACTION = 0.20

OUTPUT_DIR = Path(
    "/kaggle/working/statistical_inference"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_ORDER = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

KEY_METRICS = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "Recall_At_Top_20pct",
    "Lift_At_Top_20pct"
]


# ------------------------------------------------------------
# 2. Locate restored checkpoint
# ------------------------------------------------------------

candidate_roots = [Path("/kaggle/working")]

existing_roots = [
    path
    for path in candidate_roots
    if path.exists()
]

if not existing_roots:
    raise FileNotFoundError(
        "Could not locate the restored checkpoint. "
        "Checked:\n"
        + "\n".join(
            str(path)
            for path in candidate_roots
        )
    )

RESTORED_ROOT = existing_roots[0]

print("=" * 105)
print("RESTORED CHECKPOINT LOCATION")
print("=" * 105)

print(RESTORED_ROOT)


def find_unique_file(
    root,
    filename,
    preferred_parent=None
):
    matches = list(
        root.rglob(filename)
    )

    if preferred_parent is not None:
        preferred_matches = [
            path
            for path in matches
            if preferred_parent
            in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} "
            f"under {root}."
        )

    raise RuntimeError(
        f"Found multiple copies of {filename}:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


FINAL_PREDICTIONS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_locked_test_predictions.csv",
    preferred_parent="final_locked_holdout"
)

FINAL_METRICS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_locked_test_metrics.csv",
    preferred_parent="final_locked_holdout"
)

RAW_PREDICTIONS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_test_raw_probabilities.csv",
    preferred_parent="final_locked_holdout"
)

EVALUATION_LOCK_PATH = find_unique_file(
    RESTORED_ROOT,
    "test_evaluation_lock.json",
    preferred_parent="final_locked_holdout"
)


# ------------------------------------------------------------
# 3. Verify original SHA-256 hashes
# ------------------------------------------------------------

def sha256_file(
    filepath,
    block_size=1024 * 1024
):
    digest = hashlib.sha256()

    with open(filepath, "rb") as file:
        while True:
            block = file.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


with open(
    EVALUATION_LOCK_PATH,
    "r",
    encoding="utf-8"
) as file:
    evaluation_lock = json.load(file)


verification_records = [
    {
        "Artifact":
            "Raw test probabilities",

        "Path":
            str(RAW_PREDICTIONS_PATH),

        "Expected_SHA256":
            evaluation_lock[
                "raw_predictions_sha256"
            ],

        "Observed_SHA256":
            sha256_file(
                RAW_PREDICTIONS_PATH
            )
    },
    {
        "Artifact":
            "Final locked test predictions",

        "Path":
            str(FINAL_PREDICTIONS_PATH),

        "Expected_SHA256":
            evaluation_lock[
                "final_predictions_sha256"
            ],

        "Observed_SHA256":
            sha256_file(
                FINAL_PREDICTIONS_PATH
            )
    },
    {
        "Artifact":
            "Final locked test metrics",

        "Path":
            str(FINAL_METRICS_PATH),

        "Expected_SHA256":
            evaluation_lock[
                "final_metrics_sha256"
            ],

        "Observed_SHA256":
            sha256_file(
                FINAL_METRICS_PATH
            )
    }
]

verification_table = pd.DataFrame(
    verification_records
)

verification_table["Hash_Match"] = (
    verification_table[
        "Expected_SHA256"
    ]
    == verification_table[
        "Observed_SHA256"
    ]
)

print("\n" + "=" * 130)
print("RESTORED ARTIFACT VERIFICATION")
print("=" * 130)

print(
    verification_table[
        [
            "Artifact",
            "Hash_Match",
            "Path"
        ]
    ].to_string(index=False)
)

assert verification_table[
    "Hash_Match"
].all(), (
    "At least one restored file does not match "
    "the original locked-test hash."
)


# ------------------------------------------------------------
# 4. Load locked predictions and metrics
# ------------------------------------------------------------

final_predictions = pd.read_csv(
    FINAL_PREDICTIONS_PATH
)

saved_final_metrics = pd.read_csv(
    FINAL_METRICS_PATH,
    keep_default_na=False
)

required_base_columns = [
    "Original_Index",
    "CustomerId",
    "y_true"
]

required_model_columns = []

for model_name in MODEL_ORDER:
    required_model_columns.extend([
        f"{model_name}_Raw_Probability",
        f"{model_name}_Calibrated_Probability",
        f"{model_name}_Locked_Prediction"
    ])

missing_columns = [
    column
    for column in (
        required_base_columns
        + required_model_columns
    )
    if column not in final_predictions.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# 5. Prediction integrity
# ------------------------------------------------------------

y_true = (
    final_predictions["y_true"]
    .astype(int)
    .to_numpy()
)

assert len(y_true) == 2000
assert y_true.sum() == 407
assert set(np.unique(y_true)) == {0, 1}

probability_map = {}
prediction_map = {}

integrity_records = []

for model_name in MODEL_ORDER:

    probabilities = (
        final_predictions[
            f"{model_name}_"
            "Calibrated_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    predictions = (
        final_predictions[
            f"{model_name}_"
            "Locked_Prediction"
        ]
        .astype(int)
        .to_numpy()
    )

    probability_map[
        model_name
    ] = probabilities

    prediction_map[
        model_name
    ] = predictions

    integrity_records.append({
        "Model": model_name,
        "Rows": len(probabilities),
        "Finite_Probabilities":
            int(
                np.isfinite(
                    probabilities
                ).sum()
            ),
        "Probability_Min":
            probabilities.min(),
        "Probability_Max":
            probabilities.max(),
        "Positive_Predictions":
            int(predictions.sum()),
        "Prediction_Values":
            str(
                sorted(
                    np.unique(
                        predictions
                    ).tolist()
                )
            )
    })


prediction_integrity = pd.DataFrame(
    integrity_records
)

print("\n" + "=" * 120)
print("LOCKED PREDICTION INTEGRITY")
print("=" * 120)

print(
    prediction_integrity
    .round(6)
    .to_string(index=False)
)

assert (
    prediction_integrity["Rows"] == 2000
).all()

assert (
    prediction_integrity[
        "Finite_Probabilities"
    ] == 2000
).all()


# ------------------------------------------------------------
# 6. Metric utilities
# ------------------------------------------------------------

def top_fraction_metrics(
    y,
    probabilities,
    fraction=0.20
):
    y = np.asarray(y)
    probabilities = np.asarray(
        probabilities
    )

    top_n = int(
        math.ceil(
            len(y) * fraction
        )
    )

    ranked_positions = np.argsort(
        -probabilities
    )

    selected_positions = (
        ranked_positions[:top_n]
    )

    selected_y = y[
        selected_positions
    ]

    total_positives = y.sum()

    captured_positives = (
        selected_y.sum()
    )

    recall_at_fraction = (
        captured_positives
        / total_positives
        if total_positives > 0
        else np.nan
    )

    population_prevalence = (
        y.mean()
    )

    selected_prevalence = (
        selected_y.mean()
    )

    lift = (
        selected_prevalence
        / population_prevalence
        if population_prevalence > 0
        else np.nan
    )

    return {
        "Recall_At_Top_20pct":
            recall_at_fraction,

        "Lift_At_Top_20pct":
            lift
    }


def compute_metrics(
    y,
    probabilities,
    predictions
):
    y = np.asarray(y)
    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float
        ),
        1e-7,
        1 - 1e-7
    )

    predictions = np.asarray(
        predictions,
        dtype=int
    )

    tn, fp, fn, tp = confusion_matrix(
        y,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    result = {
        "ROC_AUC":
            roc_auc_score(
                y,
                probabilities
            ),

        "PR_AUC":
            average_precision_score(
                y,
                probabilities
            ),

        "Brier_Score":
            brier_score_loss(
                y,
                probabilities
            ),

        "Log_Loss":
            log_loss(
                y,
                probabilities,
                labels=[0, 1]
            ),

        "Accuracy":
            accuracy_score(
                y,
                predictions
            ),

        "Precision":
            precision_score(
                y,
                predictions,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y,
                predictions,
                zero_division=0
            ),

        "Specificity":
            specificity,

        "F1":
            f1_score(
                y,
                predictions,
                zero_division=0
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y,
                predictions
            ),

        "MCC":
            matthews_corrcoef(
                y,
                predictions
            )
    }

    result.update(
        top_fraction_metrics(
            y=y,
            probabilities=probabilities,
            fraction=TOP_FRACTION
        )
    )

    return result


# ------------------------------------------------------------
# 7. Recompute point estimates from saved predictions
# ------------------------------------------------------------

point_estimate_records = []

for model_name in MODEL_ORDER:

    metrics = compute_metrics(
        y=y_true,
        probabilities=
            probability_map[
                model_name
            ],
        predictions=
            prediction_map[
                model_name
            ]
    )

    point_estimate_records.append({
        "Model": model_name,
        **metrics
    })


point_estimates = pd.DataFrame(
    point_estimate_records
)


# Confirm that the reconstructed key metrics
# match the immutable metrics file.
comparison_columns = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "Recall_At_Top_20pct",
    "Lift_At_Top_20pct"
]

reconstruction_check = (
    point_estimates
    .merge(
        saved_final_metrics[
            ["Model", *comparison_columns]
        ],
        on="Model",
        suffixes=(
            "_Reconstructed",
            "_Saved"
        )
    )
)

maximum_difference = 0.0

for metric in comparison_columns:

    difference = np.abs(
        reconstruction_check[
            f"{metric}_Reconstructed"
        ]
        - reconstruction_check[
            f"{metric}_Saved"
        ]
    ).max()

    maximum_difference = max(
        maximum_difference,
        float(difference)
    )

print("\nMaximum reconstructed metric difference:")
print(f"{maximum_difference:.12f}")

assert maximum_difference < 1e-10


# ------------------------------------------------------------
# 8. Generate paired bootstrap replicates
# ------------------------------------------------------------

print("\n" + "=" * 105)
print("GENERATING PAIRED BOOTSTRAP REPLICATES")
print("=" * 105)

print("Bootstrap samples:", N_BOOTSTRAPS)
print("Seed:", BOOTSTRAP_SEED)
print("Test observations:", len(y_true))
print("Models:", len(MODEL_ORDER))

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_records = []

bootstrap_start = time.perf_counter()

n_observations = len(y_true)

for bootstrap_id in range(
    1,
    N_BOOTSTRAPS + 1
):

    sampled_positions = rng.integers(
        low=0,
        high=n_observations,
        size=n_observations
    )

    y_bootstrap = y_true[
        sampled_positions
    ]

    # Defensive check. With this dataset size,
    # a one-class bootstrap is exceedingly unlikely.
    if np.unique(
        y_bootstrap
    ).size < 2:
        continue

    for model_name in MODEL_ORDER:

        probability_bootstrap = (
            probability_map[
                model_name
            ][sampled_positions]
        )

        prediction_bootstrap = (
            prediction_map[
                model_name
            ][sampled_positions]
        )

        metrics = compute_metrics(
            y=y_bootstrap,
            probabilities=
                probability_bootstrap,
            predictions=
                prediction_bootstrap
        )

        bootstrap_records.append({
            "Bootstrap_ID":
                bootstrap_id,
            "Model":
                model_name,
            **metrics
        })

    if (
        bootstrap_id == 1
        or bootstrap_id % 500 == 0
    ):
        elapsed = (
            time.perf_counter()
            - bootstrap_start
        )

        print(
            f"Completed "
            f"{bootstrap_id}/"
            f"{N_BOOTSTRAPS} "
            f"bootstraps | "
            f"elapsed={elapsed / 60:.2f} min"
        )


bootstrap_replicates = pd.DataFrame(
    bootstrap_records
)

expected_rows = (
    N_BOOTSTRAPS
    * len(MODEL_ORDER)
)

assert len(
    bootstrap_replicates
) == expected_rows

assert (
    bootstrap_replicates[
        "Bootstrap_ID"
    ].nunique()
    == N_BOOTSTRAPS
)


# ------------------------------------------------------------
# 9. Percentile confidence intervals
# ------------------------------------------------------------

alpha = 1.0 - CONFIDENCE_LEVEL

lower_quantile = alpha / 2.0
upper_quantile = 1.0 - alpha / 2.0

confidence_interval_records = []

for model_name in MODEL_ORDER:

    model_point = point_estimates[
        point_estimates["Model"]
        == model_name
    ].iloc[0]

    model_bootstrap = (
        bootstrap_replicates[
            bootstrap_replicates["Model"]
            == model_name
        ]
    )

    for metric in KEY_METRICS:

        bootstrap_values = (
            model_bootstrap[metric]
            .dropna()
            .to_numpy()
        )

        confidence_interval_records.append({
            "Model": model_name,
            "Metric": metric,

            "Point_Estimate":
                float(
                    model_point[metric]
                ),

            "CI_Lower":
                float(
                    np.quantile(
                        bootstrap_values,
                        lower_quantile
                    )
                ),

            "CI_Upper":
                float(
                    np.quantile(
                        bootstrap_values,
                        upper_quantile
                    )
                ),

            "Bootstrap_Replicates":
                len(
                    bootstrap_values
                )
        })


bootstrap_ci_long = pd.DataFrame(
    confidence_interval_records
)


# ------------------------------------------------------------
# 10. Create compact display table
# ------------------------------------------------------------

def format_ci(
    estimate,
    lower,
    upper
):
    return (
        f"{estimate:.4f} "
        f"[{lower:.4f}, {upper:.4f}]"
    )


display_metrics = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "Recall_At_Top_20pct"
]

display_records = []

for model_name in MODEL_ORDER:

    row = {
        "Model": model_name
    }

    for metric in display_metrics:

        result = bootstrap_ci_long[
            (
                bootstrap_ci_long["Model"]
                == model_name
            )
            &
            (
                bootstrap_ci_long["Metric"]
                == metric
            )
        ].iloc[0]

        row[
            f"{metric}_95CI"
        ] = format_ci(
            result["Point_Estimate"],
            result["CI_Lower"],
            result["CI_Upper"]
        )

    display_records.append(row)


bootstrap_display = (
    pd.DataFrame(
        display_records
    )
    .merge(
        point_estimates[
            [
                "Model",
                "ROC_AUC"
            ]
        ],
        on="Model"
    )
    .sort_values(
        "ROC_AUC",
        ascending=False
    )
    .drop(
        columns=["ROC_AUC"]
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 190)
print(
    "MODEL-LEVEL BOOTSTRAP "
    "95% CONFIDENCE INTERVALS"
)
print(
    "Percentile intervals from "
    f"{N_BOOTSTRAPS:,} paired bootstrap samples."
)
print("=" * 190)

print(
    bootstrap_display
    .to_string(index=False)
)


# ------------------------------------------------------------
# 11. Save artifacts
# ------------------------------------------------------------

verification_table.to_csv(
    OUTPUT_DIR
    / "locked_artifact_verification.csv",
    index=False
)

prediction_integrity.to_csv(
    OUTPUT_DIR
    / "locked_prediction_integrity.csv",
    index=False
)

point_estimates.to_csv(
    OUTPUT_DIR
    / "reconstructed_locked_test_metrics.csv",
    index=False
)

bootstrap_replicates.to_csv(
    OUTPUT_DIR
    / "model_metric_bootstrap_replicates.csv",
    index=False
)

bootstrap_ci_long.to_csv(
    OUTPUT_DIR
    / "model_level_bootstrap_confidence_intervals.csv",
    index=False
)

bootstrap_display.to_csv(
    OUTPUT_DIR
    / "model_level_bootstrap_ci_display.csv",
    index=False
)


bootstrap_elapsed = (
    time.perf_counter()
    - bootstrap_start
)

run_summary = {
    "restored_root":
        str(RESTORED_ROOT),

    "test_rows":
        int(len(y_true)),

    "test_positive_count":
        int(y_true.sum()),

    "models":
        MODEL_ORDER,

    "bootstrap_seed":
        BOOTSTRAP_SEED,

    "bootstrap_replicates":
        N_BOOTSTRAPS,

    "confidence_level":
        CONFIDENCE_LEVEL,

    "maximum_metric_reconstruction_difference":
        maximum_difference,

    "bootstrap_elapsed_seconds":
        bootstrap_elapsed,

    "post_test_tuning_performed":
        False
}

with open(
    OUTPUT_DIR
    / "bootstrap_run_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        run_summary,
        file,
        indent=4
    )


print("\n" + "=" * 105)
print("CHECKPOINT 8A1 SAVED")
print("=" * 105)

print("Output directory:")
print(OUTPUT_DIR)

print(
    f"\nBootstrap elapsed time: "
    f"{bootstrap_elapsed / 60:.2f} minutes"
)

print(
    "\nNo model, calibrator, "
    "hyperparameter or threshold was changed."
)

In [ ]:
# ============================================================
# PAIRWISE STATISTICAL COMPARISONS
#
# - DeLong tests for ROC-AUC
# - Paired bootstrap tests and difference CIs
# - Exact McNemar tests
# - Holm multiple-comparison correction
#
# Uses frozen locked-test predictions and existing
# bootstrap replicates only.
# No model training or post-test tuning.
# ============================================================

from pathlib import Path
from itertools import combinations
import json

import numpy as np
import pandas as pd

from scipy.stats import norm, binomtest


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ALPHA = 0.05
REFERENCE_MODEL = "XGBoost"

STATS_DIR = Path(
    "/kaggle/working/statistical_inference"
)

STATS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_ORDER = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

BOOTSTRAP_METRICS = [
    "PR_AUC",
    "Brier_Score",
    "Log_Loss",
    "F1",
    "Balanced_Accuracy",
    "MCC",
    "Recall_At_Top_20pct"
]

HIGHER_IS_BETTER = {
    "PR_AUC": True,
    "Brier_Score": False,
    "Log_Loss": False,
    "F1": True,
    "Balanced_Accuracy": True,
    "MCC": True,
    "Recall_At_Top_20pct": True
}

MODEL_PAIRS = list(
    combinations(
        MODEL_ORDER,
        2
    )
)

print("=" * 105)
print("PAIRWISE INFERENCE CONFIGURATION")
print("=" * 105)

print("Models:", len(MODEL_ORDER))
print("Pairwise comparisons:", len(MODEL_PAIRS))
print("Reference model:", REFERENCE_MODEL)
print("Family-wise alpha:", ALPHA)


# ------------------------------------------------------------
# 2. Locate locked predictions
# ------------------------------------------------------------

candidate_roots = [Path("/kaggle/working")]

existing_roots = [
    path
    for path in candidate_roots
    if path.exists()
]

if not existing_roots:
    raise FileNotFoundError(
        "Restored checkpoint directory "
        "could not be located."
    )

RESTORED_ROOT = existing_roots[0]


def find_unique_file(
    root,
    filename,
    preferred_parent=None
):
    matches = list(
        root.rglob(filename)
    )

    if preferred_parent is not None:

        preferred_matches = [
            path
            for path in matches
            if preferred_parent in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} "
            f"under {root}."
        )

    raise RuntimeError(
        f"Multiple copies of {filename} found:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


FINAL_PREDICTIONS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_locked_test_predictions.csv",
    preferred_parent="final_locked_holdout"
)

BOOTSTRAP_PATH = (
    STATS_DIR
    / "model_metric_bootstrap_replicates.csv"
)

POINT_ESTIMATE_PATH = (
    STATS_DIR
    / "reconstructed_locked_test_metrics.csv"
)

if not BOOTSTRAP_PATH.exists():
    raise FileNotFoundError(
        "Bootstrap replicate file is missing. "
        "Stage must be completed first."
    )

if not POINT_ESTIMATE_PATH.exists():
    raise FileNotFoundError(
        "Reconstructed point-estimate file is missing."
    )


# ------------------------------------------------------------
# 3. Load data
# ------------------------------------------------------------

final_predictions = pd.read_csv(
    FINAL_PREDICTIONS_PATH
)

bootstrap_replicates = pd.read_csv(
    BOOTSTRAP_PATH
)

point_estimates = pd.read_csv(
    POINT_ESTIMATE_PATH
)

y_true = (
    final_predictions["y_true"]
    .astype(int)
    .to_numpy()
)

probability_map = {}
prediction_map = {}

for model_name in MODEL_ORDER:

    probability_map[model_name] = (
        final_predictions[
            f"{model_name}_"
            "Calibrated_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    prediction_map[model_name] = (
        final_predictions[
            f"{model_name}_"
            "Locked_Prediction"
        ]
        .astype(int)
        .to_numpy()
    )

assert len(y_true) == 2000
assert y_true.sum() == 407

assert (
    bootstrap_replicates[
        "Bootstrap_ID"
    ].nunique()
    == 5000
)

print("\nLocked observations:", len(y_true))
print("Churners:", int(y_true.sum()))
print(
    "Bootstrap replicates:",
    bootstrap_replicates[
        "Bootstrap_ID"
    ].nunique()
)


# ------------------------------------------------------------
# 4. Holm correction
# ------------------------------------------------------------

def holm_adjust(
    p_values
):
    p_values = np.asarray(
        p_values,
        dtype=float
    )

    number_of_tests = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty(
        number_of_tests,
        dtype=float
    )

    running_maximum = 0.0

    for rank, original_position in enumerate(
        order
    ):
        multiplier = (
            number_of_tests - rank
        )

        current_adjusted = (
            multiplier
            * p_values[
                original_position
            ]
        )

        running_maximum = max(
            running_maximum,
            current_adjusted
        )

        adjusted[
            original_position
        ] = min(
            running_maximum,
            1.0
        )

    return adjusted


# ------------------------------------------------------------
# 5. Fast DeLong implementation
# ------------------------------------------------------------

def compute_midrank(values):

    values = np.asarray(
        values,
        dtype=float
    )

    sorted_positions = np.argsort(
        values
    )

    sorted_values = values[
        sorted_positions
    ]

    n_values = len(values)

    temporary_ranks = np.zeros(
        n_values,
        dtype=float
    )

    start = 0

    while start < n_values:

        end = start + 1

        while (
            end < n_values
            and
            sorted_values[end]
            == sorted_values[start]
        ):
            end += 1

        temporary_ranks[
            start:end
        ] = 0.5 * (
            start + end - 1
        )

        start = end

    ranks = np.empty(
        n_values,
        dtype=float
    )

    ranks[
        sorted_positions
    ] = temporary_ranks + 1.0

    return ranks


def fast_delong(
    predictions_sorted_transposed,
    positive_count
):
    model_count = (
        predictions_sorted_transposed
        .shape[0]
    )

    negative_count = (
        predictions_sorted_transposed
        .shape[1]
        - positive_count
    )

    positive_predictions = (
        predictions_sorted_transposed[
            :,
            :positive_count
        ]
    )

    negative_predictions = (
        predictions_sorted_transposed[
            :,
            positive_count:
        ]
    )

    positive_midranks = np.empty(
        (
            model_count,
            positive_count
        ),
        dtype=float
    )

    negative_midranks = np.empty(
        (
            model_count,
            negative_count
        ),
        dtype=float
    )

    combined_midranks = np.empty(
        (
            model_count,
            positive_count
            + negative_count
        ),
        dtype=float
    )

    for model_position in range(
        model_count
    ):

        positive_midranks[
            model_position
        ] = compute_midrank(
            positive_predictions[
                model_position
            ]
        )

        negative_midranks[
            model_position
        ] = compute_midrank(
            negative_predictions[
                model_position
            ]
        )

        combined_midranks[
            model_position
        ] = compute_midrank(
            predictions_sorted_transposed[
                model_position
            ]
        )

    aucs = (
        combined_midranks[
            :,
            :positive_count
        ].sum(axis=1)
        / positive_count
        / negative_count
        -
        (
            positive_count + 1.0
        )
        / (
            2.0 * negative_count
        )
    )

    v_positive = (
        combined_midranks[
            :,
            :positive_count
        ]
        -
        positive_midranks
    ) / negative_count

    v_negative = (
        1.0
        -
        (
            combined_midranks[
                :,
                positive_count:
            ]
            -
            negative_midranks
        )
        / positive_count
    )

    covariance_positive = np.cov(
        v_positive
    )

    covariance_negative = np.cov(
        v_negative
    )

    covariance = (
        covariance_positive
        / positive_count
        +
        covariance_negative
        / negative_count
    )

    return aucs, covariance


def paired_delong_test(
    y,
    probability_a,
    probability_b,
    confidence_level=0.95
):
    y = np.asarray(
        y,
        dtype=int
    )

    probability_a = np.asarray(
        probability_a,
        dtype=float
    )

    probability_b = np.asarray(
        probability_b,
        dtype=float
    )

    descending_label_order = np.argsort(
        -y
    )

    positive_count = int(
        y.sum()
    )

    predictions_sorted = np.vstack([
        probability_a[
            descending_label_order
        ],
        probability_b[
            descending_label_order
        ]
    ])

    aucs, covariance = fast_delong(
        predictions_sorted_transposed=
            predictions_sorted,
        positive_count=positive_count
    )

    delta = float(
        aucs[0] - aucs[1]
    )

    contrast = np.array(
        [1.0, -1.0]
    )

    variance = float(
        contrast
        @ covariance
        @ contrast.T
    )

    variance = max(
        variance,
        0.0
    )

    standard_error = np.sqrt(
        variance
    )

    if standard_error <= 1e-15:

        z_statistic = (
            0.0
            if abs(delta) <= 1e-15
            else np.sign(delta) * np.inf
        )

        p_value = (
            1.0
            if abs(delta) <= 1e-15
            else 0.0
        )

    else:

        z_statistic = (
            delta
            / standard_error
        )

        p_value = float(
            2.0
            * norm.sf(
                abs(z_statistic)
            )
        )

    alpha = (
        1.0 - confidence_level
    )

    critical_value = norm.ppf(
        1.0 - alpha / 2.0
    )

    ci_lower = max(
        -1.0,
        delta
        - critical_value
        * standard_error
    )

    ci_upper = min(
        1.0,
        delta
        + critical_value
        * standard_error
    )

    return {
        "ROC_AUC_A":
            float(aucs[0]),

        "ROC_AUC_B":
            float(aucs[1]),

        "Delta_A_minus_B":
            delta,

        "Standard_Error":
            standard_error,

        "CI_Lower":
            ci_lower,

        "CI_Upper":
            ci_upper,

        "Z_Statistic":
            z_statistic,

        "P_Value":
            p_value
    }


# ------------------------------------------------------------
# 6. Run all-pair DeLong tests
# ------------------------------------------------------------

delong_records = []

for model_a, model_b in MODEL_PAIRS:

    result = paired_delong_test(
        y=y_true,
        probability_a=
            probability_map[
                model_a
            ],
        probability_b=
            probability_map[
                model_b
            ]
    )

    delong_records.append({
        "Model_A": model_a,
        "Model_B": model_b,
        **result
    })


delong_results = pd.DataFrame(
    delong_records
)

delong_results[
    "P_Value_Holm"
] = holm_adjust(
    delong_results[
        "P_Value"
    ].to_numpy()
)

delong_results[
    "Significant_Holm_0_05"
] = (
    delong_results[
        "P_Value_Holm"
    ] < ALPHA
)

delong_results[
    "Higher_ROC_AUC_Model"
] = np.where(
    delong_results[
        "Delta_A_minus_B"
    ] > 0,
    delong_results["Model_A"],
    np.where(
        delong_results[
            "Delta_A_minus_B"
        ] < 0,
        delong_results["Model_B"],
        "Tie"
    )
)


# ------------------------------------------------------------
# 7. Paired bootstrap comparisons
# ------------------------------------------------------------

point_estimate_lookup = (
    point_estimates
    .set_index("Model")
)

bootstrap_comparison_records = []

for metric in BOOTSTRAP_METRICS:

    metric_bootstrap = (
        bootstrap_replicates
        .pivot(
            index="Bootstrap_ID",
            columns="Model",
            values=metric
        )
        .sort_index()
    )

    assert (
        metric_bootstrap.shape[0]
        == 5000
    )

    for model_a, model_b in MODEL_PAIRS:

        replicate_difference = (
            metric_bootstrap[
                model_a
            ].to_numpy()
            -
            metric_bootstrap[
                model_b
            ].to_numpy()
        )

        point_difference = float(
            point_estimate_lookup.loc[
                model_a,
                metric
            ]
            -
            point_estimate_lookup.loc[
                model_b,
                metric
            ]
        )

        ci_lower = float(
            np.quantile(
                replicate_difference,
                0.025
            )
        )

        ci_upper = float(
            np.quantile(
                replicate_difference,
                0.975
            )
        )

        # Bootstrap hypothesis test under H0:
        # center the replicate distribution at zero.
        centered_null_distribution = (
            replicate_difference
            - point_difference
        )

        p_value = (
            np.sum(
                np.abs(
                    centered_null_distribution
                )
                >= abs(
                    point_difference
                )
            )
            + 1
        ) / (
            len(
                centered_null_distribution
            )
            + 1
        )

        higher_better = (
            HIGHER_IS_BETTER[
                metric
            ]
        )

        if abs(point_difference) <= 1e-15:

            favored_model = "Tie"

        elif higher_better:

            favored_model = (
                model_a
                if point_difference > 0
                else model_b
            )

        else:

            favored_model = (
                model_a
                if point_difference < 0
                else model_b
            )

        bootstrap_comparison_records.append({
            "Metric": metric,
            "Higher_Is_Better":
                higher_better,
            "Model_A": model_a,
            "Model_B": model_b,
            "Estimate_A":
                float(
                    point_estimate_lookup.loc[
                        model_a,
                        metric
                    ]
                ),
            "Estimate_B":
                float(
                    point_estimate_lookup.loc[
                        model_b,
                        metric
                    ]
                ),
            "Delta_A_minus_B":
                point_difference,
            "CI_Lower":
                ci_lower,
            "CI_Upper":
                ci_upper,
            "Bootstrap_P_Value":
                float(p_value),
            "Favored_Model":
                favored_model,
            "Bootstrap_Replicates":
                len(
                    replicate_difference
                )
        })


bootstrap_pairwise = pd.DataFrame(
    bootstrap_comparison_records
)


# Holm adjustment separately within each metric family.
adjusted_bootstrap_groups = []

for metric, metric_table in (
    bootstrap_pairwise.groupby(
        "Metric",
        sort=False
    )
):

    metric_table = (
        metric_table
        .copy()
        .reset_index(drop=True)
    )

    metric_table[
        "P_Value_Holm"
    ] = holm_adjust(
        metric_table[
            "Bootstrap_P_Value"
        ].to_numpy()
    )

    metric_table[
        "Significant_Holm_0_05"
    ] = (
        metric_table[
            "P_Value_Holm"
        ] < ALPHA
    )

    metric_table[
        "Difference_CI_Excludes_Zero"
    ] = (
        (
            metric_table["CI_Lower"] > 0
        )
        |
        (
            metric_table["CI_Upper"] < 0
        )
    )

    adjusted_bootstrap_groups.append(
        metric_table
    )


bootstrap_pairwise = pd.concat(
    adjusted_bootstrap_groups,
    ignore_index=True
)


# ------------------------------------------------------------
# 8. Exact McNemar tests
# ------------------------------------------------------------

mcnemar_records = []

correctness_map = {
    model_name: (
        prediction_map[
            model_name
        ] == y_true
    )
    for model_name
    in MODEL_ORDER
}

for model_a, model_b in MODEL_PAIRS:

    correct_a = correctness_map[
        model_a
    ]

    correct_b = correctness_map[
        model_b
    ]

    a_correct_b_wrong = int(
        np.sum(
            correct_a
            & ~correct_b
        )
    )

    a_wrong_b_correct = int(
        np.sum(
            ~correct_a
            & correct_b
        )
    )

    discordant_total = (
        a_correct_b_wrong
        + a_wrong_b_correct
    )

    if discordant_total == 0:

        p_value = 1.0

    else:

        p_value = float(
            binomtest(
                k=a_correct_b_wrong,
                n=discordant_total,
                p=0.5,
                alternative="two-sided"
            ).pvalue
        )

    accuracy_a = float(
        correct_a.mean()
    )

    accuracy_b = float(
        correct_b.mean()
    )

    accuracy_delta = (
        accuracy_a
        - accuracy_b
    )

    favored_model = (
        model_a
        if accuracy_delta > 0
        else (
            model_b
            if accuracy_delta < 0
            else "Tie"
        )
    )

    mcnemar_records.append({
        "Model_A": model_a,
        "Model_B": model_b,
        "Accuracy_A": accuracy_a,
        "Accuracy_B": accuracy_b,
        "Delta_Accuracy_A_minus_B":
            accuracy_delta,
        "A_Correct_B_Wrong":
            a_correct_b_wrong,
        "A_Wrong_B_Correct":
            a_wrong_b_correct,
        "Discordant_Total":
            discordant_total,
        "Exact_P_Value":
            p_value,
        "Favored_Model":
            favored_model
    })


mcnemar_results = pd.DataFrame(
    mcnemar_records
)

mcnemar_results[
    "P_Value_Holm"
] = holm_adjust(
    mcnemar_results[
        "Exact_P_Value"
    ].to_numpy()
)

mcnemar_results[
    "Significant_Holm_0_05"
] = (
    mcnemar_results[
        "P_Value_Holm"
    ] < ALPHA
)


# ------------------------------------------------------------
# 9. Standardize comparisons versus XGBoost
# ------------------------------------------------------------

def delong_against_reference(
    table,
    reference_model
):

    records = []

    relevant = table[
        (
            table["Model_A"]
            == reference_model
        )
        |
        (
            table["Model_B"]
            == reference_model
        )
    ]

    for _, row in relevant.iterrows():

        if row["Model_B"] == reference_model:

            comparator = row["Model_A"]

            delta = (
                row["Delta_A_minus_B"]
            )

            ci_lower = row["CI_Lower"]
            ci_upper = row["CI_Upper"]

        else:

            comparator = row["Model_B"]

            delta = -row[
                "Delta_A_minus_B"
            ]

            ci_lower = -row[
                "CI_Upper"
            ]

            ci_upper = -row[
                "CI_Lower"
            ]

        records.append({
            "Comparator": comparator,
            "Comparator_ROC_AUC":
                float(
                    point_estimate_lookup.loc[
                        comparator,
                        "ROC_AUC"
                    ]
                ),
            "XGBoost_ROC_AUC":
                float(
                    point_estimate_lookup.loc[
                        reference_model,
                        "ROC_AUC"
                    ]
                ),
            "Delta_Comparator_minus_XGBoost":
                float(delta),
            "CI_Lower":
                float(ci_lower),
            "CI_Upper":
                float(ci_upper),
            "DeLong_P_Value":
                float(
                    row["P_Value"]
                ),
            "P_Value_Holm":
                float(
                    row["P_Value_Holm"]
                ),
            "Significant_Holm_0_05":
                bool(
                    row[
                        "Significant_Holm_0_05"
                    ]
                )
        })

    return (
        pd.DataFrame(records)
        .sort_values(
            "Delta_Comparator_minus_XGBoost",
            ascending=False
        )
        .reset_index(drop=True)
    )


def bootstrap_against_reference(
    table,
    reference_model
):

    records = []

    relevant = table[
        (
            table["Model_A"]
            == reference_model
        )
        |
        (
            table["Model_B"]
            == reference_model
        )
    ]

    for _, row in relevant.iterrows():

        if row["Model_B"] == reference_model:

            comparator = row["Model_A"]

            delta = row[
                "Delta_A_minus_B"
            ]

            ci_lower = row[
                "CI_Lower"
            ]

            ci_upper = row[
                "CI_Upper"
            ]

        else:

            comparator = row["Model_B"]

            delta = -row[
                "Delta_A_minus_B"
            ]

            ci_lower = -row[
                "CI_Upper"
            ]

            ci_upper = -row[
                "CI_Lower"
            ]

        records.append({
            "Metric": row["Metric"],
            "Higher_Is_Better":
                row["Higher_Is_Better"],
            "Comparator": comparator,
            "Delta_Comparator_minus_XGBoost":
                float(delta),
            "CI_Lower":
                float(ci_lower),
            "CI_Upper":
                float(ci_upper),
            "Bootstrap_P_Value":
                float(
                    row[
                        "Bootstrap_P_Value"
                    ]
                ),
            "P_Value_Holm":
                float(
                    row["P_Value_Holm"]
                ),
            "Significant_Holm_0_05":
                bool(
                    row[
                        "Significant_Holm_0_05"
                    ]
                )
        })

    return (
        pd.DataFrame(records)
        .sort_values(
            [
                "Metric",
                "Delta_Comparator_minus_XGBoost"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(drop=True)
    )


def mcnemar_against_reference(
    table,
    reference_model
):

    records = []

    relevant = table[
        (
            table["Model_A"]
            == reference_model
        )
        |
        (
            table["Model_B"]
            == reference_model
        )
    ]

    for _, row in relevant.iterrows():

        if row["Model_B"] == reference_model:

            comparator = row["Model_A"]

            delta = row[
                "Delta_Accuracy_A_minus_B"
            ]

            comparator_correct_xgb_wrong = (
                row[
                    "A_Correct_B_Wrong"
                ]
            )

            comparator_wrong_xgb_correct = (
                row[
                    "A_Wrong_B_Correct"
                ]
            )

        else:

            comparator = row["Model_B"]

            delta = -row[
                "Delta_Accuracy_A_minus_B"
            ]

            comparator_correct_xgb_wrong = (
                row[
                    "A_Wrong_B_Correct"
                ]
            )

            comparator_wrong_xgb_correct = (
                row[
                    "A_Correct_B_Wrong"
                ]
            )

        records.append({
            "Comparator": comparator,
            "Delta_Accuracy_Comparator_minus_XGBoost":
                float(delta),
            "Comparator_Correct_XGBoost_Wrong":
                int(
                    comparator_correct_xgb_wrong
                ),
            "Comparator_Wrong_XGBoost_Correct":
                int(
                    comparator_wrong_xgb_correct
                ),
            "Exact_P_Value":
                float(
                    row[
                        "Exact_P_Value"
                    ]
                ),
            "P_Value_Holm":
                float(
                    row["P_Value_Holm"]
                ),
            "Significant_Holm_0_05":
                bool(
                    row[
                        "Significant_Holm_0_05"
                    ]
                )
        })

    return (
        pd.DataFrame(records)
        .sort_values(
            "Delta_Accuracy_Comparator_minus_XGBoost",
            ascending=False
        )
        .reset_index(drop=True)
    )


delong_vs_xgboost = (
    delong_against_reference(
        delong_results,
        REFERENCE_MODEL
    )
)

bootstrap_vs_xgboost = (
    bootstrap_against_reference(
        bootstrap_pairwise,
        REFERENCE_MODEL
    )
)

mcnemar_vs_xgboost = (
    mcnemar_against_reference(
        mcnemar_results,
        REFERENCE_MODEL
    )
)


# ------------------------------------------------------------
# 10. Significant-result summaries
# ------------------------------------------------------------

significant_delong = (
    delong_results[
        delong_results[
            "Significant_Holm_0_05"
        ]
    ]
    .copy()
    .sort_values(
        "P_Value_Holm"
    )
)

significant_bootstrap = (
    bootstrap_pairwise[
        bootstrap_pairwise[
            "Significant_Holm_0_05"
        ]
    ]
    .copy()
    .sort_values(
        [
            "Metric",
            "P_Value_Holm"
        ]
    )
)

significant_mcnemar = (
    mcnemar_results[
        mcnemar_results[
            "Significant_Holm_0_05"
        ]
    ]
    .copy()
    .sort_values(
        "P_Value_Holm"
    )
)


significance_counts = pd.DataFrame({
    "Test_Family": [
        "DeLong ROC-AUC",
        *[
            f"Paired bootstrap {metric}"
            for metric
            in BOOTSTRAP_METRICS
        ],
        "Exact McNemar accuracy"
    ],

    "Pairwise_Tests": [
        len(delong_results),
        *[
            len(
                bootstrap_pairwise[
                    bootstrap_pairwise[
                        "Metric"
                    ] == metric
                ]
            )
            for metric
            in BOOTSTRAP_METRICS
        ],
        len(mcnemar_results)
    ],

    "Holm_Significant_Pairs": [
        int(
            delong_results[
                "Significant_Holm_0_05"
            ].sum()
        ),
        *[
            int(
                bootstrap_pairwise.loc[
                    bootstrap_pairwise[
                        "Metric"
                    ] == metric,
                    "Significant_Holm_0_05"
                ].sum()
            )
            for metric
            in BOOTSTRAP_METRICS
        ],
        int(
            mcnemar_results[
                "Significant_Holm_0_05"
            ].sum()
        )
    ]
})


# ------------------------------------------------------------
# 11. Print compact results
# ------------------------------------------------------------

print("\n" + "=" * 145)
print("DELONG ROC-AUC COMPARISONS AGAINST XGBOOST")
print(
    "Delta = comparator ROC-AUC minus "
    "XGBoost ROC-AUC."
)
print("=" * 145)

print(
    delong_vs_xgboost
    .round(6)
    .to_string(index=False)
)


selected_bootstrap_metrics = [
    "PR_AUC",
    "Brier_Score",
    "F1",
    "MCC"
]

print("\n" + "=" * 155)
print(
    "PAIRED BOOTSTRAP COMPARISONS "
    "AGAINST XGBOOST"
)
print(
    "Deltas are comparator minus XGBoost. "
    "Lower is better for Brier Score."
)
print("=" * 155)

print(
    bootstrap_vs_xgboost[
        bootstrap_vs_xgboost[
            "Metric"
        ].isin(
            selected_bootstrap_metrics
        )
    ]
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 130)
print("EXACT MCNEMAR COMPARISONS AGAINST XGBOOST")
print(
    "Uses classifications produced by "
    "the locked model-specific thresholds."
)
print("=" * 130)

print(
    mcnemar_vs_xgboost
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 100)
print("HOLM-SIGNIFICANT PAIR COUNTS")
print("=" * 100)

print(
    significance_counts
    .to_string(index=False)
)


print("\n" + "=" * 120)
print("HOLM-SIGNIFICANT DELONG PAIRS")
print("=" * 120)

if significant_delong.empty:

    print("None")

else:

    print(
        significant_delong[
            [
                "Model_A",
                "Model_B",
                "Delta_A_minus_B",
                "CI_Lower",
                "CI_Upper",
                "P_Value",
                "P_Value_Holm",
                "Higher_ROC_AUC_Model"
            ]
        ]
        .round(6)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 12. Save complete statistical artifacts
# ------------------------------------------------------------

delong_results.to_csv(
    STATS_DIR
    / "delong_all_pairwise_roc_auc.csv",
    index=False
)

bootstrap_pairwise.to_csv(
    STATS_DIR
    / "paired_bootstrap_all_pairwise_metrics.csv",
    index=False
)

mcnemar_results.to_csv(
    STATS_DIR
    / "mcnemar_all_pairwise_locked_predictions.csv",
    index=False
)

delong_vs_xgboost.to_csv(
    STATS_DIR
    / "delong_comparisons_vs_xgboost.csv",
    index=False
)

bootstrap_vs_xgboost.to_csv(
    STATS_DIR
    / "paired_bootstrap_comparisons_vs_xgboost.csv",
    index=False
)

mcnemar_vs_xgboost.to_csv(
    STATS_DIR
    / "mcnemar_comparisons_vs_xgboost.csv",
    index=False
)

significant_delong.to_csv(
    STATS_DIR
    / "holm_significant_delong_pairs.csv",
    index=False
)

significant_bootstrap.to_csv(
    STATS_DIR
    / "holm_significant_bootstrap_pairs.csv",
    index=False
)

significant_mcnemar.to_csv(
    STATS_DIR
    / "holm_significant_mcnemar_pairs.csv",
    index=False
)

significance_counts.to_csv(
    STATS_DIR
    / "holm_significant_pair_counts.csv",
    index=False
)


run_summary = {
    "test_rows":
        int(len(y_true)),

    "test_positive_count":
        int(y_true.sum()),

    "models":
        MODEL_ORDER,

    "pairwise_model_comparisons":
        len(MODEL_PAIRS),

    "delong_family_holm_tests":
        len(delong_results),

    "bootstrap_holm_tests_per_metric":
        len(MODEL_PAIRS),

    "mcnemar_family_holm_tests":
        len(mcnemar_results),

    "alpha":
        ALPHA,

    "reference_model":
        REFERENCE_MODEL,

    "post_test_tuning_performed":
        False
}

with open(
    STATS_DIR
    / "pairwise_inference_run_summary.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        run_summary,
        file,
        indent=4
    )


print("\n" + "=" * 105)
print("CHECKPOINT 8A2 SAVED")
print("=" * 105)

print("Output directory:")
print(STATS_DIR)

print(
    "\nNo model, calibrator, "
    "hyperparameter or threshold was changed."
)

In [ ]:
# ============================================================
# MULTICOLLINEARITY AND FEATURE-ASSOCIATION AUDIT
#
# Uses development observations only.
# No model training, tuning, or test-result modification.
# ============================================================

from pathlib import Path
import hashlib
import json
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

EXPECTED_DATA_SHA256 = (
    "3996cd1fa372e0db0cd9c0ebac35bbd4"
    "e8e3c65fb942bb010c826e7b1eeef0a0"
)

EXPECTED_COLUMNS = [
    "RowNumber",
    "CustomerId",
    "Surname",
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Exited"
]

MODEL_FEATURES = [
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

NUMERIC_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

OUTPUT_DIR = Path(
    "/kaggle/working/explainability_audit"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Utilities
# ------------------------------------------------------------

def sha256_file(filepath, block_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(filepath, "rb") as file:

        while True:

            block = file.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def find_unique_file(
    root,
    filename,
    preferred_parent=None
):

    matches = list(
        root.rglob(filename)
    )

    if preferred_parent is not None:

        preferred_matches = [
            path
            for path in matches
            if preferred_parent in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}."
        )

    raise RuntimeError(
        f"Multiple copies of {filename} found:\n"
        + "\n".join(str(path) for path in matches)
    )


# ------------------------------------------------------------
# 3. Locate the exact original dataset
# ------------------------------------------------------------

observed_dataset_hash = sha256_file(DATASET_PATH)

if observed_dataset_hash != EXPECTED_DATA_SHA256:
    raise RuntimeError(
        "DATASET_PATH does not match the dataset used in the study. "
        f"Observed SHA-256: {observed_dataset_hash}"
    )

print("=" * 105)
print("LOCKED DATASET VERIFICATION")
print("=" * 105)

print("Dataset:", DATASET_PATH)
print("SHA-256:", sha256_file(DATASET_PATH))
print("Hash match:", True)


# ------------------------------------------------------------
# 4. Locate restored final predictions
# ------------------------------------------------------------

RESTORED_ROOT = Path("/kaggle/working")

FINAL_PREDICTIONS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_locked_test_predictions.csv",
    preferred_parent="final_locked_holdout"
)


# ------------------------------------------------------------
# 5. Reconstruct development observations
# ------------------------------------------------------------

data = pd.read_csv(
    DATASET_PATH
)

assert list(data.columns) == EXPECTED_COLUMNS
assert len(data) == 10000
assert data.index.is_unique

final_predictions = pd.read_csv(
    FINAL_PREDICTIONS_PATH
)

test_indices = (
    final_predictions["Original_Index"]
    .astype(int)
    .to_numpy()
)

assert len(test_indices) == 2000
assert len(np.unique(test_indices)) == 2000

# Confirm that restored test indices still point
# to the same customers in the original dataset.
assert np.array_equal(
    data.loc[
        test_indices,
        "CustomerId"
    ].to_numpy(),
    final_predictions[
        "CustomerId"
    ].to_numpy()
)

development_mask = ~data.index.isin(
    test_indices
)

development_data = (
    data.loc[development_mask]
    .copy()
)

assert len(development_data) == 8000

assert (
    development_data["Exited"]
    .value_counts()
    .sort_index()
    .to_dict()
    == {
        0: 6370,
        1: 1630
    }
)

X_development = development_data[
    MODEL_FEATURES
].copy()


print("\n" + "=" * 105)
print("DEVELOPMENT DATA RECONSTRUCTION")
print("=" * 105)

print("Development rows:", len(X_development))
print("Test rows excluded:", len(test_indices))
print(
    "Development churn rate:",
    round(
        development_data["Exited"].mean(),
        6
    )
)


# ------------------------------------------------------------
# 6. Numeric correlation matrices
# ------------------------------------------------------------

pearson_matrix = (
    X_development[
        NUMERIC_FEATURES
    ]
    .corr(method="pearson")
)

spearman_matrix = (
    X_development[
        NUMERIC_FEATURES
    ]
    .corr(method="spearman")
)


def extract_correlation_pairs(
    correlation_matrix,
    method_name
):

    records = []

    for feature_a, feature_b in combinations(
        correlation_matrix.columns,
        2
    ):

        correlation = float(
            correlation_matrix.loc[
                feature_a,
                feature_b
            ]
        )

        records.append({
            "Method": method_name,
            "Feature_A": feature_a,
            "Feature_B": feature_b,
            "Correlation": correlation,
            "Absolute_Correlation":
                abs(correlation)
        })

    return pd.DataFrame(records)


pearson_pairs = extract_correlation_pairs(
    pearson_matrix,
    "Pearson"
)

spearman_pairs = extract_correlation_pairs(
    spearman_matrix,
    "Spearman"
)

correlation_pairs = pd.concat(
    [
        pearson_pairs,
        spearman_pairs
    ],
    ignore_index=True
)

high_correlation_pairs = (
    correlation_pairs[
        correlation_pairs[
            "Absolute_Correlation"
        ] >= 0.70
    ]
    .sort_values(
        "Absolute_Correlation",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. One-hot design matrix for VIF
# ------------------------------------------------------------

design_matrix = pd.get_dummies(
    X_development,
    columns=CATEGORICAL_FEATURES,
    drop_first=True,
    dtype=float
)

design_matrix = design_matrix.astype(
    float
)

standard_deviation = design_matrix.std(
    axis=0,
    ddof=0
)

constant_columns = standard_deviation[
    standard_deviation <= 1e-15
].index.tolist()

assert len(constant_columns) == 0

standardized_design = (
    design_matrix
    - design_matrix.mean(axis=0)
) / standard_deviation

standardized_array = (
    standardized_design
    .to_numpy(dtype=float)
)


# ------------------------------------------------------------
# 8. Variance Inflation Factors
# ------------------------------------------------------------

def calculate_vif(
    standardized_frame
):

    values = standardized_frame.to_numpy(
        dtype=float
    )

    column_names = (
        standardized_frame.columns.tolist()
    )

    records = []

    for feature_position, feature_name in enumerate(
        column_names
    ):

        target = values[
            :,
            feature_position
        ]

        predictors = np.delete(
            values,
            feature_position,
            axis=1
        )

        predictors_with_intercept = np.column_stack([
            np.ones(len(predictors)),
            predictors
        ])

        coefficients, _, _, _ = np.linalg.lstsq(
            predictors_with_intercept,
            target,
            rcond=None
        )

        fitted = (
            predictors_with_intercept
            @ coefficients
        )

        residual_sum_squares = np.sum(
            (target - fitted) ** 2
        )

        total_sum_squares = np.sum(
            (
                target
                - target.mean()
            ) ** 2
        )

        r_squared = (
            1.0
            - residual_sum_squares
            / total_sum_squares
        )

        r_squared = min(
            max(r_squared, 0.0),
            1.0
        )

        vif = (
            np.inf
            if 1.0 - r_squared <= 1e-12
            else 1.0 / (1.0 - r_squared)
        )

        if vif < 5:
            interpretation = "Low"
        elif vif < 10:
            interpretation = "Moderate"
        else:
            interpretation = "High"

        records.append({
            "Feature": feature_name,
            "R_Squared_Against_Other_Features":
                r_squared,
            "VIF": vif,
            "VIF_Interpretation":
                interpretation
        })

    return (
        pd.DataFrame(records)
        .sort_values(
            "VIF",
            ascending=False
        )
        .reset_index(drop=True)
    )


vif_table = calculate_vif(
    standardized_design
)


# ------------------------------------------------------------
# 9. Condition number
# ------------------------------------------------------------

singular_values = np.linalg.svd(
    standardized_array,
    compute_uv=False
)

condition_number = float(
    singular_values.max()
    / singular_values.min()
)

if condition_number < 30:
    condition_interpretation = "Low"
elif condition_number < 100:
    condition_interpretation = "Moderate"
else:
    condition_interpretation = "High"


# ------------------------------------------------------------
# 10. Cramér's V for categorical features
# ------------------------------------------------------------

def corrected_cramers_v(
    series_a,
    series_b
):

    contingency = pd.crosstab(
        series_a,
        series_b
    )

    chi_squared = chi2_contingency(
        contingency,
        correction=False
    )[0]

    n = contingency.to_numpy().sum()

    phi_squared = chi_squared / n

    rows, columns = contingency.shape

    corrected_phi_squared = max(
        0.0,
        phi_squared
        - (
            (columns - 1)
            * (rows - 1)
        )
        / (n - 1)
    )

    corrected_rows = (
        rows
        - (
            (rows - 1) ** 2
        )
        / (n - 1)
    )

    corrected_columns = (
        columns
        - (
            (columns - 1) ** 2
        )
        / (n - 1)
    )

    denominator = min(
        corrected_columns - 1,
        corrected_rows - 1
    )

    return (
        np.sqrt(
            corrected_phi_squared
            / denominator
        )
        if denominator > 0
        else 0.0
    )


categorical_association_records = []

for feature_a, feature_b in combinations(
    CATEGORICAL_FEATURES,
    2
):

    categorical_association_records.append({
        "Feature_A": feature_a,
        "Feature_B": feature_b,
        "Corrected_Cramers_V":
            corrected_cramers_v(
                X_development[feature_a],
                X_development[feature_b]
            )
    })


categorical_associations = pd.DataFrame(
    categorical_association_records
)


# ------------------------------------------------------------
# 11. Summary
# ------------------------------------------------------------

maximum_absolute_pearson = float(
    pearson_pairs[
        "Absolute_Correlation"
    ].max()
)

maximum_absolute_spearman = float(
    spearman_pairs[
        "Absolute_Correlation"
    ].max()
)

maximum_vif = float(
    vif_table["VIF"].max()
)

features_vif_over_5 = int(
    (vif_table["VIF"] >= 5).sum()
)

features_vif_over_10 = int(
    (vif_table["VIF"] >= 10).sum()
)


audit_summary = pd.DataFrame({
    "Measure": [
        "Development rows",
        "Encoded feature count",
        "Maximum absolute Pearson correlation",
        "Maximum absolute Spearman correlation",
        "Pairs with absolute correlation >= 0.70",
        "Maximum VIF",
        "Features with VIF >= 5",
        "Features with VIF >= 10",
        "Standardized design condition number",
        "Condition-number interpretation"
    ],
    "Value": [
        len(X_development),
        design_matrix.shape[1],
        maximum_absolute_pearson,
        maximum_absolute_spearman,
        len(high_correlation_pairs),
        maximum_vif,
        features_vif_over_5,
        features_vif_over_10,
        condition_number,
        condition_interpretation
    ]
})


print("\n" + "=" * 105)
print("MULTICOLLINEARITY AUDIT SUMMARY")
print("=" * 105)

print(
    audit_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 105)
print("VIF TABLE")
print("=" * 105)

print(
    vif_table
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 105)
print("HIGH-CORRELATION PAIRS | ABSOLUTE CORRELATION >= 0.70")
print("=" * 105)

if high_correlation_pairs.empty:

    print("None")

else:

    print(
        high_correlation_pairs
        .round(6)
        .to_string(index=False)
    )


print("\n" + "=" * 105)
print("CATEGORICAL ASSOCIATIONS")
print("=" * 105)

print(
    categorical_associations
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 12. Save artifacts
# ------------------------------------------------------------

pearson_matrix.to_csv(
    OUTPUT_DIR
    / "numeric_pearson_correlation_matrix.csv"
)

spearman_matrix.to_csv(
    OUTPUT_DIR
    / "numeric_spearman_correlation_matrix.csv"
)

correlation_pairs.to_csv(
    OUTPUT_DIR
    / "all_numeric_correlation_pairs.csv",
    index=False
)

high_correlation_pairs.to_csv(
    OUTPUT_DIR
    / "high_correlation_pairs.csv",
    index=False
)

vif_table.to_csv(
    OUTPUT_DIR
    / "variance_inflation_factors.csv",
    index=False
)

categorical_associations.to_csv(
    OUTPUT_DIR
    / "categorical_cramers_v.csv",
    index=False
)

audit_summary.to_csv(
    OUTPUT_DIR
    / "multicollinearity_audit_summary.csv",
    index=False
)

run_summary = {
    "dataset_path":
        str(DATASET_PATH),

    "dataset_sha256":
        EXPECTED_DATA_SHA256,

    "development_rows":
        int(len(X_development)),

    "test_rows_excluded":
        int(len(test_indices)),

    "encoded_feature_count":
        int(design_matrix.shape[1]),

    "maximum_absolute_pearson":
        maximum_absolute_pearson,

    "maximum_absolute_spearman":
        maximum_absolute_spearman,

    "maximum_vif":
        maximum_vif,

    "condition_number":
        condition_number,

    "post_test_tuning_performed":
        False
}

with open(
    OUTPUT_DIR
    / "multicollinearity_run_summary.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        run_summary,
        file,
        indent=4
    )


print("\nSaved directory:")
print(OUTPUT_DIR)

print(
    "\nNo model, calibrator, threshold "
    "or test result was changed."
)

In [ ]:
# ============================================================
# REPRESENTATIVE MODEL RESTORATION
# + PREDICTION FIDELITY CHECK
# + SHAP COHORT PREPARATION
#
# No model training, tuning, threshold changes,
# or holdout-performance changes.
# ============================================================

from pathlib import Path
import hashlib
import json
import joblib

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import (
    StratifiedShuffleSplit
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

EXPECTED_DATA_SHA256 = (
    "3996cd1fa372e0db0cd9c0ebac35bbd4"
    "e8e3c65fb942bb010c826e7b1eeef0a0"
)

RESTORED_ROOT_CANDIDATES = [Path("/kaggle/working")]

OUTPUT_DIR = Path(
    "/kaggle/working/explainability_shap"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_FEATURES = [
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

CATEGORICAL_FEATURES = [
    "Geography",
    "Gender"
]

NUMERIC_FEATURES = [
    feature
    for feature in MODEL_FEATURES
    if feature not in CATEGORICAL_FEATURES
]

REPRESENTATIVE_MODELS = [
    "XGBoost",
    "CatBoost",
    "MLP",
    "FTTransformer"
]

EXPLANATION_SAMPLE_SIZE = 300
BACKGROUND_SAMPLE_SIZE = 100
EXPLANATION_SEED = 2031
BACKGROUND_SEED = 2032

PREDICTION_TOLERANCE = 1e-5

torch.set_num_threads(4)


# ------------------------------------------------------------
# 2. Utilities
# ------------------------------------------------------------

def sha256_file(
    filepath,
    block_size=1024 * 1024
):
    digest = hashlib.sha256()

    with open(filepath, "rb") as file:

        while True:
            block = file.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def find_unique_file(
    root,
    filename,
    preferred_parent=None
):
    matches = list(
        root.rglob(filename)
    )

    if preferred_parent is not None:

        preferred_matches = [
            path
            for path in matches
            if preferred_parent in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} "
            f"under {root}."
        )

    raise RuntimeError(
        f"Found multiple copies of {filename}:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


existing_roots = [
    path
    for path in RESTORED_ROOT_CANDIDATES
    if path.exists()
]

if not existing_roots:
    raise FileNotFoundError(
        "Working artifact directory was not found."
    )

RESTORED_ROOT = existing_roots[0]


# ------------------------------------------------------------
# 3. Verify and load original data
# ------------------------------------------------------------

observed_hash = sha256_file(
    DATASET_PATH
)

assert (
    observed_hash
    == EXPECTED_DATA_SHA256
)

data = pd.read_csv(
    DATASET_PATH
)

assert len(data) == 10000

print("=" * 105)
print("DATA AND ARTIFACT VERIFICATION")
print("=" * 105)

print("Dataset:", DATASET_PATH)
print("Dataset hash match:", True)
print("Artifact root:", RESTORED_ROOT)


# ------------------------------------------------------------
# 4. Locate restored artifacts
# ------------------------------------------------------------

FINAL_PREDICTIONS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_locked_test_predictions.csv",
    preferred_parent="final_locked_holdout"
)

RAW_PREDICTIONS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_test_raw_probabilities.csv",
    preferred_parent="final_locked_holdout"
)

XGB_ARTIFACT_PATH = find_unique_file(
    RESTORED_ROOT,
    "XGBoost.joblib",
    preferred_parent="model_artifacts"
)

CATBOOST_ARTIFACT_PATH = find_unique_file(
    RESTORED_ROOT,
    "CatBoost.joblib",
    preferred_parent="model_artifacts"
)

MLP_PREPROCESSOR_PATH = find_unique_file(
    RESTORED_ROOT,
    "MLP_preprocessor.joblib",
    preferred_parent="model_artifacts"
)

MLP_STATE_PATH = find_unique_file(
    RESTORED_ROOT,
    "MLP_state.pt",
    preferred_parent="model_artifacts"
)

FTT_ARTIFACT_PATH = find_unique_file(
    RESTORED_ROOT,
    "FTTransformer.joblib",
    preferred_parent="model_artifacts"
)


artifact_paths = {
    "Final predictions":
        FINAL_PREDICTIONS_PATH,

    "Raw predictions":
        RAW_PREDICTIONS_PATH,

    "XGBoost artifact":
        XGB_ARTIFACT_PATH,

    "CatBoost artifact":
        CATBOOST_ARTIFACT_PATH,

    "MLP preprocessor":
        MLP_PREPROCESSOR_PATH,

    "MLP state":
        MLP_STATE_PATH,

    "FTTransformer artifact":
        FTT_ARTIFACT_PATH
}

print("\nLocated artifacts:")

for artifact_name, artifact_path in (
    artifact_paths.items()
):
    print(
        f"{artifact_name}: "
        f"{artifact_path}"
    )


# ------------------------------------------------------------
# 5. Reconstruct development and test data
# ------------------------------------------------------------

final_predictions = pd.read_csv(
    FINAL_PREDICTIONS_PATH
)

raw_predictions = pd.read_csv(
    RAW_PREDICTIONS_PATH
)

test_indices = (
    final_predictions[
        "Original_Index"
    ]
    .astype(int)
    .to_numpy()
)

assert len(test_indices) == 2000
assert len(np.unique(test_indices)) == 2000

X_test = (
    data.loc[
        test_indices,
        MODEL_FEATURES
    ]
    .copy()
    .reset_index(drop=True)
)

y_test = (
    final_predictions["y_true"]
    .astype(int)
    .to_numpy()
)

test_customer_ids = (
    final_predictions["CustomerId"]
    .to_numpy()
)

assert np.array_equal(
    data.loc[
        test_indices,
        "CustomerId"
    ].to_numpy(),
    test_customer_ids
)

development_mask = ~data.index.isin(
    test_indices
)

development_data = (
    data.loc[development_mask]
    .copy()
)

X_development = (
    development_data[
        MODEL_FEATURES
    ]
    .copy()
    .reset_index(drop=True)
)

y_development = (
    development_data["Exited"]
    .astype(int)
    .to_numpy()
)

assert len(X_development) == 8000
assert y_development.sum() == 1630


# ------------------------------------------------------------
# 6. Create numeric explanation representation
#
# Categorical features are represented by integer codes.
# The prediction wrappers decode the codes back to the
# original category labels before calling each model.
# ------------------------------------------------------------

category_levels = {
    feature: sorted(
        data[feature]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )
    for feature in CATEGORICAL_FEATURES
}

category_to_code = {
    feature: {
        category: code
        for code, category in enumerate(
            category_levels[feature]
        )
    }
    for feature in CATEGORICAL_FEATURES
}

code_to_category = {
    feature: {
        code: category
        for code, category in enumerate(
            category_levels[feature]
        )
    }
    for feature in CATEGORICAL_FEATURES
}


def encode_explanation_frame(
    dataframe
):
    encoded = pd.DataFrame(
        index=dataframe.index
    )

    for feature in MODEL_FEATURES:

        if feature in CATEGORICAL_FEATURES:

            encoded[feature] = (
                dataframe[feature]
                .astype(str)
                .map(
                    category_to_code[
                        feature
                    ]
                )
                .astype(float)
            )

        else:

            encoded[feature] = (
                dataframe[feature]
                .astype(float)
            )

    return encoded[
        MODEL_FEATURES
    ]


def decode_explanation_matrix(
    matrix
):
    matrix = np.asarray(
        matrix,
        dtype=float
    )

    if matrix.ndim == 1:
        matrix = matrix.reshape(1, -1)

    decoded = pd.DataFrame(
        matrix,
        columns=MODEL_FEATURES
    )

    for feature in NUMERIC_FEATURES:
        decoded[feature] = (
            decoded[feature]
            .astype(float)
        )

    for feature in CATEGORICAL_FEATURES:

        maximum_code = (
            len(
                category_levels[
                    feature
                ]
            )
            - 1
        )

        rounded_codes = (
            np.rint(
                decoded[feature]
                .to_numpy(dtype=float)
            )
            .astype(int)
        )

        rounded_codes = np.clip(
            rounded_codes,
            0,
            maximum_code
        )

        decoded[feature] = [
            code_to_category[
                feature
            ][int(code)]
            for code in rounded_codes
        ]

    return decoded[
        MODEL_FEATURES
    ]


X_test_encoded = (
    encode_explanation_frame(
        X_test
    )
)

X_development_encoded = (
    encode_explanation_frame(
        X_development
    )
)


print("\nCategory mappings:")

for feature in CATEGORICAL_FEATURES:
    print(
        feature,
        category_to_code[feature]
    )


# ------------------------------------------------------------
# 7. Load XGBoost and CatBoost
# ------------------------------------------------------------

xgb_bundle = joblib.load(
    XGB_ARTIFACT_PATH
)

xgb_preprocessor = (
    xgb_bundle["preprocessor"]
)

xgb_model = (
    xgb_bundle["model"]
)

catboost_model = joblib.load(
    CATBOOST_ARTIFACT_PATH
)


# ------------------------------------------------------------
# 8. Restore MLP
# ------------------------------------------------------------

class RestoredMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout,
        use_batch_norm,
        activation_name
    ):
        super().__init__()

        activation_classes = {
            "relu": nn.ReLU,
            "gelu": nn.GELU,
            "silu": nn.SiLU
        }

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:

            layers.append(
                nn.Linear(
                    previous_dim,
                    hidden_dim
                )
            )

            if use_batch_norm:

                layers.append(
                    nn.BatchNorm1d(
                        hidden_dim
                    )
                )

            layers.append(
                activation_classes[
                    activation_name
                ]()
            )

            if dropout > 0:

                layers.append(
                    nn.Dropout(
                        dropout
                    )
                )

            previous_dim = hidden_dim

        layers.append(
            nn.Linear(
                previous_dim,
                1
            )
        )

        self.network = nn.Sequential(
            *layers
        )

    def forward(self, x):

        return self.network(
            x
        ).squeeze(1)


mlp_preprocessor = joblib.load(
    MLP_PREPROCESSOR_PATH
)

try:

    mlp_state_bundle = torch.load(
        MLP_STATE_PATH,
        map_location="cpu",
        weights_only=False
    )

except TypeError:

    mlp_state_bundle = torch.load(
        MLP_STATE_PATH,
        map_location="cpu"
    )


mlp_model = RestoredMLP(
    input_dim=int(
        mlp_state_bundle[
            "input_dim"
        ]
    ),

    hidden_dims=tuple(
        int(value)
        for value in mlp_state_bundle[
            "hidden_dims"
        ]
    ),

    dropout=float(
        mlp_state_bundle[
            "dropout"
        ]
    ),

    use_batch_norm=bool(
        mlp_state_bundle[
            "use_batch_norm"
        ]
    ),

    activation_name=str(
        mlp_state_bundle[
            "activation"
        ]
    )
)

mlp_model.load_state_dict(
    mlp_state_bundle[
        "state_dict"
    ]
)

mlp_model.eval()


# ------------------------------------------------------------
# 9. Load FT-Transformer
# ------------------------------------------------------------

ftt_model = joblib.load(
    FTT_ARTIFACT_PATH
)


# ------------------------------------------------------------
# 10. Prediction wrappers in common raw-feature space
# ------------------------------------------------------------

def predict_xgboost(
    encoded_matrix
):
    decoded = (
        decode_explanation_matrix(
            encoded_matrix
        )
    )

    processed = (
        xgb_preprocessor
        .transform(decoded)
    )

    return (
        xgb_model
        .predict_proba(
            processed
        )[:, 1]
    )


def predict_catboost(
    encoded_matrix
):
    decoded = (
        decode_explanation_matrix(
            encoded_matrix
        )
    )

    return (
        catboost_model
        .predict_proba(
            decoded
        )[:, 1]
    )


def predict_mlp(
    encoded_matrix
):
    decoded = (
        decode_explanation_matrix(
            encoded_matrix
        )
    )

    processed = (
        mlp_preprocessor
        .transform(decoded)
        .astype(np.float32)
    )

    with torch.no_grad():

        probabilities = torch.sigmoid(
            mlp_model(
                torch.from_numpy(
                    processed
                )
            )
        ).cpu().numpy()

    return probabilities


def predict_fttransformer(
    encoded_matrix
):
    decoded = (
        decode_explanation_matrix(
            encoded_matrix
        )
    )

    return (
        ftt_model
        .predict_proba(
            decoded
        )[:, 1]
    )


prediction_functions = {
    "XGBoost":
        predict_xgboost,

    "CatBoost":
        predict_catboost,

    "MLP":
        predict_mlp,

    "FTTransformer":
        predict_fttransformer
}


# ------------------------------------------------------------
# 11. Compare restored predictions with immutable raw outputs
# ------------------------------------------------------------

encoded_test_matrix = (
    X_test_encoded
    .to_numpy(dtype=float)
)

fidelity_records = []

for model_name in REPRESENTATIVE_MODELS:

    restored_probability = (
        prediction_functions[
            model_name
        ](
            encoded_test_matrix
        )
    )

    saved_probability = (
        raw_predictions[
            f"{model_name}_"
            "Raw_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    absolute_difference = np.abs(
        restored_probability
        - saved_probability
    )

    correlation = np.corrcoef(
        restored_probability,
        saved_probability
    )[0, 1]

    fidelity_records.append({
        "Model": model_name,

        "Rows":
            len(
                restored_probability
            ),

        "Maximum_Absolute_Difference":
            float(
                absolute_difference.max()
            ),

        "Mean_Absolute_Difference":
            float(
                absolute_difference.mean()
            ),

        "Prediction_Correlation":
            float(correlation),

        "Within_Tolerance":
            bool(
                absolute_difference.max()
                <= PREDICTION_TOLERANCE
            )
    })


fidelity_table = pd.DataFrame(
    fidelity_records
)


print("\n" + "=" * 120)
print("RESTORED MODEL PREDICTION FIDELITY")
print("=" * 120)

print(
    fidelity_table
    .round(12)
    .to_string(index=False)
)

assert fidelity_table[
    "Within_Tolerance"
].all(), (
    "At least one restored model does not "
    "reproduce its locked raw predictions."
)


# ------------------------------------------------------------
# 12. Create natural-prevalence SHAP explanation cohort
# ------------------------------------------------------------

explanation_splitter = (
    StratifiedShuffleSplit(
        n_splits=1,
        train_size=
            EXPLANATION_SAMPLE_SIZE,
        random_state=
            EXPLANATION_SEED
    )
)

explanation_positions, _ = next(
    explanation_splitter.split(
        X_test_encoded,
        y_test
    )
)


background_splitter = (
    StratifiedShuffleSplit(
        n_splits=1,
        train_size=
            BACKGROUND_SAMPLE_SIZE,
        random_state=
            BACKGROUND_SEED
    )
)

background_positions, _ = next(
    background_splitter.split(
        X_development_encoded,
        y_development
    )
)


X_explanation = (
    X_test_encoded.iloc[
        explanation_positions
    ]
    .copy()
    .reset_index(drop=True)
)

X_background = (
    X_development_encoded.iloc[
        background_positions
    ]
    .copy()
    .reset_index(drop=True)
)


explanation_cohort = pd.DataFrame({
    "Explanation_Row":
        np.arange(
            len(explanation_positions)
        ),

    "Test_Position":
        explanation_positions,

    "Original_Index":
        test_indices[
            explanation_positions
        ],

    "CustomerId":
        test_customer_ids[
            explanation_positions
        ],

    "y_true":
        y_test[
            explanation_positions
        ]
})


background_cohort = pd.DataFrame({
    "Background_Row":
        np.arange(
            len(background_positions)
        ),

    "Development_Position":
        background_positions,

    "Original_Index":
        development_data.index.to_numpy()[
            background_positions
        ],

    "y_true":
        y_development[
            background_positions
        ]
})


cohort_summary = pd.DataFrame({
    "Cohort": [
        "Full locked test",
        "SHAP explanation sample",
        "Full development set",
        "SHAP background sample"
    ],

    "Rows": [
        len(y_test),
        len(explanation_positions),
        len(y_development),
        len(background_positions)
    ],

    "Positive_Count": [
        int(y_test.sum()),
        int(
            y_test[
                explanation_positions
            ].sum()
        ),
        int(y_development.sum()),
        int(
            y_development[
                background_positions
            ].sum()
        )
    ],

    "Positive_Rate": [
        float(y_test.mean()),
        float(
            y_test[
                explanation_positions
            ].mean()
        ),
        float(
            y_development.mean()
        ),
        float(
            y_development[
                background_positions
            ].mean()
        )
    ]
})


print("\n" + "=" * 105)
print("SHAP COHORT SUMMARY")
print("=" * 105)

print(
    cohort_summary
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 13. Save reproducibility artifacts
# ------------------------------------------------------------

fidelity_table.to_csv(
    OUTPUT_DIR
    / "representative_model_prediction_fidelity.csv",
    index=False
)

cohort_summary.to_csv(
    OUTPUT_DIR
    / "shap_cohort_summary.csv",
    index=False
)

explanation_cohort.to_csv(
    OUTPUT_DIR
    / "shap_explanation_cohort_manifest.csv",
    index=False
)

background_cohort.to_csv(
    OUTPUT_DIR
    / "shap_background_cohort_manifest.csv",
    index=False
)

X_explanation.to_csv(
    OUTPUT_DIR
    / "shap_explanation_encoded_features.csv",
    index=False
)

X_background.to_csv(
    OUTPUT_DIR
    / "shap_background_encoded_features.csv",
    index=False
)


mapping_manifest = {
    "model_features":
        MODEL_FEATURES,

    "categorical_features":
        CATEGORICAL_FEATURES,

    "category_levels":
        category_levels,

    "category_to_code":
        category_to_code,

    "explanation_sample_size":
        EXPLANATION_SAMPLE_SIZE,

    "background_sample_size":
        BACKGROUND_SAMPLE_SIZE,

    "explanation_seed":
        EXPLANATION_SEED,

    "background_seed":
        BACKGROUND_SEED,

    "prediction_tolerance":
        PREDICTION_TOLERANCE,

    "representative_models":
        REPRESENTATIVE_MODELS,

    "post_test_tuning_performed":
        False
}


with open(
    OUTPUT_DIR
    / "shap_input_mapping_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        mapping_manifest,
        file,
        indent=4
    )


print("\nSaved directory:")
print(OUTPUT_DIR)

print(
    "\nStage completed. "
    "No model or result was modified."
)

In [ ]:
# ============================================================
# MODEL-AGNOSTIC PERMUTATION SHAP
# + CROSS-MODEL RANK AGREEMENT
# + BOOTSTRAP RANKING STABILITY
#
# Requires Stage variables in memory.
# Explains final calibrated probabilities.
# No model training, tuning, or threshold modification.
# ============================================================

from pathlib import Path
from itertools import combinations
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import shap

from scipy.stats import (
    rankdata,
    spearmanr,
    kendalltau
)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SHAP_OUTPUT_DIR = Path(
    "/kaggle/working/explainability_shap"
)

SHAP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SHAP_VALUE_DIR = (
    SHAP_OUTPUT_DIR
    / "model_shap_values"
)

SHAP_VALUE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPRESENTATIVE_MODELS = [
    "XGBoost",
    "CatBoost",
    "MLP",
    "FTTransformer"
]

MODEL_SEEDS = {
    "XGBoost": 2041,
    "CatBoost": 2042,
    "MLP": 2043,
    "FTTransformer": 2044
}

N_PERMUTATION_CYCLES = 5

N_FEATURES = len(
    MODEL_FEATURES
)

MAX_EVALS = (
    N_PERMUTATION_CYCLES
    * (
        2 * N_FEATURES + 1
    )
)

SHAP_BATCH_SIZE = 256

N_STABILITY_BOOTSTRAPS = 2000
STABILITY_SEED = 2045

ADDITIVITY_TOLERANCE = 1e-4
CALIBRATION_EPSILON = 1e-6

TOP_K_VALUES = [3, 5]


# ------------------------------------------------------------
# 2. State checks
# ------------------------------------------------------------

required_names = [
    "RESTORED_ROOT",
    "MODEL_FEATURES",
    "NUMERIC_FEATURES",
    "CATEGORICAL_FEATURES",
    "category_levels",
    "prediction_functions",
    "X_explanation",
    "X_background",
    "X_test_encoded",
    "final_predictions"
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Stage variables are missing:\n"
        + "\n".join(missing_names)
        + "\nRerun Stage first."
    )


assert list(
    X_explanation.columns
) == MODEL_FEATURES

assert list(
    X_background.columns
) == MODEL_FEATURES

assert len(X_explanation) == 300
assert len(X_background) == 100


print("=" * 110)
print("PERMUTATION SHAP CONFIGURATION")
print("=" * 110)

print("SHAP version:", shap.__version__)
print("Representative models:", REPRESENTATIVE_MODELS)
print("Explanation rows:", len(X_explanation))
print("Background rows:", len(X_background))
print("Features:", N_FEATURES)
print("Permutation cycles:", N_PERMUTATION_CYCLES)
print("Maximum evaluations per row:", MAX_EVALS)
print("Stability bootstraps:", N_STABILITY_BOOTSTRAPS)


# ------------------------------------------------------------
# 3. Locate locked calibrators
# ------------------------------------------------------------

def locate_calibrator(
    model_name
):
    return find_unique_file(
        RESTORED_ROOT,
        f"{model_name}_calibrator.joblib",
        preferred_parent=
            "calibration_threshold_locked"
    )


calibrator_bundles = {}

for model_name in REPRESENTATIVE_MODELS:

    calibrator_path = locate_calibrator(
        model_name
    )

    calibrator_bundles[
        model_name
    ] = joblib.load(
        calibrator_path
    )


print("\nLocked calibration methods:")

for model_name in REPRESENTATIVE_MODELS:

    print(
        model_name,
        calibrator_bundles[
            model_name
        ]["method"]
    )


# ------------------------------------------------------------
# 4. Calibration utilities
# ------------------------------------------------------------

def clip_probability(
    probability
):
    return np.clip(
        np.asarray(
            probability,
            dtype=float
        ),
        CALIBRATION_EPSILON,
        1.0 - CALIBRATION_EPSILON
    )


def probability_to_logit(
    probability
):
    probability = clip_probability(
        probability
    )

    return np.log(
        probability
        / (
            1.0 - probability
        )
    )


def apply_calibrator_bundle(
    raw_probability,
    bundle
):
    raw_probability = clip_probability(
        raw_probability
    )

    method = bundle["method"]

    calibrator = bundle[
        "calibrator"
    ]

    if method == "None":
        return raw_probability

    if method == "Sigmoid":

        calibrated_probability = (
            calibrator.predict_proba(
                probability_to_logit(
                    raw_probability
                ).reshape(-1, 1)
            )[:, 1]
        )

        return clip_probability(
            calibrated_probability
        )

    if method == "Isotonic":

        calibrated_probability = (
            calibrator.predict(
                raw_probability
            )
        )

        return clip_probability(
            calibrated_probability
        )

    raise ValueError(
        f"Unknown calibration method: {method}"
    )


# ------------------------------------------------------------
# 5. Final calibrated prediction wrappers
# ------------------------------------------------------------

def make_calibrated_prediction_function(
    model_name
):
    raw_prediction_function = (
        prediction_functions[
            model_name
        ]
    )

    calibrator_bundle = (
        calibrator_bundles[
            model_name
        ]
    )

    def calibrated_prediction_function(
        encoded_matrix
    ):
        raw_probability = (
            raw_prediction_function(
                encoded_matrix
            )
        )

        return apply_calibrator_bundle(
            raw_probability=
                raw_probability,
            bundle=
                calibrator_bundle
        )

    return calibrated_prediction_function


calibrated_prediction_functions = {
    model_name:
        make_calibrated_prediction_function(
            model_name
        )
    for model_name in REPRESENTATIVE_MODELS
}


# ------------------------------------------------------------
# 6. Fidelity check against immutable calibrated outputs
# ------------------------------------------------------------

full_encoded_test = (
    X_test_encoded
    .to_numpy(dtype=float)
)

calibrated_fidelity_records = []

for model_name in REPRESENTATIVE_MODELS:

    restored_probability = (
        calibrated_prediction_functions[
            model_name
        ](
            full_encoded_test
        )
    )

    saved_probability = (
        final_predictions[
            f"{model_name}_"
            "Calibrated_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    absolute_difference = np.abs(
        restored_probability
        - saved_probability
    )

    calibrated_fidelity_records.append({
        "Model": model_name,
        "Calibration_Method":
            calibrator_bundles[
                model_name
            ]["method"],
        "Maximum_Absolute_Difference":
            float(
                absolute_difference.max()
            ),
        "Mean_Absolute_Difference":
            float(
                absolute_difference.mean()
            ),
        "Prediction_Correlation":
            float(
                np.corrcoef(
                    restored_probability,
                    saved_probability
                )[0, 1]
            ),
        "Within_Tolerance":
            bool(
                absolute_difference.max()
                <= 1e-5
            )
    })


calibrated_fidelity = pd.DataFrame(
    calibrated_fidelity_records
)


print("\n" + "=" * 125)
print("CALIBRATED PIPELINE PREDICTION FIDELITY")
print("=" * 125)

print(
    calibrated_fidelity
    .round(12)
    .to_string(index=False)
)

assert calibrated_fidelity[
    "Within_Tolerance"
].all()


# ------------------------------------------------------------
# 7. Common independent masker
# ------------------------------------------------------------

background_matrix = (
    X_background
    .to_numpy(dtype=float)
)

explanation_matrix = (
    X_explanation
    .to_numpy(dtype=float)
)

independent_masker = (
    shap.maskers.Independent(
        background_matrix,
        max_samples=len(
            background_matrix
        )
    )
)


# ------------------------------------------------------------
# 8. Compute or restore SHAP values
# ------------------------------------------------------------

model_shap_values = {}
model_base_values = {}
model_explanation_predictions = {}

shap_run_records = []

overall_shap_start = (
    time.perf_counter()
)


for model_name in REPRESENTATIVE_MODELS:

    print("\n" + "=" * 110)
    print(
        f"PERMUTATION SHAP: {model_name}"
    )
    print("=" * 110)

    shap_archive_path = (
        SHAP_VALUE_DIR
        / f"{model_name}_permutation_shap.npz"
    )

    if shap_archive_path.exists():

        restored_archive = np.load(
            shap_archive_path,
            allow_pickle=False
        )

        shap_values = (
            restored_archive[
                "shap_values"
            ]
        )

        base_values = (
            restored_archive[
                "base_values"
            ]
        )

        explained_predictions = (
            restored_archive[
                "predictions"
            ]
        )

        elapsed_seconds = float(
            restored_archive[
                "elapsed_seconds"
            ]
        )

        run_status = "RESTORED"

        print(
            "Existing SHAP archive restored."
        )

    else:

        model_start = (
            time.perf_counter()
        )

        explainer = (
            shap.PermutationExplainer(
                model=
                    calibrated_prediction_functions[
                        model_name
                    ],
                masker=
                    independent_masker,
                feature_names=
                    MODEL_FEATURES,
                seed=
                    MODEL_SEEDS[
                        model_name
                    ]
            )
        )

        explanation = explainer(
            explanation_matrix,
            max_evals=MAX_EVALS,
            batch_size=SHAP_BATCH_SIZE,
            silent=False
        )

        shap_values = np.asarray(
            explanation.values,
            dtype=float
        )

        base_values = np.asarray(
            explanation.base_values,
            dtype=float
        )

        explained_predictions = (
            calibrated_prediction_functions[
                model_name
            ](
                explanation_matrix
            )
        )

        if (
            shap_values.ndim == 3
            and
            shap_values.shape[-1] == 1
        ):
            shap_values = (
                shap_values[:, :, 0]
            )

        if (
            base_values.ndim == 2
            and
            base_values.shape[-1] == 1
        ):
            base_values = (
                base_values[:, 0]
            )

        if base_values.ndim == 0:
            base_values = np.repeat(
                base_values,
                len(
                    explanation_matrix
                )
            )

        elapsed_seconds = (
            time.perf_counter()
            - model_start
        )

        np.savez_compressed(
            shap_archive_path,
            shap_values=shap_values,
            base_values=base_values,
            predictions=
                explained_predictions,
            elapsed_seconds=
                np.asarray(
                    elapsed_seconds
                )
        )

        run_status = "COMPUTED"

        del explainer
        del explanation


    assert shap_values.shape == (
        len(X_explanation),
        N_FEATURES
    )

    assert len(base_values) == len(
        X_explanation
    )

    reconstructed_probability = (
        base_values
        + shap_values.sum(axis=1)
    )

    additivity_residual = (
        explained_predictions
        - reconstructed_probability
    )

    maximum_additivity_residual = float(
        np.max(
            np.abs(
                additivity_residual
            )
        )
    )

    model_shap_values[
        model_name
    ] = shap_values

    model_base_values[
        model_name
    ] = base_values

    model_explanation_predictions[
        model_name
    ] = (
        explained_predictions
    )

    shap_run_records.append({
        "Model": model_name,
        "Run_Status": run_status,
        "SHAP_Rows":
            shap_values.shape[0],
        "Features":
            shap_values.shape[1],
        "Permutation_Cycles":
            N_PERMUTATION_CYCLES,
        "Max_Evals":
            MAX_EVALS,
        "Elapsed_Seconds":
            elapsed_seconds,
        "Maximum_Additivity_Residual":
            maximum_additivity_residual,
        "Mean_Additivity_Residual":
            float(
                np.mean(
                    np.abs(
                        additivity_residual
                    )
                )
            ),
        "Additivity_Within_Tolerance":
            bool(
                maximum_additivity_residual
                <= ADDITIVITY_TOLERANCE
            )
    })

    print(
        f"Status: {run_status}"
    )

    print(
        f"Elapsed: "
        f"{elapsed_seconds / 60:.2f} min"
    )

    print(
        "Maximum additivity residual:",
        f"{maximum_additivity_residual:.10f}"
    )


shap_run_summary = pd.DataFrame(
    shap_run_records
)


print("\n" + "=" * 135)
print("SHAP COMPUTATION SUMMARY")
print("=" * 135)

print(
    shap_run_summary
    .round(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. Save local SHAP values in long format
# ------------------------------------------------------------

local_shap_records = []

for model_name in REPRESENTATIVE_MODELS:

    values = model_shap_values[
        model_name
    ]

    base_values = model_base_values[
        model_name
    ]

    predictions = (
        model_explanation_predictions[
            model_name
        ]
    )

    for row_position in range(
        values.shape[0]
    ):

        for feature_position, feature_name in enumerate(
            MODEL_FEATURES
        ):

            local_shap_records.append({
                "Model": model_name,
                "Explanation_Row":
                    row_position,
                "Feature":
                    feature_name,
                "Feature_Value_Encoded":
                    float(
                        explanation_matrix[
                            row_position,
                            feature_position
                        ]
                    ),
                "SHAP_Value":
                    float(
                        values[
                            row_position,
                            feature_position
                        ]
                    ),
                "Absolute_SHAP_Value":
                    float(
                        abs(
                            values[
                                row_position,
                                feature_position
                            ]
                        )
                    ),
                "Base_Value":
                    float(
                        base_values[
                            row_position
                        ]
                    ),
                "Calibrated_Prediction":
                    float(
                        predictions[
                            row_position
                        ]
                    )
            })


local_shap_long = pd.DataFrame(
    local_shap_records
)


# ------------------------------------------------------------
# 10. Global importance by model
# ------------------------------------------------------------

global_importance_records = []

for model_name in REPRESENTATIVE_MODELS:

    values = model_shap_values[
        model_name
    ]

    mean_absolute_shap = np.mean(
        np.abs(values),
        axis=0
    )

    mean_signed_shap = np.mean(
        values,
        axis=0
    )

    standard_deviation_absolute = np.std(
        np.abs(values),
        axis=0,
        ddof=1
    )

    normalized_importance = (
        mean_absolute_shap
        / mean_absolute_shap.sum()
    )

    importance_rank = rankdata(
        -mean_absolute_shap,
        method="average"
    )

    for feature_position, feature_name in enumerate(
        MODEL_FEATURES
    ):

        global_importance_records.append({
            "Model": model_name,
            "Feature": feature_name,
            "Mean_Absolute_SHAP":
                float(
                    mean_absolute_shap[
                        feature_position
                    ]
                ),
            "Normalized_Importance":
                float(
                    normalized_importance[
                        feature_position
                    ]
                ),
            "Mean_Signed_SHAP":
                float(
                    mean_signed_shap[
                        feature_position
                    ]
                ),
            "SD_Absolute_SHAP":
                float(
                    standard_deviation_absolute[
                        feature_position
                    ]
                ),
            "Importance_Rank":
                float(
                    importance_rank[
                        feature_position
                    ]
                )
        })


global_importance = pd.DataFrame(
    global_importance_records
)


print("\n" + "=" * 125)
print("GLOBAL SHAP IMPORTANCE BY MODEL")
print("=" * 125)

print(
    global_importance
    .sort_values(
        [
            "Model",
            "Importance_Rank"
        ]
    )[
        [
            "Model",
            "Feature",
            "Mean_Absolute_SHAP",
            "Normalized_Importance",
            "Importance_Rank"
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 11. Cross-model ranking agreement
# ------------------------------------------------------------

importance_pivot = (
    global_importance
    .pivot(
        index="Feature",
        columns="Model",
        values="Mean_Absolute_SHAP"
    )
    .loc[MODEL_FEATURES]
)

rank_pivot = (
    global_importance
    .pivot(
        index="Feature",
        columns="Model",
        values="Importance_Rank"
    )
    .loc[MODEL_FEATURES]
)


rank_agreement_records = []

for model_a, model_b in combinations(
    REPRESENTATIVE_MODELS,
    2
):

    importance_a = (
        importance_pivot[
            model_a
        ].to_numpy()
    )

    importance_b = (
        importance_pivot[
            model_b
        ].to_numpy()
    )

    ranks_a = (
        rank_pivot[
            model_a
        ].to_numpy()
    )

    ranks_b = (
        rank_pivot[
            model_b
        ].to_numpy()
    )

    spearman_result = spearmanr(
        importance_a,
        importance_b
    )

    kendall_result = kendalltau(
        importance_a,
        importance_b
    )

    record = {
        "Model_A": model_a,
        "Model_B": model_b,
        "Spearman_Rho":
            float(
                spearman_result.statistic
            ),
        "Spearman_P_Value":
            float(
                spearman_result.pvalue
            ),
        "Kendall_Tau":
            float(
                kendall_result.statistic
            ),
        "Kendall_P_Value":
            float(
                kendall_result.pvalue
            )
    }

    for top_k in TOP_K_VALUES:

        top_a = set(
            np.argsort(
                ranks_a
            )[:top_k]
        )

        top_b = set(
            np.argsort(
                ranks_b
            )[:top_k]
        )

        intersection_size = len(
            top_a.intersection(
                top_b
            )
        )

        union_size = len(
            top_a.union(
                top_b
            )
        )

        record[
            f"Top_{top_k}_Overlap_Count"
        ] = intersection_size

        record[
            f"Top_{top_k}_Overlap_Rate"
        ] = (
            intersection_size
            / top_k
        )

        record[
            f"Top_{top_k}_Jaccard"
        ] = (
            intersection_size
            / union_size
        )

    rank_agreement_records.append(
        record
    )


rank_agreement = pd.DataFrame(
    rank_agreement_records
)


print("\n" + "=" * 145)
print("CROSS-MODEL SHAP RANK AGREEMENT")
print("=" * 145)

print(
    rank_agreement
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 12. Consensus importance
# ------------------------------------------------------------

normalized_pivot = (
    global_importance
    .pivot(
        index="Feature",
        columns="Model",
        values="Normalized_Importance"
    )
    .loc[MODEL_FEATURES]
)

consensus_records = []

for feature_name in MODEL_FEATURES:

    normalized_values = (
        normalized_pivot.loc[
            feature_name
        ].to_numpy(dtype=float)
    )

    model_ranks = (
        rank_pivot.loc[
            feature_name
        ].to_numpy(dtype=float)
    )

    consensus_records.append({
        "Feature": feature_name,
        "Mean_Normalized_Importance":
            float(
                normalized_values.mean()
            ),
        "SD_Normalized_Importance":
            float(
                normalized_values.std(
                    ddof=1
                )
            ),
        "Mean_Rank":
            float(
                model_ranks.mean()
            ),
        "Best_Rank":
            float(
                model_ranks.min()
            ),
        "Worst_Rank":
            float(
                model_ranks.max()
            ),
        "Models_In_Top_3":
            int(
                np.sum(
                    model_ranks <= 3
                )
            ),
        "Models_In_Top_5":
            int(
                np.sum(
                    model_ranks <= 5
                )
            )
    })


consensus_importance = pd.DataFrame(
    consensus_records
)

consensus_importance[
    "Consensus_Rank"
] = rankdata(
    -consensus_importance[
        "Mean_Normalized_Importance"
    ].to_numpy(),
    method="average"
)

consensus_importance = (
    consensus_importance
    .sort_values(
        "Consensus_Rank"
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 120)
print("CROSS-MODEL CONSENSUS FEATURE IMPORTANCE")
print("=" * 120)

print(
    consensus_importance
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 13. Bootstrap SHAP ranking stability
# ------------------------------------------------------------

stability_rng = np.random.default_rng(
    STABILITY_SEED
)

bootstrap_importance_records = []
bootstrap_rank_summary_records = []
bootstrap_selection_records = []

for model_name in REPRESENTATIVE_MODELS:

    absolute_values = np.abs(
        model_shap_values[
            model_name
        ]
    )

    full_importance = (
        absolute_values.mean(axis=0)
    )

    full_ranks = rankdata(
        -full_importance,
        method="average"
    )

    full_top_sets = {
        top_k: set(
            np.argsort(
                full_ranks
            )[:top_k]
        )
        for top_k in TOP_K_VALUES
    }

    bootstrap_importance_matrix = np.zeros(
        (
            N_STABILITY_BOOTSTRAPS,
            N_FEATURES
        ),
        dtype=float
    )

    spearman_values = []
    kendall_values = []

    top_overlap_values = {
        top_k: []
        for top_k in TOP_K_VALUES
    }

    top_jaccard_values = {
        top_k: []
        for top_k in TOP_K_VALUES
    }

    top_selection_counts = {
        top_k: np.zeros(
            N_FEATURES,
            dtype=int
        )
        for top_k in TOP_K_VALUES
    }

    for bootstrap_id in range(
        N_STABILITY_BOOTSTRAPS
    ):

        sampled_rows = (
            stability_rng.integers(
                low=0,
                high=absolute_values.shape[0],
                size=absolute_values.shape[0]
            )
        )

        bootstrap_importance = (
            absolute_values[
                sampled_rows
            ].mean(axis=0)
        )

        bootstrap_importance_matrix[
            bootstrap_id
        ] = bootstrap_importance

        bootstrap_ranks = rankdata(
            -bootstrap_importance,
            method="average"
        )

        spearman_values.append(
            float(
                spearmanr(
                    full_ranks,
                    bootstrap_ranks
                ).statistic
            )
        )

        kendall_values.append(
            float(
                kendalltau(
                    full_ranks,
                    bootstrap_ranks
                ).statistic
            )
        )

        for top_k in TOP_K_VALUES:

            bootstrap_top_set = set(
                np.argsort(
                    bootstrap_ranks
                )[:top_k]
            )

            intersection_size = len(
                full_top_sets[
                    top_k
                ].intersection(
                    bootstrap_top_set
                )
            )

            union_size = len(
                full_top_sets[
                    top_k
                ].union(
                    bootstrap_top_set
                )
            )

            top_overlap_values[
                top_k
            ].append(
                intersection_size
                / top_k
            )

            top_jaccard_values[
                top_k
            ].append(
                intersection_size
                / union_size
            )

            for feature_position in (
                bootstrap_top_set
            ):

                top_selection_counts[
                    top_k
                ][feature_position] += 1

    bootstrap_rank_summary_records.append({
        "Model": model_name,
        "Mean_Spearman_Rho":
            float(
                np.mean(
                    spearman_values
                )
            ),
        "Spearman_2_5pct":
            float(
                np.quantile(
                    spearman_values,
                    0.025
                )
            ),
        "Spearman_97_5pct":
            float(
                np.quantile(
                    spearman_values,
                    0.975
                )
            ),
        "Mean_Kendall_Tau":
            float(
                np.mean(
                    kendall_values
                )
            ),
        "Kendall_2_5pct":
            float(
                np.quantile(
                    kendall_values,
                    0.025
                )
            ),
        "Kendall_97_5pct":
            float(
                np.quantile(
                    kendall_values,
                    0.975
                )
            ),
        "Mean_Top_3_Overlap":
            float(
                np.mean(
                    top_overlap_values[3]
                )
            ),
        "Mean_Top_5_Overlap":
            float(
                np.mean(
                    top_overlap_values[5]
                )
            ),
        "Mean_Top_3_Jaccard":
            float(
                np.mean(
                    top_jaccard_values[3]
                )
            ),
        "Mean_Top_5_Jaccard":
            float(
                np.mean(
                    top_jaccard_values[5]
                )
            )
    })

    for feature_position, feature_name in enumerate(
        MODEL_FEATURES
    ):

        feature_bootstrap_values = (
            bootstrap_importance_matrix[
                :,
                feature_position
            ]
        )

        bootstrap_selection_records.append({
            "Model": model_name,
            "Feature": feature_name,
            "Point_Mean_Absolute_SHAP":
                float(
                    full_importance[
                        feature_position
                    ]
                ),
            "Bootstrap_Mean":
                float(
                    feature_bootstrap_values.mean()
                ),
            "Bootstrap_CI_Lower":
                float(
                    np.quantile(
                        feature_bootstrap_values,
                        0.025
                    )
                ),
            "Bootstrap_CI_Upper":
                float(
                    np.quantile(
                        feature_bootstrap_values,
                        0.975
                    )
                ),
            "Top_3_Selection_Frequency":
                float(
                    top_selection_counts[
                        3
                    ][feature_position]
                    / N_STABILITY_BOOTSTRAPS
                ),
            "Top_5_Selection_Frequency":
                float(
                    top_selection_counts[
                        5
                    ][feature_position]
                    / N_STABILITY_BOOTSTRAPS
                )
        })

        for bootstrap_id in range(
            N_STABILITY_BOOTSTRAPS
        ):

            bootstrap_importance_records.append({
                "Bootstrap_ID":
                    bootstrap_id + 1,
                "Model":
                    model_name,
                "Feature":
                    feature_name,
                "Mean_Absolute_SHAP":
                    float(
                        bootstrap_importance_matrix[
                            bootstrap_id,
                            feature_position
                        ]
                    )
            })


bootstrap_rank_stability = pd.DataFrame(
    bootstrap_rank_summary_records
)

bootstrap_feature_stability = pd.DataFrame(
    bootstrap_selection_records
)

bootstrap_importance_long = pd.DataFrame(
    bootstrap_importance_records
)


print("\n" + "=" * 135)
print("BOOTSTRAP SHAP RANKING STABILITY")
print("=" * 135)

print(
    bootstrap_rank_stability
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 145)
print("BOOTSTRAP TOP-FEATURE SELECTION STABILITY")
print("=" * 145)

print(
    bootstrap_feature_stability[
        bootstrap_feature_stability[
            "Top_5_Selection_Frequency"
        ] >= 0.50
    ]
    .sort_values(
        [
            "Model",
            "Top_3_Selection_Frequency",
            "Point_Mean_Absolute_SHAP"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 14. Numeric directionality diagnostics
# ------------------------------------------------------------

numeric_direction_records = []

for model_name in REPRESENTATIVE_MODELS:

    values = model_shap_values[
        model_name
    ]

    model_importance = (
        global_importance[
            global_importance["Model"]
            == model_name
        ]
        .set_index("Feature")
    )

    for feature_name in NUMERIC_FEATURES:

        feature_position = (
            MODEL_FEATURES.index(
                feature_name
            )
        )

        feature_values = (
            X_explanation[
                feature_name
            ].to_numpy(dtype=float)
        )

        feature_shap = (
            values[
                :,
                feature_position
            ]
        )

        result = spearmanr(
            feature_values,
            feature_shap
        )

        numeric_direction_records.append({
            "Model": model_name,
            "Feature": feature_name,
            "Importance_Rank":
                float(
                    model_importance.loc[
                        feature_name,
                        "Importance_Rank"
                    ]
                ),
            "Spearman_Feature_vs_SHAP":
                float(
                    result.statistic
                ),
            "Descriptive_P_Value":
                float(
                    result.pvalue
                ),
            "Mean_SHAP":
                float(
                    feature_shap.mean()
                ),
            "Mean_Absolute_SHAP":
                float(
                    np.abs(
                        feature_shap
                    ).mean()
                )
        })


numeric_directionality = pd.DataFrame(
    numeric_direction_records
)


# ------------------------------------------------------------
# 15. Categorical level attribution
# ------------------------------------------------------------

categorical_level_records = []

for model_name in REPRESENTATIVE_MODELS:

    values = model_shap_values[
        model_name
    ]

    for feature_name in CATEGORICAL_FEATURES:

        feature_position = (
            MODEL_FEATURES.index(
                feature_name
            )
        )

        encoded_feature_values = (
            X_explanation[
                feature_name
            ].to_numpy(dtype=int)
        )

        for code, category_name in enumerate(
            category_levels[
                feature_name
            ]
        ):

            level_mask = (
                encoded_feature_values
                == code
            )

            if level_mask.sum() == 0:
                continue

            level_shap = values[
                level_mask,
                feature_position
            ]

            categorical_level_records.append({
                "Model": model_name,
                "Feature": feature_name,
                "Category_Code": code,
                "Category_Label":
                    category_name,
                "Rows":
                    int(
                        level_mask.sum()
                    ),
                "Mean_SHAP":
                    float(
                        level_shap.mean()
                    ),
                "Median_SHAP":
                    float(
                        np.median(
                            level_shap
                        )
                    ),
                "Mean_Absolute_SHAP":
                    float(
                        np.abs(
                            level_shap
                        ).mean()
                    ),
                "Positive_SHAP_Rate":
                    float(
                        np.mean(
                            level_shap > 0
                        )
                    )
            })


categorical_level_attribution = (
    pd.DataFrame(
        categorical_level_records
    )
)


# ------------------------------------------------------------
# 16. Save all artifacts
# ------------------------------------------------------------

calibrated_fidelity.to_csv(
    SHAP_OUTPUT_DIR
    / "calibrated_pipeline_prediction_fidelity.csv",
    index=False
)

shap_run_summary.to_csv(
    SHAP_OUTPUT_DIR
    / "permutation_shap_run_summary.csv",
    index=False
)

local_shap_long.to_csv(
    SHAP_OUTPUT_DIR
    / "local_permutation_shap_values_long.csv",
    index=False
)

global_importance.to_csv(
    SHAP_OUTPUT_DIR
    / "global_shap_importance_by_model.csv",
    index=False
)

importance_pivot.to_csv(
    SHAP_OUTPUT_DIR
    / "cross_model_importance_matrix.csv"
)

rank_pivot.to_csv(
    SHAP_OUTPUT_DIR
    / "cross_model_rank_matrix.csv"
)

rank_agreement.to_csv(
    SHAP_OUTPUT_DIR
    / "cross_model_shap_rank_agreement.csv",
    index=False
)

consensus_importance.to_csv(
    SHAP_OUTPUT_DIR
    / "cross_model_consensus_importance.csv",
    index=False
)

bootstrap_rank_stability.to_csv(
    SHAP_OUTPUT_DIR
    / "bootstrap_shap_rank_stability.csv",
    index=False
)

bootstrap_feature_stability.to_csv(
    SHAP_OUTPUT_DIR
    / "bootstrap_shap_feature_stability.csv",
    index=False
)

bootstrap_importance_long.to_csv(
    SHAP_OUTPUT_DIR
    / "bootstrap_shap_importance_replicates.csv",
    index=False
)

numeric_directionality.to_csv(
    SHAP_OUTPUT_DIR
    / "numeric_shap_directionality.csv",
    index=False
)

categorical_level_attribution.to_csv(
    SHAP_OUTPUT_DIR
    / "categorical_level_shap_attribution.csv",
    index=False
)


run_manifest = {
    "shap_version":
        shap.__version__,

    "representative_models":
        REPRESENTATIVE_MODELS,

    "explained_output":
        "Final calibrated probability",

    "explanation_rows":
        int(
            len(X_explanation)
        ),

    "background_rows":
        int(
            len(X_background)
        ),

    "feature_count":
        N_FEATURES,

    "permutation_cycles":
        N_PERMUTATION_CYCLES,

    "max_evals":
        MAX_EVALS,

    "batch_size":
        SHAP_BATCH_SIZE,

    "stability_bootstraps":
        N_STABILITY_BOOTSTRAPS,

    "stability_seed":
        STABILITY_SEED,

    "post_test_tuning_performed":
        False
}


with open(
    SHAP_OUTPUT_DIR
    / "permutation_shap_run_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        run_manifest,
        file,
        indent=4
    )


total_shap_elapsed = (
    time.perf_counter()
    - overall_shap_start
)


print("\n" + "=" * 110)
print("CHECKPOINT 8B2B SAVED")
print("=" * 110)

print("Output directory:")
print(SHAP_OUTPUT_DIR)

print(
    f"\nTotal SHAP and stability time: "
    f"{total_shap_elapsed / 60:.2f} minutes"
)

print(
    "\nNo model, calibration rule, threshold, "
    "or locked-test metric was modified."
)

In [ ]:
# ============================================================
# SUPPLEMENTARY EXPLAINABILITY
# TABNET + EXACT IN-MEMORY REALTABR
#
# Uses the same explanation cohort, background sample,
# feature space, and permutation budget as the primary SHAP
# analysis. No model selection or post-test tuning occurs.
# ============================================================

from pathlib import Path
import gc
import joblib
import shutil
import time

import numpy as np
import pandas as pd
import shap
from scipy.stats import rankdata, spearmanr
from pytorch_tabnet.tab_model import TabNetClassifier


SUPPLEMENTARY_SHAP_DIR = Path(
    "/kaggle/working/explainability_shap/supplementary"
)
SUPPLEMENTARY_SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUPPLEMENTARY_MODELS = [
    "TabNet",
    "RealTabR"
]

SUPPLEMENTARY_SHAP_SEEDS = {
    "TabNet": 2046,
    "RealTabR": 2047
}

PREDICTION_TOLERANCE = 1e-5


# ------------------------------------------------------------
# 1. Restore the exact final TabNet artifact
# ------------------------------------------------------------

TABNET_MODEL_PATH = Path(
    "/kaggle/working/final_locked_holdout/"
    "model_artifacts/TabNet_model.zip"
)

TABNET_PREPROCESSOR_PATH = Path(
    "/kaggle/working/final_locked_holdout/"
    "model_artifacts/TabNet_preprocessor.joblib"
)

if not TABNET_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing final TabNet model: {TABNET_MODEL_PATH}"
    )

tabnet_supp_model = TabNetClassifier()
tabnet_supp_model.load_model(
    str(TABNET_MODEL_PATH)
)

tabnet_supp_preprocessor = joblib.load(
    TABNET_PREPROCESSOR_PATH
)

tabnet_numeric_count = len(
    NUMERIC_FEATURES
)

tabnet_supp_cat_idxs = list(
    range(
        tabnet_numeric_count,
        tabnet_numeric_count
        + len(CATEGORICAL_FEATURES)
    )
)


def predict_tabnet_supp_raw(
    encoded_matrix
):
    decoded = decode_explanation_matrix(
        encoded_matrix
    )

    transformed = (
        tabnet_supp_preprocessor
        .transform(decoded)
        .astype(np.float32)
    )

    transformed[
        :, tabnet_supp_cat_idxs
    ] += 1

    return (
        tabnet_supp_model
        .predict_proba(
            transformed
        )[:, 1]
    )


# ------------------------------------------------------------
# 2. Use the exact final in-memory RealTabR estimator
# ------------------------------------------------------------

if "realtabr_model" not in globals():
    raise RuntimeError(
        "The exact final RealTabR estimator is no longer in "
        "memory. Run the notebook sequentially from the final "
        "training stage before running this cell."
    )


def predict_realtabr_supp_raw(
    encoded_matrix
):
    decoded = decode_explanation_matrix(
        encoded_matrix
    )

    return (
        realtabr_model
        .predict_proba(
            decoded
        )[:, 1]
    )


supp_raw_functions = {
    "TabNet":
        predict_tabnet_supp_raw,
    "RealTabR":
        predict_realtabr_supp_raw
}


# ------------------------------------------------------------
# 3. Fidelity check against immutable raw probabilities
# ------------------------------------------------------------

encoded_test_matrix = (
    X_test_encoded
    .to_numpy(dtype=float)
)

supp_fidelity_records = []

for model_name in SUPPLEMENTARY_MODELS:

    regenerated = (
        supp_raw_functions[
            model_name
        ](
            encoded_test_matrix
        )
    )

    immutable = (
        raw_predictions[
            f"{model_name}_Raw_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    absolute_difference = np.abs(
        regenerated
        - immutable
    )

    supp_fidelity_records.append({
        "Model":
            model_name,
        "Maximum_Absolute_Difference":
            float(
                absolute_difference.max()
            ),
        "Mean_Absolute_Difference":
            float(
                absolute_difference.mean()
            ),
        "Prediction_Correlation":
            float(
                np.corrcoef(
                    regenerated,
                    immutable
                )[0, 1]
            ),
        "Within_Tolerance":
            bool(
                absolute_difference.max()
                <= PREDICTION_TOLERANCE
            )
    })


supp_fidelity = pd.DataFrame(
    supp_fidelity_records
)

print("\n" + "=" * 110)
print("SUPPLEMENTARY MODEL FIDELITY")
print("=" * 110)
print(
    supp_fidelity
    .round(12)
    .to_string(index=False)
)

assert supp_fidelity[
    "Within_Tolerance"
].all(), (
    "A supplementary model does not reproduce its "
    "immutable final probabilities."
)


# ------------------------------------------------------------
# 4. Locked calibration wrappers
# ------------------------------------------------------------

supp_calibrator_bundles = {}

for model_name in SUPPLEMENTARY_MODELS:

    calibrator_path = Path(
        "/kaggle/working/"
        "calibration_threshold_locked/"
        f"{model_name}_calibrator.joblib"
    )

    supp_calibrator_bundles[
        model_name
    ] = joblib.load(
        calibrator_path
    )


def make_supp_calibrated_function(
    model_name
):
    raw_function = (
        supp_raw_functions[
            model_name
        ]
    )

    calibrator_bundle = (
        supp_calibrator_bundles[
            model_name
        ]
    )

    def calibrated_function(
        encoded_matrix
    ):
        raw_probability = (
            raw_function(
                encoded_matrix
            )
        )

        return apply_calibrator_bundle(
            raw_probability,
            calibrator_bundle
        )

    return calibrated_function


supp_calibrated_functions = {
    model_name:
        make_supp_calibrated_function(
            model_name
        )
    for model_name
    in SUPPLEMENTARY_MODELS
}


# ------------------------------------------------------------
# 5. Permutation SHAP using the primary protocol
# ------------------------------------------------------------

background_matrix = (
    X_background
    .to_numpy(dtype=float)
)

explanation_matrix = (
    X_explanation
    .to_numpy(dtype=float)
)

supp_masker = shap.maskers.Independent(
    background_matrix,
    max_samples=len(
        background_matrix
    )
)

supp_importance_records = []
supp_runtime_records = []
supp_shap_values = {}


for model_name in SUPPLEMENTARY_MODELS:

    print("\n" + "=" * 110)
    print(
        "SUPPLEMENTARY PERMUTATION SHAP:",
        model_name
    )
    print("=" * 110)

    start_time = time.perf_counter()

    explainer = shap.PermutationExplainer(
        model=
            supp_calibrated_functions[
                model_name
            ],
        masker=supp_masker,
        feature_names=
            MODEL_FEATURES,
        seed=
            SUPPLEMENTARY_SHAP_SEEDS[
                model_name
            ]
    )

    explanation = explainer(
        explanation_matrix,
        max_evals=MAX_EVALS,
        batch_size=SHAP_BATCH_SIZE,
        silent=False
    )

    values = np.asarray(
        explanation.values,
        dtype=float
    )

    base_values = np.asarray(
        explanation.base_values,
        dtype=float
    )

    if (
        values.ndim == 3
        and values.shape[-1] == 1
    ):
        values = values[:, :, 0]

    if (
        base_values.ndim > 1
        and base_values.shape[-1] == 1
    ):
        base_values = (
            base_values[:, 0]
        )

    calibrated_probability = (
        supp_calibrated_functions[
            model_name
        ](
            explanation_matrix
        )
    )

    reconstructed_probability = (
        base_values
        + values.sum(axis=1)
    )

    maximum_additivity_error = float(
        np.max(
            np.abs(
                reconstructed_probability
                - calibrated_probability
            )
        )
    )

    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )

    mean_absolute_shap = np.mean(
        np.abs(values),
        axis=0
    )

    normalized_importance = (
        mean_absolute_shap
        / mean_absolute_shap.sum()
    )

    ranks = rankdata(
        -mean_absolute_shap,
        method="average"
    )

    supp_shap_values[
        model_name
    ] = values

    np.savez_compressed(
        SUPPLEMENTARY_SHAP_DIR
        / f"{model_name}_permutation_shap.npz",
        shap_values=values,
        base_values=base_values,
        predictions=calibrated_probability
    )

    for feature_index, feature_name in enumerate(
        MODEL_FEATURES
    ):
        supp_importance_records.append({
            "Model":
                model_name,
            "Feature":
                feature_name,
            "Mean_Absolute_SHAP":
                float(
                    mean_absolute_shap[
                        feature_index
                    ]
                ),
            "Normalized_Importance":
                float(
                    normalized_importance[
                        feature_index
                    ]
                ),
            "Importance_Rank":
                float(
                    ranks[
                        feature_index
                    ]
                )
        })

    supp_runtime_records.append({
        "Model":
            model_name,
        "Rows":
            len(explanation_matrix),
        "Background_Rows":
            len(background_matrix),
        "Max_Evals":
            MAX_EVALS,
        "Elapsed_Seconds":
            elapsed_seconds,
        "Maximum_Additivity_Error":
            maximum_additivity_error
    })

    print(
        f"Elapsed: {elapsed_seconds:.3f} s | "
        f"max additivity error: "
        f"{maximum_additivity_error:.3e}"
    )

    del explainer, explanation
    gc.collect()


supp_importance = pd.DataFrame(
    supp_importance_records
)

supp_runtime = pd.DataFrame(
    supp_runtime_records
)


# ------------------------------------------------------------
# 6. Agreement with the original four-model consensus
# ------------------------------------------------------------

primary_consensus_rank = dict(
    zip(
        consensus_importance[
            "Feature"
        ],
        consensus_importance[
            "Consensus_Rank"
        ]
    )
)

primary_top3 = {
    feature
    for feature, rank
    in primary_consensus_rank.items()
    if rank <= 3
}

primary_top5 = {
    feature
    for feature, rank
    in primary_consensus_rank.items()
    if rank <= 5
}

agreement_records = []

for model_name in SUPPLEMENTARY_MODELS:

    model_table = (
        supp_importance[
            supp_importance["Model"]
            == model_name
        ]
        .copy()
    )

    model_table[
        "Primary_Consensus_Rank"
    ] = (
        model_table[
            "Feature"
        ]
        .map(
            primary_consensus_rank
        )
    )

    rho = spearmanr(
        model_table[
            "Importance_Rank"
        ],
        model_table[
            "Primary_Consensus_Rank"
        ]
    )

    ordered = (
        model_table
        .sort_values(
            "Importance_Rank"
        )
    )

    top3 = set(
        ordered
        .head(3)[
            "Feature"
        ]
    )

    top5 = set(
        ordered
        .head(5)[
            "Feature"
        ]
    )

    agreement_records.append({
        "Model":
            model_name,
        "Spearman_Rho":
            float(
                rho.statistic
            ),
        "Spearman_P_Value":
            float(
                rho.pvalue
            ),
        "Top3_Overlap":
            len(
                top3
                & primary_top3
            ),
        "Top5_Overlap":
            len(
                top5
                & primary_top5
            ),
        "Top3_Features":
            ", ".join(
                ordered
                .head(3)[
                    "Feature"
                ]
                .tolist()
            ),
        "Top5_Features":
            ", ".join(
                ordered
                .head(5)[
                    "Feature"
                ]
                .tolist()
            )
    })


supp_agreement = pd.DataFrame(
    agreement_records
)


# ------------------------------------------------------------
# 7. Save compact supplementary evidence
# ------------------------------------------------------------

supp_fidelity.to_csv(
    SUPPLEMENTARY_SHAP_DIR
    / "supplementary_prediction_fidelity.csv",
    index=False
)

supp_importance.to_csv(
    SUPPLEMENTARY_SHAP_DIR
    / "supplementary_shap_importance.csv",
    index=False
)

supp_agreement.to_csv(
    SUPPLEMENTARY_SHAP_DIR
    / "supplementary_shap_agreement.csv",
    index=False
)

supp_runtime.to_csv(
    SUPPLEMENTARY_SHAP_DIR
    / "supplementary_shap_runtime.csv",
    index=False
)


print("\n" + "=" * 120)
print("SUPPLEMENTARY SHAP IMPORTANCE")
print("=" * 120)
print(
    supp_importance
    .sort_values(
        [
            "Model",
            "Importance_Rank"
        ]
    )
    .round(6)
    .to_string(index=False)
)

print("\n" + "=" * 120)
print("AGREEMENT WITH PRIMARY FOUR-MODEL CONSENSUS")
print("=" * 120)
print(
    supp_agreement
    .round(6)
    .to_string(index=False)
)

print(
    "\nSaved supplementary SHAP evidence to:"
)
print(SUPPLEMENTARY_SHAP_DIR)


# ------------------------------------------------------------
# 8. Cleanup the non-serializable RealTabR training object
# ------------------------------------------------------------

try:
    del realtabr_model
except NameError:
    pass

try:
    shutil.rmtree(
        realtabr_temp,
        ignore_errors=True
    )
except NameError:
    pass

gc.collect()


In [ ]:
# ============================================================
# SHAP DIRECTIONALITY AND NONLINEAR PROFILE AUDIT
#
# Reads previously saved permutation SHAP artifacts.
# No SHAP recomputation, training, tuning, or test changes.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from scipy.stats import spearmanr


# ------------------------------------------------------------
# 1. Locate saved SHAP directory
# ------------------------------------------------------------

candidate_directories = [Path("/kaggle/working/explainability_shap")]

existing_directories = [
    path
    for path in candidate_directories
    if path.exists()
]

if not existing_directories:
    raise FileNotFoundError(
        "Could not locate the saved SHAP directory."
    )

SHAP_DIR = existing_directories[0]

OUTPUT_DIR = (
    Path("/kaggle/working")
    / "explainability_shap"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def find_saved_file(filename):

    direct_path = SHAP_DIR / filename

    if direct_path.exists():
        return direct_path

    matches = list(
        SHAP_DIR.rglob(filename)
    )

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Missing SHAP artifact: {filename}"
        )

    raise RuntimeError(
        f"Multiple copies of {filename} found:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


LOCAL_SHAP_PATH = find_saved_file(
    "local_permutation_shap_values_long.csv"
)

EXPLANATION_FEATURES_PATH = find_saved_file(
    "shap_explanation_encoded_features.csv"
)

CONSENSUS_IMPORTANCE_PATH = find_saved_file(
    "cross_model_consensus_importance.csv"
)

MAPPING_MANIFEST_PATH = find_saved_file(
    "shap_input_mapping_manifest.json"
)


# ------------------------------------------------------------
# 2. Load artifacts
# ------------------------------------------------------------

local_shap = pd.read_csv(
    LOCAL_SHAP_PATH
)

X_explanation = pd.read_csv(
    EXPLANATION_FEATURES_PATH
)

consensus_importance = pd.read_csv(
    CONSENSUS_IMPORTANCE_PATH
)

with open(
    MAPPING_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:
    mapping_manifest = json.load(file)


MODEL_FEATURES = mapping_manifest[
    "model_features"
]

CATEGORICAL_FEATURES = mapping_manifest[
    "categorical_features"
]

CATEGORY_LEVELS = mapping_manifest[
    "category_levels"
]

REPRESENTATIVE_MODELS = mapping_manifest[
    "representative_models"
]

NUMERIC_FEATURES = [
    feature
    for feature in MODEL_FEATURES
    if feature not in CATEGORICAL_FEATURES
]

CONTINUOUS_FEATURES = [
    "CreditScore",
    "Age",
    "Balance",
    "EstimatedSalary"
]

DISCRETE_NUMERIC_FEATURES = [
    "Tenure",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember"
]


# ------------------------------------------------------------
# 3. Integrity checks
# ------------------------------------------------------------

assert len(X_explanation) == 300

assert list(
    X_explanation.columns
) == MODEL_FEATURES

assert set(
    local_shap["Model"].unique()
) == set(REPRESENTATIVE_MODELS)

assert set(
    local_shap["Feature"].unique()
) == set(MODEL_FEATURES)

expected_local_rows = (
    len(REPRESENTATIVE_MODELS)
    * len(X_explanation)
    * len(MODEL_FEATURES)
)

assert len(local_shap) == expected_local_rows

print("=" * 105)
print("DIRECTIONALITY INPUT VERIFICATION")
print("=" * 105)

print("SHAP directory:", SHAP_DIR)
print("Models:", len(REPRESENTATIVE_MODELS))
print("Explanation rows:", len(X_explanation))
print("Features:", len(MODEL_FEATURES))
print("Local SHAP rows:", len(local_shap))


# ------------------------------------------------------------
# 4. Add feature values from the explanation cohort
# ------------------------------------------------------------

feature_value_long = (
    X_explanation
    .reset_index(
        names="Explanation_Row"
    )
    .melt(
        id_vars="Explanation_Row",
        value_vars=MODEL_FEATURES,
        var_name="Feature",
        value_name="Feature_Value"
    )
)

shap_with_values = (
    local_shap
    .drop(
        columns=[
            "Feature_Value_Encoded"
        ],
        errors="ignore"
    )
    .merge(
        feature_value_long,
        on=[
            "Explanation_Row",
            "Feature"
        ],
        how="left",
        validate="many_to_one"
    )
)

assert (
    shap_with_values[
        "Feature_Value"
    ].notna().all()
)


# ------------------------------------------------------------
# 5. Model-specific numeric direction diagnostics
# ------------------------------------------------------------

model_direction_records = []

for model_name in REPRESENTATIVE_MODELS:

    model_data = shap_with_values[
        shap_with_values["Model"]
        == model_name
    ]

    for feature_name in NUMERIC_FEATURES:

        feature_data = (
            model_data[
                model_data["Feature"]
                == feature_name
            ]
            .sort_values(
                "Explanation_Row"
            )
        )

        feature_values = (
            feature_data["Feature_Value"]
            .astype(float)
            .to_numpy()
        )

        shap_values = (
            feature_data["SHAP_Value"]
            .astype(float)
            .to_numpy()
        )

        spearman_result = spearmanr(
            feature_values,
            shap_values
        )

        if (
            feature_name
            in [
                "HasCrCard",
                "IsActiveMember"
            ]
        ):

            low_mask = (
                feature_values
                == feature_values.min()
            )

            high_mask = (
                feature_values
                == feature_values.max()
            )

        else:

            lower_cutoff = np.quantile(
                feature_values,
                0.20
            )

            upper_cutoff = np.quantile(
                feature_values,
                0.80
            )

            low_mask = (
                feature_values
                <= lower_cutoff
            )

            high_mask = (
                feature_values
                >= upper_cutoff
            )

        low_mean_shap = float(
            shap_values[
                low_mask
            ].mean()
        )

        high_mean_shap = float(
            shap_values[
                high_mask
            ].mean()
        )

        model_direction_records.append({
            "Model": model_name,
            "Feature": feature_name,

            "Spearman_Feature_vs_SHAP":
                float(
                    spearman_result.statistic
                ),

            "Spearman_P_Value_Descriptive":
                float(
                    spearman_result.pvalue
                ),

            "Low_Value_Group_Rows":
                int(low_mask.sum()),

            "High_Value_Group_Rows":
                int(high_mask.sum()),

            "Low_Value_Group_Mean_SHAP":
                low_mean_shap,

            "High_Value_Group_Mean_SHAP":
                high_mean_shap,

            "High_minus_Low_SHAP":
                high_mean_shap
                - low_mean_shap
        })


model_numeric_directionality = pd.DataFrame(
    model_direction_records
)


# ------------------------------------------------------------
# 6. Consensus row-level SHAP
# ------------------------------------------------------------

consensus_row_shap = (
    shap_with_values
    .groupby(
        [
            "Explanation_Row",
            "Feature",
            "Feature_Value"
        ],
        as_index=False
    )
    .agg(
        Consensus_SHAP=(
            "SHAP_Value",
            "mean"
        ),
        Model_SHAP_SD=(
            "SHAP_Value",
            "std"
        )
    )
)

assert len(consensus_row_shap) == (
    len(X_explanation)
    * len(MODEL_FEATURES)
)


# ------------------------------------------------------------
# 7. Build numeric feature profiles
# ------------------------------------------------------------

profile_records = []

for feature_name in NUMERIC_FEATURES:

    feature_rows = (
        consensus_row_shap[
            consensus_row_shap["Feature"]
            == feature_name
        ]
        .copy()
        .sort_values(
            "Explanation_Row"
        )
    )

    feature_values = (
        feature_rows["Feature_Value"]
        .astype(float)
    )

    if feature_name in CONTINUOUS_FEATURES:

        bin_codes = pd.qcut(
            feature_values,
            q=5,
            labels=False,
            duplicates="drop"
        )

        feature_rows[
            "Profile_Order"
        ] = bin_codes.astype(int)

        feature_rows[
            "Profile_Type"
        ] = "Quintile"

        grouped = (
            feature_rows
            .groupby(
                "Profile_Order",
                as_index=False
            )
            .agg(
                Rows=(
                    "Explanation_Row",
                    "count"
                ),
                Value_Min=(
                    "Feature_Value",
                    "min"
                ),
                Value_Mean=(
                    "Feature_Value",
                    "mean"
                ),
                Value_Max=(
                    "Feature_Value",
                    "max"
                ),
                Consensus_Mean_SHAP=(
                    "Consensus_SHAP",
                    "mean"
                ),
                Consensus_Median_SHAP=(
                    "Consensus_SHAP",
                    "median"
                ),
                Mean_Model_SHAP_SD=(
                    "Model_SHAP_SD",
                    "mean"
                )
            )
        )

        grouped[
            "Profile_Label"
        ] = (
            "Q"
            + (
                grouped[
                    "Profile_Order"
                ] + 1
            ).astype(str)
        )

    else:

        feature_rows[
            "Profile_Order"
        ] = (
            feature_values.astype(int)
        )

        feature_rows[
            "Profile_Type"
        ] = "Exact value"

        grouped = (
            feature_rows
            .groupby(
                "Profile_Order",
                as_index=False
            )
            .agg(
                Rows=(
                    "Explanation_Row",
                    "count"
                ),
                Value_Min=(
                    "Feature_Value",
                    "min"
                ),
                Value_Mean=(
                    "Feature_Value",
                    "mean"
                ),
                Value_Max=(
                    "Feature_Value",
                    "max"
                ),
                Consensus_Mean_SHAP=(
                    "Consensus_SHAP",
                    "mean"
                ),
                Consensus_Median_SHAP=(
                    "Consensus_SHAP",
                    "median"
                ),
                Mean_Model_SHAP_SD=(
                    "Model_SHAP_SD",
                    "mean"
                )
            )
        )

        grouped[
            "Profile_Label"
        ] = (
            grouped[
                "Profile_Order"
            ].astype(str)
        )

    grouped[
        "Feature"
    ] = feature_name

    grouped[
        "Profile_Type"
    ] = (
        "Quintile"
        if feature_name
        in CONTINUOUS_FEATURES
        else "Exact value"
    )

    profile_records.append(
        grouped
    )


numeric_profiles = pd.concat(
    profile_records,
    ignore_index=True
)

numeric_profiles = numeric_profiles[
    [
        "Feature",
        "Profile_Type",
        "Profile_Order",
        "Profile_Label",
        "Rows",
        "Value_Min",
        "Value_Mean",
        "Value_Max",
        "Consensus_Mean_SHAP",
        "Consensus_Median_SHAP",
        "Mean_Model_SHAP_SD"
    ]
]


# ------------------------------------------------------------
# 8. Identify non-monotonic profiles
# ------------------------------------------------------------

profile_shape_records = []

for feature_name in NUMERIC_FEATURES:

    feature_profile = (
        numeric_profiles[
            numeric_profiles["Feature"]
            == feature_name
        ]
        .sort_values(
            "Profile_Order"
        )
    )

    profile_values = (
        feature_profile[
            "Consensus_Mean_SHAP"
        ]
        .to_numpy(dtype=float)
    )

    adjacent_differences = np.diff(
        profile_values
    )

    profile_scale = max(
        np.max(
            np.abs(
                profile_values
            )
        ),
        1e-8
    )

    meaningful_tolerance = max(
        1e-4,
        0.05 * profile_scale
    )

    meaningful_differences = (
        adjacent_differences[
            np.abs(
                adjacent_differences
            ) > meaningful_tolerance
        ]
    )

    difference_signs = np.sign(
        meaningful_differences
    )

    sign_changes = int(
        np.sum(
            difference_signs[1:]
            != difference_signs[:-1]
        )
    ) if len(difference_signs) > 1 else 0

    if len(difference_signs) == 0:

        profile_shape = "Approximately flat"

    elif np.all(
        difference_signs > 0
    ):

        profile_shape = "Generally increasing"

    elif np.all(
        difference_signs < 0
    ):

        profile_shape = "Generally decreasing"

    else:

        profile_shape = "Non-monotonic"

    profile_shape_records.append({
        "Feature": feature_name,
        "Profile_Levels":
            len(feature_profile),
        "Meaningful_Adjacent_Changes":
            len(
                meaningful_differences
            ),
        "Direction_Sign_Changes":
            sign_changes,
        "Consensus_Profile_Shape":
            profile_shape
    })


profile_shape_table = pd.DataFrame(
    profile_shape_records
)


# ------------------------------------------------------------
# 9. Cross-model directional consensus
# ------------------------------------------------------------

direction_summary_records = []

for feature_name in NUMERIC_FEATURES:

    feature_models = (
        model_numeric_directionality[
            model_numeric_directionality[
                "Feature"
            ] == feature_name
        ]
    )

    deltas = (
        feature_models[
            "High_minus_Low_SHAP"
        ].to_numpy(dtype=float)
    )

    positive_models = int(
        np.sum(deltas > 0)
    )

    negative_models = int(
        np.sum(deltas < 0)
    )

    zero_models = int(
        np.sum(
            np.isclose(
                deltas,
                0.0,
                atol=1e-8
            )
        )
    )

    profile_shape = (
        profile_shape_table.loc[
            profile_shape_table[
                "Feature"
            ] == feature_name,
            "Consensus_Profile_Shape"
        ]
        .iloc[0]
    )

    if profile_shape == "Non-monotonic":

        interpretation = (
            "Non-monotonic relationship; "
            "avoid a single directional claim."
        )

    elif positive_models == len(
        REPRESENTATIVE_MODELS
    ):

        interpretation = (
            "Higher values generally increase "
            "predicted churn probability."
        )

    elif negative_models == len(
        REPRESENTATIVE_MODELS
    ):

        interpretation = (
            "Higher values generally decrease "
            "predicted churn probability."
        )

    else:

        interpretation = (
            "Directional effect varies "
            "across models."
        )

    direction_summary_records.append({
        "Feature": feature_name,

        "Mean_Spearman_Feature_vs_SHAP":
            float(
                feature_models[
                    "Spearman_Feature_vs_SHAP"
                ].mean()
            ),

        "Minimum_Model_Spearman":
            float(
                feature_models[
                    "Spearman_Feature_vs_SHAP"
                ].min()
            ),

        "Maximum_Model_Spearman":
            float(
                feature_models[
                    "Spearman_Feature_vs_SHAP"
                ].max()
            ),

        "Mean_High_minus_Low_SHAP":
            float(
                deltas.mean()
            ),

        "Models_With_Positive_Delta":
            positive_models,

        "Models_With_Negative_Delta":
            negative_models,

        "Models_With_Near_Zero_Delta":
            zero_models,

        "Consensus_Profile_Shape":
            profile_shape,

        "Interpretation":
            interpretation
    })


numeric_directional_consensus = (
    pd.DataFrame(
        direction_summary_records
    )
    .merge(
        consensus_importance[
            [
                "Feature",
                "Consensus_Rank",
                "Mean_Normalized_Importance"
            ]
        ],
        on="Feature",
        how="left"
    )
    .sort_values(
        "Consensus_Rank"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 10. Categorical level effects
# ------------------------------------------------------------

categorical_records = []

for feature_name in CATEGORICAL_FEATURES:

    feature_rows = shap_with_values[
        shap_with_values["Feature"]
        == feature_name
    ].copy()

    feature_rows[
        "Category_Code"
    ] = (
        feature_rows[
            "Feature_Value"
        ]
        .round()
        .astype(int)
    )

    feature_rows[
        "Category_Label"
    ] = feature_rows[
        "Category_Code"
    ].map(
        {
            code: label
            for code, label
            in enumerate(
                CATEGORY_LEVELS[
                    feature_name
                ]
            )
        }
    )

    model_level_summary = (
        feature_rows
        .groupby(
            [
                "Model",
                "Category_Code",
                "Category_Label"
            ],
            as_index=False
        )
        .agg(
            Rows=(
                "Explanation_Row",
                "count"
            ),
            Mean_SHAP=(
                "SHAP_Value",
                "mean"
            ),
            Median_SHAP=(
                "SHAP_Value",
                "median"
            ),
            Mean_Absolute_SHAP=(
                "Absolute_SHAP_Value",
                "mean"
            ),
            Positive_SHAP_Rate=(
                "SHAP_Value",
                lambda values:
                    float(
                        np.mean(
                            values > 0
                        )
                    )
            )
        )
    )

    consensus_level_summary = (
        model_level_summary
        .groupby(
            [
                "Category_Code",
                "Category_Label"
            ],
            as_index=False
        )
        .agg(
            Rows_Per_Model=(
                "Rows",
                "first"
            ),
            Consensus_Mean_SHAP=(
                "Mean_SHAP",
                "mean"
            ),
            SD_Model_Mean_SHAP=(
                "Mean_SHAP",
                "std"
            ),
            Minimum_Model_Mean_SHAP=(
                "Mean_SHAP",
                "min"
            ),
            Maximum_Model_Mean_SHAP=(
                "Mean_SHAP",
                "max"
            ),
            Models_With_Positive_Mean_SHAP=(
                "Mean_SHAP",
                lambda values:
                    int(
                        np.sum(
                            values > 0
                        )
                    )
            )
        )
    )

    consensus_level_summary[
        "Feature"
    ] = feature_name

    categorical_records.append(
        consensus_level_summary
    )


categorical_level_effects = pd.concat(
    categorical_records,
    ignore_index=True
)

categorical_level_effects = categorical_level_effects[
    [
        "Feature",
        "Category_Code",
        "Category_Label",
        "Rows_Per_Model",
        "Consensus_Mean_SHAP",
        "SD_Model_Mean_SHAP",
        "Minimum_Model_Mean_SHAP",
        "Maximum_Model_Mean_SHAP",
        "Models_With_Positive_Mean_SHAP"
    ]
]


# ------------------------------------------------------------
# 11. Publication-oriented profile table
# ------------------------------------------------------------

top_six_features = (
    consensus_importance
    .sort_values(
        "Consensus_Rank"
    )
    .head(6)[
        "Feature"
    ]
    .tolist()
)

top_numeric_profiles = (
    numeric_profiles[
        numeric_profiles["Feature"]
        .isin(top_six_features)
    ]
    .merge(
        consensus_importance[
            [
                "Feature",
                "Consensus_Rank"
            ]
        ],
        on="Feature",
        how="left"
    )
    .sort_values(
        [
            "Consensus_Rank",
            "Profile_Order"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. Print results
# ------------------------------------------------------------

print("\n" + "=" * 170)
print("NUMERIC SHAP DIRECTIONAL CONSENSUS")
print("=" * 170)

print(
    numeric_directional_consensus[
        [
            "Consensus_Rank",
            "Feature",
            "Mean_Normalized_Importance",
            "Mean_Spearman_Feature_vs_SHAP",
            "Minimum_Model_Spearman",
            "Maximum_Model_Spearman",
            "Mean_High_minus_Low_SHAP",
            "Models_With_Positive_Delta",
            "Models_With_Negative_Delta",
            "Consensus_Profile_Shape",
            "Interpretation"
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 155)
print("CONSENSUS SHAP PROFILES FOR TOP NUMERIC FEATURES")
print("=" * 155)

print(
    top_numeric_profiles[
        [
            "Consensus_Rank",
            "Feature",
            "Profile_Type",
            "Profile_Label",
            "Rows",
            "Value_Min",
            "Value_Mean",
            "Value_Max",
            "Consensus_Mean_SHAP",
            "Mean_Model_SHAP_SD"
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 145)
print("CATEGORICAL SHAP LEVEL EFFECTS")
print("=" * 145)

print(
    categorical_level_effects
    .sort_values(
        [
            "Feature",
            "Category_Code"
        ]
    )
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 13. Save artifacts
# ------------------------------------------------------------

model_numeric_directionality.to_csv(
    OUTPUT_DIR
    / "model_specific_numeric_shap_directionality.csv",
    index=False
)

numeric_profiles.to_csv(
    OUTPUT_DIR
    / "consensus_numeric_shap_profiles.csv",
    index=False
)

profile_shape_table.to_csv(
    OUTPUT_DIR
    / "numeric_shap_profile_shapes.csv",
    index=False
)

numeric_directional_consensus.to_csv(
    OUTPUT_DIR
    / "numeric_shap_directional_consensus.csv",
    index=False
)

categorical_level_effects.to_csv(
    OUTPUT_DIR
    / "categorical_shap_level_consensus.csv",
    index=False
)

top_numeric_profiles.to_csv(
    OUTPUT_DIR
    / "publication_top_numeric_shap_profiles.csv",
    index=False
)


run_summary = {
    "models":
        REPRESENTATIVE_MODELS,

    "explanation_rows":
        int(len(X_explanation)),

    "numeric_features":
        NUMERIC_FEATURES,

    "categorical_features":
        CATEGORICAL_FEATURES,

    "continuous_profile_method":
        "Explanation-sample quintiles",

    "discrete_profile_method":
        "Exact observed values",

    "interpretation_scope":
        (
            "Associations with model predictions; "
            "not causal effects"
        ),

    "post_test_tuning_performed":
        False
}


with open(
    OUTPUT_DIR
    / "shap_directionality_run_summary.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        run_summary,
        file,
        indent=4
    )


print("\nSaved directory:")
print(OUTPUT_DIR)

print(
    "\nNo model, calibrator, threshold, "
    "or test metric was modified."
)

In [ ]:
# ============================================================
# INPUT-PERTURBATION ROBUSTNESS AUDIT
#
# Representative models:
# XGBoost, CatBoost, MLP, FTTransformer
#
# Fixed perturbation protocols, 20 repetitions.
# No model training, tuning, calibration changes,
# threshold changes, or post-test model selection.
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/kaggle/working/robustness_audit"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPRESENTATIVE_MODELS = [
    "XGBoost",
    "CatBoost",
    "MLP",
    "FTTransformer"
]

N_REPETITIONS = 20
ROBUSTNESS_SEED = 2050

NUMERIC_NOISE_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "EstimatedSalary"
]

INTEGER_NOISE_FEATURES = [
    "CreditScore",
    "Age",
    "Tenure"
]

SCENARIOS = {
    "NumericNoise_Mild": {
        "kind": "numeric_noise",
        "CreditScore_SD": 5.0,
        "Age_SD": 1.0,
        "Tenure_SD": 1.0,
        "Balance_IQR_Fraction": 0.01,
        "EstimatedSalary_IQR_Fraction": 0.01
    },

    "NumericNoise_Moderate": {
        "kind": "numeric_noise",
        "CreditScore_SD": 20.0,
        "Age_SD": 3.0,
        "Tenure_SD": 2.0,
        "Balance_IQR_Fraction": 0.05,
        "EstimatedSalary_IQR_Fraction": 0.05
    },

    "MedianModeMasking_5pct": {
        "kind": "masking",
        "mask_probability": 0.05
    },

    "MedianModeMasking_10pct": {
        "kind": "masking",
        "mask_probability": 0.10
    },

    "CategoricalNoise_5pct": {
        "kind": "categorical_noise",
        "noise_probability": 0.05
    },

    "CategoricalNoise_10pct": {
        "kind": "categorical_noise",
        "noise_probability": 0.10
    },

    "Combined_Stress": {
        "kind": "combined",
        "CreditScore_SD": 20.0,
        "Age_SD": 3.0,
        "Tenure_SD": 2.0,
        "Balance_IQR_Fraction": 0.05,
        "EstimatedSalary_IQR_Fraction": 0.05,
        "mask_probability": 0.05,
        "noise_probability": 0.05
    }
}


# ------------------------------------------------------------
# 2. State verification
# ------------------------------------------------------------

required_variables = [
    "RESTORED_ROOT",
    "MODEL_FEATURES",
    "NUMERIC_FEATURES",
    "CATEGORICAL_FEATURES",
    "category_levels",
    "X_test_encoded",
    "X_development_encoded",
    "calibrated_prediction_functions",
    "final_predictions"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required variables from Checkpoints 8B2A–8B2B "
        "are missing:\n"
        + "\n".join(missing_variables)
        + "\nRerun Checkpoints 8B2A and 8B2B first."
    )


assert set(REPRESENTATIVE_MODELS).issubset(
    calibrated_prediction_functions
)

assert list(
    X_test_encoded.columns
) == MODEL_FEATURES

assert list(
    X_development_encoded.columns
) == MODEL_FEATURES

assert len(X_test_encoded) == 2000
assert len(final_predictions) == 2000


# ------------------------------------------------------------
# 3. Load locked thresholds
# ------------------------------------------------------------

def find_unique_file(
    root,
    filename,
    preferred_parent=None
):

    matches = list(
        root.rglob(filename)
    )

    if preferred_parent is not None:

        preferred_matches = [
            path
            for path in matches
            if preferred_parent in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}."
        )

    raise RuntimeError(
        f"Multiple copies of {filename} found:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


LOCKED_MANIFEST_PATH = find_unique_file(
    RESTORED_ROOT,
    "locked_calibration_threshold_manifest.json",
    preferred_parent="calibration_threshold_locked"
)

with open(
    LOCKED_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:
    locked_manifest = json.load(file)


locked_thresholds = {
    model_name: float(
        locked_manifest[
            model_name
        ]["locked_mcc_threshold"]
    )
    for model_name in REPRESENTATIVE_MODELS
}


# ------------------------------------------------------------
# 4. Baseline arrays and development statistics
# ------------------------------------------------------------

test_matrix = (
    X_test_encoded
    .to_numpy(dtype=float)
)

development_matrix = (
    X_development_encoded
    .to_numpy(dtype=float)
)

y_true = (
    final_predictions["y_true"]
    .astype(int)
    .to_numpy()
)

baseline_probability_map = {
    model_name:
        final_predictions[
            f"{model_name}_Calibrated_Probability"
        ]
        .astype(float)
        .to_numpy()

    for model_name in REPRESENTATIVE_MODELS
}

baseline_prediction_map = {
    model_name:
        final_predictions[
            f"{model_name}_Locked_Prediction"
        ]
        .astype(int)
        .to_numpy()

    for model_name in REPRESENTATIVE_MODELS
}


feature_position = {
    feature: position
    for position, feature
    in enumerate(MODEL_FEATURES)
}


development_minimum = {
    feature: float(
        X_development_encoded[
            feature
        ].min()
    )
    for feature in MODEL_FEATURES
}

development_maximum = {
    feature: float(
        X_development_encoded[
            feature
        ].max()
    )
    for feature in MODEL_FEATURES
}

development_iqr = {
    feature: float(
        X_development_encoded[
            feature
        ].quantile(0.75)
        -
        X_development_encoded[
            feature
        ].quantile(0.25)
    )
    for feature in MODEL_FEATURES
}


replacement_values = {}

for feature in MODEL_FEATURES:

    if feature in CATEGORICAL_FEATURES:

        replacement_values[feature] = float(
            X_development_encoded[
                feature
            ]
            .mode()
            .iloc[0]
        )

    else:

        replacement_values[feature] = float(
            X_development_encoded[
                feature
            ].median()
        )


replacement_vector = np.asarray(
    [
        replacement_values[feature]
        for feature in MODEL_FEATURES
    ],
    dtype=float
)


# ------------------------------------------------------------
# 5. Metric utilities
# ------------------------------------------------------------

def expected_calibration_error(
    y,
    probability,
    n_bins=10
):

    y = np.asarray(y)
    probability = np.asarray(
        probability,
        dtype=float
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    bin_ids = np.digitize(
        probability,
        bin_edges[1:-1],
        right=True
    )

    result = 0.0

    for bin_id in range(n_bins):

        mask = bin_ids == bin_id

        if mask.sum() == 0:
            continue

        result += (
            mask.mean()
            * abs(
                probability[mask].mean()
                - y[mask].mean()
            )
        )

    return float(result)


def evaluate_predictions(
    y,
    probability,
    threshold
):

    probability = np.clip(
        np.asarray(
            probability,
            dtype=float
        ),
        1e-7,
        1 - 1e-7
    )

    prediction = (
        probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        prediction,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "ROC_AUC":
            roc_auc_score(
                y,
                probability
            ),

        "PR_AUC":
            average_precision_score(
                y,
                probability
            ),

        "Brier_Score":
            brier_score_loss(
                y,
                probability
            ),

        "Log_Loss":
            log_loss(
                y,
                probability,
                labels=[0, 1]
            ),

        "ECE_10":
            expected_calibration_error(
                y,
                probability
            ),

        "Accuracy":
            accuracy_score(
                y,
                prediction
            ),

        "Precision":
            precision_score(
                y,
                prediction,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y,
                prediction,
                zero_division=0
            ),

        "Specificity":
            specificity,

        "F1":
            f1_score(
                y,
                prediction,
                zero_division=0
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y,
                prediction
            ),

        "MCC":
            matthews_corrcoef(
                y,
                prediction
            )
    }


baseline_metric_records = []

baseline_metrics = {}

for model_name in REPRESENTATIVE_MODELS:

    metrics = evaluate_predictions(
        y=y_true,
        probability=
            baseline_probability_map[
                model_name
            ],
        threshold=
            locked_thresholds[
                model_name
            ]
    )

    baseline_metrics[
        model_name
    ] = metrics

    baseline_metric_records.append({
        "Model": model_name,
        "Locked_Threshold":
            locked_thresholds[
                model_name
            ],
        **metrics
    })


baseline_metric_table = pd.DataFrame(
    baseline_metric_records
)


# ------------------------------------------------------------
# 6. Perturbation functions
# ------------------------------------------------------------

def apply_numeric_noise(
    matrix,
    config,
    rng
):

    perturbed = np.asarray(
        matrix,
        dtype=float
    ).copy()

    fixed_sd = {
        "CreditScore":
            config["CreditScore_SD"],

        "Age":
            config["Age_SD"],

        "Tenure":
            config["Tenure_SD"]
    }

    for feature in [
        "CreditScore",
        "Age",
        "Tenure"
    ]:

        position = feature_position[
            feature
        ]

        noise = rng.normal(
            loc=0.0,
            scale=fixed_sd[feature],
            size=len(perturbed)
        )

        perturbed[:, position] += noise

        perturbed[:, position] = np.rint(
            perturbed[:, position]
        )

        perturbed[:, position] = np.clip(
            perturbed[:, position],
            development_minimum[feature],
            development_maximum[feature]
        )


    for feature, fraction_key in [
        (
            "Balance",
            "Balance_IQR_Fraction"
        ),
        (
            "EstimatedSalary",
            "EstimatedSalary_IQR_Fraction"
        )
    ]:

        position = feature_position[
            feature
        ]

        scale = max(
            development_iqr[feature]
            * config[fraction_key],
            1e-8
        )

        perturbed[:, position] += (
            rng.normal(
                loc=0.0,
                scale=scale,
                size=len(perturbed)
            )
        )

        perturbed[:, position] = np.clip(
            perturbed[:, position],
            development_minimum[feature],
            development_maximum[feature]
        )

    return perturbed


def apply_median_mode_masking(
    matrix,
    probability,
    rng
):

    perturbed = np.asarray(
        matrix,
        dtype=float
    ).copy()

    mask = (
        rng.random(
            perturbed.shape
        ) < probability
    )

    replacement_matrix = np.broadcast_to(
        replacement_vector,
        perturbed.shape
    )

    perturbed[mask] = (
        replacement_matrix[mask]
    )

    return perturbed


def apply_categorical_noise(
    matrix,
    probability,
    rng
):

    perturbed = np.asarray(
        matrix,
        dtype=float
    ).copy()

    for feature in CATEGORICAL_FEATURES:

        position = feature_position[
            feature
        ]

        category_count = len(
            category_levels[
                feature
            ]
        )

        noise_mask = (
            rng.random(
                len(perturbed)
            ) < probability
        )

        current_codes = (
            np.rint(
                perturbed[
                    noise_mask,
                    position
                ]
            )
            .astype(int)
        )

        if category_count == 2:

            new_codes = (
                1 - current_codes
            )

        else:

            offsets = rng.integers(
                low=1,
                high=category_count,
                size=len(current_codes)
            )

            new_codes = (
                current_codes
                + offsets
            ) % category_count

        perturbed[
            noise_mask,
            position
        ] = new_codes

    return perturbed


def generate_perturbed_matrix(
    matrix,
    scenario_config,
    rng
):

    kind = scenario_config[
        "kind"
    ]

    if kind == "numeric_noise":

        return apply_numeric_noise(
            matrix,
            scenario_config,
            rng
        )

    if kind == "masking":

        return apply_median_mode_masking(
            matrix,
            scenario_config[
                "mask_probability"
            ],
            rng
        )

    if kind == "categorical_noise":

        return apply_categorical_noise(
            matrix,
            scenario_config[
                "noise_probability"
            ],
            rng
        )

    if kind == "combined":

        perturbed = apply_numeric_noise(
            matrix,
            scenario_config,
            rng
        )

        perturbed = apply_median_mode_masking(
            perturbed,
            scenario_config[
                "mask_probability"
            ],
            rng
        )

        perturbed = apply_categorical_noise(
            perturbed,
            scenario_config[
                "noise_probability"
            ],
            rng
        )

        return perturbed

    raise ValueError(
        f"Unknown perturbation kind: {kind}"
    )


# ------------------------------------------------------------
# 7. Run repeated perturbation audit
# ------------------------------------------------------------

robustness_records = []

overall_start = time.perf_counter()

scenario_names = list(
    SCENARIOS
)

for scenario_position, scenario_name in enumerate(
    scenario_names
):

    scenario_config = SCENARIOS[
        scenario_name
    ]

    print("\n" + "=" * 110)
    print(f"ROBUSTNESS SCENARIO: {scenario_name}")
    print("=" * 110)

    for repetition in range(
        1,
        N_REPETITIONS + 1
    ):

        repetition_seed = (
            ROBUSTNESS_SEED
            + scenario_position * 1000
            + repetition
        )

        rng = np.random.default_rng(
            repetition_seed
        )

        perturbed_matrix = (
            generate_perturbed_matrix(
                matrix=test_matrix,
                scenario_config=
                    scenario_config,
                rng=rng
            )
        )

        for model_name in REPRESENTATIVE_MODELS:

            perturbed_probability = (
                calibrated_prediction_functions[
                    model_name
                ](
                    perturbed_matrix
                )
            )

            baseline_probability = (
                baseline_probability_map[
                    model_name
                ]
            )

            baseline_prediction = (
                baseline_prediction_map[
                    model_name
                ]
            )

            threshold = locked_thresholds[
                model_name
            ]

            perturbed_prediction = (
                perturbed_probability
                >= threshold
            ).astype(int)

            perturbed_metrics = (
                evaluate_predictions(
                    y=y_true,
                    probability=
                        perturbed_probability,
                    threshold=threshold
                )
            )

            baseline_model_metrics = (
                baseline_metrics[
                    model_name
                ]
            )

            absolute_probability_shift = (
                np.abs(
                    perturbed_probability
                    - baseline_probability
                )
            )

            ranking_correlation = (
                spearmanr(
                    baseline_probability,
                    perturbed_probability
                ).statistic
            )

            robustness_records.append({
                "Scenario":
                    scenario_name,

                "Scenario_Kind":
                    scenario_config["kind"],

                "Repetition":
                    repetition,

                "Seed":
                    repetition_seed,

                "Model":
                    model_name,

                "Mean_Absolute_Probability_Shift":
                    float(
                        absolute_probability_shift.mean()
                    ),

                "Median_Absolute_Probability_Shift":
                    float(
                        np.median(
                            absolute_probability_shift
                        )
                    ),

                "P95_Absolute_Probability_Shift":
                    float(
                        np.quantile(
                            absolute_probability_shift,
                            0.95
                        )
                    ),

                "Maximum_Absolute_Probability_Shift":
                    float(
                        absolute_probability_shift.max()
                    ),

                "Prediction_Rank_Spearman":
                    float(
                        ranking_correlation
                    ),

                "Locked_Class_Flip_Rate":
                    float(
                        np.mean(
                            perturbed_prediction
                            != baseline_prediction
                        )
                    ),

                **{
                    metric:
                        value
                    for metric, value
                    in perturbed_metrics.items()
                },

                **{
                    f"Delta_{metric}":
                        perturbed_metrics[metric]
                        - baseline_model_metrics[
                            metric
                        ]

                    for metric in [
                        "ROC_AUC",
                        "PR_AUC",
                        "Brier_Score",
                        "Log_Loss",
                        "ECE_10",
                        "Accuracy",
                        "Precision",
                        "Recall",
                        "Specificity",
                        "F1",
                        "Balanced_Accuracy",
                        "MCC"
                    ]
                }
            })

        if (
            repetition == 1
            or repetition % 5 == 0
        ):

            print(
                f"Completed repetition "
                f"{repetition}/"
                f"{N_REPETITIONS}"
            )


robustness_replicates = pd.DataFrame(
    robustness_records
)


expected_rows = (
    len(SCENARIOS)
    * N_REPETITIONS
    * len(REPRESENTATIVE_MODELS)
)

assert len(
    robustness_replicates
) == expected_rows


# ------------------------------------------------------------
# 8. Aggregate repeated results
# ------------------------------------------------------------

summary_metrics = [
    "Mean_Absolute_Probability_Shift",
    "P95_Absolute_Probability_Shift",
    "Prediction_Rank_Spearman",
    "Locked_Class_Flip_Rate",
    "Delta_ROC_AUC",
    "Delta_PR_AUC",
    "Delta_Brier_Score",
    "Delta_Log_Loss",
    "Delta_ECE_10",
    "Delta_F1",
    "Delta_Balanced_Accuracy",
    "Delta_MCC"
]


robustness_summary = (
    robustness_replicates
    .groupby(
        [
            "Model",
            "Scenario",
            "Scenario_Kind"
        ]
    )[summary_metrics]
    .agg(["mean", "std"])
)

robustness_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat
    in robustness_summary.columns
]

robustness_summary = (
    robustness_summary
    .reset_index()
)


scenario_order = {
    scenario_name: position
    for position, scenario_name
    in enumerate(scenario_names)
}

model_order = {
    model_name: position
    for position, model_name
    in enumerate(REPRESENTATIVE_MODELS)
}

robustness_summary[
    "_Scenario_Order"
] = robustness_summary[
    "Scenario"
].map(scenario_order)

robustness_summary[
    "_Model_Order"
] = robustness_summary[
    "Model"
].map(model_order)

robustness_summary = (
    robustness_summary
    .sort_values(
        [
            "_Scenario_Order",
            "_Model_Order"
        ]
    )
    .drop(
        columns=[
            "_Scenario_Order",
            "_Model_Order"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Robustness ranking by scenario
#
# Descriptive only. This does not alter model selection.
# ------------------------------------------------------------

robustness_summary[
    "Absolute_ROC_AUC_Degradation"
] = np.maximum(
    0.0,
    -robustness_summary[
        "Delta_ROC_AUC_mean"
    ]
)

robustness_summary[
    "Absolute_PR_AUC_Degradation"
] = np.maximum(
    0.0,
    -robustness_summary[
        "Delta_PR_AUC_mean"
    ]
)

robustness_summary[
    "Robustness_Composite"
] = (
    robustness_summary[
        "Absolute_ROC_AUC_Degradation"
    ]
    +
    robustness_summary[
        "Absolute_PR_AUC_Degradation"
    ]
    +
    robustness_summary[
        "Locked_Class_Flip_Rate_mean"
    ]
)


robustness_summary[
    "Scenario_Robustness_Rank"
] = (
    robustness_summary
    .groupby("Scenario")[
        "Robustness_Composite"
    ]
    .rank(
        method="min",
        ascending=True
    )
)


# ------------------------------------------------------------
# 10. Print results
# ------------------------------------------------------------

print("\n" + "=" * 185)
print("INPUT-PERTURBATION ROBUSTNESS SUMMARY")
print(
    "Metric deltas are perturbed minus locked baseline. "
    "Negative ROC-AUC, PR-AUC, F1 and MCC deltas indicate degradation."
)
print("=" * 185)

display_columns = [
    "Scenario",
    "Model",
    "Mean_Absolute_Probability_Shift_mean",
    "P95_Absolute_Probability_Shift_mean",
    "Prediction_Rank_Spearman_mean",
    "Locked_Class_Flip_Rate_mean",
    "Delta_ROC_AUC_mean",
    "Delta_PR_AUC_mean",
    "Delta_Brier_Score_mean",
    "Delta_F1_mean",
    "Delta_MCC_mean",
    "Scenario_Robustness_Rank"
]

print(
    robustness_summary[
        display_columns
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 11. Worst-case descriptive summary
# ------------------------------------------------------------

worst_case_records = []

for model_name in REPRESENTATIVE_MODELS:

    model_rows = robustness_summary[
        robustness_summary["Model"]
        == model_name
    ]

    worst_roc_row = (
        model_rows
        .sort_values(
            "Delta_ROC_AUC_mean"
        )
        .iloc[0]
    )

    worst_pr_row = (
        model_rows
        .sort_values(
            "Delta_PR_AUC_mean"
        )
        .iloc[0]
    )

    highest_flip_row = (
        model_rows
        .sort_values(
            "Locked_Class_Flip_Rate_mean",
            ascending=False
        )
        .iloc[0]
    )

    worst_case_records.append({
        "Model": model_name,

        "Worst_ROC_AUC_Scenario":
            worst_roc_row[
                "Scenario"
            ],

        "Worst_Delta_ROC_AUC":
            worst_roc_row[
                "Delta_ROC_AUC_mean"
            ],

        "Worst_PR_AUC_Scenario":
            worst_pr_row[
                "Scenario"
            ],

        "Worst_Delta_PR_AUC":
            worst_pr_row[
                "Delta_PR_AUC_mean"
            ],

        "Highest_Flip_Scenario":
            highest_flip_row[
                "Scenario"
            ],

        "Highest_Class_Flip_Rate":
            highest_flip_row[
                "Locked_Class_Flip_Rate_mean"
            ]
    })


worst_case_summary = pd.DataFrame(
    worst_case_records
)


print("\n" + "=" * 130)
print("MODEL WORST-CASE PERTURBATION SUMMARY")
print("=" * 130)

print(
    worst_case_summary
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 12. Save artifacts
# ------------------------------------------------------------

baseline_metric_table.to_csv(
    OUTPUT_DIR
    / "representative_model_locked_baselines.csv",
    index=False
)

robustness_replicates.to_csv(
    OUTPUT_DIR
    / "input_perturbation_replicate_metrics.csv",
    index=False
)

robustness_summary.to_csv(
    OUTPUT_DIR
    / "input_perturbation_robustness_summary.csv",
    index=False
)

worst_case_summary.to_csv(
    OUTPUT_DIR
    / "model_worst_case_perturbation_summary.csv",
    index=False
)


run_manifest = {
    "models":
        REPRESENTATIVE_MODELS,

    "test_rows":
        int(len(y_true)),

    "perturbation_scenarios":
        SCENARIOS,

    "repetitions_per_scenario":
        N_REPETITIONS,

    "base_seed":
        ROBUSTNESS_SEED,

    "classification_thresholds":
        locked_thresholds,

    "comparison_baseline":
        (
            "Immutable calibrated locked-test "
            "probabilities and predictions"
        ),

    "inference_scope":
        (
            "Descriptive robustness audit; "
            "not used for model tuning or selection"
        ),

    "post_test_tuning_performed":
        False
}


with open(
    OUTPUT_DIR
    / "input_perturbation_run_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        run_manifest,
        file,
        indent=4
    )


elapsed_seconds = (
    time.perf_counter()
    - overall_start
)


print("\n" + "=" * 105)
print("CHECKPOINT 8C1 SAVED")
print("=" * 105)

print("Output directory:")
print(OUTPUT_DIR)

print(
    f"\nElapsed time: "
    f"{elapsed_seconds / 60:.2f} minutes"
)

print(
    "\nNo model, calibrator, threshold, "
    "or locked-test metric was modified."
)

In [ ]:
# ============================================================
# SUBGROUP PERFORMANCE AND DISPARITY AUDIT
#
# - All 9 final models
# - Gender, Geography, AgeBand, ActivityStatus
# - Locked calibrated probabilities
# - Locked model-specific classifications
# - 1,000 bootstrap samples for disparity uncertainty
#
# No model training, tuning, calibration, or threshold changes.
# ============================================================

from pathlib import Path
import hashlib
import json
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

EXPECTED_DATA_SHA256 = (
    "3996cd1fa372e0db0cd9c0ebac35bbd4"
    "e8e3c65fb942bb010c826e7b1eeef0a0"
)

RESTORED_ROOT_CANDIDATES = [Path("/kaggle/working")]

OUTPUT_DIR = Path(
    "/kaggle/working/subgroup_fairness_audit"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_ORDER = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

SUBGROUP_ATTRIBUTES = [
    "Gender",
    "Geography",
    "AgeBand",
    "ActivityStatus"
]

N_BOOTSTRAPS = 1000
BOOTSTRAP_SEED = 2060
N_ECE_BINS = 5

FAIRNESS_METRICS = [
    "Demographic_Parity_Gap",
    "Equal_Opportunity_TPR_Gap",
    "FPR_Gap",
    "Equalized_Odds_Gap",
    "Predictive_Parity_Gap",
    "Brier_Range",
    "Calibration_In_Large_Range"
]


# ------------------------------------------------------------
# 2. Utilities
# ------------------------------------------------------------

def sha256_file(
    filepath,
    block_size=1024 * 1024
):
    digest = hashlib.sha256()

    with open(filepath, "rb") as file:

        while True:
            block = file.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def find_unique_file(
    root,
    filename,
    preferred_parent=None
):
    matches = list(
        root.rglob(filename)
    )

    if preferred_parent is not None:

        preferred_matches = [
            path
            for path in matches
            if preferred_parent in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}."
        )

    raise RuntimeError(
        f"Multiple copies of {filename} found:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


existing_roots = [
    root
    for root in RESTORED_ROOT_CANDIDATES
    if root.exists()
]

if not existing_roots:
    raise FileNotFoundError(
        "Working artifact directory was not found."
    )

RESTORED_ROOT = existing_roots[0]


# ------------------------------------------------------------
# 3. Load immutable predictions and original attributes
# ------------------------------------------------------------

assert sha256_file(
    DATASET_PATH
) == EXPECTED_DATA_SHA256

FINAL_PREDICTIONS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_locked_test_predictions.csv",
    preferred_parent="final_locked_holdout"
)

final_predictions = pd.read_csv(
    FINAL_PREDICTIONS_PATH
)

data = pd.read_csv(
    DATASET_PATH
)

assert len(final_predictions) == 2000
assert len(data) == 10000

test_indices = (
    final_predictions[
        "Original_Index"
    ]
    .astype(int)
    .to_numpy()
)

test_attributes = (
    data.loc[
        test_indices,
        [
            "CustomerId",
            "Gender",
            "Geography",
            "Age",
            "IsActiveMember"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

assert np.array_equal(
    test_attributes[
        "CustomerId"
    ].to_numpy(),
    final_predictions[
        "CustomerId"
    ].to_numpy()
)

audit_data = pd.concat(
    [
        final_predictions.reset_index(
            drop=True
        ),
        test_attributes[
            [
                "Gender",
                "Geography",
                "Age",
                "IsActiveMember"
            ]
        ]
    ],
    axis=1
)


# Fixed descriptive age bands.
audit_data["AgeBand"] = pd.cut(
    audit_data["Age"],
    bins=[
        17,
        34,
        44,
        54,
        np.inf
    ],
    labels=[
        "18-34",
        "35-44",
        "45-54",
        "55+"
    ],
    right=True,
    include_lowest=True
).astype(str)

audit_data[
    "ActivityStatus"
] = audit_data[
    "IsActiveMember"
].map({
    0: "Inactive",
    1: "Active"
})

assert audit_data[
    SUBGROUP_ATTRIBUTES
].notna().all().all()


# ------------------------------------------------------------
# 4. Cohort summary
# ------------------------------------------------------------

cohort_records = []

for attribute in SUBGROUP_ATTRIBUTES:

    for group_name, group_data in (
        audit_data.groupby(
            attribute,
            observed=True
        )
    ):

        cohort_records.append({
            "Attribute": attribute,
            "Group": str(group_name),
            "Rows": len(group_data),
            "Positive_Count":
                int(
                    group_data[
                        "y_true"
                    ].sum()
                ),
            "Churn_Prevalence":
                float(
                    group_data[
                        "y_true"
                    ].mean()
                )
        })


cohort_summary = pd.DataFrame(
    cohort_records
)

print("=" * 105)
print("SUBGROUP COHORT COUNTS")
print("=" * 105)

print(
    cohort_summary
    .sort_values(
        [
            "Attribute",
            "Group"
        ]
    )
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Metric utilities
# ------------------------------------------------------------

def safe_roc_auc(
    y,
    probability
):
    if np.unique(y).size < 2:
        return np.nan

    return float(
        roc_auc_score(
            y,
            probability
        )
    )


def safe_pr_auc(
    y,
    probability
):
    if np.sum(y) == 0:
        return np.nan

    return float(
        average_precision_score(
            y,
            probability
        )
    )


def expected_calibration_error(
    y,
    probability,
    n_bins=N_ECE_BINS
):
    y = np.asarray(y)
    probability = np.asarray(
        probability,
        dtype=float
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    bin_ids = np.digitize(
        probability,
        edges[1:-1],
        right=True
    )

    ece = 0.0

    for bin_id in range(n_bins):

        mask = bin_ids == bin_id

        if mask.sum() == 0:
            continue

        ece += (
            mask.mean()
            * abs(
                probability[mask].mean()
                - y[mask].mean()
            )
        )

    return float(ece)


def compute_group_metrics(
    y,
    probability,
    prediction
):
    y = np.asarray(
        y,
        dtype=int
    )

    probability = np.asarray(
        probability,
        dtype=float
    )

    prediction = np.asarray(
        prediction,
        dtype=int
    )

    tn, fp, fn, tp = confusion_matrix(
        y,
        prediction,
        labels=[0, 1]
    ).ravel()

    tpr = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    tnr = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )

    prevalence = float(
        y.mean()
    )

    mean_probability = float(
        probability.mean()
    )

    return {
        "Rows":
            len(y),

        "Positive_Count":
            int(y.sum()),

        "Churn_Prevalence":
            prevalence,

        "Mean_Probability":
            mean_probability,

        "Calibration_In_Large":
            mean_probability
            - prevalence,

        "Absolute_Calibration_Gap":
            abs(
                mean_probability
                - prevalence
            ),

        "Predicted_Positive_Rate":
            float(
                prediction.mean()
            ),

        "ROC_AUC":
            safe_roc_auc(
                y,
                probability
            ),

        "PR_AUC":
            safe_pr_auc(
                y,
                probability
            ),

        "Brier_Score":
            float(
                brier_score_loss(
                    y,
                    probability
                )
            ),

        "ECE_5":
            expected_calibration_error(
                y,
                probability
            ),

        "Accuracy":
            float(
                accuracy_score(
                    y,
                    prediction
                )
            ),

        "Precision":
            float(
                precision_score(
                    y,
                    prediction,
                    zero_division=0
                )
            ),

        "TPR_Recall":
            float(tpr),

        "Specificity":
            float(tnr),

        "FPR":
            float(fpr),

        "FNR":
            float(fnr),

        "F1":
            float(
                f1_score(
                    y,
                    prediction,
                    zero_division=0
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y,
                    prediction
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y,
                    prediction
                )
            ),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


# ------------------------------------------------------------
# 6. Subgroup metrics for all models
# ------------------------------------------------------------

subgroup_metric_records = []

for model_name in MODEL_ORDER:

    probability = (
        audit_data[
            f"{model_name}_"
            "Calibrated_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    prediction = (
        audit_data[
            f"{model_name}_"
            "Locked_Prediction"
        ]
        .astype(int)
        .to_numpy()
    )

    y_true = (
        audit_data["y_true"]
        .astype(int)
        .to_numpy()
    )

    for attribute in SUBGROUP_ATTRIBUTES:

        group_values = (
            audit_data[
                attribute
            ]
            .astype(str)
            .to_numpy()
        )

        for group_name in sorted(
            np.unique(group_values)
        ):

            mask = (
                group_values
                == group_name
            )

            metrics = compute_group_metrics(
                y=y_true[mask],
                probability=
                    probability[mask],
                prediction=
                    prediction[mask]
            )

            subgroup_metric_records.append({
                "Model": model_name,
                "Attribute": attribute,
                "Group": group_name,
                **metrics
            })


subgroup_metrics = pd.DataFrame(
    subgroup_metric_records
)


# ------------------------------------------------------------
# 7. Disparity calculation
# ------------------------------------------------------------

def metric_range(
    table,
    metric
):
    valid = table[
        [
            "Group",
            metric
        ]
    ].dropna()

    if valid.empty:
        return (
            np.nan,
            None,
            None
        )

    minimum_row = valid.loc[
        valid[metric].idxmin()
    ]

    maximum_row = valid.loc[
        valid[metric].idxmax()
    ]

    return (
        float(
            maximum_row[metric]
            - minimum_row[metric]
        ),
        str(
            minimum_row["Group"]
        ),
        str(
            maximum_row["Group"]
        )
    )


def calculate_disparity_row(
    model_name,
    attribute,
    attribute_table
):
    dp_gap, dp_low, dp_high = (
        metric_range(
            attribute_table,
            "Predicted_Positive_Rate"
        )
    )

    tpr_gap, tpr_low, tpr_high = (
        metric_range(
            attribute_table,
            "TPR_Recall"
        )
    )

    fpr_gap, fpr_low, fpr_high = (
        metric_range(
            attribute_table,
            "FPR"
        )
    )

    precision_gap, _, _ = (
        metric_range(
            attribute_table,
            "Precision"
        )
    )

    brier_range, _, _ = (
        metric_range(
            attribute_table,
            "Brier_Score"
        )
    )

    calibration_range, _, _ = (
        metric_range(
            attribute_table,
            "Calibration_In_Large"
        )
    )

    roc_range, _, _ = (
        metric_range(
            attribute_table,
            "ROC_AUC"
        )
    )

    worst_brier_row = (
        attribute_table
        .sort_values(
            "Brier_Score",
            ascending=False
        )
        .iloc[0]
    )

    return {
        "Model": model_name,
        "Attribute": attribute,

        "Group_Count":
            attribute_table[
                "Group"
            ].nunique(),

        "Minimum_Group_Rows":
            int(
                attribute_table[
                    "Rows"
                ].min()
            ),

        "Demographic_Parity_Gap":
            dp_gap,

        "Lowest_Positive_Rate_Group":
            dp_low,

        "Highest_Positive_Rate_Group":
            dp_high,

        "Equal_Opportunity_TPR_Gap":
            tpr_gap,

        "Lowest_TPR_Group":
            tpr_low,

        "Highest_TPR_Group":
            tpr_high,

        "FPR_Gap":
            fpr_gap,

        "Lowest_FPR_Group":
            fpr_low,

        "Highest_FPR_Group":
            fpr_high,

        "Equalized_Odds_Gap":
            max(
                tpr_gap,
                fpr_gap
            ),

        "Predictive_Parity_Gap":
            precision_gap,

        "Brier_Range":
            brier_range,

        "Highest_Brier_Group":
            str(
                worst_brier_row[
                    "Group"
                ]
            ),

        "Calibration_In_Large_Range":
            calibration_range,

        "ROC_AUC_Range":
            roc_range
    }


disparity_records = []

for (
    model_name,
    attribute
), attribute_table in subgroup_metrics.groupby(
    [
        "Model",
        "Attribute"
    ],
    sort=False
):

    disparity_records.append(
        calculate_disparity_row(
            model_name=model_name,
            attribute=attribute,
            attribute_table=
                attribute_table
        )
    )


disparity_summary = pd.DataFrame(
    disparity_records
)


# ------------------------------------------------------------
# 8. Gender directional differences
#
# Difference = Female minus Male.
# ------------------------------------------------------------

GENDER_METRICS = [
    "Predicted_Positive_Rate",
    "TPR_Recall",
    "FPR",
    "Precision",
    "Brier_Score",
    "Calibration_In_Large"
]

gender_direction_records = []

for model_name in MODEL_ORDER:

    gender_table = (
        subgroup_metrics[
            (
                subgroup_metrics[
                    "Model"
                ] == model_name
            )
            &
            (
                subgroup_metrics[
                    "Attribute"
                ] == "Gender"
            )
        ]
        .set_index("Group")
    )

    assert {
        "Female",
        "Male"
    }.issubset(
        gender_table.index
    )

    record = {
        "Model": model_name
    }

    for metric in GENDER_METRICS:

        record[
            f"Female_minus_Male_{metric}"
        ] = (
            gender_table.loc[
                "Female",
                metric
            ]
            -
            gender_table.loc[
                "Male",
                metric
            ]
        )

    gender_direction_records.append(
        record
    )


gender_directional_gaps = pd.DataFrame(
    gender_direction_records
)


# ------------------------------------------------------------
# 9. Bootstrap disparity uncertainty
# ------------------------------------------------------------

print("\n" + "=" * 105)
print("BOOTSTRAPPING SUBGROUP DISPARITIES")
print("=" * 105)

print("Bootstrap samples:", N_BOOTSTRAPS)
print("Models:", len(MODEL_ORDER))
print("Attributes:", len(SUBGROUP_ATTRIBUTES))

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

n_rows = len(audit_data)

y_full = (
    audit_data["y_true"]
    .astype(int)
    .to_numpy()
)

attribute_arrays = {
    attribute:
        audit_data[
            attribute
        ]
        .astype(str)
        .to_numpy()

    for attribute in SUBGROUP_ATTRIBUTES
}

probability_arrays = {
    model_name:
        audit_data[
            f"{model_name}_"
            "Calibrated_Probability"
        ]
        .astype(float)
        .to_numpy()

    for model_name in MODEL_ORDER
}

prediction_arrays = {
    model_name:
        audit_data[
            f"{model_name}_"
            "Locked_Prediction"
        ]
        .astype(int)
        .to_numpy()

    for model_name in MODEL_ORDER
}


def fast_group_values(
    y,
    probability,
    prediction
):
    tn, fp, fn, tp = confusion_matrix(
        y,
        prediction,
        labels=[0, 1]
    ).ravel()

    tpr = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    prevalence = y.mean()
    mean_probability = (
        probability.mean()
    )

    return {
        "Predicted_Positive_Rate":
            prediction.mean(),

        "TPR_Recall":
            tpr,

        "FPR":
            fpr,

        "Precision":
            precision,

        "Brier_Score":
            np.mean(
                (
                    probability
                    - y
                ) ** 2
            ),

        "Calibration_In_Large":
            mean_probability
            - prevalence
    }


bootstrap_disparity_records = []
bootstrap_gender_records = []

bootstrap_start = time.perf_counter()

for bootstrap_id in range(
    1,
    N_BOOTSTRAPS + 1
):

    sampled_positions = rng.integers(
        low=0,
        high=n_rows,
        size=n_rows
    )

    y_bootstrap = y_full[
        sampled_positions
    ]

    for model_name in MODEL_ORDER:

        probability_bootstrap = (
            probability_arrays[
                model_name
            ][sampled_positions]
        )

        prediction_bootstrap = (
            prediction_arrays[
                model_name
            ][sampled_positions]
        )

        for attribute in SUBGROUP_ATTRIBUTES:

            groups_bootstrap = (
                attribute_arrays[
                    attribute
                ][sampled_positions]
            )

            group_metric_values = {}

            for group_name in np.unique(
                groups_bootstrap
            ):

                mask = (
                    groups_bootstrap
                    == group_name
                )

                group_metric_values[
                    group_name
                ] = fast_group_values(
                    y=y_bootstrap[mask],
                    probability=
                        probability_bootstrap[
                            mask
                        ],
                    prediction=
                        prediction_bootstrap[
                            mask
                        ]
                )

            metric_arrays = {
                metric: np.asarray(
                    [
                        values[metric]
                        for values
                        in group_metric_values.values()
                    ],
                    dtype=float
                )

                for metric in [
                    "Predicted_Positive_Rate",
                    "TPR_Recall",
                    "FPR",
                    "Precision",
                    "Brier_Score",
                    "Calibration_In_Large"
                ]
            }

            def valid_range(values):

                values = values[
                    np.isfinite(values)
                ]

                if len(values) == 0:
                    return np.nan

                return float(
                    values.max()
                    - values.min()
                )

            dp_gap = valid_range(
                metric_arrays[
                    "Predicted_Positive_Rate"
                ]
            )

            tpr_gap = valid_range(
                metric_arrays[
                    "TPR_Recall"
                ]
            )

            fpr_gap = valid_range(
                metric_arrays[
                    "FPR"
                ]
            )

            bootstrap_disparity_records.append({
                "Bootstrap_ID":
                    bootstrap_id,

                "Model":
                    model_name,

                "Attribute":
                    attribute,

                "Demographic_Parity_Gap":
                    dp_gap,

                "Equal_Opportunity_TPR_Gap":
                    tpr_gap,

                "FPR_Gap":
                    fpr_gap,

                "Equalized_Odds_Gap":
                    max(
                        tpr_gap,
                        fpr_gap
                    ),

                "Predictive_Parity_Gap":
                    valid_range(
                        metric_arrays[
                            "Precision"
                        ]
                    ),

                "Brier_Range":
                    valid_range(
                        metric_arrays[
                            "Brier_Score"
                        ]
                    ),

                "Calibration_In_Large_Range":
                    valid_range(
                        metric_arrays[
                            "Calibration_In_Large"
                        ]
                    )
            })

            if attribute == "Gender":

                if {
                    "Female",
                    "Male"
                }.issubset(
                    group_metric_values
                ):

                    gender_record = {
                        "Bootstrap_ID":
                            bootstrap_id,

                        "Model":
                            model_name
                    }

                    for metric in GENDER_METRICS:

                        gender_record[
                            f"Female_minus_Male_{metric}"
                        ] = (
                            group_metric_values[
                                "Female"
                            ][metric]
                            -
                            group_metric_values[
                                "Male"
                            ][metric]
                        )

                    bootstrap_gender_records.append(
                        gender_record
                    )

    if (
        bootstrap_id == 1
        or bootstrap_id % 100 == 0
    ):

        elapsed = (
            time.perf_counter()
            - bootstrap_start
        )

        print(
            f"Completed "
            f"{bootstrap_id}/"
            f"{N_BOOTSTRAPS} | "
            f"elapsed="
            f"{elapsed / 60:.2f} min"
        )


bootstrap_disparities = pd.DataFrame(
    bootstrap_disparity_records
)

bootstrap_gender = pd.DataFrame(
    bootstrap_gender_records
)

assert len(
    bootstrap_disparities
) == (
    N_BOOTSTRAPS
    * len(MODEL_ORDER)
    * len(SUBGROUP_ATTRIBUTES)
)

assert len(
    bootstrap_gender
) == (
    N_BOOTSTRAPS
    * len(MODEL_ORDER)
)


# ------------------------------------------------------------
# 10. Disparity confidence intervals
# ------------------------------------------------------------

disparity_ci_records = []

for (
    model_name,
    attribute
), bootstrap_table in (
    bootstrap_disparities.groupby(
        [
            "Model",
            "Attribute"
        ]
    )
):

    point_row = (
        disparity_summary[
            (
                disparity_summary[
                    "Model"
                ] == model_name
            )
            &
            (
                disparity_summary[
                    "Attribute"
                ] == attribute
            )
        ]
        .iloc[0]
    )

    for metric in FAIRNESS_METRICS:

        values = (
            bootstrap_table[metric]
            .dropna()
            .to_numpy()
        )

        disparity_ci_records.append({
            "Model":
                model_name,

            "Attribute":
                attribute,

            "Metric":
                metric,

            "Point_Estimate":
                float(
                    point_row[
                        metric
                    ]
                ),

            "CI_Lower":
                float(
                    np.quantile(
                        values,
                        0.025
                    )
                ),

            "CI_Upper":
                float(
                    np.quantile(
                        values,
                        0.975
                    )
                ),

            "Bootstrap_Replicates":
                len(values)
        })


disparity_confidence_intervals = (
    pd.DataFrame(
        disparity_ci_records
    )
)


# ------------------------------------------------------------
# 11. Gender directional confidence intervals
# ------------------------------------------------------------

gender_ci_records = []

for model_name in MODEL_ORDER:

    point_row = (
        gender_directional_gaps[
            gender_directional_gaps[
                "Model"
            ] == model_name
        ]
        .iloc[0]
    )

    model_bootstrap = (
        bootstrap_gender[
            bootstrap_gender[
                "Model"
            ] == model_name
        ]
    )

    for metric in GENDER_METRICS:

        column = (
            f"Female_minus_Male_{metric}"
        )

        values = (
            model_bootstrap[
                column
            ]
            .dropna()
            .to_numpy()
        )

        gender_ci_records.append({
            "Model":
                model_name,

            "Metric":
                metric,

            "Female_minus_Male":
                float(
                    point_row[
                        column
                    ]
                ),

            "CI_Lower":
                float(
                    np.quantile(
                        values,
                        0.025
                    )
                ),

            "CI_Upper":
                float(
                    np.quantile(
                        values,
                        0.975
                    )
                )
        })


gender_directional_ci = pd.DataFrame(
    gender_ci_records
)


# ------------------------------------------------------------
# 12. Compact display table
# ------------------------------------------------------------

DISPLAY_GAPS = [
    "Demographic_Parity_Gap",
    "Equal_Opportunity_TPR_Gap",
    "FPR_Gap",
    "Equalized_Odds_Gap"
]


def format_estimate_ci(
    estimate,
    lower,
    upper
):
    return (
        f"{estimate:.4f} "
        f"[{lower:.4f}, {upper:.4f}]"
    )


display_records = []

for model_name in MODEL_ORDER:

    for attribute in SUBGROUP_ATTRIBUTES:

        record = {
            "Model":
                model_name,

            "Attribute":
                attribute
        }

        for metric in DISPLAY_GAPS:

            row = (
                disparity_confidence_intervals[
                    (
                        disparity_confidence_intervals[
                            "Model"
                        ] == model_name
                    )
                    &
                    (
                        disparity_confidence_intervals[
                            "Attribute"
                        ] == attribute
                    )
                    &
                    (
                        disparity_confidence_intervals[
                            "Metric"
                        ] == metric
                    )
                ]
                .iloc[0]
            )

            record[
                f"{metric}_95CI"
            ] = format_estimate_ci(
                row[
                    "Point_Estimate"
                ],
                row[
                    "CI_Lower"
                ],
                row[
                    "CI_Upper"
                ]
            )

        display_records.append(
            record
        )


disparity_display = pd.DataFrame(
    display_records
)


# ------------------------------------------------------------
# 13. Print results
# ------------------------------------------------------------

print("\n" + "=" * 185)
print("FAIRNESS AND SUBGROUP DISPARITY SUMMARY")
print(
    "All classification metrics use the locked "
    "model-specific thresholds."
)
print("=" * 185)

summary_columns = [
    "Model",
    "Attribute",
    "Minimum_Group_Rows",
    "Demographic_Parity_Gap",
    "Equal_Opportunity_TPR_Gap",
    "FPR_Gap",
    "Equalized_Odds_Gap",
    "Predictive_Parity_Gap",
    "Brier_Range",
    "Calibration_In_Large_Range",
    "ROC_AUC_Range",
    "Lowest_TPR_Group",
    "Highest_TPR_Group",
    "Highest_FPR_Group",
    "Highest_Brier_Group"
]

print(
    disparity_summary[
        summary_columns
    ]
    .sort_values(
        [
            "Attribute",
            "Model"
        ]
    )
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 180)
print("BOOTSTRAP DISPARITY 95% INTERVALS")
print(
    "Intervals quantify sampling uncertainty. "
    "They are not legal or normative fairness thresholds."
)
print("=" * 180)

print(
    disparity_display
    .sort_values(
        [
            "Attribute",
            "Model"
        ]
    )
    .to_string(index=False)
)


print("\n" + "=" * 170)
print("GENDER DIRECTIONAL GAPS")
print(
    "All differences are Female minus Male."
)
print("=" * 170)

gender_display = (
    gender_directional_ci
    .pivot(
        index="Model",
        columns="Metric",
        values=[
            "Female_minus_Male",
            "CI_Lower",
            "CI_Upper"
        ]
    )
)

gender_display_records = []

for model_name in MODEL_ORDER:

    record = {
        "Model": model_name
    }

    for metric in GENDER_METRICS:

        row = gender_directional_ci[
            (
                gender_directional_ci[
                    "Model"
                ] == model_name
            )
            &
            (
                gender_directional_ci[
                    "Metric"
                ] == metric
            )
        ].iloc[0]

        record[
            f"{metric}_Difference_95CI"
        ] = format_estimate_ci(
            row[
                "Female_minus_Male"
            ],
            row[
                "CI_Lower"
            ],
            row[
                "CI_Upper"
            ]
        )

    gender_display_records.append(
        record
    )


gender_display_table = pd.DataFrame(
    gender_display_records
)

print(
    gender_display_table
    .to_string(index=False)
)


# ------------------------------------------------------------
# 14. Save artifacts
# ------------------------------------------------------------

cohort_summary.to_csv(
    OUTPUT_DIR
    / "subgroup_cohort_counts.csv",
    index=False
)

subgroup_metrics.to_csv(
    OUTPUT_DIR
    / "all_model_subgroup_metrics.csv",
    index=False
)

disparity_summary.to_csv(
    OUTPUT_DIR
    / "all_model_subgroup_disparity_summary.csv",
    index=False
)

gender_directional_gaps.to_csv(
    OUTPUT_DIR
    / "gender_directional_point_estimates.csv",
    index=False
)

bootstrap_disparities.to_csv(
    OUTPUT_DIR
    / "subgroup_disparity_bootstrap_replicates.csv",
    index=False
)

bootstrap_gender.to_csv(
    OUTPUT_DIR
    / "gender_directional_bootstrap_replicates.csv",
    index=False
)

disparity_confidence_intervals.to_csv(
    OUTPUT_DIR
    / "subgroup_disparity_confidence_intervals.csv",
    index=False
)

gender_directional_ci.to_csv(
    OUTPUT_DIR
    / "gender_directional_confidence_intervals.csv",
    index=False
)

disparity_display.to_csv(
    OUTPUT_DIR
    / "subgroup_disparity_ci_display.csv",
    index=False
)

gender_display_table.to_csv(
    OUTPUT_DIR
    / "gender_directional_ci_display.csv",
    index=False
)


run_manifest = {
    "dataset_sha256":
        EXPECTED_DATA_SHA256,

    "test_rows":
        int(len(audit_data)),

    "models":
        MODEL_ORDER,

    "subgroup_attributes":
        SUBGROUP_ATTRIBUTES,

    "age_bands": [
        "18-34",
        "35-44",
        "45-54",
        "55+"
    ],

    "bootstrap_samples":
        N_BOOTSTRAPS,

    "bootstrap_seed":
        BOOTSTRAP_SEED,

    "classification_source":
        (
            "Immutable locked model-specific "
            "predictions"
        ),

    "interpretation_scope":
        (
            "Descriptive subgroup and fairness audit; "
            "not a causal or legal determination"
        ),

    "post_test_tuning_performed":
        False
}


with open(
    OUTPUT_DIR
    / "subgroup_fairness_run_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        run_manifest,
        file,
        indent=4
    )


elapsed_seconds = (
    time.perf_counter()
    - bootstrap_start
)


print("\n" + "=" * 105)
print("CHECKPOINT 8D1 SAVED")
print("=" * 105)

print("Output directory:")
print(OUTPUT_DIR)

print(
    f"\nBootstrap elapsed time: "
    f"{elapsed_seconds / 60:.2f} minutes"
)

print(
    "\nNo model, calibrator, threshold, "
    "or locked-test metric was modified."
)

In [ ]:
# ============================================================
# DIRECT SENSITIVITY TO GENDER AND GEOGRAPHY
#
# Holds all other observed features fixed.
# Measures direct prediction sensitivity to changing only:
# - Gender
# - Geography
#
# This is a diagnostic sensitivity analysis, not a causal
# or legal counterfactual-fairness determination.
# ============================================================

from pathlib import Path
from itertools import combinations
import json
import time

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "subgroup_fairness_audit"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPRESENTATIVE_MODELS = [
    "XGBoost",
    "CatBoost",
    "MLP",
    "FTTransformer"
]

N_BOOTSTRAPS = 2000
BOOTSTRAP_SEED = 2070


# ------------------------------------------------------------
# 2. State verification
# ------------------------------------------------------------

required_variables = [
    "MODEL_FEATURES",
    "CATEGORICAL_FEATURES",
    "category_levels",
    "X_test_encoded",
    "calibrated_prediction_functions",
    "final_predictions",
    "RESTORED_ROOT"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required variables are missing:\n"
        + "\n".join(missing_variables)
        + "\nRerun Checkpoints 8B2A and 8B2B."
    )


assert len(X_test_encoded) == 2000
assert list(X_test_encoded.columns) == MODEL_FEATURES

test_matrix = (
    X_test_encoded
    .to_numpy(dtype=float)
)


# ------------------------------------------------------------
# 3. Load locked thresholds
# ------------------------------------------------------------

def find_unique_file(
    root,
    filename,
    preferred_parent=None
):
    matches = list(
        root.rglob(filename)
    )

    if preferred_parent is not None:

        preferred_matches = [
            path
            for path in matches
            if preferred_parent in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}."
        )

    raise RuntimeError(
        f"Multiple copies of {filename} found:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


LOCKED_MANIFEST_PATH = find_unique_file(
    RESTORED_ROOT,
    "locked_calibration_threshold_manifest.json",
    preferred_parent=
        "calibration_threshold_locked"
)

with open(
    LOCKED_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:
    locked_manifest = json.load(file)


locked_thresholds = {
    model_name: float(
        locked_manifest[
            model_name
        ]["locked_mcc_threshold"]
    )
    for model_name in REPRESENTATIVE_MODELS
}


# ------------------------------------------------------------
# 4. Category mappings
# ------------------------------------------------------------

category_code_maps = {
    feature: {
        label: code
        for code, label in enumerate(
            category_levels[feature]
        )
    }
    for feature in CATEGORICAL_FEATURES
}

assert {
    "Female",
    "Male"
}.issubset(
    category_code_maps["Gender"]
)

assert {
    "France",
    "Germany",
    "Spain"
}.issubset(
    category_code_maps["Geography"]
)


gender_position = MODEL_FEATURES.index(
    "Gender"
)

geography_position = MODEL_FEATURES.index(
    "Geography"
)

female_code = category_code_maps[
    "Gender"
]["Female"]

male_code = category_code_maps[
    "Gender"
]["Male"]

geography_labels = [
    "France",
    "Germany",
    "Spain"
]


print("=" * 105)
print("COUNTERFACTUAL AUDIT CONFIGURATION")
print("=" * 105)

print("Rows:", len(test_matrix))
print("Models:", REPRESENTATIVE_MODELS)
print("Gender codes:", category_code_maps["Gender"])
print(
    "Geography codes:",
    category_code_maps["Geography"]
)
print("Bootstrap samples:", N_BOOTSTRAPS)


# ------------------------------------------------------------
# 5. Construct counterfactual input matrices
# ------------------------------------------------------------

female_matrix = test_matrix.copy()
female_matrix[:, gender_position] = (
    female_code
)

male_matrix = test_matrix.copy()
male_matrix[:, gender_position] = (
    male_code
)


geography_matrices = {}

for geography_label in geography_labels:

    matrix = test_matrix.copy()

    matrix[
        :,
        geography_position
    ] = category_code_maps[
        "Geography"
    ][geography_label]

    geography_matrices[
        geography_label
    ] = matrix


# ------------------------------------------------------------
# 6. Generate counterfactual predictions
# ------------------------------------------------------------

counterfactual_probabilities = {}

prediction_start = time.perf_counter()

for model_name in REPRESENTATIVE_MODELS:

    prediction_function = (
        calibrated_prediction_functions[
            model_name
        ]
    )

    counterfactual_probabilities[
        model_name
    ] = {
        "Female":
            prediction_function(
                female_matrix
            ),

        "Male":
            prediction_function(
                male_matrix
            )
    }

    for geography_label in geography_labels:

        counterfactual_probabilities[
            model_name
        ][geography_label] = (
            prediction_function(
                geography_matrices[
                    geography_label
                ]
            )
        )


prediction_elapsed = (
    time.perf_counter()
    - prediction_start
)


# ------------------------------------------------------------
# 7. Gender sensitivity
#
# Difference is P(Female) minus P(Male), while every
# other feature is held constant.
# ------------------------------------------------------------

gender_point_records = []
gender_row_records = []

gender_difference_map = {}

for model_name in REPRESENTATIVE_MODELS:

    female_probability = (
        counterfactual_probabilities[
            model_name
        ]["Female"]
    )

    male_probability = (
        counterfactual_probabilities[
            model_name
        ]["Male"]
    )

    difference = (
        female_probability
        - male_probability
    )

    gender_difference_map[
        model_name
    ] = difference

    threshold = locked_thresholds[
        model_name
    ]

    female_prediction = (
        female_probability
        >= threshold
    ).astype(int)

    male_prediction = (
        male_probability
        >= threshold
    ).astype(int)

    gender_point_records.append({
        "Model": model_name,

        "Mean_Female_minus_Male":
            float(
                difference.mean()
            ),

        "Median_Female_minus_Male":
            float(
                np.median(
                    difference
                )
            ),

        "Mean_Absolute_Difference":
            float(
                np.abs(
                    difference
                ).mean()
            ),

        "P95_Absolute_Difference":
            float(
                np.quantile(
                    np.abs(difference),
                    0.95
                )
            ),

        "Maximum_Absolute_Difference":
            float(
                np.abs(
                    difference
                ).max()
            ),

        "Female_Probability_Higher_Rate":
            float(
                np.mean(
                    difference > 0
                )
            ),

        "Gender_Assignment_Decision_Flip_Rate":
            float(
                np.mean(
                    female_prediction
                    != male_prediction
                )
            ),

        "Female_Predicted_Positive_Rate":
            float(
                female_prediction.mean()
            ),

        "Male_Predicted_Positive_Rate":
            float(
                male_prediction.mean()
            )
    })

    for row_position in range(
        len(test_matrix)
    ):

        gender_row_records.append({
            "Test_Position":
                row_position,

            "Original_Index":
                int(
                    final_predictions.loc[
                        row_position,
                        "Original_Index"
                    ]
                ),

            "CustomerId":
                final_predictions.loc[
                    row_position,
                    "CustomerId"
                ],

            "Model":
                model_name,

            "Probability_If_Female":
                float(
                    female_probability[
                        row_position
                    ]
                ),

            "Probability_If_Male":
                float(
                    male_probability[
                        row_position
                    ]
                ),

            "Female_minus_Male":
                float(
                    difference[
                        row_position
                    ]
                ),

            "Decision_If_Female":
                int(
                    female_prediction[
                        row_position
                    ]
                ),

            "Decision_If_Male":
                int(
                    male_prediction[
                        row_position
                    ]
                )
        })


gender_point_summary = pd.DataFrame(
    gender_point_records
)

gender_row_results = pd.DataFrame(
    gender_row_records
)


# ------------------------------------------------------------
# 8. Geography sensitivity
# ------------------------------------------------------------

geography_point_records = []
geography_pair_records = []
geography_row_records = []

geography_range_map = {}
geography_pair_difference_map = {}


for model_name in REPRESENTATIVE_MODELS:

    probability_matrix = np.column_stack([
        counterfactual_probabilities[
            model_name
        ][label]
        for label in geography_labels
    ])

    threshold = locked_thresholds[
        model_name
    ]

    decision_matrix = (
        probability_matrix
        >= threshold
    ).astype(int)

    row_probability_range = (
        probability_matrix.max(axis=1)
        - probability_matrix.min(axis=1)
    )

    any_decision_change = (
        decision_matrix.max(axis=1)
        != decision_matrix.min(axis=1)
    )

    geography_range_map[
        model_name
    ] = row_probability_range

    geography_point_records.append({
        "Model": model_name,

        "Mean_Within_Row_Probability_Range":
            float(
                row_probability_range.mean()
            ),

        "Median_Within_Row_Probability_Range":
            float(
                np.median(
                    row_probability_range
                )
            ),

        "P95_Within_Row_Probability_Range":
            float(
                np.quantile(
                    row_probability_range,
                    0.95
                )
            ),

        "Maximum_Within_Row_Probability_Range":
            float(
                row_probability_range.max()
            ),

        "Any_Geography_Assignment_Decision_Flip_Rate":
            float(
                any_decision_change.mean()
            )
    })

    for label_a, label_b in combinations(
        geography_labels,
        2
    ):

        probability_a = (
            counterfactual_probabilities[
                model_name
            ][label_a]
        )

        probability_b = (
            counterfactual_probabilities[
                model_name
            ][label_b]
        )

        difference = (
            probability_a
            - probability_b
        )

        contrast_name = (
            f"{label_a}_minus_{label_b}"
        )

        geography_pair_difference_map[
            (
                model_name,
                contrast_name
            )
        ] = difference

        decision_a = (
            probability_a
            >= threshold
        ).astype(int)

        decision_b = (
            probability_b
            >= threshold
        ).astype(int)

        geography_pair_records.append({
            "Model": model_name,

            "Contrast":
                contrast_name,

            "Mean_Probability_Difference":
                float(
                    difference.mean()
                ),

            "Median_Probability_Difference":
                float(
                    np.median(
                        difference
                    )
                ),

            "Mean_Absolute_Difference":
                float(
                    np.abs(
                        difference
                    ).mean()
                ),

            "P95_Absolute_Difference":
                float(
                    np.quantile(
                        np.abs(difference),
                        0.95
                    )
                ),

            "First_Country_Higher_Rate":
                float(
                    np.mean(
                        difference > 0
                    )
                ),

            "Pairwise_Decision_Flip_Rate":
                float(
                    np.mean(
                        decision_a
                        != decision_b
                    )
                )
        })

    for row_position in range(
        len(test_matrix)
    ):

        geography_row_records.append({
            "Test_Position":
                row_position,

            "Original_Index":
                int(
                    final_predictions.loc[
                        row_position,
                        "Original_Index"
                    ]
                ),

            "CustomerId":
                final_predictions.loc[
                    row_position,
                    "CustomerId"
                ],

            "Model":
                model_name,

            "Probability_If_France":
                float(
                    probability_matrix[
                        row_position,
                        0
                    ]
                ),

            "Probability_If_Germany":
                float(
                    probability_matrix[
                        row_position,
                        1
                    ]
                ),

            "Probability_If_Spain":
                float(
                    probability_matrix[
                        row_position,
                        2
                    ]
                ),

            "Within_Row_Probability_Range":
                float(
                    row_probability_range[
                        row_position
                    ]
                ),

            "Any_Decision_Change":
                bool(
                    any_decision_change[
                        row_position
                    ]
                )
        })


geography_point_summary = pd.DataFrame(
    geography_point_records
)

geography_pair_summary = pd.DataFrame(
    geography_pair_records
)

geography_row_results = pd.DataFrame(
    geography_row_records
)


# ------------------------------------------------------------
# 9. Bootstrap confidence intervals
# ------------------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

n_rows = len(test_matrix)

bootstrap_indices = rng.integers(
    low=0,
    high=n_rows,
    size=(
        N_BOOTSTRAPS,
        n_rows
    )
)


gender_ci_records = []

for model_name in REPRESENTATIVE_MODELS:

    difference = gender_difference_map[
        model_name
    ]

    bootstrap_means = (
        difference[
            bootstrap_indices
        ].mean(axis=1)
    )

    point_row = (
        gender_point_summary[
            gender_point_summary[
                "Model"
            ] == model_name
        ]
        .iloc[0]
    )

    gender_ci_records.append({
        "Model": model_name,

        "Mean_Female_minus_Male":
            float(
                point_row[
                    "Mean_Female_minus_Male"
                ]
            ),

        "CI_Lower":
            float(
                np.quantile(
                    bootstrap_means,
                    0.025
                )
            ),

        "CI_Upper":
            float(
                np.quantile(
                    bootstrap_means,
                    0.975
                )
            ),

        "Bootstrap_Replicates":
            N_BOOTSTRAPS
    })


gender_confidence_intervals = pd.DataFrame(
    gender_ci_records
)


geography_range_ci_records = []

for model_name in REPRESENTATIVE_MODELS:

    row_ranges = geography_range_map[
        model_name
    ]

    bootstrap_means = (
        row_ranges[
            bootstrap_indices
        ].mean(axis=1)
    )

    point_row = (
        geography_point_summary[
            geography_point_summary[
                "Model"
            ] == model_name
        ]
        .iloc[0]
    )

    geography_range_ci_records.append({
        "Model": model_name,

        "Mean_Within_Row_Probability_Range":
            float(
                point_row[
                    "Mean_Within_Row_Probability_Range"
                ]
            ),

        "CI_Lower":
            float(
                np.quantile(
                    bootstrap_means,
                    0.025
                )
            ),

        "CI_Upper":
            float(
                np.quantile(
                    bootstrap_means,
                    0.975
                )
            ),

        "Bootstrap_Replicates":
            N_BOOTSTRAPS
    })


geography_range_confidence_intervals = (
    pd.DataFrame(
        geography_range_ci_records
    )
)


geography_pair_ci_records = []

for (
    model_name,
    contrast_name
), difference in (
    geography_pair_difference_map.items()
):

    bootstrap_means = (
        difference[
            bootstrap_indices
        ].mean(axis=1)
    )

    point_row = (
        geography_pair_summary[
            (
                geography_pair_summary[
                    "Model"
                ] == model_name
            )
            &
            (
                geography_pair_summary[
                    "Contrast"
                ] == contrast_name
            )
        ]
        .iloc[0]
    )

    geography_pair_ci_records.append({
        "Model": model_name,

        "Contrast":
            contrast_name,

        "Mean_Probability_Difference":
            float(
                point_row[
                    "Mean_Probability_Difference"
                ]
            ),

        "CI_Lower":
            float(
                np.quantile(
                    bootstrap_means,
                    0.025
                )
            ),

        "CI_Upper":
            float(
                np.quantile(
                    bootstrap_means,
                    0.975
                )
            ),

        "Bootstrap_Replicates":
            N_BOOTSTRAPS
    })


geography_pair_confidence_intervals = (
    pd.DataFrame(
        geography_pair_ci_records
    )
)


# ------------------------------------------------------------
# 10. Merge CI results for display
# ------------------------------------------------------------

gender_display = (
    gender_point_summary
    .merge(
        gender_confidence_intervals[
            [
                "Model",
                "CI_Lower",
                "CI_Upper"
            ]
        ],
        on="Model",
        how="left"
    )
)


geography_display = (
    geography_point_summary
    .merge(
        geography_range_confidence_intervals[
            [
                "Model",
                "CI_Lower",
                "CI_Upper"
            ]
        ],
        on="Model",
        how="left"
    )
)


geography_pair_display = (
    geography_pair_summary
    .merge(
        geography_pair_confidence_intervals[
            [
                "Model",
                "Contrast",
                "CI_Lower",
                "CI_Upper"
            ]
        ],
        on=[
            "Model",
            "Contrast"
        ],
        how="left"
    )
)


# ------------------------------------------------------------
# 11. Print results
# ------------------------------------------------------------

print("\n" + "=" * 155)
print("GENDER COUNTERFACTUAL SENSITIVITY")
print(
    "Difference = predicted probability if Female "
    "minus predicted probability if Male, holding "
    "all other features fixed."
)
print("=" * 155)

print(
    gender_display[
        [
            "Model",
            "Mean_Female_minus_Male",
            "CI_Lower",
            "CI_Upper",
            "Median_Female_minus_Male",
            "Mean_Absolute_Difference",
            "P95_Absolute_Difference",
            "Female_Probability_Higher_Rate",
            "Gender_Assignment_Decision_Flip_Rate",
            "Female_Predicted_Positive_Rate",
            "Male_Predicted_Positive_Rate"
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 155)
print("GEOGRAPHY COUNTERFACTUAL SENSITIVITY")
print(
    "Within-row range across France, Germany, and Spain "
    "while all other features are fixed."
)
print("=" * 155)

print(
    geography_display[
        [
            "Model",
            "Mean_Within_Row_Probability_Range",
            "CI_Lower",
            "CI_Upper",
            "Median_Within_Row_Probability_Range",
            "P95_Within_Row_Probability_Range",
            "Maximum_Within_Row_Probability_Range",
            "Any_Geography_Assignment_Decision_Flip_Rate"
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 165)
print("PAIRWISE GEOGRAPHY COUNTERFACTUAL CONTRASTS")
print(
    "Positive values indicate that the first country "
    "produced the higher predicted churn probability."
)
print("=" * 165)

print(
    geography_pair_display[
        [
            "Model",
            "Contrast",
            "Mean_Probability_Difference",
            "CI_Lower",
            "CI_Upper",
            "Median_Probability_Difference",
            "Mean_Absolute_Difference",
            "P95_Absolute_Difference",
            "First_Country_Higher_Rate",
            "Pairwise_Decision_Flip_Rate"
        ]
    ]
    .sort_values(
        [
            "Model",
            "Contrast"
        ]
    )
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 12. Save artifacts
# ------------------------------------------------------------

gender_point_summary.to_csv(
    OUTPUT_DIR
    / "gender_counterfactual_point_summary.csv",
    index=False
)

gender_confidence_intervals.to_csv(
    OUTPUT_DIR
    / "gender_counterfactual_mean_difference_ci.csv",
    index=False
)

gender_row_results.to_csv(
    OUTPUT_DIR
    / "gender_counterfactual_row_predictions.csv",
    index=False
)

geography_point_summary.to_csv(
    OUTPUT_DIR
    / "geography_counterfactual_point_summary.csv",
    index=False
)

geography_range_confidence_intervals.to_csv(
    OUTPUT_DIR
    / "geography_counterfactual_range_ci.csv",
    index=False
)

geography_pair_summary.to_csv(
    OUTPUT_DIR
    / "geography_counterfactual_pairwise_summary.csv",
    index=False
)

geography_pair_confidence_intervals.to_csv(
    OUTPUT_DIR
    / "geography_counterfactual_pairwise_ci.csv",
    index=False
)

geography_row_results.to_csv(
    OUTPUT_DIR
    / "geography_counterfactual_row_predictions.csv",
    index=False
)


run_manifest = {
    "models":
        REPRESENTATIVE_MODELS,

    "rows":
        int(len(test_matrix)),

    "gender_contrast":
        "Female minus Male",

    "geography_assignments":
        geography_labels,

    "bootstrap_samples":
        N_BOOTSTRAPS,

    "bootstrap_seed":
        BOOTSTRAP_SEED,

    "all_non_audited_features_held_fixed":
        True,

    "interpretation_scope":
        (
            "Direct model sensitivity diagnostic; "
            "not a causal or legal counterfactual-"
            "fairness determination"
        ),

    "post_test_tuning_performed":
        False
}


with open(
    OUTPUT_DIR
    / "counterfactual_sensitivity_run_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        run_manifest,
        file,
        indent=4
    )


print("\n" + "=" * 105)
print("CHECKPOINT 8D2 SAVED")
print("=" * 105)

print("Output directory:")
print(OUTPUT_DIR)

print(
    f"\nPrediction elapsed time: "
    f"{prediction_elapsed:.2f} seconds"
)

print(
    "\nNo model, calibrator, threshold, "
    "or locked-test metric was modified."
)

In [ ]:
# ============================================================
# COMPUTATIONAL-EFFICIENCY AND MODEL-COMPLEXITY AUDIT
#
# Uses saved training, inference, performance, model-artifact,
# and SHAP timing records.
#
# No retraining, tuning, threshold changes, or test changes.
# ============================================================

from pathlib import Path
import json
import os
import platform

import joblib
import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/kaggle/working/computational_efficiency_audit"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESTORED_ROOT_CANDIDATES = [Path("/kaggle/working")]

MODEL_ORDER = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

REFERENCE_MODEL = "XGBoost"
TEST_ROWS = 2000
SHAP_ROWS = 300


# ------------------------------------------------------------
# 2. Locate current pipeline artifacts
# ------------------------------------------------------------

existing_roots = [
    path
    for path in RESTORED_ROOT_CANDIDATES
    if path.exists()
]

if not existing_roots:
    raise FileNotFoundError(
        "Working artifact directory was not found."
    )

RESTORED_ROOT = existing_roots[0]


def find_unique_file(
    roots,
    filename,
    preferred_parent=None,
    required=False
):
    if isinstance(roots, Path):
        roots = [roots]

    matches = []

    for root in roots:

        if not root.exists():
            continue

        matches.extend(
            root.rglob(filename)
        )

    matches = list(
        dict.fromkeys(matches)
    )

    if preferred_parent is not None:

        preferred_matches = [
            path
            for path in matches
            if preferred_parent in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:

        if required:
            raise FileNotFoundError(
                f"Could not find {filename}."
            )

        return None

    raise RuntimeError(
        f"Multiple copies of {filename} found:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


FINAL_METRICS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_locked_test_metrics.csv",
    preferred_parent="final_locked_holdout",
    required=True
)

TRAINING_SUMMARY_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_model_training_summary.csv",
    preferred_parent="final_locked_holdout",
    required=True
)

ARTIFACT_STATUS_PATH = find_unique_file(
    RESTORED_ROOT,
    "final_model_artifact_status.csv",
    preferred_parent="final_locked_holdout",
    required=True
)

SHAP_SUMMARY_PATH = find_unique_file(
    [
        Path("/kaggle/working"),
        RESTORED_ROOT
    ],
    "permutation_shap_run_summary.csv",
    preferred_parent="explainability_shap",
    required=True
)


print("=" * 105)
print("EFFICIENCY ARTIFACT LOCATIONS")
print("=" * 105)

print("Artifact root:", RESTORED_ROOT)
print("Final metrics:", FINAL_METRICS_PATH)
print("Training summary:", TRAINING_SUMMARY_PATH)
print("Artifact status:", ARTIFACT_STATUS_PATH)
print("SHAP summary:", SHAP_SUMMARY_PATH)


# ------------------------------------------------------------
# 3. Load saved records
# ------------------------------------------------------------

final_metrics = pd.read_csv(
    FINAL_METRICS_PATH,
    keep_default_na=False
)

training_summary = pd.read_csv(
    TRAINING_SUMMARY_PATH
)

artifact_status = pd.read_csv(
    ARTIFACT_STATUS_PATH,
    keep_default_na=False
)

shap_summary = pd.read_csv(
    SHAP_SUMMARY_PATH
)


assert set(
    final_metrics["Model"]
) == set(MODEL_ORDER)

assert set(
    training_summary["Model"]
) == set(MODEL_ORDER)


# ------------------------------------------------------------
# 4. Core time and throughput table
# ------------------------------------------------------------

efficiency_table = (
    final_metrics[
        [
            "Model",
            "ROC_AUC",
            "PR_AUC",
            "Brier_Score",
            "MCC"
        ]
    ]
    .merge(
        training_summary,
        on="Model",
        how="left",
        validate="one_to_one"
    )
)


efficiency_table[
    "Inference_Throughput_Rows_Per_Second"
] = (
    TEST_ROWS
    / efficiency_table[
        "Prediction_Time_Seconds"
    ]
)


efficiency_table[
    "Inference_Latency_Milliseconds_Per_Row"
] = (
    efficiency_table[
        "Prediction_Time_Seconds"
    ]
    * 1000.0
    / TEST_ROWS
)


reference_row = (
    efficiency_table[
        efficiency_table["Model"]
        == REFERENCE_MODEL
    ]
    .iloc[0]
)


efficiency_table[
    "Relative_Final_Fit_Time_vs_XGBoost"
] = (
    efficiency_table[
        "Fit_Time_Seconds"
    ]
    / float(
        reference_row[
            "Fit_Time_Seconds"
        ]
    )
)


efficiency_table[
    "Relative_Inference_Time_vs_XGBoost"
] = (
    efficiency_table[
        "Prediction_Time_Seconds"
    ]
    / float(
        reference_row[
            "Prediction_Time_Seconds"
        ]
    )
)


best_roc_auc = float(
    efficiency_table[
        "ROC_AUC"
    ].max()
)

best_pr_auc = float(
    efficiency_table[
        "PR_AUC"
    ].max()
)


efficiency_table[
    "ROC_AUC_Gap_From_Best"
] = (
    best_roc_auc
    - efficiency_table[
        "ROC_AUC"
    ]
)


efficiency_table[
    "PR_AUC_Gap_From_Best"
] = (
    best_pr_auc
    - efficiency_table[
        "PR_AUC"
    ]
)


# ------------------------------------------------------------
# 5. Locate model artifacts
# ------------------------------------------------------------

artifact_filenames = {
    "LogisticRegression":
        "LogisticRegression.joblib",

    "RandomForest":
        "RandomForest.joblib",

    "XGBoost":
        "XGBoost.joblib",

    "LightGBM":
        "LightGBM.joblib",

    "CatBoost":
        "CatBoost.joblib",

    "MLP":
        "MLP_state.pt",

    "TabNet":
        "TabNet_model.zip",

    "FTTransformer":
        "FTTransformer.joblib",

    "RealTabR":
        "RealTabR.joblib"
}


artifact_paths = {}

for model_name, filename in (
    artifact_filenames.items()
):

    artifact_paths[model_name] = (
        find_unique_file(
            RESTORED_ROOT,
            filename,
            preferred_parent=
                "model_artifacts",
            required=False
        )
    )


# ------------------------------------------------------------
# 6. Complexity utilities
# ------------------------------------------------------------

def count_xgboost_tree_structure(
    tree_node
):
    if "leaf" in tree_node:
        return {
            "nodes": 1,
            "leaves": 1
        }

    node_count = 1
    leaf_count = 0

    for child in tree_node.get(
        "children",
        []
    ):

        child_counts = (
            count_xgboost_tree_structure(
                child
            )
        )

        node_count += child_counts[
            "nodes"
        ]

        leaf_count += child_counts[
            "leaves"
        ]

    return {
        "nodes": node_count,
        "leaves": leaf_count
    }


complexity_records = []


for _, row in training_summary.iterrows():

    model_name = row["Model"]

    artifact_path = artifact_paths[
        model_name
    ]

    artifact_exists = (
        artifact_path is not None
        and artifact_path.exists()
    )

    artifact_size_bytes = (
        artifact_path.stat().st_size
        if artifact_exists
        else np.nan
    )

    recorded_parameter_count = (
        row["Parameter_Count"]
        if pd.notna(
            row["Parameter_Count"]
        )
        else np.nan
    )

    complexity_type = "Unavailable"
    primary_complexity_count = np.nan
    secondary_complexity_count = np.nan
    complexity_note = ""

    try:

        # ----------------------------------------------------
        # Logistic regression
        # ----------------------------------------------------

        if (
            model_name
            == "LogisticRegression"
            and artifact_exists
        ):

            bundle = joblib.load(
                artifact_path
            )

            model = bundle["model"]

            coefficient_count = int(
                model.coef_.size
                + model.intercept_.size
            )

            complexity_type = (
                "Trainable coefficients"
            )

            primary_complexity_count = (
                coefficient_count
            )

            complexity_note = (
                "Coefficient and intercept count."
            )


        # ----------------------------------------------------
        # Random forest
        # ----------------------------------------------------

        elif (
            model_name
            == "RandomForest"
            and artifact_exists
        ):

            bundle = joblib.load(
                artifact_path
            )

            model = bundle["model"]

            total_nodes = int(
                sum(
                    estimator.tree_.node_count
                    for estimator
                    in model.estimators_
                )
            )

            total_leaves = int(
                sum(
                    estimator.tree_.n_leaves
                    for estimator
                    in model.estimators_
                )
            )

            complexity_type = (
                "Trees / total nodes"
            )

            primary_complexity_count = int(
                len(model.estimators_)
            )

            secondary_complexity_count = (
                total_nodes
            )

            complexity_note = (
                f"Total leaves={total_leaves}."
            )


        # ----------------------------------------------------
        # XGBoost
        # ----------------------------------------------------

        elif (
            model_name
            == "XGBoost"
            and artifact_exists
        ):

            bundle = joblib.load(
                artifact_path
            )

            booster = bundle[
                "model"
            ].get_booster()

            tree_dumps = booster.get_dump(
                dump_format="json"
            )

            total_nodes = 0
            total_leaves = 0

            for tree_text in tree_dumps:

                tree_object = json.loads(
                    tree_text
                )

                counts = (
                    count_xgboost_tree_structure(
                        tree_object
                    )
                )

                total_nodes += counts[
                    "nodes"
                ]

                total_leaves += counts[
                    "leaves"
                ]

            complexity_type = (
                "Boosted trees / total nodes"
            )

            primary_complexity_count = (
                len(tree_dumps)
            )

            secondary_complexity_count = (
                total_nodes
            )

            complexity_note = (
                f"Total leaves={total_leaves}."
            )


        # ----------------------------------------------------
        # LightGBM
        # ----------------------------------------------------

        elif (
            model_name
            == "LightGBM"
            and artifact_exists
        ):

            bundle = joblib.load(
                artifact_path
            )

            model = bundle["model"]

            model_dump = (
                model.booster_.dump_model()
            )

            tree_information = (
                model_dump["tree_info"]
            )

            total_leaves = int(
                sum(
                    tree["num_leaves"]
                    for tree
                    in tree_information
                )
            )

            complexity_type = (
                "Boosted trees / total leaves"
            )

            primary_complexity_count = (
                len(tree_information)
            )

            secondary_complexity_count = (
                total_leaves
            )

            complexity_note = (
                "Leaf count from LightGBM model dump."
            )


        # ----------------------------------------------------
        # CatBoost
        # ----------------------------------------------------

        elif (
            model_name
            == "CatBoost"
            and artifact_exists
        ):

            model = joblib.load(
                artifact_path
            )

            tree_count = int(
                model.tree_count_
            )

            leaf_value_count = int(
                len(
                    model.get_leaf_values()
                )
            )

            complexity_type = (
                "Boosted trees / leaf values"
            )

            primary_complexity_count = (
                tree_count
            )

            secondary_complexity_count = (
                leaf_value_count
            )

            complexity_note = (
                "Number of stored leaf values."
            )


        # ----------------------------------------------------
        # MLP
        # ----------------------------------------------------

        elif (
            model_name
            == "MLP"
            and artifact_exists
        ):

            try:

                state_bundle = torch.load(
                    artifact_path,
                    map_location="cpu",
                    weights_only=False
                )

            except TypeError:

                state_bundle = torch.load(
                    artifact_path,
                    map_location="cpu"
                )

            state_dict = state_bundle[
                "state_dict"
            ]

            trainable_parameter_count = int(
                sum(
                    tensor.numel()
                    for tensor
                    in state_dict.values()
                )
            )

            complexity_type = (
                "Trainable parameters"
            )

            primary_complexity_count = (
                trainable_parameter_count
            )

            complexity_note = (
                "Parameter count reconstructed "
                "from the saved state dictionary."
            )


        # ----------------------------------------------------
        # TabNet
        # ----------------------------------------------------

        elif model_name == "TabNet":

            complexity_type = (
                "Trainable parameters"
            )

            primary_complexity_count = (
                recorded_parameter_count
            )

            complexity_note = (
                "Recorded during final model training."
            )


        # ----------------------------------------------------
        # FT-Transformer
        # ----------------------------------------------------

        elif model_name == "FTTransformer":

            complexity_type = (
                "Not exposed in saved summary"
            )

            complexity_note = (
                "Artifact size and timing are available; "
                "parameter count was not exported."
            )


        # ----------------------------------------------------
        # RealTabR
        # ----------------------------------------------------

        elif model_name == "RealTabR":

            complexity_type = (
                "Artifact unavailable"
            )

            complexity_note = (
                "Final predictions and timing were saved, "
                "but model serialization failed because "
                "the implementation contained a local lambda."
            )


    except Exception as error:

        complexity_note = (
            f"Complexity extraction failed: "
            f"{repr(error)}"
        )


    complexity_records.append({
        "Model":
            model_name,

        "Artifact_Path":
            (
                str(artifact_path)
                if artifact_exists
                else None
            ),

        "Artifact_Exists":
            artifact_exists,

        "Artifact_Size_Bytes":
            artifact_size_bytes,

        "Artifact_Size_MB":
            (
                artifact_size_bytes
                / (1024 ** 2)
                if artifact_exists
                else np.nan
            ),

        "Complexity_Type":
            complexity_type,

        "Primary_Complexity_Count":
            primary_complexity_count,

        "Secondary_Complexity_Count":
            secondary_complexity_count,

        "Recorded_Parameter_Count":
            recorded_parameter_count,

        "Complexity_Note":
            complexity_note
    })


complexity_table = pd.DataFrame(
    complexity_records
)


# ------------------------------------------------------------
# 7. SHAP computational cost
# ------------------------------------------------------------

shap_efficiency = (
    shap_summary[
        [
            "Model",
            "SHAP_Rows",
            "Features",
            "Permutation_Cycles",
            "Max_Evals",
            "Elapsed_Seconds",
            "Maximum_Additivity_Residual"
        ]
    ]
    .copy()
)


shap_efficiency[
    "Seconds_Per_Explained_Row"
] = (
    shap_efficiency[
        "Elapsed_Seconds"
    ]
    / shap_efficiency[
        "SHAP_Rows"
    ]
)


shap_efficiency[
    "Explained_Rows_Per_Second"
] = (
    shap_efficiency[
        "SHAP_Rows"
    ]
    / shap_efficiency[
        "Elapsed_Seconds"
    ]
)


xgb_shap_seconds = float(
    shap_efficiency.loc[
        shap_efficiency["Model"]
        == "XGBoost",
        "Elapsed_Seconds"
    ].iloc[0]
)


shap_efficiency[
    "Relative_SHAP_Time_vs_XGBoost"
] = (
    shap_efficiency[
        "Elapsed_Seconds"
    ]
    / xgb_shap_seconds
)


# ------------------------------------------------------------
# 8. Merge operational and complexity tables
# ------------------------------------------------------------

complete_efficiency = (
    efficiency_table
    .merge(
        complexity_table,
        on="Model",
        how="left",
        validate="one_to_one"
    )
)


model_order_map = {
    model_name: position
    for position, model_name
    in enumerate(MODEL_ORDER)
}


complete_efficiency[
    "_Model_Order"
] = complete_efficiency[
    "Model"
].map(model_order_map)


complete_efficiency = (
    complete_efficiency
    .sort_values(
        "_Model_Order"
    )
    .drop(
        columns="_Model_Order"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Current execution environment
# ------------------------------------------------------------

environment_record = {
    "python_version":
        platform.python_version(),

    "platform":
        platform.platform(),

    "processor":
        platform.processor(),

    "logical_cpu_count":
        os.cpu_count(),

    "torch_version":
        torch.__version__,

    "cuda_available":
        bool(
            torch.cuda.is_available()
        ),

    "timing_source":
        (
            "Immutable final-training and prediction "
            "records from the locked model run"
        ),

    "test_rows_for_prediction_time":
        TEST_ROWS,

    "memory_measurement_source":
        (
            "Direct 2-ms process-tree RSS sampling during "
            "the original final fit and inference operations"
        ),

    "post_test_tuning_performed":
        False
}


# ------------------------------------------------------------
# 10. Print publication-oriented tables
# ------------------------------------------------------------

print("\n" + "=" * 190)
print("FINAL MODEL PERFORMANCE–EFFICIENCY TRADE-OFF")
print(
    "Prediction time refers to one saved inference pass "
    f"over {TEST_ROWS:,} locked-test observations."
)
print("=" * 190)

display_columns = [
    "Model",
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "Training_Rows",
    "Early_Stopping_Rows",
    "Completed_Epochs",
    "Fit_Time_Seconds",
    "Prediction_Time_Seconds",
    "Inference_Throughput_Rows_Per_Second",
    "Inference_Latency_Milliseconds_Per_Row",
    "Relative_Final_Fit_Time_vs_XGBoost",
    "Relative_Inference_Time_vs_XGBoost",
    "ROC_AUC_Gap_From_Best",
    "PR_AUC_Gap_From_Best"
]

memory_display_columns = [
    "Training_Baseline_RSS_MB",
    "Training_Peak_RSS_MB",
    "Training_Incremental_Peak_RSS_MB",
    "Inference_Baseline_RSS_MB",
    "Inference_Peak_RSS_MB",
    "Inference_Incremental_Peak_RSS_MB"
]

display_columns += [
    column
    for column in memory_display_columns
    if column in complete_efficiency.columns
]

print(
    complete_efficiency[
        display_columns
    ]
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 165)
print("MODEL ARTIFACT SIZE AND COMPLEXITY")
print("=" * 165)

print(
    complexity_table[
        [
            "Model",
            "Artifact_Exists",
            "Artifact_Size_MB",
            "Complexity_Type",
            "Primary_Complexity_Count",
            "Secondary_Complexity_Count",
            "Recorded_Parameter_Count",
            "Complexity_Note"
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 145)
print("PERMUTATION SHAP COMPUTATIONAL COST")
print(
    f"Common protocol: {SHAP_ROWS} rows, "
    "10 features, 5 permutation cycles."
)
print("=" * 145)

print(
    shap_efficiency
    .sort_values(
        "Elapsed_Seconds"
    )
    .round(6)
    .to_string(index=False)
)


print("\n" + "=" * 105)
print("EFFICIENCY ENVIRONMENT RECORD")
print("=" * 105)

print(
    json.dumps(
        environment_record,
        indent=4
    )
)


# ------------------------------------------------------------
# 11. Save artifacts
# ------------------------------------------------------------

efficiency_table.to_csv(
    OUTPUT_DIR
    / "final_model_performance_efficiency.csv",
    index=False
)

complexity_table.to_csv(
    OUTPUT_DIR
    / "model_artifact_size_and_complexity.csv",
    index=False
)

complete_efficiency.to_csv(
    OUTPUT_DIR
    / "complete_model_efficiency_audit.csv",
    index=False
)

shap_efficiency.to_csv(
    OUTPUT_DIR
    / "permutation_shap_computational_cost.csv",
    index=False
)

artifact_status.to_csv(
    OUTPUT_DIR
    / "restored_model_artifact_status.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "efficiency_environment_record.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        environment_record,
        file,
        indent=4
    )


run_manifest = {
    "models":
        MODEL_ORDER,

    "reference_model":
        REFERENCE_MODEL,

    "performance_source":
        str(FINAL_METRICS_PATH),

    "training_timing_source":
        str(TRAINING_SUMMARY_PATH),

    "artifact_status_source":
        str(ARTIFACT_STATUS_PATH),

    "shap_timing_source":
        str(SHAP_SUMMARY_PATH),

    "test_rows":
        TEST_ROWS,

    "interpretation_scope":
        (
            "Descriptive computational-efficiency audit. "
            "Final deep and classical models used different "
            "early-stopping protocols and effective gradient-"
            "training sample counts."
        ),

    "memory_measurement_source":
        (
            "Direct 2-ms process-tree RSS sampling during "
            "the original final fit and inference operations"
        ),

    "post_test_tuning_performed":
        False
}


with open(
    OUTPUT_DIR
    / "computational_efficiency_run_manifest.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        run_manifest,
        file,
        indent=4
    )


print("\n" + "=" * 105)
print("CHECKPOINT 8E1 SAVED")
print("=" * 105)

print("Output directory:")
print(OUTPUT_DIR)

print(
    "\nNo model, calibrator, threshold, "
    "or locked-test result was modified."
)

In [ ]:
# ============================================================
# FINAL PUBLICATION OUTPUTS
# FIGURES + TABLES
# Generated from the current end-to-end pipeline
#
# No model training, tuning, calibration, or threshold changes.
# ============================================================

from pathlib import Path
import hashlib
import json
import math
import shutil
import textwrap
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve
)

from sklearn.calibration import (
    calibration_curve
)


# ------------------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------------------

RESTORED_ROOT_CANDIDATES = [Path("/kaggle/working")]

existing_roots = [
    path
    for path in RESTORED_ROOT_CANDIDATES
    if path.exists()
]

if not existing_roots:
    raise FileNotFoundError(
        "Working artifact directory was not found."
    )

RESTORED_ROOT = existing_roots[0]

WORKING_ROOT = Path(
    "/kaggle/working"
)

PUBLICATION_DIR = (
    WORKING_ROOT
    / "publication_outputs"
)

TABLE_DIR = (
    PUBLICATION_DIR
    / "tables"
)

FIGURE_DIR = (
    PUBLICATION_DIR
    / "figures"
)

SUPPLEMENT_DIR = (
    PUBLICATION_DIR
    / "supplementary_tables"
)

for directory in [
    PUBLICATION_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    SUPPLEMENT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


MODEL_ORDER = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
    "TabNet",
    "FTTransformer",
    "RealTabR"
]

MODEL_DISPLAY_NAMES = {
    "LogisticRegression":
        "Logistic Regression",

    "RandomForest":
        "Random Forest",

    "XGBoost":
        "XGBoost",

    "LightGBM":
        "LightGBM",

    "CatBoost":
        "CatBoost",

    "MLP":
        "MLP",

    "TabNet":
        "TabNet",

    "FTTransformer":
        "FT-Transformer",

    "RealTabR":
        "RealTabR"
}

REPRESENTATIVE_MODELS = [
    "XGBoost",
    "CatBoost",
    "MLP",
    "FTTransformer"
]

FIGURE_DPI = 350


# ------------------------------------------------------------
# 2. File utilities
# ------------------------------------------------------------

def sha256_file(
    filepath,
    block_size=1024 * 1024
):
    digest = hashlib.sha256()

    with open(filepath, "rb") as file:

        while True:

            block = file.read(
                block_size
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def locate_file(
    filename,
    preferred_parent=None,
    required=True
):
    roots = [
        WORKING_ROOT,
        RESTORED_ROOT
    ]

    matches = []

    for root in roots:

        if not root.exists():
            continue

        matches.extend(
            root.rglob(filename)
        )

    matches = list(
        dict.fromkeys(matches)
    )

    if preferred_parent is not None:

        preferred_matches = [
            path
            for path in matches
            if preferred_parent in path.parts
        ]

        if len(preferred_matches) == 1:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:

        if required:
            raise FileNotFoundError(
                f"Missing required artifact: "
                f"{filename}"
            )

        return None

    raise RuntimeError(
        f"Multiple copies of {filename} found:\n"
        + "\n".join(
            str(path)
            for path in matches
        )
    )


# ------------------------------------------------------------
# 3. Locate required evidence
# ------------------------------------------------------------

paths = {
    "final_predictions":
        locate_file(
            "final_locked_test_predictions.csv",
            "final_locked_holdout"
        ),

    "final_metrics":
        locate_file(
            "final_locked_test_metrics.csv",
            "final_locked_holdout"
        ),

    "calibration_effects":
        locate_file(
            "final_test_calibration_effects.csv",
            "final_locked_holdout"
        ),

    "bootstrap_ci":
        locate_file(
            "model_level_bootstrap_"
            "confidence_intervals.csv",
            "statistical_inference"
        ),

    "delong_vs_xgb":
        locate_file(
            "delong_comparisons_vs_xgboost.csv",
            "statistical_inference"
        ),

    "bootstrap_vs_xgb":
        locate_file(
            "paired_bootstrap_"
            "comparisons_vs_xgboost.csv",
            "statistical_inference"
        ),

    "shap_consensus":
        locate_file(
            "cross_model_consensus_importance.csv",
            "explainability_shap"
        ),

    "shap_rank_agreement":
        locate_file(
            "cross_model_shap_rank_agreement.csv",
            "explainability_shap"
        ),

    "shap_stability":
        locate_file(
            "bootstrap_shap_rank_stability.csv",
            "explainability_shap"
        ),

    "robustness_summary":
        locate_file(
            "input_perturbation_"
            "robustness_summary.csv",
            "robustness_audit"
        ),

    "robustness_worst":
        locate_file(
            "model_worst_case_"
            "perturbation_summary.csv",
            "robustness_audit"
        ),

    "fairness_summary":
        locate_file(
            "all_model_subgroup_"
            "disparity_summary.csv",
            "subgroup_fairness_audit"
        ),

    "gender_counterfactual":
        locate_file(
            "gender_counterfactual_"
            "point_summary.csv",
            "subgroup_fairness_audit"
        ),

    "geography_counterfactual":
        locate_file(
            "geography_counterfactual_"
            "point_summary.csv",
            "subgroup_fairness_audit"
        ),

    "efficiency":
        locate_file(
            "complete_model_efficiency_audit.csv",
            "computational_efficiency_audit"
        ),

    "complexity":
        locate_file(
            "model_artifact_size_and_complexity.csv",
            "computational_efficiency_audit"
        ),

    "shap_cost":
        locate_file(
            "permutation_shap_"
            "computational_cost.csv",
            "computational_efficiency_audit"
        )
}


print("=" * 110)
print("PUBLICATION INPUT ARTIFACTS")
print("=" * 110)

for artifact_name, artifact_path in (
    paths.items()
):
    print(
        f"{artifact_name}: "
        f"{artifact_path}"
    )


# ------------------------------------------------------------
# 4. Load evidence
# ------------------------------------------------------------

final_predictions = pd.read_csv(
    paths["final_predictions"]
)

final_metrics = pd.read_csv(
    paths["final_metrics"],
    keep_default_na=False
)

calibration_effects = pd.read_csv(
    paths["calibration_effects"],
    keep_default_na=False
)

bootstrap_ci = pd.read_csv(
    paths["bootstrap_ci"]
)

delong_vs_xgb = pd.read_csv(
    paths["delong_vs_xgb"]
)

bootstrap_vs_xgb = pd.read_csv(
    paths["bootstrap_vs_xgb"]
)

shap_consensus = pd.read_csv(
    paths["shap_consensus"]
)

shap_rank_agreement = pd.read_csv(
    paths["shap_rank_agreement"]
)

shap_stability = pd.read_csv(
    paths["shap_stability"]
)

robustness_summary = pd.read_csv(
    paths["robustness_summary"]
)

robustness_worst = pd.read_csv(
    paths["robustness_worst"]
)

fairness_summary = pd.read_csv(
    paths["fairness_summary"]
)

gender_counterfactual = pd.read_csv(
    paths["gender_counterfactual"]
)

geography_counterfactual = pd.read_csv(
    paths["geography_counterfactual"]
)

efficiency = pd.read_csv(
    paths["efficiency"],
    keep_default_na=False
)

complexity = pd.read_csv(
    paths["complexity"],
    keep_default_na=False
)

shap_cost = pd.read_csv(
    paths["shap_cost"]
)


y_true = (
    final_predictions[
        "y_true"
    ]
    .astype(int)
    .to_numpy()
)

assert len(y_true) == 2000
assert int(y_true.sum()) == 407


# ------------------------------------------------------------
# 5. Correct artifact reporting
# ------------------------------------------------------------

complexity_publication = (
    complexity.copy()
)

real_tab_r_mask = (
    complexity_publication[
        "Model"
    ] == "RealTabR"
)

complexity_publication.loc[
    real_tab_r_mask,
    "Artifact_Exists"
] = False

complexity_publication.loc[
    real_tab_r_mask,
    "Artifact_Size_Bytes"
] = np.nan

complexity_publication.loc[
    real_tab_r_mask,
    "Artifact_Size_MB"
] = np.nan

complexity_publication.loc[
    real_tab_r_mask,
    "Complexity_Type"
] = (
    "Serialization unavailable"
)

complexity_publication.loc[
    real_tab_r_mask,
    "Complexity_Note"
] = (
    "Final predictions and timing were retained, "
    "but the estimator could not be serialized "
    "because the implementation contained "
    "a local lambda."
)


# ------------------------------------------------------------
# Main Table 4. Locked-test performance with bootstrap intervals
# ------------------------------------------------------------

TABLE_1_METRICS = [
    "ROC_AUC",
    "PR_AUC",
    "Brier_Score",
    "F1",
    "Balanced_Accuracy",
    "MCC"
]


def format_estimate_ci(
    estimate,
    lower,
    upper
):
    return (
        f"{estimate:.4f} "
        f"[{lower:.4f}, {upper:.4f}]"
    )


table_1_records = []

for model_name in MODEL_ORDER:

    record = {
        "Model":
            MODEL_DISPLAY_NAMES[
                model_name
            ]
    }

    for metric in TABLE_1_METRICS:

        row = bootstrap_ci[
            (
                bootstrap_ci["Model"]
                == model_name
            )
            &
            (
                bootstrap_ci["Metric"]
                == metric
            )
        ].iloc[0]

        record[
            metric
        ] = format_estimate_ci(
            row["Point_Estimate"],
            row["CI_Lower"],
            row["CI_Upper"]
        )

    table_1_records.append(
        record
    )


table_1 = pd.DataFrame(
    table_1_records
)

roc_lookup = (
    final_metrics
    .set_index("Model")[
        "ROC_AUC"
    ]
)

table_1[
    "_ROC"
] = [
    roc_lookup.loc[
        model_name
    ]
    for model_name
    in MODEL_ORDER
]

table_1 = (
    table_1
    .sort_values(
        "_ROC",
        ascending=False
    )
    .drop(
        columns="_ROC"
    )
    .reset_index(drop=True)
)

table_1.to_csv(
    TABLE_DIR
    / "Table_4_Locked_Test_Performance_with_95CI.csv",
    index=False
)


# ------------------------------------------------------------
# Main Table 5. ROC-AUC comparison against XGBoost
# ------------------------------------------------------------

table_2 = delong_vs_xgb.copy()

table_2[
    "Comparator"
] = table_2[
    "Comparator"
].map(
    MODEL_DISPLAY_NAMES
)

table_2 = table_2[
    [
        "Comparator",
        "Comparator_ROC_AUC",
        "XGBoost_ROC_AUC",
        "Delta_Comparator_minus_XGBoost",
        "CI_Lower",
        "CI_Upper",
        "DeLong_P_Value",
        "P_Value_Holm",
        "Significant_Holm_0_05"
    ]
]

table_2.to_csv(
    TABLE_DIR
    / "Table_5_DeLong_Comparisons_Against_XGBoost.csv",
    index=False
)


# ------------------------------------------------------------
# Main Table 6. SHAP consensus
# ------------------------------------------------------------

table_3 = (
    shap_consensus[
        [
            "Feature",
            "Mean_Normalized_Importance",
            "SD_Normalized_Importance",
            "Mean_Rank",
            "Best_Rank",
            "Worst_Rank",
            "Models_In_Top_3",
            "Models_In_Top_5",
            "Consensus_Rank"
        ]
    ]
    .sort_values(
        "Consensus_Rank"
    )
    .reset_index(drop=True)
)

table_3.to_csv(
    TABLE_DIR
    / "Table_6_Cross_Model_SHAP_Consensus.csv",
    index=False
)


# ------------------------------------------------------------
# Supplementary robustness summary
# ------------------------------------------------------------

table_4 = (
    robustness_worst.copy()
)

table_4[
    "Model"
] = table_4[
    "Model"
].map(
    MODEL_DISPLAY_NAMES
)

table_4.to_csv(
    TABLE_DIR
    / "Supplementary_Robustness_Worst_Case.csv",
    index=False
)


# ------------------------------------------------------------
# Main Table 7. Efficiency and complexity
# ------------------------------------------------------------

efficiency_publication = (
    efficiency.merge(
        complexity_publication[
            [
                "Model",
                "Artifact_Exists",
                "Artifact_Size_MB",
                "Complexity_Type",
                "Primary_Complexity_Count",
                "Secondary_Complexity_Count",
                "Complexity_Note"
            ]
        ],
        on="Model",
        how="left",
        suffixes=(
            "",
            "_Corrected"
        )
    )
)


for corrected_column in [
    "Artifact_Exists",
    "Artifact_Size_MB",
    "Complexity_Type",
    "Primary_Complexity_Count",
    "Secondary_Complexity_Count",
    "Complexity_Note"
]:

    candidate = (
        f"{corrected_column}_Corrected"
    )

    if candidate in efficiency_publication.columns:

        efficiency_publication[
            corrected_column
        ] = efficiency_publication[
            candidate
        ]

        efficiency_publication.drop(
            columns=candidate,
            inplace=True
        )


table_5_columns = [
    "Model",
    "ROC_AUC",
    "PR_AUC",
    "Fit_Time_Seconds",
    "Prediction_Time_Seconds",
    "Inference_Throughput_Rows_Per_Second",
    "Relative_Final_Fit_Time_vs_XGBoost",
    "Relative_Inference_Time_vs_XGBoost",
    "Artifact_Exists",
    "Artifact_Size_MB",
    "Complexity_Type",
    "Primary_Complexity_Count",
    "Secondary_Complexity_Count"
]

for memory_column in [
    "Training_Baseline_RSS_MB",
    "Training_Peak_RSS_MB",
    "Training_Incremental_Peak_RSS_MB",
    "Inference_Baseline_RSS_MB",
    "Inference_Peak_RSS_MB",
    "Inference_Incremental_Peak_RSS_MB"
]:
    if memory_column in efficiency_publication.columns:
        table_5_columns.append(memory_column)

table_5 = efficiency_publication[
    table_5_columns
].copy()

table_5[
    "Model"
] = table_5[
    "Model"
].map(
    MODEL_DISPLAY_NAMES
)

table_5.to_csv(
    TABLE_DIR
    / "Table_7_Performance_Efficiency_Tradeoff.csv",
    index=False
)


# ------------------------------------------------------------
# 11. Supplementary fairness and calibration tables
# ------------------------------------------------------------

fairness_publication = fairness_summary[
    fairness_summary[
        "Model"
    ].isin(
        REPRESENTATIVE_MODELS
    )
].copy()

fairness_publication[
    "Model"
] = fairness_publication[
    "Model"
].map(
    MODEL_DISPLAY_NAMES
)

fairness_publication.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_Subgroup_"
      "Disparity_Representative_Models.csv",
    index=False
)


calibration_effects.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_Final_"
      "Calibration_Effects.csv",
    index=False
)

bootstrap_vs_xgb.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_Paired_Bootstrap_"
      "Comparisons_Against_XGBoost.csv",
    index=False
)

shap_rank_agreement.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_Cross_Model_"
      "SHAP_Rank_Agreement.csv",
    index=False
)

shap_stability.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_SHAP_"
      "Bootstrap_Stability.csv",
    index=False
)

gender_counterfactual.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_Gender_"
      "Counterfactual_Sensitivity.csv",
    index=False
)

geography_counterfactual.to_csv(
    SUPPLEMENT_DIR
    / "Supplementary_Geography_"
      "Counterfactual_Sensitivity.csv",
    index=False
)

# ------------------------------------------------------------
# Export compact supplementary evidence
# ------------------------------------------------------------

supplementary_shap_source = Path(
    "/kaggle/working/explainability_shap/supplementary"
)

for filename in [
    "supplementary_prediction_fidelity.csv",
    "supplementary_shap_importance.csv",
    "supplementary_shap_agreement.csv",
    "supplementary_shap_runtime.csv"
]:
    source = (
        supplementary_shap_source
        / filename
    )

    if source.exists():
        shutil.copy2(
            source,
            SUPPLEMENT_DIR / filename
        )

memory_columns = [
    "Model",
    "Training_Baseline_RSS_MB",
    "Training_Peak_RSS_MB",
    "Training_Incremental_Peak_RSS_MB",
    "Inference_Baseline_RSS_MB",
    "Inference_Peak_RSS_MB",
    "Inference_Incremental_Peak_RSS_MB"
]

if all(
    column in efficiency.columns
    for column in memory_columns
):
    (
        efficiency[
            memory_columns
        ]
        .to_csv(
            SUPPLEMENT_DIR
            / "Supplementary_Peak_Memory_Benchmark.csv",
            index=False
        )
    )


# ============================================================
# FINAL PUBLICATION FIGURES
# Generated directly from the current pipeline outputs.
# Figure 1 is the conceptual workflow diagram prepared manually.
# ============================================================

from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve

FIGURE_DPI = 350

primary_cv = pd.read_csv(
    "/kaggle/working/unified_benchmark/"
    "unified_primary_repeated_cv.csv"
)

primary_cv[
    [
        "Model",
        "ROC_AUC_mean",
        "ROC_AUC_std",
        "PR_AUC_mean",
        "PR_AUC_std",
        "Brier_Score_mean",
        "ECE_10_mean",
        "Fit_Time_Seconds_mean",
        "Predict_Time_Seconds_mean"
    ]
].to_csv(
    TABLE_DIR
    / "Table_3_Repeated_CV_Performance.csv",
    index=False
)

representative_models = [
    "XGBoost",
    "CatBoost",
    "MLP",
    "FTTransformer"
]


def save_publication_figure(
    figure,
    filename
):
    filepath = (
        FIGURE_DIR
        / filename
    )

    figure.savefig(
        filepath,
        dpi=FIGURE_DPI,
        bbox_inches="tight"
    )

    plt.close(figure)

    print("Saved:", filepath)


# ------------------------------------------------------------
# Figure 2. Development repeated-CV benchmark
# ------------------------------------------------------------

cv_plot = (
    primary_cv
    .copy()
    .sort_values(
        "ROC_AUC_mean",
        ascending=True
    )
)

cv_labels = [
    MODEL_DISPLAY_NAMES.get(
        model,
        model
    )
    for model in cv_plot["Model"]
]

figure, axes = plt.subplots(
    1,
    2,
    figsize=(12, 6),
    sharey=True
)

axes[0].scatter(
    cv_plot["ROC_AUC_mean"],
    np.arange(len(cv_plot))
)

for row_index, value in enumerate(
    cv_plot["ROC_AUC_mean"]
):
    axes[0].text(
        value + 0.0008,
        row_index,
        f"{value:.3f}",
        va="center",
        fontsize=8
    )

axes[0].set_yticks(
    np.arange(len(cv_plot))
)
axes[0].set_yticklabels(
    cv_labels
)
axes[0].set_xlabel(
    "Mean ROC-AUC"
)
axes[0].set_title(
    "Discrimination"
)
axes[0].grid(
    axis="x",
    alpha=0.25
)

axes[1].scatter(
    cv_plot["PR_AUC_mean"],
    np.arange(len(cv_plot))
)

for row_index, value in enumerate(
    cv_plot["PR_AUC_mean"]
):
    axes[1].text(
        value + 0.0015,
        row_index,
        f"{value:.3f}",
        va="center",
        fontsize=8
    )

axes[1].set_xlabel(
    "Mean PR-AUC"
)
axes[1].set_title(
    "Precision-recall performance"
)
axes[1].grid(
    axis="x",
    alpha=0.25
)

figure.suptitle(
    "Development-stage repeated cross-validation performance"
)
figure.tight_layout()

save_publication_figure(
    figure,
    "Figure_02_Development_Repeated_CV_Benchmark.png"
)


# ------------------------------------------------------------
# Figure 3. Calibration curves
# ------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(8, 6)
)

for model_name in representative_models:

    probability = (
        final_predictions[
            f"{model_name}_Calibrated_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    observed_rate, predicted_mean = (
        calibration_curve(
            y_true,
            probability,
            n_bins=10,
            strategy="quantile"
        )
    )

    axis.plot(
        predicted_mean,
        observed_rate,
        marker="o",
        label=MODEL_DISPLAY_NAMES[
            model_name
        ]
    )

axis.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

axis.set_xlabel(
    "Mean predicted probability"
)
axis.set_ylabel(
    "Observed churn rate"
)
axis.set_title(
    "Calibration of representative models"
)
axis.legend()
axis.grid(alpha=0.2)

save_publication_figure(
    figure,
    "Figure_03_Calibration_Curves.png"
)


# ------------------------------------------------------------
# Figure 4. Locked-test ROC and precision-recall curves
# ------------------------------------------------------------

figure, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

for model_name in MODEL_ORDER:

    probability = (
        final_predictions[
            f"{model_name}_Calibrated_Probability"
        ]
        .astype(float)
        .to_numpy()
    )

    roc_row = (
        final_metrics[
            final_metrics["Model"]
            == model_name
        ]
        .iloc[0]
    )

    false_positive_rate, true_positive_rate, _ = (
        roc_curve(
            y_true,
            probability
        )
    )

    precision, recall, _ = (
        precision_recall_curve(
            y_true,
            probability
        )
    )

    axes[0].plot(
        false_positive_rate,
        true_positive_rate,
        label=(
            f"{MODEL_DISPLAY_NAMES[model_name]} "
            f"({float(roc_row['ROC_AUC']):.3f})"
        )
    )

    axes[1].plot(
        recall,
        precision,
        label=(
            f"{MODEL_DISPLAY_NAMES[model_name]} "
            f"({float(roc_row['PR_AUC']):.3f})"
        )
    )

axes[0].plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)
axes[0].set_xlabel(
    "False positive rate"
)
axes[0].set_ylabel(
    "True positive rate"
)
axes[0].set_title(
    "ROC curves"
)
axes[0].legend(
    fontsize=7
)
axes[0].grid(alpha=0.2)

prevalence = y_true.mean()
axes[1].axhline(
    prevalence,
    linestyle="--",
    label=f"Prevalence ({prevalence:.3f})"
)
axes[1].set_xlabel(
    "Recall"
)
axes[1].set_ylabel(
    "Precision"
)
axes[1].set_title(
    "Precision-recall curves"
)
axes[1].legend(
    fontsize=7
)
axes[1].grid(alpha=0.2)

figure.suptitle(
    "Locked-test discrimination curves"
)
figure.tight_layout()

save_publication_figure(
    figure,
    "Figure_04_Locked_Test_Discrimination_Curves.png"
)


# ------------------------------------------------------------
# Figure 5. Locked-test ROC-AUC uncertainty
# ------------------------------------------------------------

roc_ci = (
    bootstrap_ci[
        bootstrap_ci["Metric"]
        == "ROC_AUC"
    ]
    .copy()
    .sort_values(
        "Point_Estimate",
        ascending=True
    )
)

roc_ci["Display_Model"] = (
    roc_ci["Model"]
    .map(
        MODEL_DISPLAY_NAMES
    )
)

lower_error = (
    roc_ci["Point_Estimate"]
    - roc_ci["CI_Lower"]
)

upper_error = (
    roc_ci["CI_Upper"]
    - roc_ci["Point_Estimate"]
)

figure, axis = plt.subplots(
    figsize=(8, 6)
)

axis.errorbar(
    roc_ci["Point_Estimate"],
    np.arange(len(roc_ci)),
    xerr=np.vstack(
        [
            lower_error,
            upper_error
        ]
    ),
    fmt="o",
    capsize=3
)

axis.set_yticks(
    np.arange(len(roc_ci))
)
axis.set_yticklabels(
    roc_ci["Display_Model"]
)
axis.set_xlabel(
    "ROC-AUC with bootstrap 95% CI"
)
axis.set_title(
    "Locked-test ROC-AUC uncertainty"
)
axis.grid(
    axis="x",
    alpha=0.25
)

save_publication_figure(
    figure,
    "Figure_05_ROC_AUC_Forest_Plot.png"
)


# ------------------------------------------------------------
# Figure 6. Cross-model SHAP consensus
# ------------------------------------------------------------

shap_plot = (
    shap_consensus
    .copy()
    .sort_values(
        "Consensus_Rank",
        ascending=False
    )
)

figure, axis = plt.subplots(
    figsize=(8, 6)
)

axis.barh(
    shap_plot["Feature"],
    shap_plot[
        "Mean_Normalized_Importance"
    ],
    xerr=shap_plot[
        "SD_Normalized_Importance"
    ]
)

axis.set_xlabel(
    "Mean normalized SHAP importance"
)
axis.set_title(
    "Cross-model consensus feature importance"
)

save_publication_figure(
    figure,
    "Figure_06_SHAP_Consensus_Importance.png"
)


# ------------------------------------------------------------
# Figure 7. Corrected robustness heatmap
# ------------------------------------------------------------

scenario_order = list(
    robustness_summary[
        "Scenario"
    ]
    .drop_duplicates()
)

robustness_matrix = (
    robustness_summary
    .pivot(
        index="Model",
        columns="Scenario",
        values="Delta_ROC_AUC_mean"
    )
    .reindex(
        representative_models
    )
    .reindex(
        columns=scenario_order
    )
)

figure, axis = plt.subplots(
    figsize=(12, 5)
)

image = axis.imshow(
    robustness_matrix.to_numpy(),
    aspect="auto"
)

axis.set_yticks(
    np.arange(
        len(
            robustness_matrix.index
        )
    )
)
axis.set_yticklabels([
    MODEL_DISPLAY_NAMES.get(
        model,
        model
    )
    for model
    in robustness_matrix.index
])

axis.set_xticks(
    np.arange(
        len(
            robustness_matrix.columns
        )
    )
)
axis.set_xticklabels(
    robustness_matrix.columns,
    rotation=35,
    ha="right"
)

axis.set_xlabel(
    "Perturbation scenario"
)
axis.set_ylabel(
    "Model"
)
axis.set_title(
    "ROC-AUC sensitivity under input perturbations"
)

for row_index in range(
    robustness_matrix.shape[0]
):
    for column_index in range(
        robustness_matrix.shape[1]
    ):
        value = robustness_matrix.iloc[
            row_index,
            column_index
        ]

        axis.text(
            column_index,
            row_index,
            f"{value:.3f}",
            ha="center",
            va="center",
            fontsize=8
        )

figure.colorbar(
    image,
    ax=axis,
    label="Mean change in ROC-AUC"
)
figure.tight_layout()

save_publication_figure(
    figure,
    "Figure_07_Robustness_ROC_AUC_Heatmap.png"
)


# ------------------------------------------------------------
# Figure 8. Direct sensitivity to categorical assignments
# ------------------------------------------------------------

gender_plot = (
    gender_counterfactual
    .set_index("Model")
    .reindex(
        representative_models
    )
)

geography_plot = (
    geography_counterfactual
    .set_index("Model")
    .reindex(
        representative_models
    )
)

x_positions = np.arange(
    len(representative_models)
)

bar_width = 0.36

figure, axis = plt.subplots(
    figsize=(9, 5)
)

axis.bar(
    x_positions
    - bar_width / 2,
    gender_plot[
        "Mean_Female_minus_Male"
    ],
    width=bar_width,
    label="Gender assignment"
)

axis.bar(
    x_positions
    + bar_width / 2,
    geography_plot[
        "Mean_Within_Row_Probability_Range"
    ],
    width=bar_width,
    label="Geography assignment"
)

axis.set_xticks(
    x_positions
)
axis.set_xticklabels([
    MODEL_DISPLAY_NAMES[
        model
    ]
    for model
    in representative_models
])

axis.set_ylabel(
    "Mean probability sensitivity"
)
axis.set_title(
    "Direct model sensitivity to categorical assignments"
)
axis.legend()
axis.grid(
    axis="y",
    alpha=0.2
)

save_publication_figure(
    figure,
    "Figure_08_Direct_Sensitivity.png"
)


# ------------------------------------------------------------
# Figure 9. Performance-efficiency trade-off
# ------------------------------------------------------------

efficiency_plot = (
    efficiency_publication
    .copy()
)

figure, axis = plt.subplots(
    figsize=(8, 6)
)

axis.scatter(
    efficiency_plot[
        "Fit_Time_Seconds"
    ],
    efficiency_plot[
        "ROC_AUC"
    ]
)

for _, row in (
    efficiency_plot.iterrows()
):
    axis.annotate(
        MODEL_DISPLAY_NAMES.get(
            row["Model"],
            row["Model"]
        ),
        (
            float(
                row[
                    "Fit_Time_Seconds"
                ]
            ),
            float(
                row[
                    "ROC_AUC"
                ]
            )
        ),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8
    )

axis.set_xscale(
    "log"
)
axis.set_xlabel(
    "Final fit time (seconds, log scale)"
)
axis.set_ylabel(
    "Locked-test ROC-AUC"
)
axis.set_title(
    "Predictive performance versus training cost"
)
axis.grid(alpha=0.2)

save_publication_figure(
    figure,
    "Figure_09_Performance_Efficiency_Tradeoff.png"
)


# ------------------------------------------------------------
# Compact final output manifest
# ------------------------------------------------------------

final_output_manifest = []

for directory in [
    TABLE_DIR,
    FIGURE_DIR,
    SUPPLEMENT_DIR
]:
    for filepath in sorted(
        directory.glob("*")
    ):
        if filepath.is_file():
            final_output_manifest.append({
                "Category":
                    directory.name,
                "File":
                    filepath.name,
                "Bytes":
                    filepath.stat().st_size
            })

final_output_manifest = pd.DataFrame(
    final_output_manifest
)

final_output_manifest.to_csv(
    PUBLICATION_DIR
    / "final_output_manifest.csv",
    index=False
)

print("\n" + "=" * 100)
print("FINAL PUBLICATION OUTPUTS")
print("=" * 100)
print(
    final_output_manifest
    .to_string(index=False)
)

print("\nPublication outputs saved to:")
print(PUBLICATION_DIR)
